# 00 — YouTube Automation: Kalıcı Master Notebook (Orkestratör)

Bu, normal Colab kullanımında **açman gereken tek notebook**. Notebook 01–06'yı ayrıca yüklemene veya
çalıştırmana gerek yoktur — altı fazın hepsi burada, aynı `local_pipeline` kanonik modülleri üzerinden
gerçekten çalışır.

**Mimari sınır:** Ana üretim hattı yalnızca verilen kaynak videoyu `analiz → script → ses → render`
zinciriyle Shorts'a dönüştürür. YouTube keşfi ve viral tahmin bağımsız araştırma yan koludur;
kaynak videoyu, seçilen hot moment'i veya render kararını otomatik olarak değiştirmez.

## Maliyet politikası (ücretli sağlayıcı fan-out'u yok)

Her ücretli faz **tek bir seçili sağlayıcı/model** çağırır. Fallback kapalıyken bir istek en fazla
**bir** ücretli çağrı yapar; fallback açıkken adaylar **sırayla** denenir ve **ilk geçerli sonuçta
durulur** — bir sonraki aday yalnızca öncekinin gerçekten başarısız olduğu durumda çağrılır.
Karşılaştırma/benchmark amaçlı çok-modelli fan-out **sadece** standalone Notebook 02/03'te,
açık karşılaştırma sarmalayıcıları (`call_*_script`, `call_*_tts`) ile yapılır.

Denenen her sağlayıcı/model ve başarısızlık nedeni artefaktın `provider_attempts` alanına yazılır.

## Onay politikası (üretimden SONRA, içeriğe bağlı)

Üretim fonksiyonları **onay argümanı almaz** ve asla `approved` üretemez. Yeni üretilen içerik her zaman
`review_required` (gTTS için `development_placeholder`) olur — çünkü henüz kimse okumadı/dinlemedi/izlemedi.

Her faz için sıra: **üret → incele (görüntüle/dinle/izle) → ayrı onay hücresi**. Onay hücresi, inceleme
hücresinin yazdırdığı **SHA-256** değerini istemenizi gerektirir; bu, içeriği gerçekten gördüğünüzün
kaydıdır. İçerik sonradan değişirse (yeniden üretim, yeniden render) onay **otomatik olarak geçersiz**
olur (`approval.check_approval`). Hiçbir yerde otomatik `production_ready` yoktur ve
**bu notebook hiçbir şeyi YouTube'a yüklemez veya yayınlamaz.**

## Tek dosya Colab çalışması

`local_pipeline` modülleri bu notebook'a gömülüdür (base64 + SHA-256 doğrulamalı). Yükleme sırası:
repo → Drive/üst dizin kopyası → `PROJECT_REPO_URL` klonu → gömülü hash-doğrulanmış yedek.

## Bağımlılık farkındalığı

Bir faz kapalıysa, sonraki açık faz gerekli artefaktı diskte arar ve şemasını doğrular; bulamazsa
**sadece o fazı** net bir hatayla durdurur. Her faz iki durum bildirir:
`execution_status` (completed/skipped/failed) ve `artifact_status`
(review_required/approved/insufficient_data/failed/development_placeholder).

In [ ]:
!pip -q install requests pandas numpy google-api-python-client faster-whisper scenedetect librosa openai anthropic google-genai elevenlabs gTTS imageio-ffmpeg > /dev/null
print('Kurulum tamam.')

## Ana üretim hattı ve bağımsız araştırma anahtarları

In [ ]:
# --- ANA URETIM HATTI ---------------------------------------------------------
# Varsayilan: kaynak video -> moment secimi -> orijinal ses + gomulu altyazi.
# 'narrated_recap': istege bagli Gemini script -> TTS -> anlatimli render.
# NOT: PIPELINE_MODE artik ZORUNLU Faz 0 isleme planindan TURETILIR
# (direct_clip_series -> original_audio_subtitles, narrated_theory -> narrated_recap).
# Asagidaki deger yalnizca plan yokken gecerli olan guvenli baslangictir.
PIPELINE_MODE = 'original_audio_subtitles'  # 'original_audio_subtitles' | 'narrated_recap'
RUN_VIDEO_ANALYSIS = False
RUN_SCRIPT_GENERATION = False
RUN_TTS = False
RUN_RENDER = False

# --- BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline'a kaynak/karar BESLEMEZ -------
RUN_YOUTUBE_DATA = False
RUN_VIRAL_PREDICTION = False

# Her faz icin IKI ayri durum tutulur:
#   execution_status: completed | skipped | failed   (faz hucresi calisti mi?)
#   artifact_status : review_required | approved | insufficient_data | failed |
#                     development_placeholder | None
# Temiz calisip 'insufficient_data' artefakti yazan bir faz sadece "ok" gorunmez.
EXECUTION_STATUS_VALUES = {'completed', 'skipped', 'failed'}
ARTIFACT_STATUS_VALUES = {
    'review_required', 'approved', 'insufficient_data', 'failed',
    'development_placeholder', 'experimental', 'production_ready', None,
}
phase_report = {}


def record_phase(name, execution_status, artifact_status=None, detail=''):
    assert execution_status in EXECUTION_STATUS_VALUES, f'gecersiz execution_status: {execution_status!r}'
    assert artifact_status in ARTIFACT_STATUS_VALUES, f'gecersiz artifact_status: {artifact_status!r}'
    phase_report[name] = {
        'execution_status': execution_status,
        'artifact_status': artifact_status,
        'detail': detail,
    }
    suffix = f' | artifact_status={artifact_status}' if artifact_status is not None else ''
    print(f'[faz:{name}] execution_status={execution_status}{suffix}' + (f' | {detail}' if detail else ''))


## Google model router yapilandirmasi (tek merkez)

Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider. Notebook icinde **hicbir**
`genai.Client(...)` / sabit `model='...'` cagrisi yoktur.

**Kota proje bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits).
Bu yuzden ikinci bir hesap/proje anahtari eklemek limitleri artirmaz; router yalnizca **ayni proje
icinde model degistirir**. `ALLOW_ACCOUNT_QUOTA_ROTATION` politika geregi `False`'tir ve
`google_ai.assert_no_account_quota_rotation()` True yapilirsa calismayi durdurur.

**Ucretsiz katman veri politikasi:** gonderdigin icerik Google tarafindan urun gelistirmede
kullanilabilir ve insan gozden gecirenler okuyabilir. Bu yuzden `ALLOW_RAW_MEDIA_TO_FREE_TIER`
varsayilan `False`: ham telifli video/ses **yuklenmez**, yalnizca yerel olarak uretilmis
transcript pencereleri / zaman damgalari / sahne-ses skorlari gonderilir.


In [ ]:
# --- Google model router yapilandirmasi (BURADAN AC/KAPA) --------------------
# Manuel credential profili: kota dolunca otomatik hesap gecisi YOKTUR.
# Profiller farkli proje/kanal ortamlari icindir; kota sinirini asmak icin profil degistirme.
# Profil degistirmek icin yalnizca asagidaki iki etiketi elle degistirip hucreleri yeniden calistir.
GOOGLE_CREDENTIAL_PROFILE = 'primary'
YOUTUBE_CREDENTIAL_PROFILE = 'primary'
GOOGLE_CREDENTIAL_PROFILES = {
    'primary': ('GeminiKey', 'GOOGLE_API_KEY'),
    'secondary': ('GeminiKeySecondary',),
}
YOUTUBE_CREDENTIAL_PROFILES = {
    'primary': ('YoutubeKEY', 'YOUTUBE_DATA_API_KEY'),
    'secondary': ('YoutubeKEYSecondary',),
}

GOOGLE_ROUTE_MODE = 'auto'              # 'auto' | 'explicit'
GOOGLE_TASK_MODEL_OVERRIDES = {}        # ornek: {'script_generation': ['gemini-3.5-flash-lite']}
ENABLE_GOOGLE_MODEL_FALLBACK = True     # False: gorev basina TEK Google modeli denenir
ENABLE_CROSS_PROVIDER_FALLBACK = True   # script/TTS/embedding: Groq/Cloudflare/Azure (kimlik varsa)
MAX_GOOGLE_MODEL_ATTEMPTS = 4
MAX_PROVIDER_ATTEMPTS = 3
RUN_GOOGLE_CONNECTION_TEST = False      # True: SADECE secili birincil modellere minik bir ping
ENABLE_GOOGLE_SEARCH_GROUNDING = False  # Gemini 2.5 arama-temelli arastirma
RUN_ANTIGRAVITY_RESEARCH = False        # deneysel ajan raporu (varsayilan KAPALI)
ALLOW_RAW_MEDIA_TO_FREE_TIER = False    # ham telifli medya ucretsiz katmana YUKLENMEZ

# POLITIKA -- ASLA True yapma. Birden fazla Google hesabi/projesi arasinda anahtar
# dondurerek RPD/RPM/TPM limitlerini asmak kota ihlalidir. Kota proje bazlidir;
# ikinci bir anahtar limiti artirmaz. Router yalnizca AYNI proje icinde MODEL degistirir.
ALLOW_ACCOUNT_QUOTA_ROTATION = False

# --- Her Google yetenegi icin BAGIMSIZ komut anahtari -----------------------
RUN_GOOGLE_HOTMOMENT_REVIEW = False     # Faz 1 adaylarinin nihai LLM incelemesi
RUN_GOOGLE_SCRIPT_GENERATION = True     # Faz 2 script uretimi (Google = kanonik varsayilan)
RUN_GOOGLE_BULK_CLASSIFICATION = False  # baslik/aciklama/hashtag, trend satiri siniflama
RUN_GOOGLE_GROUNDED_RESEARCH = False    # arama-temelli guncellik dogrulamasi
RUN_GOOGLE_TTS = True                   # Ana pipeline seslendirmesi: Gemini TTS (preview -> review_required)
RUN_GOOGLE_EMBEDDINGS = False           # duplikasyon/novelty/kumeleme

# Gemini TTS ayarlari (RUN_GOOGLE_TTS ve TTS_PROVIDER='gemini' ile kullanilir)
GEMINI_TTS_VOICE = 'Kore'               # bkz. google_ai.GEMINI_TTS_VOICES
GEMINI_TTS_LANGUAGE_CODE = 'tr-TR'
GEMINI_TTS_NARRATION_DIRECTION = 'Gergin, merak uyandiran bir tonda, net ve akici oku:'

# Antigravity deneysel arastirma icin ZORUNLU token butcesi (sessiz tuketim yasak)
ANTIGRAVITY_MAX_TOTAL_TOKENS = 8000


## Kurulum — Drive bir kez bağlanır, yollar bir kez tanımlanır, Secrets bir kez okunur

In [ ]:
import os, sys, json
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()

USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"

# Colab Secrets erisim hatalari (panelde 'Notebook erisimi' kapali, secret hic
# eklenmemis, vb.) ACIKLAYICI olmali ama DEGERI asla gostermemeli.
_SECRET_ACCESS_NOTES = []


def get_secret(name):
    """Colab Secrets -> os.environ sirasiyla dener. Deger asla yazdirilmaz/loglanmaz."""
    try:
        from google.colab import userdata
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception as exc:
            _SECRET_ACCESS_NOTES.append(
                f"{name}: Colab Secrets okunamadi ({type(exc).__name__}). "
                "Sol paneldeki anahtar simgesi -> ilgili secret -> 'Notebook erisimi' ACIK olmali."
            )
    except Exception:
        pass
    return os.environ.get(name)


def resolve_alias(*names):
    """Ilk dolu alias'i dondurur: (deger, kullanilan_ad). Deger asla yazdirilmaz."""
    for name in names:
        value = get_secret(name)
        if value:
            return value, name
    return None, None


def resolve_profile_alias(profile, profiles, kind):
    """Yalnizca ELLE secilen profili cozer; baska profile otomatik gecmez."""
    if profile not in profiles:
        raise ValueError(f'Bilinmeyen {kind} profili: {profile!r}; mevcut={sorted(profiles)}')
    names = profiles[profile]
    names = (names,) if isinstance(names, str) else tuple(names)
    return resolve_alias(*names)


# Her servis icin tam olarak bir MANUEL profil secilir. Kota/429 baska profile gecis yaptirmaz.
YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS = resolve_profile_alias(
    YOUTUBE_CREDENTIAL_PROFILE, YOUTUBE_CREDENTIAL_PROFILES, 'YouTube')
GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS = resolve_profile_alias(
    GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, 'Gemini')
TMDB_API_KEY, _ = resolve_alias('TMDB_API_KEY')

# Opsiyonel / varsayilan olarak KULLANILMAZ (capraz-saglayici fallback, TTS alternatifleri):
OPENAI_API_KEY, _ = resolve_alias('OPENAI_API_KEY')
ANTHROPIC_API_KEY, _ = resolve_alias('ANTHROPIC_API_KEY')
ELEVENLABS_API_KEY, _ = resolve_alias('ELEVENLABS_API_KEY')


def describe(value, alias=None):
    if not value:
        return 'missing'
    return f'configured ({alias})' if alias else 'configured'


print('--- Zorunlu/onerilen (Google-agirlikli ucretsiz strateji) ---')
print(f'Gemini profil={GOOGLE_CREDENTIAL_PROFILE}:', describe(GOOGLE_API_KEY, GOOGLE_SECRET_ALIAS))
print(f'YouTube profil={YOUTUBE_CREDENTIAL_PROFILE}:', describe(YOUTUBE_DATA_API_KEY, YOUTUBE_SECRET_ALIAS))
print('TMDB                                :', describe(TMDB_API_KEY))
print('--- Opsiyonel (varsayilan kapali) ---')
print('OpenAI     :', describe(OPENAI_API_KEY))
print('Anthropic  :', describe(ANTHROPIC_API_KEY))
print('ElevenLabs :', describe(ELEVENLABS_API_KEY))

if _SECRET_ACCESS_NOTES:
    print()
    print('!!! Colab Secrets erisim uyarilari (deger GOSTERILMEZ) !!!')
    for _note in _SECRET_ACCESS_NOTES:
        print(' -', _note)
if not GOOGLE_API_KEY:
    print()
    print("Gemini anahtari YOK. Duzeltme: Colab sol panel -> anahtar (Secrets) simgesi ->")
    print("  '+ Yeni gizli anahtar' -> Ad: 'GeminiKey' -> Deger: https://aistudio.google.com/apikey")
    print("  -> 'Notebook erisimi' anahtarini AC. Yerelde: ortam degiskeni GeminiKey veya GOOGLE_API_KEY.")


## `local_pipeline` modullerini yukle (tum standalone notebooklarla ayni tek kaynak)

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['video_analysis', 'script_generation', 'tts', 'render_utils', 'youtube_data', 'viral_prediction', 'approval', 'google_ai', 'provider_adapters', 'narration_presets', 'source_strategy']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'video_analysis': {
        'sha256': '719335b9ce67dfea3d9d5c4055d03f42981a35651571f9c11eee8b4de554f553',
        'encoding': 'zlib+b85',
        'data': (
            'c-qZ<YjfK;w%~XD3XHmHEv6+!c6a*Db&q;0lQ=V}$)hVyy6>#lB~ub*aZHiAeArQPUA4b#zijRQ+VcVkfTXNsy0@mQW@3@V0dR0|'
            'aNg+m`|qNv2<J(pZsTQ?jlwicc12vM$7-FGqfNGn(o!v>GFp^zmZ~JXju)y}hk3MA^IdOpJfns)-EaoYxYan`?$Wsm)1{hBX1lB^'
            't9djFt1{b!2y3<pi!#djvvIHYCM%<PmffhyvAT((ZK2|_P}yCoHdRu_qXj^Umth&Hd6X{Jn=rqL(`)tNr5Y@<%{I*OR}tMtd6@Km'
            'UB|^X%GD-YMu{roPf;;c#Ue^0nM{_$0~}kn#leJ$diClxe39i*UO<B-bhD#>gJEx8m8x7vDqNHmd>JO!Sss_`4Gys$O`%33rw@P&'
            'eN@Xh8*QWTrnik3IArb@;U(Du!145o_!>uHVI2p9b6RCNd=et`B)*N*I?5weWW9M-u2m{nD-?iA2x{~Z#;J;M6R8DEU&SflIfR+6'
            ')FzD668?u^m(@0j7l6-UFHV=yHiG|4%ValN=JBe;FBaK0jtW?s_kc}AWd(GSB8C;T&B7%tPp-DrJc)~SRKT=$<r*6Ph-k^uWT)~7'
            'KMG}{L@B*h3{VgQcYs4^#qYLxbQ?!^YLUcSq&}QT9+g#|0-?p}XcKL+e5YXjLsgV{1uKz9<K8ovG%nb{;2}^zo6xk_W@!;4Wdr`G'
            'FPsvfcgE-~#{VdQpvz1x<KkxA>-YP;Ri16sY__Tp_-v-)%{I$R7%V_1nk{-gd6q}|ucDf7^K20ng?>`Tn}}g;!*ZR(a}DTc_{lGj'
            'j`1`7M*kjC<4;*?z^}ssLFk`2Xfuek+H7|!EL6Jf_0HcvegENn_RG^(AI{FzRDJF#_^%&*gnba>jKicqRQ)BaUy^O{!EBp^i)fuC'
            '%P7ZJyacm6`p+tc{oNljgm4S<yT#wC542RRR`DXn=C~O6a23Lm(rfr|SrOsSaOrmd;vPo!;pd;<zI%W6e0Ki!!@Fl^vtM35KYKen'
            '`;Yf$Z_Z!7ePfMwyxD$_fX18bmjAusfA6CChW-d|V}9}dEkl3t_8Cn3-PvCR(x6|IVP4`#n|QfQBK!-88~<5U1+brf(ChV<(F&;~'
            'OKzi@dF5ex9St4<tH`Q+5zUrW&b+FK7DFH;Xb%&+h9}cEuttoaj{lI7YXF;AnkXK_j0Raheh?CW5963lh8}Euc=G)uP@|tz2|F{o'
            'Si$ZrVZiv$<td{Uc|A=R_T#$Rgz0FN!!j;mn<iO$J)+&Icy_ogfZt|mjC&K{XXz3d0RlSv7#Fmc^p+#egPRb!Hg4QFmnNfmSitrT'
            '^XrPYoO%!2`WcW0@LvM<4E9t8dlWh|e5!y!sjDj++gDd=n^eGq;<N&u!Ek7?6mX_6xeIqik5LtFw$LSv-(uLjn(&Y6CvfPf<B5jU'
            '$DUy#&}no7+ZA4(y?rs(7^8pVm9z3NPBq}A?DvCHvv9OPK!1evfxBkNCntgVRIi--wY{F$0GHG>fkAKwe^L{CX+ZpGswSrn_2gk('
            'MCz9?siLzy&+<Xv8_ZA8L;)Q%d;JN4Cz~*d`*qjMhPnHp>9`d&h)F<?vnA`nH3qio&*LOcH_<Li9YCk*v;FfA`Mr#%%y$;X-A!t+'
            '*|K@<+5Be}0ii8t53B-ENT436A131oHI6<mqHU=L?{`}s{?M9AfLs{9*fZHaZ?FQ6mLf*<BiD&P=?6u)iwmGtjebSv1c20t_GjxZ'
            'e6zdiQ<d_OtFmQy9V_74sk#YwX?UZYvG5|qyDS$h$E!9pJ=CE)pJjLx--IxoFo*soM@!h@^J-V!xCCY`0wCO9x?#&Wf>z3#PL=Te'
            '=8iNgn6$N$L!#NtAB-BA(FP!)oa}{)2Tl`YH$(wdM0F#F)>}A?caAv3`V|D*zNpzJke`8#C%|k$^5<23kX(I^^|igE(KTRag;cZ&'
            'KUq9Uqk?FL8Pg{GIDqL7_aHf70K(AntV)-Ix*38AhUz=`AHS-p@H;Ir%xLVhcr}A_V-dl*w;X`ba}6Y4(D{o81J8D`jutl{Do@pQ'
            '1kz)fn-5XcYJdUp1lt<wmgY6)<iVgGN|nw;@|^L@<zTT{4wV^Deewe<2mwNzER{4-cWV^H*ICM11|F9SK8=q+naDB_8)5dqr-011'
            'Ky%@>2mn`Cpl@xel#~ZJnKwX)Y6&xf4UQ_wHr|3*|Lb30zB(hQx<eQ*oZLTWNkUppoEIg*LtOx+MSL2rKqb5j^JPII!4e*WesBYW'
            'g%je%3pm@aRkT6HFiNxPdJV#Rc%5b-kSq!~^a&m)7@(ukFvcLQpq2yr$cOhYMt_8UNaiK|Doj8iK)nM+6V}3TWGFDa{@X7EY6B}r'
            'DhHmp5o%MQQo>xqYnUNX{7uZ$3DvKlMP+vd^-J>s-PVyK5D&-$jJ`1b8IYU~>J`StE6OFP%2WHRmp`A;^C-_7&ueZGeJrQ%^9sZg'
            'P)IYx=CogxtI;2UvLf7O#k4QeZ#ztx91wuJSsPr4w;v8t4+<uQV;WmzF+ep2=P?<d1dZnN9Mxg_iSR0GcJCPgtRLzk%^`rSRyYs$'
            'Zq0@;Q+M<%+knUnC(O_M!0e&Lm_Np4A9_;`Y?0v4AmyIXpD4Q643MB~f#L@N3aT}8Ftfl9faHOXA=ZL!=HH3o?}FafVUZvr{SJct'
            'Jq7Ne?<r5hZb_mT*gs~=I5*swpP)WG1*sW1B`U)W)dcxPoNlX9B$T_jT*LSk>!iHV(14Jlbpv=i8Y6bdS%fz|g-3eFUtKl&25xl+'
            'Y6i2DqT0fj(Q>TbLbv%H9!l7M8r?<0nm_{@MVoE8qmM!52GxQG2F%Wb53>dEu*yZ0M2iyPzIUPJXg`4u&^m-Gj(XHW4K||q<Cyf^'
            '0yYw9%?f$p7N$>TDC-P^;o!8?JX&QrOO&X`HuRSUB2F!u5p+#yMFx#e9UTM(eBELU3Nwy(5VRfdFW}9atbBndsh0NtW(%M5Dyd+q'
            '!1qDg*D&w<PIsm&IGoEt0RPO;mUoEK<M);o9o+~=(65+|v<wmFt$2aw=Ab6+(R_|bfq!hA9Egt_P))PEa`93F#v!v@q3xgr4DF&<'
            '^FyEmG<DG*K%VxPfZk|O4sU|*>LoO9knHFc8G1gBlk5(K4utri9j66|yKbOg(l?hj^6Cj$0<+m}AWNgU=N}nq2l#04;5dTdLF7Bc'
            'v+CQ3CPir(xvEmVDr5B=6_Oj+sIXzHPhkpjLzv@Y=|SK34R~2JAu<#;>*)^FO|+YWXggnqgsxLH@UTX_VV1(D^a5FfUJ;mN?;CZF'
            'j4;|{c=U|!vV5u5QI#`i0SO=Z8OV3nG1`_>%fhaW%Xm<ZduOlTz6Zq3etrAyxn+$j)*yQ!`sok&9RHq2WghP8zu2sZ=ZOB7ZGj-W'
            'm23IK*b+gsBa2~ty(SACsOEqg{Hx3>{+tZV_(wR;=vaiA>E=mxOXj>apif&KLER+NBVvWRL$eV5HBYjPhmwca2n2dr=WqsVaPc}!'
            'x%VQYd1gtq;eSE)r|02qSl}Pm@u!3gj{tiS(Z{H+Xh>zWPU0IH$W2H+MgIUfVNLeYGQ`QEt&$E-5Y+N;QO?NR0K_mO7r|h)Lg{uk'
            'kJD4MaDx&O-dSvT$b>k9(@=g>v-<Zw3!*Z{pHn5aLEAcF9%^}sY?7Wn1INj;3Iyc~18Z1={njzIC_J*Wl{W)mA!VjQIaqVl8$ZDY'
            '|LgeVhwmRhefo^-QF0Cnj?i~-{KL&(KdBAqDeBj!zp&I@0s+wHEbs%@gU{1qk;f#1J!YqiIS$o3<N}2PH3;_;aR3r6pjnlojI->C'
            'Sc~E;$Q8U8O6Y5>k5YFD^wHoxW!~d1j{eep8nnP1Z^Im&XPX;j*!-i=ay?pJXW0$?Z0qI=8uyH08N^-ZAgSpPNOwdVeYA!#&gNm7'
            '^2Qn^*>!?`1R`|AX1^xCZ`4Z^;6~&_BeN=qJU~-0mKc~2E*9FG<$Quu8Afluzq%R~#nqJs>@KT7X=UY$7k@H2olI1<MI-M3!9^cK'
            '6yXCg>Z+6PwLm0p9}65ZL{dG~d{=^g_$%saBhr}Au?V~sMOLk90viQgQY77?_O4j~je&|hH?Sc#Th?M}?xa~~q6SBch?HF5`l}o{'
            'xdGFMS@&@}<pZ;&0z~u*^=@K7-(E-QwxmBtv3%?9uV84wE9DbsU)mpR!Y#s@oZg;J2&j1wo=*OI-~RM=MXV5n(P&B=W)R;!J_#P3'
            'oXW2?{c_qYPRVbR#Pg4^y{Q90K*8J=h4xigl>9jUI9qM#?_G8K{mEqO5YzMQF5Sc*5sT<9j<T_Os-BFGjZweKZ)08@q%&T~5M8(*'
            ')nbh#n-n%Vtuv4ZUOoN$+Yj$+^*%;7WKq_h*%TO%LY^c5LHZ#B$r`O3qmc#BI4-NQB<F69j?^5zwZzHtXjLKa0STKp&e4ao7H(vK'
            '7PurtPr2&<U<Thm!Wjd~N^-`WuhH{|UJtch19Jv0w;av$?5==Lfpp@POP^u7y9;+j)QmrDOLiKz_~byApjL$WbyS+^lk--FLMoO8'
            'a0v?nlof>=K~9bkud5I^a2c`hC0w8)YSVYPzy<bcgpS9M-;KiCEMCso<Og~s)A}a7iDsXoJYyQoZtD$!6eO-b%PyGv?09HNoOwik'
            ')2TV=L`j#w3zA3i$Rks%n!Tx#<Y#$3FcN=TgHsDbsP+kU-jPv=-CHFnhAZGPAO)s7?SLX)I9BIX4(k>P-#M}=b`ynYQeO(6BsQbi'
            'as?coO`OXt;-Mrc-T{~V1e6Cy<mB-W^&$}S94q6%MlzjMz~PGhRtG?r{qY#>j(kWA8Zw7{c<9sR;>qcyC0g(rG{u!NE5pQfYO<l*'
            'VPOqphjXTP+qCV^K=aZnS3Hju2z>=e=T%ZGql<grqV1aI1ug9+Sef=vsLduD+oYua(Be<|l3@}&7FjMv-%a3HyfjK8%LTvn(O#cC'
            '{vN|qB5fQaPE(frkAHxH@^+_Pe@uT1C)h4Fv*P#%a@k`qp15T1<b5k(CjXl)0Xnaci7(=tFpnQx0TT1IL`p$VDB$z3SazqlB+`jV'
            '*hek50D%~>zJ&8+Q9oX6tA3Ykk8w}HL}uk~TQ>kH8%@4E%UPVRvQyyqSpx4~gh0hzz+Bni$o%ikGzwMEfX}iOMR1MoIM@orZ0fTB'
            'sSNn$3@y^Ij?p#w{&@}VD0nFf7i;|jnds*uBlR2&)e$fo{U^e}f5Y4B8BivSael-TI7+h_^cpSJGk&U$Q2i~APSu42uLegSz(4qX'
            'h)&OPcXWSw?^?@f*Ym_?GBM3EC8+E*t{rbP4lkQJZwB?+>cyP$KSO81ruGlUj>gpt4K_oCYll8Agg36qY2ci8^_svgME_Y8UPlsY'
            'GKWK&1~PpD>SMS>TMs~Hjle*`7kH9v&@5R@KlkBif>sHf?v@jMQ}j>OlgZ?Ms1S`aBkdL*I1K!VsY!-g6K0-O1rfuP>CbVsSdD|<'
            '>WJ28MBaC4G(<yhsf*`&|F?M%YzAV(9*0|um>bA%aj2lw2wTwDtD@^MJ_8;Bn&}|W<FLm-=2`DMXBsn<?HW_DL0C>4Tp9O!-G*#K'
            '7`lR;(N009*2Se~yoJ%=%P~7x#b=?U{TL&2TKyQZ@E`$Kk@LcgNF(fXqg~%@R-iEUyz;?c4}{&a&6cvY+uFgau2%RSn$Ty`2cf;?'
            'Q;Su!bz;4y5TEe``uyGlbl;$!JqYPU1}<9{E;{glP1@1WM{0wweZup;vbb&wXzOjutots1ZfWm+&<29YR91>`R=>ZuB30^x&Nu2i'
            'M-W3$9=<ip?9%}{(7t*EH?-bTuotc|{A0E={A>pAnKrXF;%Q3EgpZDH9S}pJuXZ}S@9mwnh$I}AJfj%pVqn`Abx?FIcEDza*6gZF'
            ';-b80i7dRd0}KlF^X?oWKPOl!ybdv<Pu1Owt!|E_gCm^J5oqCCRQitiEY@;FNIjz^abOu8h1dfg1V=m;W9R&)8FFX(@TQ@?4pYD?'
            '9IfbhXVXb*qp=AR*Ug{G2%pP~A%-sE1oQ(I2z0yIB0(Pf+2HZi{~i@zfe9~t@b_b&h=GP?Xx#zdV8r|5>xytqk?G6m1|U=*zG`TH'
            '$nVu)E*_0jB|{!6J-$F|6|9a-E$j_yWq?EwIk^y%vkfB~zPE8dX!&FuE|)Vsnt?sZK^w$z3mw~^5z$P&Hh|Ar{dy4S^;)T!#J7@1'
            '9o2$hPYjl9G2XSh0RtB(O%=5zHlP7YGXM>eeIp1^_Py&!x)_-L^r2uLAj=m~gCnEp)J`qrTnazX5~#?KQFv20T1uI1XFw3>cb=-_'
            '$q<bevn2pP@yK!<;M?(&JsyZP00Qc|Mi&=Ddvbh2tYsN5O2b87QBHxSWq|lqkZqT&O5_O$BwmO(1U3A8M6xId{^@p{F2g)Wd)HCE'
            'X*{ACvVuT<HP#C7Ly&66<MHLaIDg4FH{U7t%)_41xe`mXyrh{b?=nTuDBA#)AwaN?9mA7^hkzQNXGO>hi6%MR8}e|B`4oeOKwsoj'
            '(wIdy-V|1gRTNSZ1Uwz=rU0+T^0fXlxR|gYNJz4qqr?w_c}AlA01y>&_&c;40L*~9FY+$HJP*VI?%gm>JSjC^enhe(z}JXVf_ia!'
            'Iy%0zOexr&Y+6%mR}~k~`qJ@NgLXU!=f$9IFj64DZ!Y?JE&G=N#y0!Qz=MG=aWwXDJ<->GWrVXDxIV#6{~RjV&q*_~g&TOn!yl?2'
            'f_rBYh+Ag}7%a-$tk)hP<&283XgQ5>R;NuY*qnlAfJ3Bz>j7`@Z=zxNW%r`}lo(Ws4(nSyP&Bgm2dot?Ye5@JK7f5fYAxY_hd|zx'
            'b$q=xYs;UGF>Dj*%Y6ua1?H%uW+Mcl{-lnh(eX(`x`QF0p@9fi28q1#i8FtB!=Ny3cThtPj7N5heppTnn$2-3!#s|PLDOw2!iWC7'
            '*XgRV+bB<fl0em+8D)*7Ddc;X4>V_nT@Oc^vg|c9(LP2;KWtEdW;qB!OpMaP^#Q|5{V<+9G%yj{;n?P>S*<7I@18u+d<9#19xiUW'
            '0Q}pNj#QQ*9T&CKc}D97Vj|`3Ew!~>2aQpt8i?wf4QI2Zq@R<l7nc$+c1}}ahq5de`}qVG2w8EuNGdo3*l);DU928&!W}z#Hqey7'
            'k>dpChB2(<2~lYj=9o%IiGi}1z^pmml01qq1%!k^2!Q6G2xF{%#STico00WH1FS*8{Lyx3@Tvf>_-$20OBSdpfzO1xl7}Z*5}>Fk'
            'M;elvN9z#HH8~?mTw44+utV)h1$MefvV!t0*IB8({b8yy6G(TRWlPMqQ<4no*x2l#e;t0!te`>D!Vq)}%gh9R>vqUWM5nYor(Mkv'
            '*WB2JXv5L^g#)E$(LJ?rqp!Sa$Y#H=w0St2M*3}I`&crgXR2yrAx?-rKO=R)>MgJ>WtQ1ox`ruw<A>ST;qKlfv~>Zd@a^wC1>INI'
            '1Y{S_>;!-de=Q$53nk~FZdsGSg-sT9D@4wvta#0QDSLy$&py*a*VuYAXp%uqO?q=V<U}Yk+s?@5b#H7C71_ZUQJv9TnD#y}>oY*F'
            'dGn{Ki#;u=C2$hr+Wd3Sf127DzHA-%P7vQRp@7Z2&54qgSEsc!2D=jyXq!dANkT2tBbX-h#$;SLq%~?8{hzcL({YAc0%e%Q*J-pg'
            'blL#c^Fp&q7)^AI^XZ#!9je&%TO5&oH--TH$?&+_PTnBcGZeo;zgK5pHI`!J?6I#v7H`ca<XWLd?qCfsr70eaCE5B8ud|2z?O|y0'
            'N6~?#Z`AW>h4a0|_>o&6y^!QT8t-FfHas-qlqdp__`uRAow_X72sD;R#_DCsDL0&nPf|l^L=q7j@Ql{7g%(Hn3~o>iV8eMf^bXsg'
            '*MdSIGwaFFIbZ`fb!KW=T>IM$^k}q(_IdszLH}qO=Ug!%Oh$yD(e1JAK6c0CC}ckOw~gI>dtmLYX#7qaIR~6A<g|6(>~HLMYd!ZH'
            'I-8Ho(e?xDOPl`SJNqYpb>B>L^Om?ya&wjuAM1q^-H1qVX4Op?)S;H`N{v=DNyFT)!*uBhovMwO3vV7fma1<3gSU{YZ+s`&e>nz1'
            'xHhfH9!=<<Qlq|FfqGbV+B;wo#b$(5yWPp2c5yl*#bE}sMXz|f0r0yg<4Nr-aP_oj7>jvWu@h5UMGa!CM?x`g96`x$e*!ZiFbrjM'
            'dvb(Cy9R~Bj&Qf?vS?R?UF${c#Z>Pl7^`R5W*(;?8wg)s;*yp-A>a|HLiHoE(80Ega=qc40DSXZy;1O#S>b#z2mqOT5`CnZE#Yv;'
            '@oPIc`Yyuocu+VeqvOeh2dOX)p-5s1AZ0R^Q0ePDik2~)8=QAkaAYi-{BT!DAJ8#oGg-p8o||w0fvU?qG<`63L7*5r8o>N+hVF@i'
            'MDbXd-q22^X};x&VWdh};7MGgivuM<@K5n{hmT-|2-b6qb9?jly}Cy1V=W-JZVVRSi6ZlMCT}U!z8D0q7D|*flUdiMOU)y+l2;o>'
            'S+d8Y&u32{`qZpN{B+;qKa*pdwHV_TTCALY$PQ?QBDD?zZEY6-R4XUO#7|aE5Y3%dr>?vbEwrfYD$6j=dgwumRjHPH+7vlQixd~P'
            '9iz~4SorY<@Z8cd)lb%(aI4U3OOIB6(i1X(E}bk`F9PF&o!yuAs9I;Dhr*v5FRv3^QBW}Ccm)_kdvxBvcslwY;po%<`Cp^|`@fGa'
            'zwHlc2qs6zjm$uAZWCkp>cWXMxOB6V_K&;qt&a3SA3N#P>_#7Aut#gG2YqDs-&DR`EVGMZ6BajECV?Fbp#Rkd81>)uW3yO+PuuLw'
            'KwEhY?y^xLMH4RUI!(qSWjTY7$5xRPXGec&e1&B_IxxPXHWW4ED-7AA;T`?)?)$cdgfH73x5Pw%bXk#4nfM5<uHU~17;+RjU#iCr'
            'a{Y!R($wvWYGd(=KcP*qUQ;4Be9~gUc%UC%b3Q}+B1NDNYK!Srm{AuEYC0LJf1d<_$2ifMWUK@JdhvPe%|mYtH`~qLG6q9PZx(XZ'
            'arXkx>a*RR$R-N8Iq!NNzQqMPMlOQE%m@YmlSWcRrHrP1d|3Vi*}JpQCv;@t9&sXPdOCQNkJy`N=W-=?b%s7w3>)-iD<>Z6hh4ZD'
            'woTR!@wIZqK!=1uo*I|b#oYz`W|n@33+L?oOSZ4U7x>DZ=e;Fi6|QdTktE3BlVNj1kgVLY!6xJ5CwtH!A9NEAyq*l(*B=(~iLKa?'
            'KQmoMABV<Ek;0a&HYD^699&pd7GI_^Shaatjda5B1@&`@#08@0P@O!8+M2^(^!*;;5S})M(P%}F=~EsUZCze4UiC)}7&SG>$mII~'
            'M_fmGIO-4Vlod#tIeutiDMlTqkaCxvp8;D~qKY(j4CEu(nk(1%Z$7mh0+!(olX{{;+he?b&{^3D(Eh5W58r3%Opp19?tF-D)_=Fq'
            'C5)EUz5#lfgN6@wz|Ln|KKoW2(iT>@T2hZgL5wBA4Hu~Eds6Z`nc5vvcG9J7&$kc^Y7(UBaEha~q=jrJj4?p5t=HnD6{0Ys038HV'
            '#nJ$dyV{QFeC$e7<cXXY0~c!sUQp`@`W3@(mPh%?SWfq;-a$j~V}OW03&~a{x+VA3C~@NsJTsAF9MTq6ZQBB5b~?_R!ND329u-^9'
            '>wVSTdc4iHgLdhO<lCmfsR^v1;S9TsIn$2kO_)uC&b0Zc*{sc+Gi`h5Hg%WYLLn;T#5qw4u!sP)(c`X04gKBXz2cR%zaVk<Ay_q6'
            'YKNr5lDpveMgt3wSIqER)i7;70JJ%_PLN_0?E>iU2m<LpbQ*u7p4vHqb2N)pOD=~CLR*444x<s;_hV3Q=3!YTXfaY;!C<VOqKx7}'
            'WQhgf<>cH#xdqKN#(%)MpA|YAbl*rPg~Eyjm>=rFBPYJaj;TsASM$A%!lcldjGW~h>V#J~lc9GcTn`?l3Z*!Nc^Q`&tE<jS;`3M~'
            'nA1>euFDxh_>hW}da$^lCyWJn9Wc9M+_g08u#o6-S(GfPd2y{4izXFx?<g`(5sa4DZI&6QpX&xkzG!lEd2wV*mPeQOAm+7zB0lpV'
            'C`TOahD9@f7?>l(c0v<b_Cau-%nR_E(7siJexJOp9cx8;QD6O+`rF$#AI@JtReyi`HwAp7+=ZWFRe{(6y5T1J6!e2VJkhT6EOvnU'
            'B=!UH97g$2Jf6rs_6f_5k`AL|qpXJ=c5gv(T__p`meAT4Oywl)vFByBilby{0{MlN^*gG0gp}*1{_SM4b!@3*Z?5Ba*gS2I3V0OO'
            '#FmgC%ILF<!xU<XrRF5*UAM$(AE^Px{Od=QWO@{s;)3-YuC9zt`)(a$loq^7;uOn7I5oZELLqa+z${pauu4j)TrG2#dh%Q>EYkIh'
            'xww|9Qp~i!jsRuZY6WU`Ar-{KCFT=PxQH>wGh_v2wqZ;dm8(EDMFD15<ye=_NA!)=iz-P*k_B#Ft+7@&^KT5FN+P!vacba-Sldp@'
            'P2}O7RSHeX8P?aC`s|yX7g>UZFHA+UM|h3_V(bZFG?=cj+$a!MBVE_<9T@BlO|QYBxdt{Jey)Fbi3ZYvt`VnqG#|tu>vOavYIIX9'
            '(Nx2^w*-Ts|4JY4zg0G&MfAFii}k4S&7qZBDBZn9QnQ9Vro(A48S=+NAl0~>_HTev`a@lZa=PkI9)G3}?qL_ipO8BWCQ&I^5@RsC'
            '*s>E?y-FpY>2YA^xYt`&2e1el9s|zn*0az2@5K>{6-Sq+lau8=6>_iRqWFWp4U^7P$G~j5u)+hmq0j;9xbnmh$)#OBvte8RXEWZt'
            ';w+8MgJB~n-Ydo8mSgd!=zaK&dL3>}DY|4gR=<kygn9KtU6uwfl3@10u1G1zfgHjEhu^$@d;Z>36CoB8>hcsQGwzTVQbCYVR|w&t'
            '+9%^<A6kylrCgEcI+nN-eQGS@)hf!Vuqq9zgdS`51|U6PA~phB8mpuKek44a@{V;uy#j?|tf4dZ(0Oq2nq~o;(a9+sc2ad<PmLNl'
            'MDmb})OZ6wXC$p8#D*@~U@=iwTyvQEDF((?=}n4Pnfx&FOTIW|U$EK<gQ#oI<Wz-fC`TY8pRt5ua8~#gTHLZUa!!D%L3MPsx1Ekv'
            'pg$k6%`z>mrCPe`usAgCZG_9is%Q9B8#*dCOdAy-J9$iFOtoV)@$DZz5atJ$y27ol^>9qJ9(uM`2jX9aq6zVA)W23(;B$>deza?v'
            'E9*oe7{%7^V5KK?%1Ys-DC`UYF3%^3rS3Q<1oQa)hs0<6=)Ucw==Huib;OkdRnxM<cyBB{q^~_-!bI>Spr{k8#6_V+7m#oe90-e~'
            '+X8EmLd|%iL`Ij-pc=ZyOCN+7>1GEk>C7SWqBe7!1lcALpoW5R`leg12MZf&xvCP;c{g~J?F!g5Xb_7+lw*+<(WvZ(93EH3ITi#<'
            'Hsjul53gR$&YwMfBR3b!o_~1vlpICpXU{Mk=#LnN3SiC=Kbwq%#A^}<iPy0rOETqMNF}AUWY2f~R7<q*3O04c9)!pDR~Yo**K_<-'
            '@ROyuCK4`)I3YA~@C%D}#9LY>@cP56_b-2b_44c;4FAM>|N7~FSU08|JFnipK6~^2eD?F%JNX2IdvKV)l1*Kk7Os<Q&YCz~zC?FO'
            'i6ug!d#K1R$QsuyY@4d3xP}Ptt}X?_=^M3B4KfCDOUBh$okt+s&Ja{2HTJdOa%Io@FR$MI`RS|K*_*R>fBF0D*Rz*@`RjX%Zu;J<'
            'yh!Cq4y?N9RatzfMB*8DK`8q5uWw(S$x(JBd5w5PSQQO_e)gLD`p6YvPl*!rGj6YA%BfH*e`BUmv@EC`S`_(2%$tTg;{94@rO2w#'
            'o^UoL<$#hI92CAPFR)+2Kx^GZiIK5uC+fP1cao{1r3=<}lw2@oiz=@Ead@r7uP%4kCLjKG)sXx;eE;98A^j$5&KDfYUsp;=3hEYk'
            'J;;|-QbMwy*)}vBAeAk)la(o-)lW*EG?R><Wg&Wk-n~B8{0fhVE7SZ+g^N5ZP-N9x>({5h4A~$Miy)<U>hlPsM^dL?hb?t<8pyOd'
            '&GuAbVlN+5<y|(y9VQHbrRflD7rae**c9hQf?c83Rl3Y^yW`z$(RGM5E9Kt0t?&rk+RPF{+3V4WbEZcyuMsDO!$j!TuLx^2@o-ok'
            '#5^+kgQah|WesFIsaOtuid8z(Rm2si|9U0_&Tzg9Z=*UxfMe-3UV{fOkTIWjUl&6`MNSNH?L?&2j5}Ww&%yzY>DGlts&zk~_vuy?'
            'W>NOV8>~99D2d$bIx18B)D94fm&Cs6uW%vgsZ&G0jGT{X+PPb3A7e!b8f>eR2P>UxlL`>lXbwUcoGL@9Se>fR4Cs;i<2}A~3#`*%'
            '4!J_=I$B^n!Y7sGDfR|B&pQ5OUlT=iG~=vtahxAx4XbiMojhvLgJpEbi$|&r1$s;#ypaM;Vg`S>E4j~sJ%C1_8!|v>czVEH6uf&R'
            '>Jk$n=M`$U$mMm8GrWL_9sdxFLHB_N81O^W`|bpt-QS)3+Yf)VZsIXlx(Fftq>h^dkz*3?9pV9usbu=&$EF=dVKhpbHb*UgU7Rw+'
            'OTbaF203^zIv&=rI6x_cV{&@jz?L(DrmEGBH`<%p+HSHPcwHaM>LV$T<In9<Z(ZHmL{Sz0nb*-&^t*y%USlW-0^f6^JN$O)Ov1^C'
            'ua9A!8gC`DOY(x9%Wj?9oI7uQu1DV4!qB@34y$*7^_d+i=Y<(^J%rIQMpjsD0g2jj>qnE-8T^1YIWq@xW}eb%I8+a%w$~K^Ujq<8'
            'ES;KiVv?|k<{fQ$mZO2<q=&jL5(|W4`Y^>X>g*7fS8!15kmJ<|hVD~Dd0^YA3%e{Cg0X(0HI=^c1FB{1jlHoEyy{1RIx(znyr*;Z'
            '19=t^+Bi&igCOYc(y8~NX&U2SDe^Ti(hk{XW=3HWji>Mzu>F?G!)gGC64lZWt3>Y8gAW5tW(l2+^L<xZ$j>nd9(>U&tI=XsxWl5`'
            'wQXT&bZ1JyZ{#tEDE0!hQT9KhTBAW0Taw!_iUtHFk?gn#c(O4Xi^;8LmRXAvJ57tgmDw_hRc5KWJV{03xnSRu$-f6!@6Py&3Z<`h'
            '<BMuo`3E86%9!A0oD+97l!QKG+uLBU&kfy-h;DY)-(0cl3vNKvb;G{)I>9pB=o>LgEEb8^v(b%$`>ztT3Br>KtT%!4v3Dn+oE%M('
            '4H`SZDhlU9dT4g@EV+*KBwD4<f0B2hk6S3pa2;DOqoPKe<}-_-{iG(nwp6*L6+K;UxpWVT7<Dk8u&R0t)<r8xMuB*UXnRG5_EPz@'
            '0y2q*U@A*@Y)3S1!?fkX!^hlV&*c{_<ti!e6xA(Gy`fe?`*nyobpHGib-VW}K!4mpugSjl)vBb9`lm9!Z`JYekb0JFmkc)BwaNh2'
            '!qo9<&3WwwOkx0wF71~M_kC`J8M;*(<JIFc&DQ#zWi554DPvD+Wl+|WY<Yh>Qaq@YywOO)@~`e>T|nO_=}q6sbh0mng!Op{b?$Xj'
            '&|FX{g`^%tJ<8;h@src0IEx#-O#mBOW!V~nx3p|6EbqoiTP3P&dK1`I#tOx-*(AVy+&5<a)3h70O^II+Je7rZ7>$h3PiLd`b%)!T'
            'VQWp`pk?TQVoke!EofKzOyT}gQY0v#){e}1IAr;uU}ZC+?s#QWhiHn?d4t}=akQ`}&?g)%?Rapq63J8d+qR(FMl>-ur7(X&Lwg{x'
            '9_wjuZz*5~@o*@Xj=7#rT~V@y5Umy3TNkYn4X;Zm8kk;&*aIxSGlQ(zT}xuv9zD;Cnm6U`9)M_(coSMpZB&fTpGgbXcT<170<&KB'
            'i7u*F$CSwWiD}wsp-sol!2Rh*zd4#to!sB0NxTcw8{fs$^e?@9bN=%AS<}BHk!Ouc2Uo~?MsR-BsYvXoG{KK`YB~9KQIl}aGaZJD'
            'Ap$1AD^&k}!UeWM^`BK78J{4BUFetEd4r0|QPK1t5~s!Wt-HV~l-HEYz76%Iz8l4*HKeSV7eBBHZ_yx4u?3`k2>3PaR4%)TH<n<S'
            '0i}+l=<}0GfO)Jhxug|lMT+iS0v!q~OcC<UvcMz+I%l-A6Un%!v`dp0@8x-IT(Gn@b{v!LHX6xV*oFzjTu)yQtB&y_E*H9><XsAL'
            'q5DM(9eS)IA~FfLOX<!g@*x|SQ;AAp9g8EDI!{5Y6<L{|RA5tKBEmeA89Pq3Cxbq{5}!3m{jiVL;<E<S<6z<xb3h_5YEN69+#{(D'
            '^zt@XuglifelF@jh-M&u<6WKS!~32iU0_sHd67oNe-WpUK?A>+hqb7qOO%nu4q6gQjs%R-sz3_hA7u98PMqRUSSoUAeb@4tho+&C'
            'w^H>f-g-P~Xhs{H_F82$ZFQ0%gOceLs<cB&TGqW80@u>OPluD#AiW0smJ{8&<67)q5w$W0QxsEXKv{mC;rgNRyBX2-OLs^IOspZn'
            '4M0DDA?|nZ2gbADqogJF{DD}3N9v(z=stu}zF=*n4&gG&$;~0sqEB%hGK`BcPjD1{OKu%LP+FCde7VrsOw6fePoE()^5eV@JEhXS'
            'fL!5J&-<SePtBFh4+^OIOA=AhHbTNl&?%teV7W|6TL`X())s$jNKNN-;UBgDYi2I@apSdI7}z4n)4^pAhq&bIrsuH!M-SD0{aTx|'
            'T&6Q?x%w!<+vi5N-;H2i^Q?a$ptg&W%*#DjBOOw7xAQ)PgG){Q&W3i(OcZ+ykLADkG@5o;c)Y9;6U!}$Bh9ELCuZUW$g@^9NMVIY'
            'oV|Jf^4*zzrRR|9zF>iLN@LLCHPR<jou<qyy1A38E@cw<7hhLdRp>Oe!O8cN0%H^;tAh-_+M;g7Q;<9u(@oM+LJvh*n9~+7d(>mr'
            'Qngw|cjo#%Eq_Yjo-OxoGsG)^4YBC2A3A7BqLNf0vh|z+l0q=2I*#zwM`C{>>x3)d3QLqBPxZ}FAkWiX$9Tz>2_8nFTZ@epyo@P('
            'ShN>rNW!xwD_zSt(_$}1%oK9LBRWA#?eB>7y^j<HCIa_BWa!W&7fNj_S{4@5sQeVgk_^T5jyzEF^ulrD+`-v~(d1YpSQH|>mvGE+'
            'Z70ZYxWU<fygYyZ^37im`bgG%8*e$s2Sj&DM3SIAwE7SzjWLerz``B%RaBDxEB9n@YOh^zOjj8L*g{1m-3O{ij>H4{Q>{pboQueD'
            'jCcnlD-E9C)A$B(;6tNJ5Sxy$xN{n>P<I0bPHs|jXcDjJWFJ)OXnl5#=#N{eB#9C6qApOTDPL4@W$}Y6uY#9R5ACG5e_*&e?l_88'
            'q?9f6O0{fK#ySP*!uGl{+0%eaWUYPc%;{fAG?7e{YL$^Hj$t(mj6Cp^nBO39%{skw=%Dqe@y(&pg8!nfwaA1*nTv;Kmf^$ku(4?3'
            'xYp*?67P9s(Z!+}oC0c2W|Xb5!t7Z&8MgQRNbLzg4s;FeZ(#F%?SI1F>=da2qvHocgbTiU{e5NdNwJ2A)YS4xK6KAp9UiO$Qf(T0'
            '?2!EUF1+pQ07|k0Osy*ytUwVePSA`W!N>k#?Zk{6qk-Bt8v?c6>tlh!HQ_AT*t(>1Z_VDY=cYL5O}}w<YhWjG$dtQ7G(BkCST4pH'
            '*a4zLYflW-+jIIOXk6)vxfbSfUAD)d>gt3TWBdTHpeVnk^VA%vKjLZrkE*cmM@FuMVH}aVu43V)Xlnzh1TT)4?SVcp6I+161i(8%'
            '#=y)BJF|%st_;5~kV_X?oweaD7iKPE^cYMq7|&)YuKMg=-$3i<dSG$O!KzQ0q2WAClUUe=+J##QrNt^8$#sOH3{XBW_V79ebc74o'
            '3xm&g_xC#8aut(J&%eN!;?6nL6XRcz7XwcZ%f<TC2}^zbXExZu=+OFTT@USMvn&lZB_W1#&n#msE#A2M<|U7i#uXqSy%}uc0#j*S'
            '=55iK5R!M*3-FO%&vig|#@+bj-Wqo^^_~rTdsKjF%d?^5@R&OMx;-hB?&;w%20HDBR=-Wfu@ZsocK3M6*G@_JL$rvKr3rTU=Q;I>'
            '6V*ZIv&Ttg9DVP@gG2hWWS`Z;s<jo~hOiAdz>=;Z^*_*8?zvpF@!GzT&;C@#&TCXOhvVhr>YOUu7$CT4vjrS-GpTzZ%6K&A>(Y8x'
            '{Xf&v>+CR>He&26qNB5=FoXaYe6FK;F06}pFELEyhe>ch1}=0H--LF>2CBHC57(MLFtJ#Ee%27l*VLL->>kly0L7fytm~9?jQ}o|'
            'Oe5&#Li#MF31ID`htE?p*P%7y2Q+k>b@+9^<e?)HT<$rgbzRi@E@r#d?o4hOZd_O0teez&fAzF!*D==~yOx@kvQGAO$ZeJz=NfBa'
            '1AFI*A)?lOrp@4q4witR8{fvORtC{|0d6-(@zBk|Vdiep;7~{&^Vtj4xjNMC;Sg?OOksF5y7w(w^sLQfQ`c(N0>-;t1mjJR8_wL@'
            '35yH79PW~oDDIrpz6;NiOoS+OyZ;Wupc$dA!}n?^7b34;4xNYOHt+4N9M%${=x@y6pq;L~P;XQ0RAT?M2dmAYLc&X<N`Es!px*p@'
            '>hrf0fNoBpl`oODIo+2?+N@Yt+r%4gO5Y4_hQ*zeMf`Jn(2ow_(8J{49k&g*;T$a!Yj5SpakMk#M%#PO4<X{qn99p72Tl<X=YSel'
            'TNFw$+KX1qt^mwi1=Y)1c(vbasQW0<sY|*J^FrD>*OCwULTu0;B$DvHA#Q(-O2@+H+Z2<_jV>n=65n)^VSq)R-Kp@7uUxY;uS>Ft'
            'Y0J_<SevxsYBFp^!`|EQEQIXW=I;GSQ(LEF`P&y(i~tsGBFd7aIb9q{mq(N$X||KE#+lOLthG^1pH@+%^_1MTA8!KeLFMZZCCpLA'
            'JZ(wjXzh=qi3j_NAAFHA<S9h0vM(=*EI_RkNrYpqc<f7g^-4xJ&3wRg)jD3x<J`Vo0adbC^7l}w_hfGrx!cA{X0Bx%{3iMv*}Gc|'
            'ZDK)3Guu-bW)h0`1*@MWNVlqp^Dy}$dz)Jlwmn?TAbeP>;cZq(b4{{6Xz_OyTrA43xKtOzP&Y|D*G1;OU;+=8erqZPcJZa&VXiu~'
            '(GEQQ*{Z^1CK()7F^(r-*NmH*I$u4>krF%B6o@fSmbQE`(=Tfxcj`&md{i$E>@}<vgT5Y?^+%tZ%EP_2Ve5-2rv5EGQ@Wf-*Lj{k'
            'KYQ`?!>jia$Z>92UmUzo8`T}ORAD&d)b*dbr9r2Bw;Dck?Kxbi+A{$;v6{ABiO6>9g`FB|CUK%k*<S1>tmCp;(APde`(K*aPOFy0'
            '2%V?MdEI-qxNM#;0Xo5|duPd46qsd$M&C7S+}Ep!_U-0-?Xzr|8V3gNL(V<_88{S-fv|W%5J`=lXz-f2p02v7c6NE6+p!nqiEt&<'
            'ev8rYFr4<;Jv+l;LJR*s3-WB%k-H$(&yIb2k0qtpHel}~b~|Wk2(3DFT{W{?qa+1#Rbw;xsp+qqG}c8(y<z_Juh4GI_HwY|SV)Zs'
            'F25V!6*KWa3Q-wZq4^=Y6)e({%Oj4RYs6hKg(_;p8^(q&SOzVf)Z-Np_Hvu9XLAat<r7f`ZVo~RDu=Dk+P6A;WJVN}>s_m-8=@SF'
            'gBtj^fp}`=o=9j!o0PKQsgWaG#Q1&-T=uTBG-XN%NnnXi#8NL%tmSAmu3ABX#a67XFB`eB#jRzE+>l|46sO8YKB&+8!v*hubmEDr'
            '5EBZ!FetkmDrh@>pVQaDgA_v15?aa8##j?B+!Jry3b-r$xUC!7vOp^18j?*{CbFW7Ky;z(M|@oq0MsQ%sbH+NP87>Q+L}b;Tfx`f'
            'GzM7N$kX?OAGtAsseMs8tOlfq2rCguO=@X7ZqS`o9+gIG0y&&vyHWzj<s(ty0HkbT-#;>!rsN)0G;E;r-a9SY?2vR~w7uwVYix*K'
            'Q~mtmmX4jU1B(5Ptp&A>)e2xV-2w@f6|SVz(*71s(jV2YPv5<H`Q|UD_C0wRC=zY9<xUN-Vz|DGuVjwbx5FPb_+GEKNYLBl1c#W='
            'wKv%)+p26%-r(E>=a`!UG4M;_G8Yt81FB=0Rf~1Bq=E(JE)3Qru<khp>DHC?`c`FSH6D*m3)W7#g_dZRrb^!<t)%u307X^;h9uuo'
            'Tw=%9Ey@bE^}d$eoMIiGJo#Zpu6i;C;8y^f@wbexS^i&##Ts;@F@HV?>WVs4OoxT(&I&qwIYpJ;;L#c5-4`6=d}>#585~bezNZNE'
            'lkefIpZEKoVS)#vO*#b6N$8UI>*yo@HP96mY_&z=IAz|pY&pYAPdVktMFP;lIc0zl78Iy+QqPGB)SIf;(IUqCAuvFI8Lou!0X0@h'
            'Ij++-uv`{~fMW>Ocv)?>MXL@)<3aN{#{bO1ViCtv@>~unc$}`>VV8&%#*9Z?ndQZF&>y0Xeu^pS5<NM^;9nMw$2^KbzbaRwKlU9o'
            'dP^<i4O;NZN};HqCYNl`YnzDVayg;ux+w}FBySF79LPrD^?b(x%kuOejYfr~tcjjX=ah-s?GH_;RGb_2{FI7zh}XyKGc38W7f#z!'
            'Mhs|H0!Wz<MjD2?@=-$3c2{tQl<_KDloFqZQ5O0Uo?&E!r-Z9agclUzEW=PyYvlTZ=;$Lfc93e0A*vOI^*gsoNY}P!2_}nC{mmrA'
            'V-lHzm9)i?FdSRE1&)6kaphV#jZ?~UL1P%qQd>KgLi%1w3gDfakPD}xIp-Ec1QLdAir|kLyRVL*vw`d_I^KwBEFy}Z)Ag!AUvhxm'
            '8)B!$%B4Q6MRPb+;62XukGg=AG};%;$8L|=yR%<jp8b0EeD>$R1L_NvVSLV~AL|0Vi+B@b<S?N41~x5d#IZIvIo4ORS&-l*aIb9x'
            'xq(&hciW~3zG-=H4*&e@)!DQ6Fm64Mr*HnIw%Ml`=v1Md60Mj8+9!2o431$1uszEq1$fj0rtknanISmUcyOISK0RF6%iF)aH{{{q'
            'L<-h>J@1*MC+4?JdHUJu<KD8y{@BnHHqk9!f`}g`VU85X$vxhP0xP5|aClms55%z)__4_xA;Wzu_d`m(xD3RzAa#~z7SIc7Zv!$f'
            'srFkZ@-8)|eV~V4000915^p8m-~njx;zPTy3p=?>YK?Rsu~NWNWRBo)m&V8$L#KP*d3`TU%G8!ObG~qfaM^3Afp5};<i=F1(6oCG'
            '0H2+9e2{qqs`HFkv;CF~Dpps>-kqBUNHiORHbjD2OO<{HZc9@X!3X%y%#=g(I<_DjP!MqCBj1#P&x#!~U|@Pw*Rvjk?X7Ac$`wQ#'
            ')w-jEiu$<6_`y5e8!E+Cqs2dskYw}v$TXYW8~Pvedw<*aUxc63F^PHcr#Out8}$#p2OD^zU?br5j;Lbbl_R;m{W}WM$yr5rKZ2@s'
            'Y8M~qYhi&xW6;Rs%Ne$1hjlKp3-V#UY`)bR*{RaREGH7WlIV`(wDy3h+sHn>Fi_A2*m5MH<;>nw-Jdtjn)}<^%Dhwk#Mz*Y4}x|G'
            'b*Yio_WHHox_NJ!51{!)7mThU`0pDtpp4MQ!0D;~tyygMV|l#y24lB{^{U}(uzBKXv-Ez0(~+6DnX=Y*83cfGA9Q_fc$nS7JnXVY'
            'x16nHpylVzr$$Vq4_F**`yedq5AYYB)hGs|MQsd+AO-e(f_aata)F#Z!&*zoqZ`nS?X4D6`H?57>avR#fNsqfz<q&Q9CW}GHWC8T'
            'Xj~ptY$_DV+{lI_UKCag2OZ7CABj`@eD90`Rj#_%&+YV}7XZF+HD#~-3Ejk-Z@NAVt1^SNi5C{DJ>@Xs21)7Kn?~<x*N@FlTQT0J'
            'H1F48>%u5N3|3tfNde<Mj<<bn@nk8eXuBQ(>YftFL9xdPSMUVIFI+{n-9jsL7nin56c?ENh=!(t0XSqMN~03yMdQFyX;h?`qElp_'
            'k_z+2uH&SCX-|0R+`Q)>kVF4e9Ga7F*l{TBIfaJq4j;Pv`LKO8^k~y|N1`Oomu!)M%ky(;OE_0>)imNpHqlC$Qn^i}qI|QA)X`_Y'
            '5$i}rH${9?%i0aKnwN(ky8<~HFXTvZ@E%}cYO9E`JErE4+~$~^Q&P=pyHsSeQ+H{p&!)`NIS8k<Y7B{Kt*dLzu&7TXK|L^GYfo`|'
            'ZAe-Az2OQYJNJ6EsW0%@duQLa?E<83U2xJOLZiBvjvNvd_NvEbzjHMm!%mH>*R9U&4a{#wU;CJ7_SVLh59_d{<$-;+ad^`wG5T&Y'
            'Yc}N;rq|!yE+!+AAX4X}V9zecq+(`&Vpg1yfqVKzl0nD**~1}ePqp{miR6Zv4{-}Rm2XhfDwk+2R=Vu(Z<spJY*(|Gw$mv%#E#e9'
            'jZ;jhb$7V^89Qm0CFUXH+97?2eax;OAxG3MyXgmqoI{pjsL3<i#P>jqV|zx*HUR>w?L|?FnUg3-MHrkJ`BtM@QEfIMr3>v}OrNLv'
            '0A%R2;<z20SsserF^#<y<rp9k6-iXs2W*Em%N<jTRqN}mQ_%83TU3j+sbi^HOiX>d=v614J!O(G-feLDs<AM`fdePPN}y>TGKT`W'
            'x_^iT>(Ch+1C{HulPzPr<yvDkJMSBA3)hrtL3K3WVRe2??O*3|0SVO9B^@7*!+f|GH#F1q)QHYJc-V~)O&+c53v7A<|BUz+N8sKH'
            '8+`A1MRhIs&Nl*aWW*#^2#)x!z?i?QM>P`18;h6GRtm;TmBKO`X<_p*$A*?1UKTEQMv3b@)m&e_(NWq=Si`2(YS?0o8#c^q-p1m('
            'GTIDJ8<qNPy}HFxGpXHB4bWdKS5>rRFtoc0_2iL9PCr{c^VPMO@`|?MT602AR}m&G%AKmu)cGIsd-oO{9HEgUeB@lJpOpX3F6sK9'
            'q5bf23wT{IM7!;5>9=jfezi4gz|BH!zvX!KwRY{pwJ(;+3iMshGTt#Y_uy+WF+d%!;qdNZqk{_CjHzA#f9ikJbMpF#cepT{oT(jd'
            '%Xf_^ddtu+@yMsDpJa<L8Bv)cj^!QQp0peCJwCD5o=;HKI;wIMlNVO;KZ}cCxuYLBbI4L%1%9Dky?TvO42QXB7YWA-Y^ymOA8UIE'
            'ZAVD|wDn}MS~;=e^$%xiR<wBi!-m0U3j*N3+N_KX|AI{h+PLb`4>?JlP{1y8wNH^~{PiAn1=$;VkwLT(91NuSxN|UV<lH}PrE)*='
            '-Z@gCjr*YOYcGhq@yvHCn26E}<JdalN?ZLqk6k`>aQx<vEwu}pocOWFw+Z}YG!HR`)`8x#5-lAJ_`YU3>gf=Moe*y}Tw3nJ!>fnq'
            'ZyQ*=Z0ka*OXGcLG~3`uS58OKmgAE35Ks-1uw3mk2UdpH2VxzaB8qOtZa+c0%f!}xPn)?+M(t@zR4vh=jgKe{mAnFtS3ySvsF(j^'
            'c&%H`91ACXH=(P*IbA(1FzOW_pFF|G6mRN0Kb8uQ{b-ZXVt6lIzK2#P_<$G=yvD8)*l56tXUmw3pVD#yVDGKMoo>ywFD%*FDhTuC'
            '8Qnr}mzaz<1%J=ERow+Ui%pm$dn#%EZY4Uuv^r-~d%@d`_adgDyx>KmgpdcyW_E)Nm*KXA^|y)(=_@j{8;2~0P2J+JaGs6m2HGLU'
            '-(4rsNTNAM^@+}9;!MD)sh~JWxtb%FK%`}a^E+LvK{LjzkKk07Gtspofqbv4T<$C%+V^{_SQIG6y<FB^40ZhleZ4UiU>GV%@UToi'
            'Lz5+0*v1Gh1~rr_?RCMz2Q(=iLCH!I$4aIwzDy>wT~?J96*6JdEas6?;^8>n?$SBMd>3wAd{leheRwnb<>m9Ux3i~jp1%6~`OEVk'
            ')#TXQo2XceKx+fzE57fde8lxGammb>06B9EryMl$X@+iXjX;u&3ap4_tzEs=_WEH}%`FDBQ^2T>Xosa)Qz8ZiAt%>a9+zwH!sTz&'
            'm8RS)I-x)m+~Pnn)kY%aIl5aiXoJU+Y%MpPnZ5)ZReRDEz;G$&(K@`9@Kn5atq{TSM@?u2R!`*uu~^9!ufoL&qgk&mqxRt?6?6p^'
            'jPEDl7<?QraKDrb<8$FvI0&iiYoANMVL1TIQ6B#mB_@B*8k3e_*{F;q!VYeay6VxsqJ(w>Dsa@E2u%^<;yV2sXQSU>PK{j1`~_Lh'
            'Ur`FFd9nI~rk++w$t|4LS`o;dICX|oK1X`H0ZmJ-!%q}(+w!%XP?RoyK*ll$cv+bHXR0k7US-KNX|J|)Ky@Y0^1i3alI_L*Mm<d>'
            'Nyp0EiTUd5Rak}z=o(nKBVLSXTRw&@Hj-Q(DQQakbRBB`VP6qH(s_BnMu;npP-<n>SPlx4yfZmI!YG*d8dX?<rEB~WbA%=!?gFpj'
            'fOQ_VR2rtzCh8^KHNqE!m;%27%Bb#xwoIp~oY{1xOI`L*@pqQBPBTfEZ^SGuia47MIXz%{gPBi^ky~W0a5RNUc3nj%tW&{XjIkc?'
            't7%8J3a41h;fY$?9KBOVAO8OI-Ahr}Z)i<1aDADj%SsJyva|xJ50F-D<5H2c1gnz3f$-P5az$~I<p~$CJ@Pj7Q4ua9H1=RoC&0`<'
            'a1VY9+<l#5-ej@C$`R>~%5cX!m=hPZT!c42N<ju_eIv+iLT4=g&m*j31nAo#{?sNaQw~($A)xV5>u%?xmzaYdOkCn9*xX(n&JjBb'
            'X4zbCZ^f*G;Qtntw44GeP6;9B2WP0!5*<)eQkvb%niBGsRlfu0ri{qBO%PU$DUnhhABg%dt_k+FwZQh*0Bf91!SL|XUcI{ds^M@('
            '-UG6XwjvvYAOYxGfFe^oKAseY<>{LN5rhDvG$kO9XCiWGktzC!webrp+nf^?`zWhn$}#2#tus1Ej8qnvg}K^_3Nw*nqb&*`3sjV9'
            'UGpUpM$}u(F}g!Lai~rvn&%sRg>$v3de{?Uj~l!Ji9#wkbDJ`)Si(aw4Q4H)7=J|}Y@>&R+{7LQvi5*h_(U$gL1SB|!p6zK4I22#'
            'c45{!r_cr$FKM=V+)={DGSs(Du5o{St+QgZT(hH*H7307XV40Eq>lI9A>?#|*tN{H&B+B;`QI0~1CT)ZN`++}ukcm{R+>UA@U<O>'
            'bkVC4=C%VRhHg`!(+iC~p;Jy+Mz2-?Y3<x>1H`xjNtpIPA-5M4B5<&c+XCd>ByETrNdttlic-q!+sDjh3Pjf(>&;iaKGFN$LcDc5'
            'u8&H)Mg7uh(Mh(_`q(axLwAz?Q5cF_U1?9Dz^^Y5*l8x_T}}UQn_{Lm<ybX++N}Gqe`Z*48}tmk1TA;wKW#^gEv@RikK25mEhW49'
            '2bAOD10ARVm~vc%aA@s9lxsjL$800YDGB=js+Fkz6{}GF3l^Mum_&b%;#1b^t`bxaD#X)0KA3-jDpYviMSdNXBa$~p`l!XiH2aU<'
            'E_xyXs=taqMoYOwW@Oj`LJhvCycP4{=5h8=oh!Sw#q(R5ISx&`nZ@{9n*APCv3}<&B%QU6zq%sHL3NJ*#j2Ay*&r>^zFysl?ktXw'
            '-l4Tq7aWjUsSHS`XbgCWvGf|SpmL5zVY=NSOn0`qg3S^GnPP3V2-rjcl({T?L4r;4C<>DrkVh*}80gNdJX=I;MhO822_`j>n3i;B'
            'lA!i_6YV(ONa7-#G!EO_=mr;JGl@_oEbh&dz>yYkI_=Od4r&HvMVzXm&j98A$PE)l9o}RiIsJDeznF6fmxzKt?aI5jK_{Kjen(aZ'
            'M`Uy%iGspVo!}i;Cf9Kw`sk&YCBiEQmJZe7(OZ--I5atej$6|zf=%9u<;V8R4E1E2C1@KtFYb^}+bXX+nK1l$d-+&@nJr{qe1^f^'
            'Up5nBs6eTc7UKp|{y*-AZ+!'
        ),
    },
    'script_generation': {
        'sha256': '4a71343c730bf7563993694455a1890d2105b3984ffc2c4a0561a5c6cf728822',
        'encoding': 'zlib+b85',
        'data': (
            'c-q}PYjfK+w%_$Du<8d%U5Rnprgz3!?Tj{2Q{Osv)>gXH@p@!Pge<NpQp<-O)z|<1o(BK|BxO6<?&IFf#Fj|l05~{roC9~e-7{6@'
            'auF+WeEeSI%PiU!qidC@Ocqg^h&a8DmLgxvOod{x69?bSxzW686hzxyvgpVp6bA?MU0N39Ld|7aq#Jra-^jdB88-_$os+asi!{9f'
            'VB$upZ7!lB7wKIhHf3ByqivSnMxn}H@YhWms#q*kvRrRub`vGn;^UF%XJwMW=T5F}VIHwK-Ku1A^kR|}>nz<y%NOs`^cvPh-E)xx'
            'L|g#<#;L}tC=iwu>vAI#SbLhOPO(;-!D|7(1WaSIEyOZS@**pjxRRmx@AK1>QKs@OJcTAQ%K^ra0B%)gr!JN<jyrdm+`>~_Up4<+'
            'z$+0)w@R#4rbM2KMOv&yqOp=Qwndt#PPEy^YNL{ZHzR_XC>e@$CBkwWM@va7SY`4$f|b(T^ptqx5ZQ<x0j*OR0+KVaEf;Z=uNBNA'
            '*%fPO8whT+&7!2pUqH|N1x`E43v47NAC6$3c0s4x?RHjx6)~T$V3RX7p9`2`nibFt=1;TCJ01P(Z+V)S&qcIR+&7ek!f&Rh`97pJ'
            'zv8kx=d;P|<N5sO$??bOxfqN4j(~q%^$DgG5kBJXP;^6>cAReU!F(IbrCO&kARk|4>NZk$^GyA{jDU2zLxuqR0Xu}>MHjGHu2#`9'
            '!tO8&i4Rv2FiEeWVaU4<P-M6R5D%SBCsZr3D5E%>m$}O3OeFnW#cEloaK1@#HisgNmc<Zhr$FdPgNFjh2-eaC+JP~$(JE64NNFjz'
            'dUh@U3P6YLc&)PQC=oz78=&|bQSN~3q6}65^DSg{tzaD`^W;2N%R?ZT0y=(wuou3WU@VjCQeNw^{zsg^xX=@S4aDe=`1Fu5*zNua'
            'SDzD$s84cP2t<V8mK>nLtm!RCl%b$K2Odicjhj3mx<Fw3fN%x;ER&lTd0J*mHP2x+1unrnP)SH%VJD#Tivk+67#ajfln5*@H^5#y'
            'n04Q-cGaCE3|ZXQw&<dVk9(I7u+>icdb0qK+V9n;)f<Y|b~28x6L?lN91I>t&xWs8`4n{13}EMj!)HvNu?Ap!y}?5uW)^#gqP;_f'
            'U)_nv7|>7J(ZHev=uKVk3^%_yJ(*2UX7l68$-9q}chmEscsu=J^6_{!ufOULG@zRm3a@dqE6t@&p(w)kK^%y`T3kT02EdDb9WSZd'
            'S7HWKYX{0om{tVh7~~o#zaNgy&Sxa(fd`Sind7SVpu|+XIuud^1I$D*isEj%rha!>RA4g%Fioye3U`HEfocQ>nU$d0@-@kLt!>02'
            'PM0?@1r3n6E#A@Kpaw|~kN}77tR&YmUX5019H)2rP_P8!_XhkLG>irZ14ug&t@;CcmnjhLiN{CxdY>+O%;<B(4sef0eUls%Q7CUB'
            'k;^p*kn~rO>`ACVZ{D}arw3@3L^~BnH)0WG;+M3Xf#o31*FZ_Ro>3%r^4Gu{W|k&ExF{NdnBS)%d~pRWJUnL+3!*YZ#o~#}p|~!W'
            'aY-X0HpH^r#43+=u-gln$I>6{O>_fu2w^8-m^u;?xgjL<t2axS0X!ObP<rQ+x6?OM($FHL>ECyGeO+MSAP7+7;B1_)DhGv#q~{?G'
            '8vFso`X3)2{+j$X={Yk!uW)|&*W_NyL(96Xy0r18J^^*+eKZzS$l8kJJbti(9=Rb&lM(k3t4JBSuZ6wL!CG0RfYKz*HXxn=Sio#O'
            't6HHBR%l9G;R3IS6|UMBclBDJC9)INj@rs%O#ng3gs50TFz#>;8>C;H%PLNB24F@>!CD`>C1lX#@{XF&Hsc545(0N8VI^QKAV}iN'
            'Kui{-yg+c2iMEMvWeihC39u8AUynv;sS($q`9{E^<Q5ehsW}*l1SAfp5mx}V1A|oh&FDwN=7GLbw=rs2y(J|2m>>o?qi%N)z}sls'
            'XBix=sL!F@euj9#T`mC@ez-XN?%;rBTaoRY#s>M&4rob_d9jb;E(dzjO9iXx|D<-)EJK{nc3Y)C{*0KS&x7`H(3CJ!ubv|<RNww6'
            '4w@_4Hz|N=U%66?`)P8gl^Kr(?G7*o?#Rdi;|#?CTA#$m*2?NyM$mSU<%bUl40jn|kh^6406=0Y?t5BgYWTp$Ad1dtRykW#eHfCr'
            '1E>RC1SHaRS^^z{)eVq)Y=8C;G&ny47z&zZ&>loJ=o2#SfgW;bm4Gf_g)A4D@dczIy1TVv{veqHY7F=jSakzvhW?u<Cs1a}){64z'
            'gv=Pg39L6*fgta9@4JI)gOX6c%lPF3pzinLrELH^ts#uX=p_z|UH&lUr-z<RifdC274^F`Z_t3(Agar_ggugB<P6V53p2oN4amL0'
            '&AnjYFw*6~orw{*$i1k5b$6H)KBzeF!Zf_J9QV{r4HChz1AQjcO%4j}R_!D>cDXZSO`trE%TiPNg&H4(0z9h*B$uqe>R#OY)cSDA'
            '`zm)Ccxe#`U-9E)HaR~6%>n>`?tYXTu_WCU#W&)cg9C9RK?{Ph0ZkB3hw$w3ciBU8g2bww&R1fB-qI&xG4gI-z8@9o4bT-#YpL?r'
            'nuK#h!vz?c1ri+~2v#yZpB=rQ%%<l63K@O5R?8csKS;dya(09f9+Kl0?I@s9(8*bb(v}5q(D|xE#u>nk*((JH83DaHeg9!{c7&*S'
            '1M5FNeaDId9Q~w<GQcn(_oD^ID-=NTVhNx=wgKrY4pWG2w1vjNt(!70#L4vM=@}Sp)Rz?xsDLoVu6YW;R$wZNv@FnJH_WG19?OM1'
            '`UDG$Lc&(=4+Y#Uu2Y3tHA~CuHNd-K?^4SwyX|>@a3x-dtE^0#5g2$k1dflEYnVOYSX+vnuV_+i8{!V637C&Eyp_ok0}KKnl(<rq'
            'I^fO+ozoA~lgZKi{LR_XhuQr7>D%cs`rjA*?)A19y$Ze`At!=$@-jF$7>e)V=U^zRw|R<=`73()?E${M>`YE(Kc1a_IC|3tb{Weu'
            'RHL<wZpzVjqgNL6D+5}$047gVK@Gm81}|SRcv$4S)6;jy(?d1~(5o&YK<*iAiV_nY02(6GXZR=srU%*!#0R2cq9vRvki}90^#DL7'
            'VL-JtG61R|sId9?04O~MxfG%Y0K)a`!}}LA_<#1{t=<US$!m&0&1Dq)xl9Xrte|;zg^t1UCLfaE3BgzgdB7BYb*(m05{<qIUSTBZ'
            'ima5g>HE{4r*Cm}Fc3`y-u(@nHZb>daUlMVu!(D;g6yCOgI2(g0FmvWwFDl^%jHt3P%B-cZGH=Uh8_qAA04q<GM=D&zghup+Wc(#'
            '?&y4W_KS_%0V1v=q-B7+1rh@K%2OR104qDEuvLx%pbVn|jjBP=;r(io0|_D^CdR_IKu@b!f|E3){o$X3VZGI>&uKM^ftBk`fr#{6'
            'KAfHYeDrpDHaF75#-7rxN@N7qcN>eC=#q&&LbE+ouUj%+p;;R$DA2^tC@N{NL&21hU=FYiTa!{H+9agyYimHdB;BUK`UEE7a40=w'
            'v-r*A<n-j|&E(j^r+{Q1JCUTgjU*TpR*<GZEg6Vyz}Q!YSt{;$N0??NiE^e^O{hKm6j5}>NdBN>eDV6M#}+$?gtrP7i-2>aOJZDP'
            'WR`WOvmc>@0jae@76B;|K2{)a7m_?^SWjT~_=m~y@t-DdeyTS6^o$4Yn~{g^UJmW%8z)LHhP=&q4xHCFb0E`?49KF~f;&itAb2RU'
            '(7!INHE9e(L|~sF4Oy?wqr`jyHXPuJ02G>lg^O6NKndK3Gf=d^m3>gLatoSgw2HDE`V!ng{9rIa+sYAPBpb6UIz~Pk5i}MH($&)2'
            '40g8!ivm142_^?{bsjA!x&Z<aI3DoBUk%a;(vrf$)>;LF9dsb-(Pf$!*5(A$RigPj7JImmydg6i>i{8S-L*298rGR-yREtl49RTN'
            'P9;WH=khw1JFq`;?cviXU>J%1Lfyc`6}szrnt`9u-!s9%!=Uo^2d+1wCwKvF7}E|I_Gx^pH0=OdvYk&oL*5NH68l_&BFGf7SD)~B'
            '0K~I}t-%h^6v@!*#WNS8ve0e-%_h*HS!g^ar)8ti!s*rM&ZnK;|L@I&39=Ct<&I0(zRff>@^t{2D61OFLJt^ULThx5c)NWcL%Xur'
            's`EG|R^6Um=R?m$KoWvKc~8*c1^tf1F1-oFdv&`kG5C9-a1JzEXQr36n9)9i)FWF>sx8zz!~?JNZf+#QA-F|py!S|g|H)f0Nzq)G'
            'Tk%d@fad>&!wOA)xNJM9APGNxi^1s8+m9cPLF>(?Z>tah!6igAt#}>L`jF-k`Dh171v0*qJ97xayoRQa4yHHRE)ahKl|+DiY#^{%'
            'Rsxokv?|T6elNqBs4yNdCTP?}R{!tOZ#)AEKj)|rwC<*bgCVK?QI0Sp5{US8TuqxH#;pOgkpvf@kv8}n+glgM;LEDzyQo;BgCy~|'
            'X%$_UXe=&cJa;3NO_n08fd)%bXIF5b1&57fQK-!paf{$E;;lm%OMnKDA{qNJwhD(}!)<AAw}_<{%HklLJqfYF$8i;hbWR;=V#5fC'
            'W_KNFume^+{*^d4lww<>F^`KUyu2noDu_VzX&PyCXH_Pi#_;r+MA%|3^-1Z&3|*4jVlzN#I0#(1gj23@Ynx!YxU6DCEm%v`r=s6?'
            'aQ=kl1q1tA1CAf*Juj5h7^oqP+h@r9!VyS1mswj%2K<}sf;dgCIl?SO7_C-{;tyu5jxlGn>^pppAVQbsveKp$xM{WVFpSn>7Bp6D'
            'gh_2O0Skft-VXam9t=8$e)nxV2}KG*KDm+pP5?~q2Rg2~c^2w^E^!a3qoh>c3+g=Ab~As$mGLLWY12%nw*wDO3mU1_@W~Mj?sA0('
            'TxAjgNnsEl%p0KJi^>wH<QeH$OG9=ZH;Rs}?JH2qPI&s_r60+})&K~H^>Nj}6E+EC81|t{1%hX4R%?gx%vNrr^*;*rIlDmG3h1+x'
            'a>+~Eu7<`A4KG+yvrnA7iOOB_4xZj%pp29XC-sOW_&y`_Jz_;ECYozW6Hpw~M&dEVE%j-O@v!cDc9FfI3DypKsKMrC9>PcTNs{G`'
            '58r!M=MH;p+_HXvo%A6ow8VN37$c&X?h8zmF`+bZ#W=bmYk@UC<1pYw(RySNkt|<OJg1+c+f|^74bu`|&<g1&2~aHZGF!=|-YE*R'
            'U__`)__UbaHk+<gWmRca`$C`5v(<9`_9p@pMwvpL4|^#o9ts8mc%0z+$1&O(%Nt}$$GkVRJO<N~7c{!B$l#&n7@;_T#+cj_;E%o='
            'Z_c;yisu*LCI`+Y2TKemE8pm6zr>Yl8<m66tMc*1eK$)fsM19eL%&@g3&SIa=pHV8+Y{{yjHAM?jSqtV8d_qSGck?tyTxuxaPg>N'
            '&YRVJ7}lHw{HF=YU#}Pe3YzB@EoW0OFrXpDuq4QUKrb!O%W!0ZClEBm$JF6{*LH<<*w1*%j6{rUtDh&pZpa)ni@FYXKtuiQp^_&!'
            'M^<6mfWCApiIk%t?$gsiADYt<LJs8ka(-}dfd2>LYb!X!?@jZH6dvsYqNVff)!H-n`e6?d3(EuS0w4D+yy>`fjle)O>QE8%CL)DL'
            'F0p?7b=v{_;Nf|vLC@aIRTo7O@1Q;rYHs8~aPZ(tcpLHh7YKP@jM$NXBS~UJf{`FCLPLJiL^XbOaPWjL>PVl@mo>kwY~V3aHegc0'
            'f2cq(Twe)<AWF97b2#7W0@U<WpZ_q{_d>@1*SKGw^!yW8e?ND{bRag?aU+>TD=>ZZS3Ms(iSqMjO}u4))~v{5rMB}`GPcoE&%V@0'
            'Av(<CWV??>VsiYK$uH;TJXR-aBxw=>mh|~ymEx4?^Av4G8JjQQczU2AtGVXeeh>tME5bJ)wa5fbtW2{gALPm9GA+@NzeY<0#NP|_'
            '$I{hmC}@L0$jLm9R+RC9Dd}wPVNhknJVU^4WsGz|gp`DoX;(zQf>a<NP7*pW-c_8!oaW|8DX^!0R~-K#fEsBg2DFi(;!G$1pxvIW'
            'gHfw?xtFwvMl2_Nrm%xMq;Lu_TTm_MTN>F{4SE>&3?cNcptU|p(I>p%#?O8{I@f*z20(_!VCM8M&ddkgMSut7`P>VBvE84GJFu76'
            'm}jZOPdO(tNMmiUjAf7Et!0+xxjA<BgHfC_Ym#IwemgD7kYBcFLQV;SHr1glCIfhk>*fSF2AXBap;nP&m%+yk+*X`r&iwMRNmz7-'
            '05Nh=O%qe8Efbp$6(r5<{A<VuMkd~2q7Gw<OR{?AHwN$HmlY(--R8qk=yHX*H;6d^C4tiMqWO$V&;bS5{dt$-@LkNlY^q|@lzFRL'
            '#g(X*;1epYF)S|WJ7^w&zWT1-kb@G~vYzLsHPH~0&jx;9qf4v79?_o8K>u2O+0=`^vN1SxpI#1q?P-o)o3M;3HBk6F_IC*v8q7UH'
            ')(NnEol4gP{HQuM__4kAAgX$A)Gv>xcb4dD&I$NBmD32)6!kNv9OHp%^sBS-R_rlA_YLJQIP3ZXbnG_)xL0T`freHWUk@Ks;7^&<'
            'o=V5e6If0qY@P+8RfO>A5&)$`l_zJ>5HzM}=DVS9MLY`>)lE=65o+#8S!mxhjga@G=SDV<W1msNX-z+TJU+G=P0O{660I0NCUJD5'
            '!18DsabRln>0~BK5)<hjQbf{I*#yR_?ZJ@4N_R58;aBx^99&5~t)}f6rP}Bh8#(%J^zF;VDAK|rFVgLn5>^Z6bc1q~@;ngdq?f_4'
            '0tGuF<5H3lo>k|tXj93hN&e-`HmarI1VeQjr6t=JI?D|jaCYrysOEp3&LuVLP`v&0k1NsErxlDA<=T8SY|*lvM@fC$`n2lwSt`0J'
            '=6Y0!Jzt%-s=LbA(g1Y4_mcc2s{rd3fTIeV%F0oVAf1Q*CQYdE^t8^audSy2Jaids@gT$BQsaHT*4wvaqtPCtn|%c@ldOnVAX7%U'
            'T!AD8=R%R~Z@daE4f-ILumr%wi7}H?yYC<as*1#>3Bi-GhmW;Y*5A6So!@O^iK4u<pLyr@GE`;Ry4JJ#-tBD<?eyFFPS;<%eT~?`'
            'D%tvc{dcXsH2Li5XNb$`Nin1OMir7OdHs&Hmu|nRx`e#%`9gtyRJHzl8a)PwRM8V)xQ&N#mS1fg<&TDpH^&;l`ViHNnuVv6c+8$@'
            'DIhh)_w{#<D%OsVhr$T8p>Q~{QKhP@o}U#=<sNe`*fQnPx~3GzPHDT$3#WQ`t{Y+M{1f3N_ut=33FZjQ8>{^1FXykc(drjIgVrdr'
            '?(-n((~K<;B=vVHhIF#Q^Wb;_b8gIlmO||3fvfCT%Ix_92vk^RtD&CPr@csvt=})0!gI^gJ?ZVFd)3ICD$WULhV|;$D-cJv7!rxz'
            '1j{*X9OSu8RH!m;UYLS)%97DH34kV&nifW6gld8?30j$2Vj529Uf$prY^~&tvgNlrvgn@czL}hyoX&JLuhR4~?wJYnFlreUPK66b'
            '+W;n7RvjCExRtSx*>y=p-r@%k8f$20B9x&)xs%)0tKFf{8){F16Tg#mk%l`;Tf+Mha0)wBU{yuBNonFmDx6uaDd8YzOC1wFbn*mv'
            '-4t+O^B!=trM&ZcOwueue^8a^vb8T7-!xl&$o=DF^v$d9xlmf!HBwC|UHSq2OFHP*m7`&Goq#KgBFZpZMU>rC8Tk!)rS-AA)6Y;y'
            'u?@7RsZ3<N1G9JzzQNXM-Yt`xB)v=Ay8D{6omV!pYhlO)Yl!QY9q{u@SJl+6G-!6x+Na*G{c?hC<LZO%V9*>+UtRFTV5Zd*Lz!m&'
            'Kugx&?eLd%y=@JR*7m-k!KN{ougb@{W4l|;*=gA9tV-R9TH%AkJNnm2>pL^j1JuF-1jeSS>tshbFSpwmD;_|0z1Bf#7(W7*vmtc{'
            '9uLD&{Y0Rm*m3dk8v5rL{pM`%ZZ!g!|0M9P@vs0|b*qFyQ?ZWmOoHN9Ds%ZUKJ+F3{>t|D7p(|^70hPmLqf$ZW=wE-At^nK(c{qy'
            'ixiYjD95j{N=4s1!ocpZq{80cKw%H93x=+^j*=BBWy*~S+#nryr1L@k=&Z4>!YEYgKWa0znfjXaIB$4#UC)Ff0&~7MS+6e1aE$gh'
            'd-~QppcEUrw9sm!S@LLo>qRe(E&_)FdYOBH0+54!b2U&<7Rx~p<teA;^xggq6Ayqio_RaOM(x~CosMSXGbCDc-}i-Noo`;#2Yt83'
            'lx{Joa+a$rt!ynOW@qVSF~M>Mb+0iOoP)Qos>EFi{E><_)pSnG3zebhM+qiVBe8OgRF!Y2f!*Q3BMS<yl`;w)1%k|CRC1IAZ|Vhg'
            'hA_&#ix!U&(_E43`u2zn5OSgO9=H?f+PTr-)WZ><_D-flPD>v)U(FXgcw3Vq_9c-^?IlJy^GEs^Ud-1L3<R&?gU#r!WO5Zq;uTea'
            'rsh>stxb*3dP&zjv|0R!Hb2)*s|W!UQh;e2y%t9&=aUoo`DS`NeLp=v5`UWhbbR#7=@|xk@s5gJ7SY8fGK@B%bgwpYmfvDT3A};_'
            'CKAO_)~J3_uqpo@1b`i`aiFc=@U^*B#F*LY)0V1vxQgWyFawxWFM!^F^PMPqw<TH6Qncg8(fs^JZY&48m_n}BLTp#gVX)x|54K_9'
            '%+XML`oKYF<Lbb&uClTDTi#<G2Xmiwp5b~ud1cx2?<RX}xW}(8Gdj(%tv%eS35xvz21kN8KCN|$<|HSc>Fw&>ZYz5--mPiVD1v&9'
            '`^y>lR>C$;w4PqH-fQnzPCtA8>D>Oz8$PCcM&Cl_H~K)~GtTt%xrLeV7ar^3F@y)Ad9wG+?)y*k9w(D$9~<fgQ{KowwC_0ShGsUi'
            '{q}Kw_Xh|5gT;bG10dNmq=ny`FoyE@`Eus7LOk^|)0V?hlf`p-GUbvqyr_pVuJlz77>D4ZN)*64;_?R5ENp<NPp&}3F@(EyigkD?'
            'B2&i^OlZd53^Nis_$2Ekg%rxzf1^hr{u1d6t{#y9V^gk&?uxG0^A_q9?&4&)exe#koX^gV-pr1F5$DrCf1IAojwZ+O1+CDt>Fnd#'
            '$+^Aff{tM+cg)SL#rU$AXn)EnXK_wpIj&>9Q}+PYLaHCciasVPD!xIAY9DMJ2d(vvZk0jjlTtGV)e8B-Qhh5+?$@e2HR`&cm5k|@'
            '>sgkv@&;`_7v*xbUfgoB=SCUtKACE<&b2Ot5<a5;+Bu!K_rsbZ4C8fbs+^d>ouSM_UA%_22wvo2t+(9dEjEI7TvXDjBGz7%au9fF'
            'jEzX)YkS#Q6)CjWo^f@ijf8RyB3_t?wJJ^}=+)PAQwxLTo<JHSTcrZs&for|PsnqctW*Dz5B#*fTCEFu_PF^hw2*=pmJPY)h3{na'
            '5B!Z8w%C!eh0e&v1jxXPCPQ+Ai>cdk7p{-ny?kg?emW*xGx@sK3WnW8O?$C6#t^H$#<AQiLMb*dJ2TNX%Y5HdbcJ1Q-A~+V)sWq*'
            'ZmX$!-}dyu#ht*roxr@O((`T$TrNi!*fEw_Ri9s*u(^e_X~?&r*6(`JHp$CQr*RVHd!|m~M`Eg;2U{~c_xiF;^VG)4DmQIzyjcH1'
            'OGM3tuRlL@^3bWJs{jg~yx-Wr*_d5{)+>m=?6_xKe=z8@hTG|yL!W=#I5LOQ^%aZ0Pv`Hmvr@l&qblp5g*q`jUUwH44o?ezH(C&Z'
            'jrO7EfeDea7W6bxM)%o29@-;4&Sr1qv^;UL>+Jv&nER8GROm-zkNtq+W5GW@Hr(Uk`j{X4Uws(ufq`)Y5jIQ7@#>u`e6<*0;m=oH'
            'E%b@c$Z9J-#21s3Un(Wf_aufYaIYn)`=E9FdwbSxIU*GO8VI)M2B=(XHY#5Dg%jxN^%%Vf<XYW8<vIUD|Jtc~5Mb!sHRVMD9<&PQ'
            '3;aBGumd#<e_eUzjM`d1)<{e>4ZZ9+R|>gTHe6szIVUSp?Tk_Arkpho!1~*O4UOAmURKFum+lT$aM9$&Cr@}@k8d&T8f_R4THR^!'
            '#j&YD{dxf_gPT4>EePR99ESTmh@o@0f`8?X9rCw8O{Z=!Eb->2>QWlfzpu^zk&88cmpu%~b`W^M+-6Ca*I<OH7Syz$Y?%B;Zoz|j'
            'E!NQzgP}`#o#8do+bE_xMC(Gs2Cm0$)xKT2t{8mc3Wq(V{aB6X0;0@B!w%j$><w+&l5hAA^)e)L<70Z4*s7Bn#J-ob>t*Xcm8MI1'
            'z^Vq1E0lEVCK3$Ye}neo3yXZN?E!bOx`nC+<;o@5*4iEN{%jpi;9vY;0!px57#!$75_4kxKWJ2tR|&mUl&gMsMYZ8vK;;JZ8di-R'
            '@uH<lhAske>SQ`?x$s_PTdiVK=OFJ-(r;dw!7}o~klN+YH*0i+(2Ex~Z8aM+ti=$YOkwUkD^-8|SirAsHm!Nft+BpG)0dx4W8N14'
            '9PV*Id_SzQX`o>*JSlLwf5-fGPs$yCU-wMUz5W<)LFFXaPFvcTxfbNUEvrFKqn3uuCO1EG=R0`H_@@kQefgEmC4NqlSUvUEorm{o'
            'P~-hCu$ZgObSt3D6N@D0nN7+z4{S<!OTPG?eDHtJ7Jt+pf5eJ*Qvk;<Rw>bg>eE_yyESQJ+@AUB`9v+io^%EG(Kx72|1x7#X?&WF'
            'scH81<JphX6S^dYOa@dtcw1GayDMeWV2RGY%I|&r;qopMdrK(4HH)5~K`-Zu!*wlp`SUoT>i1xMrE|Gb1?Il3${Z%f_oveRVHg7B'
            'a#nju8U}!^sg>VF1(+Uui*Q<Kcnf9!Ov7Tgm3eOdFi8)Cf;3jB@M=Q5jg|a8dN`*EP_d;<93!%CQVQDEqg8t|?U^hr7qE<-E6&lf'
            '(@-f1QSY%|9s0NlY%OJf_Rfu2UPV!LE)&WvU^nkn4GqL`vlZ1D)sI(zQtdQ+G3D1Af&GhwQ|!jwb#%R+n=}|oHmkYZ-ht0!r$Mz_'
            'bL>&izB3@ded`g`xb>;ukvAh8HxK@FiF=xv04e=#ht@y&fPoyGru_qLE<+5OERW9Gqb!fkr(Z+S9XOm{z2+x%wk2UuH6r24K><e<'
            'R`2;q9Y}r?fn*B3mgboQ&fg!SaVv44uMSrf^ETAAuRa7XHXC@Bh2O&zD>N&xyrOE;simyi`}-)Z1-ie5N^cf5S5tD8diB(*bsI5j'
            '0R2zU+Kj3%*4l3Of1=jzJQ+b)b&GN%KpO6<@IfrIs(MVMaafA}Jr>j(oD3d7!6a&fu{9fuxl!~7oN_7TO#yx5(9KENgIB!!_J08m'
            '5~w}'
        ),
    },
    'tts': {
        'sha256': '7df5642ff7d3b8594b5b8eaf3a9970a364b6bd4206eb43cacd645254fe56d94b',
        'encoding': 'zlib+b85',
        'data': (
            'c-qxlTT|mmvVQlks8K&)ov;Nxy|XbvhtCP7+p|k711!+JXM!MvY?W;-vgFYP+QZ?$pD(kjq|(I=eR(*C2e758tjw&;FE3R%j&mW)'
            'Jeb8&6!KFsF4A$nk#fEgak`A=B3}iW48?3K4&L}hk^A>)G?!kq*(Ng)B;nxTz~822QO=|vltsD@iYQI|b&wY_Q^UN$;504dEKTn~'
            'MBGWa$wgG;B7I22x{Qlxyvfr0D3qC)$z;A-2iaYeEX9Z8x1v}{Vcw4CK^zYrvS723Sx&3->s-ciUdT|fmfmCCf9Gi;=4n!7!MqT&'
            'GKz_K9NkN?k{PC?VwM)GL8AG}m7F3?q=?p=SgvJKuq4F11W8eD;xq`sbS5_CEROP(Bo?;C3aolU4ccT;QsnzkUA|8|MtMPl#L@XN'
            '<lcG%$8iRWEL{uVUqES@^nF?|%?b<yH@s#(80crKAYaAN%zP!W%;$Vn7ExS%D`(KsT;{oXQbcR1mI{MH(l@h^`99(?e?n$8ZW9!='
            'm>G7CuY<wm)zQ_5OaJ51$%pBsn25)Lz>gz8LA;2y6gwl~gb+VYH}t^Y#KByyQc^j+%H(|{AN)-IQARK&XQU`#6S1xIT{uyamy1O-'
            'k7zpB3+Uk@h+_2`1H*F8c9qE>+=9f@U~qMH>Hl{2etP0xO#gO#d3Eu-#a;}4k~o;<&Y^g8)@dkXKML_hzw+7cVr*R*#Qyz{v<#b%'
            '9`)nv^})f_eEpZJ`2PJLKSy`}I{M9d(yXT&nFLXXGQ4B_db|oNTJ#VWn{Jg5L>*f5U*Gw2SX=IwIn6up4&EFX(whh)I+Xe8I8I^z'
            '!~}#NgmNK*GK^9`EHmYExttFd3n&Y&IE#`IZ>THA|0xzQ-9r42IK^Hc@<P!<z}kd0Ts^0sOu{e|9?Z|ku%O!7#JsFbhSl)v#+uIf'
            'o}`somh$e8^HuubV_&mK=De+G28@^(u6azlpg&~Y_?s8mkU75@Rcp+H4Lb{fW>XfEtE?pJ1zxF-JkB+O5tp74Vh$yWA12~pw{%6e'
            '?Yzv=G6{#QwIOlkvFi_+a9)Jq!oAjlI5*c$CCiOF5^oeA^3z;y3NhtBc-PN>+DC#lzcyKd+2K4UXMi^5S+ps9lJ$^OGZI+cC;g0s'
            'F%kX3*e`$OeGj8~p%w=2+$xf{B4S9Y*Yng1Z^}<j@MJts*H|K=sF2S74RjSl;9eo~f~i3Rc;$+Rs91?xkC^j|)K`JS1MUI~i=p9X'
            'Qgb@dyXM{wYE=-_gP{lkgX2)TK)5NSn6Jv@F82sBX|RzSrby5<G~eCsHWmjMMBwJpuc2GLV2{B&E%9@Vn-7)%=7Sc*+Maq9Rttf|'
            'u-+@@Vdl6Z$i<>&lYU@DN>C9cioNE09^*Y^(9V#@)yu@d@npVjiYmsCzV@3mFYtaCgBP1su`ga992~gfKL9^(4%)W)=$MdOBPN3>'
            '<X{Ehajgg&I4g~Hgm4sEkv1H*(0T(wf*2({^kHHV+={T1p5c`ili&>x^|z;9CnNph*<n{6g-3^C;nY?RzguQe9*J3y2lU}j;4Y>='
            'IZw7@DzyJ5r_`UjA>dF!^Rv#Xc<|2f=jgb{JIUgB>aE9W_KZa%Uczvgr-^0;f1`v@)V;MG5TEl6at~<}1KToKlZz}6j1?^3epipA'
            'RcnXb&3tWcN(laKB<b}+qTCjsVRql!le*jcdB4(W#LIl@|E@zRqZCNTUg%Jq&9FbIPsker*b0fDa}*QHls=SS?e@8@_xzek&{&jl'
            'ycL`^F?aV-!1cm8tt#G~93yK_O=><uJ|q%ZJfztjvx&r?eZb~Xxg+$rUQ_Xr1W2lfu(LEqBFps8kFNgiQjHLB_9cpxmQ-mJFDpOE'
            'd09}Ldn<0+Qfm5&WiL&VPf8KWFa(*f$Fz>bI?8kMTg4cysK%%bVlHDD*gNHU-V^)<by*N1#;O1uL_}po1rZ5NWmR5g3ncJjualMU'
            'VLYTkBCrDGd4}SISQw9)R!-WeYOWyfOKbL9h>C(sP@a$6?uDqL6W502?6yuv?KM_KB^G5-9IS&dI&4fA+hBtiNl5S40Tk;z>Fh*p'
            'ERCc5mkTcMVHM3+4a=<<YfQjSk|}eQ%5pTOc=8j~hC|n8zOkG={BsL@);D*EuLYl#ahdQVF?_Vwdvdi3w6@!JO6Eq-S+4fWe%W5i'
            'T8^rkX+=%;+N)G7zAVf<Q^Rl_V{4bHlCgqiRgrY0<PoYK;n<?bRn~gbMC?wW8oj8fsSBIW6gOr&%-TI0@XRK(V4FbAtT1_OE&1dE'
            ')lDe`QrCw+9UR;|>Fs!7IPDFHs`$0y;ST!(gsqz)l7q%~h>)nozkh(-q2%7!xTLW7x%3i1wJd|BN-3dXryz~dG#{v4q}g^t%IL<#'
            '%cJ+xcT=&WQ3DVcOA3-4fEI|K#3;NF2nMt-#EKer0TcUMnl6!gKFFDnNw`U)L?xgUL91wrfD%;kR71S#y+yg!<c7O;`-mlBERpjh'
            '+(KI*(8Qj&wM7~#hauUKv_e@y8H1f7RUw*&dc^(hTdq+l^Z*|dPK_Z=a3_VPL;*Q~;e^z>!)77QY9<G&&w#-HJ=0rwtSv|vw7I^c'
            '?i&p=tvB8YSXN`Icm)Oj5y04ge*H!aVX<>+L6#ep<09S%dCoaWy*$yJh>j|oI7Dil7`t+bWl{sBC_{UuD`br7z<y+}<K@Bqw<OrT'
            'h(L?kfWQyN+5nvYM;O36YT;|J+|Olp4|9NX1%QPV3XkF_b8T371-JZeKcIGQ+U%-jPP|k9fs$g?u;;9UoZcI}!uWrd2+zfKyofSX'
            'KU(dF!YZNryQ!YgMS*FFlGDGULiX(Z>iF#R=mgm@d~g9fn+5YbPv_(n`0R5EJR&$515U;aNYx<$0TRezG<GeHInf_4fFVm{;dQeZ'
            'wx)oM&l)($(wx0kLwb*oF~A)+q{-|@qD<llph3IlRk*0|X%oOf(}g*hFl(XSLr%tvX<?ycjD_LE+wl^ALK}Nq=t_8XI{i4k5Hl%E'
            'XB<R^xisGaiaf$UIzPWS`*`%r$<(J0$M2^Xm#+7nZDjz`??1ew&GM<yUadqmaH}I!*Rd*qyMy=BUynYVTs3A=tg_|}FzBQI>(R-{'
            'FGuhG;a|SHI6lXO>3?6*3My#*Qlh9>B@$^4l5TMqt#g?`%QJaLCp2NSL^zGeno$77<u=Vcag+q&4r;%HHIERVklo*5N(e_UlQI#G'
            'K7Gic{4k(xxC@vFf?+~#7?w*66zMX<T1@g>-cd4&;R(W8l2KeEf1L++|0Ui@m`p^^4e1<ow#4%i8w>*CUMkj!MgV{R6RU-CiCV%L'
            'xH|ar5fz`W&8ZThBO2~Ev%xn@V>(fI@Qv#rnNL%AN^p-DOUCP)SvltblwMzB`c0JxoMm`P@ivs>u-wGaoQ<>I@Y>OoQ4>T^B|~mN'
            'P6Mpe#qqx=>NokB{2EY{C{}QosunGf(gQ%wPNx-Zogq;58B1MmF=_S%?zWg|v^K*9N9W2NsYaMd<zNCPMP60;9}(1vh2@7_jHL-a'
            'Jgi_ENI@WdRNy--7`h8Rqa^-Y>68wZHmrvYw`se&=G2_t?eZHHI?Pce)pth8pxWmzrv9p?P=^1R1+4IcqF#!RL0rly1)ye`vv4TR'
            'Bx|{a29T5??(xp%!7>iEP)Cj`?9r_HL-r(w>~D!o@SMzPxQhiBMVVE#sEmqDXd<p}Is)06thwh29v$%};n0k&O6->Lw`L)op;^{e'
            '?SwJ2U-cf7ZETE|nVuFwhjS%KhX<7v@yqMO*EbD+?w(Bhh#nDTPs7U34>?b-+`F6n>nC{;6lJcSb<&(2KV^=^5q2tWeMQWmhIMo|'
            '>Cc~b{QRLZ1CUEm!M^r?Qq%j(la~?=3KVvm0y2`BX^ntLe?jvLsW^Yx4FJ|Kr`bgDAxgOC4U5oOzupRJuppB`dS<{&U47{3WQqOu'
            '>44a;N(e6FTn&;}1ci$-QF+3<qtnx~E1fxjbKca+o{9N;8o~<HEIAcaCip8r%9bT%4+wL1?qQ&0FQ?=wUs=gc5I+Rlo}7ot4U=@1'
            'hFiL2z#5k%E(#>9zm<ji7WlkWpwC*V$noU89hD@fQw-uNU$PP!LUe|3<e}?SrD&>zv*cf~>*orlsik_$xu9QVrd2wm=hQCc-YQe$'
            '7M;+tDhs*%`_cHvpZ;nRH+2q3Z5#f&D%ao_Bw*E871C_mu^>;&43&oVMY@*(!_?O^ynGda_|Pug3DzCQ^(6O9hE7>5#y>l*>#gJ`'
            'BrXVaLl=1QR|Shm42+J<9fsy5i9^A0byPp{XIoSBSPz4uQwceA$tCX@OWr}7r?_v-dm0$Pt$o8ZG)XX3*bq?GI7TlXpL&{p4fl7H'
            '#ZxWUbL*pTDBby>n*oO;i9Tj9OU~0W4#~u{KZU4^bz@w2BJz=~T(v<XZqTD}Fn}Y}T@$y>fK`fW<(WcMe{Gkat#Y#qsZQ69s4VR~'
            '9JNPQYHNIN5#I&(n6~Rxi-LWbq!wy-nOwb5&Qnim*3UG!xlkv7*i%&u;ksLWY)xJQsVJk<_HHgTc}e;(G<TZ3vY5MGl&9*R#?ald'
            'x4tm~M}?hXWfU-y>znp8h6>+ckfVb4ur@Jkym6wO!vP&~8oTwDvKZFZ<%-Ef_d1N*=}EOSsr{yiy6dV{Yp8Y`n!zAmm(^A6`Bp=8'
            'UHS6uw$^Pc3V@pwiSkx0LHSM}U+sHdJNX&5JVNHkF2g8WOO=ntVf0oUpI#oF;`h7h$@I7B<+1o>`j3<2-_I_nCe9<MJ%PtYDDzBF'
            'mL|bgI8EBYHd7^H5Ju}T7>S+1Lr^S$GPnoR$znTDR>N4^5j&{W_->7}z@;*b>-wp6eo&!~HBLRU2U%FuGPLTAWt>f*vItwxZR#(@'
            'tz9#o!>@0{t~PExcQbB>ytaON573)>?G70c()Chl;$uCaiIc4b?3W$VeF)ll)>!E?u==f#`W0~cQYba&BUzd7rGYBhH>FIYe4_3y'
            'u%xH7v2MKi>bgl!$)nSG`@Pt8OF>sAW<dL#s)u^#m}W#BQopayV~98AtID%db+ZSA89GN)!^mOuf_=lf1=Vld0v?-0n^LVqAwPj#'
            '<iy;&F-;gcw&%PVTeVde9fnm0mjaEdR!|3oDVH$?Woiks|7Aco5LDB1R)(;0QzpHmR*xyzOnezrsn9NSP=;j4+p`D>-R0HA@w=;&'
            '-^JzhpC6{DSI0*u_=2*b(%WrsYYnABb$>ig(~Y>8UVXSYy)=@kj(1+p`K-7irrX*P7ZsU=A4nZT`St`VoCS5!tp+)hkRj4gS<Jmk'
            'EUR<Y*-qU;Yjw@qS}<KMbvY?10`iG^0ks`f4OAb^?JL`Us~Tm91EvAuKz3B5d@sH_Hn;CdaD$}R)OkZRW1&~yPO0c93j6AcN<`;#'
            '<#I)hmK-zMw}M<=Nnb<B>2#&uO_ivd#2po^N?_Dw6nCUczXkM~gF0&2o?9&T;T-gm@|BiKt!Be&RhoulrDbdhj7+r<vHZ4odmg|y'
            'u&in?rCD2uqto}hT$OJCc;dLI&m9^D7T6aX%+i&$sw+YoS8JcL$3EX7c&p@A19c)>4c=Azd*Vu;@!GwMx^JOa>-Xs}jOY<p;F=5d'
            'uM^a?o4ywY?xRJcm8!$Q^`6Qo_HO7(MBPu|dtEhffw6TEO5KSN!ywW1w-Uc95yiTM>k)9FxYUgxwkIaVIfL755a_^T9~R@QYrpf>'
            'clbTrz*~|c*B3tcu7;ngE!XL0NzT*ActDnzX$L>$Da~gpb+B3yj$5@)yM(PwzI`Ul8A3B(0~@=JZASVk!sqLgR`Fo<OD3HiZOCwA'
            'oh2*Qx-(4Dr1Kobk=tIalB8d6Do^eCzZFvoAPuofBvsiBX>+>D)kxM{cVe}526o$1b;+x{UY&KUZZ|_XO?Yhnt4k-v8Cn0W(#UP<'
            '+dB3s$2DJRr0zL=80m$$A+DD6BJfHOO|2pQQU!R*R&E+Mm0lJ*npYtzX3Pmdb*9+p^n=yjux@8lqh0aUIxN$BYA)heLs<>x7@*0a'
            'x`Sm6n=}tLyVv*UizWE&{=3q(J?*A5O?O=<z5gE4+fvti)TBFVI<|qSeVb`-y?L{vO%LX;>;FXpHGWW4lsDa3ZsQ43$v%Y{qx=G8'
            '=_$`C%DS%qc_kToI~BChL&lFJ^PW+{-p$k3>8tv-<6psEpvzo-iN=)cJg2Za@#+=GcH>>8&Tbpy@!msl?>r|Q%*3>n^%<OEvn#N{'
            'O9K&&D?<%avl_OFSWn+%_m7j_VWNuC)m0vXxhJ<X`;{&rM6{-h34GdoNU6u7%uIgS&UsV?f-!u}q-U47=k)wZVy`Gl-qyx6doMi!'
            't6wd4DcxwJm{2>Z(*?6R6Z7G@egCT6gw3mPU3r@O+>WZ<G^Z$LJB?YW?>Bm9n~v7u@Cy$Ktc9C14HMlRbgYB<py12trgO%9PCVh8'
            '>aBjKI^Q2Qwt4c{`1(Wk<h~W<9Q5NzT}o9K;OU-qlbYSi)`9YsI7>J^4**+~GH?+@8Fo(vq$?>?l~yO`-NwYA(I9EUQ~O42177&%'
            '=WZ}={PY-WqaV|_>p0jAOBRoZFSq39uQI+6ZN45BEXKbb7rrAVxP$Ma!NHgMzNHTv00v*D<<9MmFH-AwP^UZiU#3|4ss'
        ),
    },
    'render_utils': {
        'sha256': 'b35f4b1004b927d6b69d715ed563a41f6c14381fd82ed2fdd02e10630b3e04c9',
        'encoding': 'zlib+b85',
        'data': (
            'c-rl~+jiSXmLU4BuZU8fX^@#9P05#Z(Z#H#wB<@qIV~?snVxE;h5|{DL<t1gxJafoTx<SdUgmS=Yvu>eFPwde2t)(`O1@OrsqV_|'
            '%A|mZjkxUCvG03-fB#)r=D{Qm#mkqgb+`~&n9Rbgw~p3f93`PxhVeSga*;2CES!nSMhu>exzSiP>PPEMG7+7dWi(yxMF~F3MUaT_'
            '-^%DZh{L4t#q%@{CcOwcuCaGgh-tdsh;%NBWt59JT}0C^bef6lFpK7!ahy(rxWC@)F<cR?)@fGc+<cYJN*MGk%ED<8Z^S&!;6*5?'
            'lbnReSLszKiZCy_{k^?6DfCPhFrzFK<vLD-SuUX4x}3yOzN8Uvs7YVEDnyz9*et|G40ImF)Mh4bvIv0gnU72QU?AX049l07>vbH#'
            'dZlHtE(@^^ie-Oqe}8{(o~0`>9?#37%);?l&I-ULX%Q4rn&f+X^4T)TmvJ;vfBrR36ZJa_)vtV67E!GKDktkKorZa?o)k-*15Ap)'
            'JPV4jh*qKcuKL>-`0tNt5^_5nFnn7zdkcT=?VX+-o&9n;{`KhfFUO~1ByRTv{Ief^01ScNNfE^RzSy6I*I}Hl@xgc<2h(tw#;`j0'
            'Dhscp@CNV)5ChO@-)9KH8b*DMe~bMn$;<gXnnu_ih8N(&c>t?Suc6_roZ>=_abg<)v4xTS^5fa7v)9Mt_dmTleS7?F{PW3+<JUAs'
            'uZWV3=L;`i!QSAHRXB^v75<8YY!UXZANF7~CaDL2o*%tEdv)?=eD>4v&&T7p?@nI6dTq?rn}zeBj0<llAJ9V>6Jo&_5I2W`!OH``'
            'dOJ^HB1y0c;koxB{A=)QDNchV_uMvl^by;9e`vf2V5-;Qc$&s(HV*O}Vf@Qa1Ni49{9}Tc1bIWdmoMFRtKh@<Ce3Ep<dO4y3Wpq@'
            'KO9&uvPG1Pui^ROq4@%EJj|x>S$Oj@i1W~R7l%aw6e3>-Q@}lFAUkgPKi&e?W1P;%fS!PJ`CaRI*tVWO2iY`?<1!bg&s!IC@ONL('
            '!>6CRpl4aRdp-Lm$RY!Tnr6|O4hF2>YK=JB(JiReiy&O3i8zDr2iYb}eDQ7>gvllg%xAwwd6;c-^U=?nJjkK|PK*ukr?kvNv*CN7'
            'wPBX#zIc%Z3m|3%?5;2V3}gPy%f)M;p765<=T(NoB2<9y2k{0zh9e2#fNlT*bnC9W#fT?0TsVE+dlxQB=-e-gc^xg5!Qta4*kkZ;'
            '@aR!6dGh%1@abgm`0=CZ?}NwFgYe12+5GVLvq$0Mr@{9JPal6D9v&P%9UKI~{QD<QXY<*?;K^h*AJjmXS&VR(MX}C@4<2N}O@9#;'
            '%W?wilue-p5Vig^T|HQ&=^_pvV5j`SDu|K?={$xXat8VI!98XZN7FC?BsCY|<mKyr@u6sH_7w-$@RbQ%fk2~yD#rmGF}y<XdvJI{'
            'fTN&EAieudQ9)I6_yp+o7$xEPM*KNk!*$E-$p^Tkrx9`@t2H1vT(A$aG>#|1^h%_3mipqTEYcNxb1MFHawamk(2<Eq(i;g67B45('
            'W0eA50T*x)6>-?Z%Zd0ZlR^ne*+ejb>#pBO3deB3KZfyV-!RX)dkZam^7QXd3%~!`wD6#RxQj0Eiu8U(5(YbR2Thninmv6ud-!N}'
            'Fn==r{?Wm~?;nPbg6aIx1c<`?_m3YQgpY$K-%p+d0nv@wU=Vyin|yy5JPHT%hmXECO<;6*z?1kdkAJ*bEQ|m0_{HCs7RW(;HW{!c'
            ')w(mEcK92|v?ccNPe;JaoxB9D?%m0o<5S7q{rUJW?@!*nK<=(n9YvN%m@9#cYDrfdJp*xrE;W2ys5=k;xRJM>{H7P-b;dC8^{l!I'
            '=})?a@mD5qUxE?AnrHJ6S=7Dm-rgQ?n!x7gX^bZ{;ENr`unIfKJ>wT={P$4gMFw;50-#g?igxcqF~)!CcV1WYegJW4S`3-xM#+zW'
            'gU~Vy;nS-qiE<=!@?)A-kvyPDC4CabCWRy@)F4DXD)=0H)Ey;u;SqNG@H$#|y8Sr43A0X@x<_*Z!tc~<h8h{H_aG~YC=t!f`q1Jn'
            'vH;i_<SEMVILp$^W{L2h`Tma)NUW<6c2iGYuI#P;=I^rYGqK;$c@<tyQNp9u0@)8b<sdSJvrhByx?3AcSN|n<d*-9q33m}Jr2af$'
            'h}r^PPv(h2;JT~%CTRvT4iNg;SR#Sk3lk-1BRqTY@gKP3rc2tr1X|h4)VT^bNY+WrR=;5b?l@PlXGY`t=nQTS|9OM#-(G!P0E`@*'
            'v$gg0s>v0;n0s;z!&?buTk(_j`n3U7LLY2GlJfG8CD1X!^yCQuF)}f)ZRZRC$*MNgY{j268p$y@ZAvBf=EMT<55LAGz$-lHE%<k?'
            'pkK)4-%4U5??Adi#`7|cJ4j}UZU7GPD$Sw=j2_lR>L&W!N21XU+#tX=g$eaU0yhD=1H9HUKtf(EG@{XTSteJA7ZDJjP8_TzvtTG&'
            '_K|z)JbC>1;ghZ}CV;N?Ch;)(<r>u&9fH&~w?MvMh9CG>M-f!GMo{fy(bF+rA07S)YXck2>j#uH0CXA8qO49le-5sYa%vh0AVhE-'
            'L^1Jvq#24ghBgaknTvIlByhq`e>&>HqV@L(*pGb43BJo<4T0x;m53Zz8(<>|I;qZ3<rUH`6n+544Xz|qk_55%6W!1R>m^=<q>&O)'
            'E@tUXf?6(ALnTvqloTlec#**++CwfEUP)aU%@w#^rhGHGn)C{lFQ^17XQ&Z_oefdf7T_3BwFeJ#*!d|b*We3qpG5O{n4yXf*QKx4'
            'jQ*=uY$VJTKi9Z>L9G-Q1I^@%{vs?o4sWM#4!jT4)1g66`8;wj$ukg<Va_bat*svAX$hP<vK9FTPUwp80(C+NOQQme2$f--3ODTc'
            '{nZux8x8=@KOYh63Ofc@>o~olKe@vU&@xI$;|Cxs@DKDvn81Hnhf^dVe9)@(IJfxL3->NH!1tq^maN0t)%r0v+UjdTj7ILa?^sy8'
            'XG}=WBGWab*UVaOIw28W3;15GILz~}@cD6;riEJcF^_uO?MosCOa!mA{15e1Y$Fbg7pwyGpjy9XPW3{qpb6#9B{Un~Zw*;*+eaPv'
            'q1D|t_pZ7G9bUL~UVKh=bIv|80Os}TG_R&>likKK%Rat^!`5JGqKRLLCf#Zrj1YL=D$p7+h&DEHJW@Rk(hOck@^anNui+HKd*IXE'
            'Bj)vQ)o8YTMJFq^pvX4%%;{#cztV?4`L44LBnnm+tW8hLV)W=SU4VeDYY+h(Uc(w<pR5Yw$)*VNj%v>rp|MDE!|E8Q;Z*4NyB+9f'
            'v&Zks1P9CvvKJ+=O1L_Ft;y>Ou9HPp!aW`2k(dOi5X_^Go;|c`Hd#oj|4t(^H?Hx$*V&X|jio*u+_BWe$$lo5-hX$(4^z}I9n*i1'
            'y2F?hY<O26);fcWxij~E`ODF}S3_R&ILd=07guo7aJkivpydMO3j^HN*?{U0DgI!r&|4oTgZ*t7Yi*2xZq>HZO_sZa!QT234S-jj'
            '?%o}U)*wusg{y#8paf3vW0b`KY(+4OMCaB%X0$^W+b*mIu0p>-G~0g}WLI%S-~y3{IiRapgj3**Ho-cI5pLg+#gwo+-P>zDX^q$A'
            'uCvxK^|@!v6r~Jx$gbc?2An=xNO7=g%Z-n!E&sZ=F#p&(iii0ejl$yzfXkv#?>lJ^@awtgR82tEn!_$lLj0rKH~#_wZX+fkv09p8'
            'OJSDc|NhVa%}c!s7GadiFS7MDUq)+E)XIl6jE&E|8MqGv7l^p<1>`_OA|ZP$4GU)ERQD*!3t-EwaRD++)76SThlA^j=VZan#i|5B'
            'r_4jej6et6ivTuT8l>mYG8d#Q2DTR%5aij#rTkhiy6jrqGF)Qi3JtM@>h>7e2CsD~@>;XkWKVWf%Z;)<U&u5ZV2<e(NL(OY*<F#s'
            'sl5z8gz{*bW2nQTcQ&fEY_*cDauBPS!vNMf!xMv&U0sNTSsqhc*fp}A^DYs)hLT~pnuIg76XA|UAoNriz%UnCuwEiOwVEBE;eD-p'
            '&qseiSU&&?K2H;%dIj%N9tULRhs}b|+yvQ7s7J`v@%VEB7>M^kbJH6TqDWR^m9sz%cp^H_hd7_-zx_7s>Mb+0EzAU}3())RH@r$7'
            'RMHCEFC9{Uh#$lOeH#BdKe#}K(Hj~{FA0=h4=%)a0)P!Y_%A&<9A22agU{qvOv)%GdS65!h>nd_>!1`#bijv`GAout6bpT^45P)e'
            'kbg|#G8^a7VinK>7=fvpZ~5nAny$vMgdpdOIovnc?VvvZOi8C@KK@WW9rPdH35`2#>`vlaAI38VVK|JHW^;cqyxzYP9PkHDw|NFP'
            '$n5T()N4#G2_JOl{Tf2Q3kje=`gRRCo8sFrSh3Iz==9+y=?l^qyagWI;b_K~X6f34Bg?8|T>1>|5X9F-h7{71dICjo1M&Ri=eMtq'
            '&yL{}0UR9iu0fIrmmF#c!yf4%U_1+a2U}POye@OnK>#zrc$lDH3L>9$QK5<EeL#FThNSFftyf<4zdVP<zPO}bm*>2K7ni(*(nlwU'
            'AFy8`;NFJCUOdO_6(es*=|h9f$dzYaCgjh7Iwd}Q@lCMF1$%mgGtuh_<hzJA;19VVmkJdweQ~KbSdPxzwo4ue+&|L*mHE>&nZm;1'
            '4GCj=!}5EKR}BzC+(4!;Nf^#@@dn+L;Of(kO4DV4^p7Y8&5}OFiYz?^*mJn;$|M37wl6O0y9#4k;dZ=y_3N=*dfr6RMdYb=C1P;6'
            'SkfU0w_-*cRNKOvw2XnK0U3{xRK-X;VO=jLi*cS!<wjl3<zM(*uGV~U*>~3rrW&ltRcl6C9lSV!N3gXyc<A9kz_r#n#IwE%K6Lod'
            '<FV~jGy(4<c+r*2a*#}ya7BEOeGgRMLDg514X4NE&K?`pgFeorPg{7H%k*hTqap*$>qIx--eS{X|IvK=;IRAc;ZVL<x-a%knVF{k'
            '15Ro%I3Tf9$VHJFAvo9Y-T6NL7~K-F-41V=&~AtD-Clar{q|NZ1$_n{FMH{-dc2{>3^$+SdYQYVZWK}t;^kq|l$yTAX1M43=eOzv'
            'Z7+C5YNLcBHx0*W7A=7Fia`!vlg0sg%Tbg}Q3D<N`wrY&W99oX8+$w-_!79kd{3>N+8SCp&)eSOptt9@`Xp~J_8nt@InxG0SxxvG'
            '@nZ^9Y=%y0XfFp;&O$N4R)FU`*Z0HT;cyo694`5EP0>sT?0P~LeFGe=i(e7BWst3a(a4D=1p>J>=mrE09m)!U+GOs7jcrhqh<so~'
            'I9niRvjJMyg$o<bH&7Em<QpXMAot9eMFRNwzm+(ApbSG17?AW=rO-$7qQlCcKKT9N0Eu3TwEN+sgNK6HU&U03gTd2*0gmW-UvXNq'
            '6gnY63Q=GYWD)}+1hg1Gs1(0A<`{`}kQk6sa#Q#ilD#P8-4Ov^P)WETJrruM^RO^hlvXUwfJ|RSNYUoz$btxVJJbXm+bS1NB6j<E'
            'Ih`W8-tT`B(!+IGj1JG4KHkFl+FqQ`jX$2bsBJa<nrEhwNcaoH@)2=jquYur+rlu~1~~t+0jdMB>r8LqBkk*>myv%+3>kIa_Zq0('
            'PrVjeXKvUIicque&i5ykF<0UU^INz0pWkI$(V2u-fWBce6b89uuGgk0-Tkh`PzU7<MJj`vhWp-FNPIm6GLy!>0M-Q%8N{swxv`mi'
            'yqnv^KD-Z-@AU;a-dv|~xl(NP9WX#bCJ2es3wh@Ws%stQIoQ89K)dU~-CgTkcL0y-%=6FD2cQiwI?+*!Ner2w6-5#V!Z><|lX_4O'
            'h<R&d*d%AoeBs$LBv=vZ0?7_FB?}mktD7ZY#btfdfctq(JeXp7frv7*1RQU0fxOcB0sfGzFHQrW0-xsaxVb?3k6damLML~WzxL!>'
            'z!0CoJW&fouJt#|6p+`LPA*xxHy{h=dBw2XJeO3TG@zbK`|$8M+qb=7v>4sGh|+ZRq9Mt_<Gbmr#<J>L1{Ndjf?o}!<5PwTPnv<q'
            '7l9(}32J3#;RXI!MIRa|7Ix4L@n6K^qfcN4SNsS)=yP;o9|5oVc*u0}wmPfZVNeu6Ge+<(jFEVb9u5ZQ*Z{<%`Um9;HKG9iHKKjb'
            'N5GJjJkwE}CTO`H&VV-IzL1A+M6V73`c9(g?#yTH$`|j5wYIb=UumY=JE}~iQp-?6O5FzVVfhf%=*2nweL>_81j+^NsO@N?bS-It'
            '%5piXaUM>U^5K85+HyqyVdAC3Ec(v@P2#W!XK(pIr4cBBi<Hb1(XHM_^p=2KaKR)ATuZ$e?e^CoJ2H#)9SyoK-r=2=Tl(V4y8>As'
            '^j;`27$+~_ddq;@M0o`y+F2L*v?NV>AH$4DlcI_5k?{R^a`uzBRI6|)Isu5glRV7eie~8jDO`I+r^{x}%leXBA|p)@7r@1l8jy63'
            'M5e@LuL7>rA34R)u(%9o3Z%&%z)69<?d5529%Rr&V(psTrbMtn2An+2#8rZDrFg3b(gzX`+kyfrP}`(LTDpRB2Wvvg5UKqw$i)|k'
            '^9q(`sJ4Q1Ccwa_H*l!{>nUNg>`swAMJUC48~`nWLBLUYw_vuO$2JL($CdjM$7;P{HK7As`Y1NwOMS6P#}1$<92UgtUzj)S;WB`u'
            'Fyng#bqi`HAutbD1RuH=c)cxy>o8|e5cD(2<v40Lh1wAO3SluI8ZJ$JYw=CP(?5*Fpd$Ly6*`EWn~GqsN6|GVYSZ5eyYUY~aQN{p'
            'V)(^gW%){{s57lj6CkpEp}+@yh45qE0_jtM>dukv8B8WEL^4?7kyjS;-c$G}eLNp|QXy+wzVX9-CYXqvXvY~0?=oD?J!OzkW>C?='
            '!zOMWaAcpNXM9x8dvG%hh8L#QPEu=ZC0$LvXpPOWd-^<L1B})i9#Fp|#bc1$J3^a}8VgdU)KP(lyhdBBy=iaBJiBLY8O#@LvSzDS'
            'Z;yRRvTDyQjkj`|xpOChqb=tsXpRoL(i>lrMD%__Ii46l#YV|kxS=G+4*89UqOg-<ypiC-)w<aDB%TK(h9aj2ND3&KkOmJ%nU;$s'
            '*%VL`#iNc3O$1~HUyD;T4@nKcH*#HwCIZogOD(<WO9vt*>SGv<ad9CgT<l4Yp}82-tW5K3&5UjW>azx~Lz^aXP#UeJ8)&O3c!;A-'
            '(titv;>Xv=gTaCI5#&$hsA!op^5>F~hW#`8)nR4Gh{p^P5&1ovE>PSR#ua3Rv`JahLFEc@%26Wp6kz$B2`@unsBi@EHDJZEj>3BM'
            'K*ZBKj^<<7Fbu;8XMi<ykGnQ*38rq?mM<K8_EiXYG4{nZ%ATl}!#G&w*t~`b5eL)pMqcep&{cUbcRb>FfU=BBI;8r~Rqizy+0k(}'
            'O3LtVPl{CpQf;n}GYzxRQK1r^+do%5`9|HNtiJZ%JUH^qJ8&MM3Wrol7PN0zMK7mD+GQRg^8&aQO7RrtYrNwS8tS^A(Ix5tgC)!X'
            ')3a<t?Q%4|lu1-bb1>=nWvmSR#9!oY*tfk%nX5x}GEqBUE!qQdFa|n}|LYIzMN_CHIVDmPbY`TqKj`{`|CZ430S;>9<%zdbR9*qZ'
            'U=fbyo{Yl2MUcb(V7`5DOVH@I$P;mu&SE%awF`u9uaP_^H$kKhl-%VXf!D9jc69?&bN2p6SK4knT`?dlkDG@z@?p^RGj4SJVU3Z#'
            'O|Y(jU^=I&Hhg^pOSjMjiL*O?rV9OO4CrL^Q}pGpDekZw^JBH6@}?MB1QV^SNDGWdN<m#kDYP&PpTsCQJVG;j(1>gXa(yDU)v>HX'
            't=(uQd6XSuPls;(YF6lCsBU{DuPK)S77g-_;X)v#Htj3m1gfy*6Fn?Kv{rFHfjsn$f+qi(<NaM+ivN0g@`mmYj=U!WYF)0v{d7Dr'
            '=0Tiqgv=&D95=f^|KI=lzqN`7HQ_9h1?cvz0|e0_M7e|T#`jRho@w6@7$REeP={@Klt_zgUZALy&gnzsO^pUemE)lAD1gl6+l7S6'
            'mjk$%xMX*#euZGW81QrB6Ua7Ro4mYdEr0cZcYyv*=lCDYTR^xVJtHnE<h+BizYhzzXPJj~v-2${RIZLXqAtsRXq`!U6mU{~N+Hov'
            'jbRcB(|Ogdqd+;`D9orUDw<LLN03{4;e3u>K!hVg4sz4rXC&C&<x(CupYUF~L*<qI7)3^P$+E)?4A5~DVoMZyPKEHD0FCZydC2!l'
            'ncnN)X4Pz}#AIp1M`D*(8VNi^g_PZuyBq<eoE*q9qnLVfCJ?De7mzFCPhXs%23Je`@ViWOkS&E2bP`tDdM~8ZGW3{zM8Jvm$dK7d'
            'aEx)6aU!qzSm178&XY9^B918519-a?n%>ol5Q(w9O>aGQ9ERfj)<YDWj*HEj-Q3YFghTyi<f|7Y#s8482W*-#o^5^6@AuVF@#LWb'
            ')bx}{#@>9~cq<j(5y|;vm|SP*=@_!>8H`E0y~(|&K|J8^D$0uZkcEqo?aJf|xhz*fQUx=!+Y;gke5cCOq0_G{gS<!ce1N}5EB#<k'
            'hgDJh(gQUCW~`<10}vZn)Cl8wWOf-~qnp&;#sRv6=<&s?Y2LGnJo7rt0`>z9R?#NS?oSd;vD_8NqZUH<jY_XW&b~P@)+{%zKxV-#'
            'CwON_0Zsh;=U-)7nK?N@lc|-sT?3%<b`$LLCdkP$ffEw1m%$_~@@FXeDmS*vOHEcUFGY%8i8qR?a^l!IDqI}BIejIFh^~P`hz@z?'
            'eF*T{7t{5@;p0cFCBy^H;;aFTWkHgXXHm!wnymPuQ;UjqOc#SFg|-qOa5lK68<M#)lDiFf_@PyRh6n((1Z*yr8|h~iqiT$g9U^{T'
            't+ZZPz|>#PUiO|oGvLYV1J#9u$-&&VF4KX?*l5IlKu4#~U%i4cWFQLW{lC!-rg>810%Ad33foE@Ls5qmAKK5uAe$~bSw;1J%fIdX'
            'Hv6tStp45k<M6jW{`JSM!ImLNTEs?pr%W!tfo?<i-}!(4ExGvikH2F3A^+ikV)N+zK^>9Nss^gEJXw=f3QUTrGX`xwH%B0!-;Y0c'
            'Nkf`%f=wjSt8zoc!(h=59xHZ0ZW*(mq^FT1H-=~0%ksvx>HP5zzx!?0g<ZNm^dE1#AM~%zAMpjOi~+BYmg#k8U<4hov&RUpj*k9_'
            'Y<JM@vdp5sO~MSWi|#J3fad;(clVbIpj1tF-g?rNrn>T^TY~1{j7=Qg_bb%L+jgB}!d7QJ+!|6<@PRSUa#R2$Wf8&;rI#2vJ$PMm'
            'Q_8eIhSVQdk^pwax8mWG!9aXRt1F@7(<k_hRWtHaT{(E|)?cv_@|WMn2be?HZY>^^1&kWkxkL0SN}%xP8~}?NNX?VJ#U5~zV9QIm'
            'E|3^2o`MYPI{wwKE*RTMo^RN?qkLa;Ua)rabj)9;0M%sHMd;}L!+$D!k#<oh0_pJCUZWX}={aBlS;CZ3R?Q(OFodN*?|ySKU2O}}'
            'L;_h-;$;Ism^^oU`X#Fj7cr703L{9+nj;Ai&c#P<Gra+(a<K?A3%)7nH41q(-Hi6nZxyiR2K&{rwQGKBnqKz7BJFmjwB0mdQ^wV+'
            'LrU#E+q7|~G-Qs?-bsnhE_ZlOl=iEz9Mj!hKQ#T`n>BQBXdA7<(j^@;+cdw%gWn%Cs545EF!B&%xPkRZqGF?XVP*Rar0){0D@Lff'
            'k_ky$><j3TugJ`ken)}u7?#H^;S=a^2xPcKuNr)XBa)2)v%MKGN>Q9MLcsZ(#w9r@vTL3aj`{4}1kA4w?>K&4?Tdpp6~cb)GsuQs'
            '6d$I@Uo-7sAr}un@SLb4L0%tkVwsvm%87XBFkThCJN3C$$J4pW>68Md5+};KX(UaRVU>@`gI3W*N~x4|1IA4$?s&=6l0yS}YD+4-'
            '{+vVrGY4*^`G%IcdhAJj<)xKz=vzr5=|-oRXF3;q2H<m#V$nDf&qE859im)s3i3BG)D4YkKJ*AJf*s7wLaCy}4`Q%q%3Y1$HVnh`'
            'T4iLowbQ$ys}l1tyyQAxAiv@Z-+5(2Fw$R4(+u@t1`Ol)C3!T7L*pH<v65fw(6`-sRJ+{W3Wz*VLpe*oIx|DHtukocqRfcR`W9oR'
            'zTrl;D3>II&a$4n)4+X1`J!S%cogRjPGLLwd+%QHo;$uri&1M>+uGett*vfv+OXa2jqBTAZ|kfN>o6}Qy9O)pb+Mq52@nN1ytvYn'
            'o?n=fmmxo}H#tf2>M0z)ockO6E+rO_jP->fd3`o0=AJM1?%Y_6T(gGVB|35EE@MAyj9rVqSRgB8yzvE6LF?`L;DRT?;zZYPc`c=j'
            'ZXE+E+@hh)z1y4Zhufu*6zC2t?A%z19a#KAGP<;$^9Z^PNek|U<(4303NiClVIh{*x8GJG6tv-&T1eRCF<^W}2GXpG>R5RfH7j4|'
            '3~+}yYTqy)cUz(AvFlARIVPI4da!r*$<}})4p7k2zssqi?~-l|zg<>M<{6=L4sZj^;{j|@=ekR$=NeCsyw}Klo~uy*I*s5T_a+j1'
            '--zDV`v;(%DrMjzOhP&1&OK6gE8b}pU<SLrZ?v<S2a?k-Z^rMA-@G_}C)RMPL*VJY?y%V?Na#DxGj<LDE@ZV<BC+J9ZGZ7fGIzqi'
            'DmmH^=(7-W7S6~$1i0W;I#ca@V$iv<DfZ`8OWoNAd0n-P<0x0!60*N`s8f%g;Lk3~_XmT)z{VCJG?iXpI5?bbhqtCuz<5MZ=x_g)'
            'KR6sb#I$8*Fv1iNa*R59KxGQ_%sc9b^dCY-^MWN5%Dt(Ac^@6pidJp3Xi@<<NBu*P0bTKsk?=j#X?oHsAv$xymmY`+`I@3|HXtbX'
            '#VLAS%OK8VBP2LM9=<XPUXO;o_g7_HL~GJwuR?TJL)ieZ7iATgpcaMkk4nczmm5X&3q52vXc}Ie^QbSBUykYcM^2EKA#rb~e~lHc'
            'EolfZrri(6s&!Mmazh>h|8aqqQ{wXJAu(r|MiR$2q=9U^yGKH4Rb?=EpWNGx2?5ZtL2&V8RNn#Iv_}GhtI1-P$(7Il8OP+?caClB'
            '24m{1mQ_O9vwf#|TlwpH(a<qr&zMwRt}xS+Y&x)fy;P<^nyTyM1%ps4jPJYk*EIU?nA@Nx;yZERE~@$coo&e_OhNsjb?4bq7NE_@'
            'f&Rc;mIkjfgw2A1xu!4<Ycszk==JW62xn}S#y9<;3f(;6u~Q6Z&T}xYzULZ5bGu#0qgL@g$E9iXO{b-K_>#;bXh|nIWe?me9eijL'
            'MXm0vYy@l<wrET=``{x)M9MIMz0D}!U`ba6j&pQia6JS-O^fuvvk!)z-g-UXziqtv0JcVj?iQ}wAH<G^v@qT6e#=Lw4U@o@7Rm#('
            '(lp3KN7$?5{g%j;rcT|uIZHV_sQmMjkWzo>OdOpmP=xt^*2yUyjgzC+xU0r%jLR#->dv|=Q-i#V#u(-olMX`zR+S&(AX${bLcXxV'
            'I_`oVB~gLNlsL7%Ipk$pFukf$?dtpVOfO3=npT4Kc+O=?^r)wITBbyZ77508U}6Eh{wwn@oh5OS(2c7iWjj(YEmXCY>4s;Bf<}o6'
            '&~Xf~pT)G_brRXo-9jO4);7_7txV+lmhKrUGdPU72+)Hvm)@Il#$L|4k6Fcz1N%p@4V7XXj+As7$&@WX9yB@AF^4%DDOW6CVf0qu'
            'l^m;;lxcL{^PfK>VpO@|2v1<Y%(tC75@9;N>u4H|yy?26?8>Wki3bEkppge_^3=1gVPZzcnA6l3W721i_*4DrjMyO#NGY0-1CASI'
            'nAMn|vPoVCGZ}>15dVO@Ex8_#sSpH=JuP$Ez>!pBs8cXfKQJ&fSkY`pPG%Mx?MEH^Z&dw~FSYNu=f36Cgkli`E}+{G5EzoOf?Qmx'
            '19NF*N19`em|2Bn6)@+U%1j@ScQCmaq|<4cQ7k<filT&__b7tYxwO8~=ao_dWMMB#{z?M7SW^@fx=TEBzsZuO?LDmEj5-=(eL3o='
            'fftOyA@-f?u5Vq`d7n5#Lz!bD*oDi_)_LJp?--lpOwyc5_!X8ALC0t~sI#&jrE;g}(Lln<sAGMvN8}9Kd-qdIx+(JP(J!Nqft#05'
            'bSN)e_nA$tv~l37i~;?lIG6@kePf<2dRemBx1>aCSFBX0npMH$Pt*lBQiYZ;)RZ{($LqQxS3yJ>S}cF}-G;%&Pa}Yu#L}_v5t)Bn'
            'iZJ-+x!2+1HR!aHL}HU(bq(o4@`Y%X7>Oqq__4fo;84-m&ZA)j9CX#=g84W+I~fMobq2oX`H`WiT6T@NG(=oVPd#dy<^wI^7(;Lb'
            '9VxC_D^?x*2-tf45q<bOIP`<A^R9*IwWEr0j((L=yW&;N5$Y}%?y=^J)@wW}sCD>eD2`Y<d#|omEO~|zmsrY#-2s{SlBX)RNYXM9'
            'WnNMu>;+~MxyjOGfv`IJs_xJ}vf%sg!X$#<s6Z&Qc`)U~A%csAHD{cjz_97M=mS86c^U%U3S+okao*5k87-E5JzlQc5*p@55Q2My'
            '{!@yWmD#aim<7V3fyyjQs5+Py^nDoyF4HWb$6BZ(FexuLd3LeWHjf>Fv8@alt0{TVl*v2}7K?D!r%VO-2m2+=vfw6{xlOQhm;v7u'
            'ZbF%JWr1<42r!X#?BFp5y8r{#BZ!oQAjNM5aPeZe6Y|95aW`vdgP@R0TvJpOU#5ChrGcMx>;1jgC-2A4Pu{$I_2T%=^W!ll_s5j~'
            '$EQD?ynaE3@b4cJlCb^<ePvUia|{O#3Ppv9HRky$=?Ebkg<cStz9x3wb!puZnpBDtb4gCD<S?f)#mtC#M%)WIkyli3F2|t1$3uL3'
            '^zQ7Gy{(XILwjfD{rApq@4xMK`rrPsOTYdf50!-dU*5cWe)8fNgaY{E#mU*x>(@%Q8Oslk`7FV}J2XA%Wv~KK3O>(v-O7VT%K%5~'
            'HTf+YvyoXV2nvR!hZuvcreG|E<CWZz9prdsa9(~6WZrd5OcvzG&5*8~bL=G5TH&ytNnh~Lb$=ueT)?9Ve0Aj;9F}pEJ^a8rDgsPR'
            'E7kAHou^RsQOr4_H!wO63*?k%wuM~z!5RD1r)MNK(=@wQa`&1q9cMh`^{7mJh-wyYkv6L4wq;I#q&AT5L+skeg+z9=(=|$ySPN!G'
            'Qe&69M^-6;ag5tr1K{?ShPnkrB%3Z{U4UpauIjpttF_HLQs6{|2I(iN-&LaqRD2nIz$3p5G4&<i&hKBH{dDrn84S3T3H+B~l%X~j'
            '{>V))4wa(`NMNXHV;G+!cA6wt4yv_y`z#E5m9n*RGVT#>BBhSW2{DrSHWLbTKo$?*)2;ac`6d#DL^>hg=~?BaP8p46LbH|VFN1mv'
            'IU!LkBeIrj3_Fxm96g$D;NPj-rvM~Sj@`xdZ7SOWBDBu)YU-#LWk_;1eh@d5{AOUxO4eq-Y^GaRVYtp^&Oj~GnIZhNE+pt@Rq{if'
            '!a-93x#(zm#4XgD9x`L@6Y4~v_M5wue_yJrPjAAKN}qvXK(dc<c;RZX2^Pj|r~)zS{&Dj*kS>G44Vo|QV2YfiOAOXQlsS4gXby{n'
            '@v$a`A*Xm@=-YIzYa|*S?gMQ!Z;Pnmbv3`~*Z^dzw$|QQsi>_l+S2*sWI!WGuG<_;N@>|T+}M{bZPG?9YXk(3b4N>+Fm0b75o4*I'
            'sK{_h!O*FQzhscGp_(Ykkk&W|HSz0PiLNEfr%gZxR{yclRjZW@oV0gSsr9Y-7;Z9{{CC79Yi!NFbKl{GQNN3^yWBD4l&Q9XzLmRn'
            '{V0axntZab^(w;8xWKzz?Hp|^kpmN#@*F{5kYND-)EA5|1`Xy0n1)i;qLl}y-q1iVblnHY*#m`q*stsEQr(;;wcI9xHE+?>j3cLE'
            '=F084VJEsZQ<XDOE!Q-&S=CNrXfA~rr6l6;m%9zKH;oI%<Rukk{R@$Xp~h!C?CPMP5g+#p4HDJ9)s6&i;&!i=wYc?+S*e=uOTrZg'
            'YOn#apb;h*07<DZbj^5sP#t9u-cVRj#hQRO3yd4kS86JGue@4G`W*)!BU!UZs*!sI-1X407;1-9>CX&w3*&k;rzrKY`j%bN`g1Z$'
            'iaShFH8_m<QMKcm0=%M}^0b5aq9^GyZ(Spj+OMRp3GVB{+Hy(of*q?rTtNCoUyf{y^{vp9(J@NvdYD#9F8kiSa%PAgI@(_0%}-@-'
            'S>26cqrC|*VajpboswzhU!_z>+rR)dhITISTVG-!izs)cxYn7jl{#2{6FzyR|H#9b?_pWq;J#2>%rd^CVjZ(r@ja3x3g^TlYrq-3'
            'V2d;P3nl2hdUH>R;Vh#14Xn!?N3&1suC(2%yxmCW3$y&c&$NWkA(hpsfs=aj=2)&c=G0O-qe%TH&+7|Q$$_!56qP<hRRuvo1%Whf'
            '++-;w(m>W7y#}zXy~@z7q8}6=6H_&Hkp~+x#E`j3`}tF9bIkjS=9vgnQCAtVNOwV!-kw@RY{I-Ai4F7U0|)LABf#?KG+iWEXIjf6'
            'jtliY2_(90?Glu1U8FA+U8a;I^YTdbR$$7k6ml6+$V-+^0!%DgY}ifv5?SPetqo`!FlHnH0uIbE10*UefcC+;$0$m=zr-vpCj}NK'
            '2s4rx-mo@64Eh6hZO4kC3(oQ_!6^V(%tb~+W>+!~xLzBcL!939s;`q83#;n&cpDTvR`{#}SyRryLJFt^Zs_#5FY9%XHa;%VK10M6'
            '$Jp0(F?eAjW++(qe8X&jv24HzFfUkR1e*^RTn5HP&;-xNScO75e|1~HaO8J!H!LJPQUeU@H#T<9)Z*SMZS0M~=TI3H6j8~UQNLqd'
            'uW1^yzNy<;1KHaCFzt)=`Rns(OJa4~wYh%~JGF#+mEKwUXKBWd`0N_rSS^4<3v6ku(UYs`3)AK=OzkfqeXu|9jX1LimP~cX@xFeQ'
            'i}3{J%xlXg?NFu?)CFZEG>WI95MybZ)jS12zJwd>Qb$>V;LX~CER;6?l<M3hrQ~3~VQP!Y(Mc$am>UTcQHoEnz#|p{VQLPKi27q3'
            'Pk?GhRL&dpHw$i%q~@7}VDLmTVH^Q!)Hqe;zRy{JYdhvfFuaGWtb0t2jWDGTvx92`xeH~iFW1F)ou4?nKaQ9~(-6(V+LC*MVQNOo'
            'Sfm;kgan}OY*6)Z3Nx9jr;-y<ASlM$pf;(Wq&Jl0=p(ZBWijpcqdaAUpEXcKWz2G4k`!qL#4z=NQ57kb%p;A1W~EFLR4;X#PnY2;'
            'FbdS0CD|2<>v-K7SL>O%65h}e!n`^?X3UW_G-VjWs4@vTz)R*J<1Bt`$JjG{2&f8+v}aU<E`p+@a`w!SjWOK<rmHfmkaMG|>~6;c'
            'zU;4@%M(laCZ_2eHl~218lek89-s+i8e-ixv`tgmam+R0?O-V3+-c+`lnc!`mij;5q?8OJuzFW%QsygIDaxUS)FDO_^D|)RiVcW&'
            'a(FPFkLJzf0SO(KYpxXwGqk-*cA(;Jq&LI}XkZEzwKlGFQMGTEJ3ix3HOL$u3mu(?SLO;yr3C8*(DduxN;Y|GX-e0N(Gr7YV6@?M'
            'sfOVy((uep+ioiIt~5E33lq3x8CjoVvmVlOJug=si-YTm2Q*^)p)VeF?<WFr1}1|e)ae2&QfV<*V|9VpkI#pX2N&&~%=NWjSvyfN'
            '`p{O&41WFYa>b3>Rpp2uwVMh5V=jG;#rEJbwdk&-qnPr5No>Pg+#LD#ymxSMUcI2ibc#09x(>{#)@4^ZuvCaX2MnpdSC^SddEeKW'
            'gFsaPvIRG39ye$*vmpjBjGg5itT9|t*-%ClV?ZyE=!l}mF=Zo{gThxgC{Us0Wz8-C1~8d-SxTK5Q~OyoLlM}Z=&lS7i{L&YEx#RJ'
            'ibK*UHSqaCuIR)0g?=K(#t*9?Qu9co{HXOwl&L}8VMZ*WdNiJFsshydAzkcuY95ek6LPxUl@3djQcfsk6?=Cac4r$92cxMfjGlMy'
            '<E?uyn(zf4z4!I&-TM^n)@S(seu)P#TkWLgLpyScbzI{4n{lQHqUA{A<u+)WBkNqzzF3vEU>P6lnUSe9Z#JcpP4dM00Q)-@6>Ahw'
            'IsqAAk5w#;42Bex&ngjRRArwZofpGa*!&0b#r>xT?tgp_+>4r%qVY&Zmod&eKDSOmbor_t%c4RO={-C7cD19Q{Ss5$+(Cl0Kd|i5'
            '5HDsKT2;GdgR2`Tr|g{6y~X0)5IHC07kPH3TMRu7Wj04E2iKjstoaa<OEf#mN}ZGbPS-ad;o3Eg5&QmO#E_c5E+emK=-;8{^=Z8r'
            'xoJkdZG@utTQbx<@`u${^K)|Qe%P|h77ytv3Aw*9W8*b0H9G7fe_jW@?fyHuhf90%XSs*F!)%EDlj;U!#H6dL<4`W)8&0cES4;oz'
            '_9nNbFZ4;rn|2jliK{S%b-cqTeO7)Hr|-_ho$lx|0!BHyw1!z(tJTW8HLSjkH}xxbv`d~g`6}ZsbFM7=&x&0dsUJ<my0+uI2Ff4~'
            'ciBC|#bpd8mf~_E$9*G)+m0aT`0ehUk>qvQ98_KlD{ts)#g?lM82IfAXR!4S2daJLRY-%}4sYcrWiUCyqo4g-6DU3M%(dj&*B#B^'
            '+iR%Of1__E#`L<it^eQu^S?af;VnMvm`~8R(d(K|e#4Z|p3nh=<WZx<50=7B`?eX&E+0!NqW$Kq#l1fq>`K>l*nip%Ym$Y*)p!xC'
            '-3(w)9#^sS)j+F=`aj{kv^|yYV^T)u%QT}5dz>(Dy@Y$R%CoBkfN6kNkYcjgi4{*oma*?Do{t7kD~E)lnuxo&r9yyHrKD=E#pK-b'
            '(xLDIbOAyQ6a&IIWeT}ajT}Q>EgXg0a=@zS$%hiy`QV}odbPm7iaQWgvl<N2mN0y$VuR$s;a!fIL<(uvvpUK|0hy}cfsN6jvD5MT'
            'h^fEBfBg~vHN<~C^BjvKXLcVGo><j}HJwC7Ab+td!^t}bOoB90cN<L<)YIB2%$|e&pH}?|(VP40t@uG^%y_I=DEBZl_s2c>L1o)G'
            'P<^cv(cH0l<*k;r+9>8a))_ZG09AN1FK0t=e04tdoQL%s0!^o5$A=^_<Q7(>ISb;fZv~>-M8Eh_##(f7uo+`o)enj~VKv@eVu0?k'
            '(LH{Dh{;cYjUJ;QrqvHG!oLQ;mf{q6g`LU~>d6Wf2KfCU63A>3CF5)L?C@~ss2T`zBjMcvLN41hje(_sb<^|sm!I$|eF^{UoCcLK'
            'Z0`N?<(<7t*y0S``d9(4@-OY`5{Ct9Hu5$3dz-gGL$=5Fh#KPP^b~EmD1EYoOkOi2BQtKnU-_DXW|)TOViU$x9Et!P2bg;o0iYLa'
            '7+^swkhSO%u`UZzka8WlTo#F^6!^G4Afd~$%F5{+VUSe8F8;xFih5z}wy(|8d}jgK-BjA8aJDbP`5dHnE{!*@bXiwH7Qj425iD{V'
            '%z~9v;(4%sUKTw{roT!l+(l*@rx94%BtBeN(W#9sFj#?+{$-%vs3MA{p8x}Pb3z;SgS{7$kH;O0V1E@u|1m(npjoh5AWy$ZuT%o)'
            'C71Ft-9)_eQ&x$;O6KW>X-nkim_BPLt{?RWgYS&DbMGyj@pobV--d$n098byyKQRs7q{KUY6oA#g5W>G(uCPjGCN%cv-D;Nm}Sxh'
            '?|l}mPcebdP#l`i&EMz0KKf2jquiR$OR_Hy#T%eMzIX`)CDFfFWA`?TfTV1m6aM+)lsHi1!AV&lf~r3&!Oi;-#1{4V$20~B`V~eW'
            'O?~kTvccF3;K6-;6`r6g2F>=r$^ZTZ{>$gn7e|<P-UdxeLi5ohfDW`i&|XP`zWAA_;cNBdo%->sFOFp@3~OWLIvx>XzlX2*x0*h0'
            'ecCKdb-VDurd{yBC5BtyGr%)N#DDyM`RV(o@@MbBAJG4STtNT3<ty5zQ{CZWi*5YJ@V`R~EL=SB|D26Vl7AT7K`SyG6D4azgxA3a'
            'C6rTEgdC&ZLoPF;$gTw?g5rxaa=$Pfh9u#%+|`*dpgi=!=RfSYS3c-nyOx}6Zrsxiog77(%7Md6YIJSkSfe?A1|9sxABcHt0?WxS'
            'J``h`+}QpKbvV~x+L&V?OmlM~+)w78{LTE_0e8{Pn%g-Icfhx@#VcCuHZyYh)qyIH5I|^A+@!X21Xmk*gH*A;a$^Daui+|hfVP6W'
            'jO)^`MNSu5ym=;&-{%;(zoL)3QPd7c%UIX|4CM0kT3)vE02v;2b@sUU%ye3=%9vU3JfyH#0dy^x^!J*Tl30+g0o|nR#L7y)zJxt|'
            'aqyYy&_-o!^A?+W+4ccC`a4c$J#Nspo3g7q(Dyr<IthTI$ksWJ`o!FIsDy#$^VoJ?Yg^Ys2XjdyMwX65f3b0;t65+&Q2Nol;P#Rz'
            'Nu!nPoO`#ozx_6?-w*%t+i#PD?e<m=ckA5=#JP1Oh~|FvSvF%u`<(R19Qr}1>na{>5Dh)rmukNMA`0SkfhK0S&0HNJ84tF#=V(FL'
            '`uh#gj7wbJul{W{=<R-kAbVFDEy*i>C#QmEg*9RIyjsm(c9I`9oL?lRM}q^6&3q@nVqDYu6excq{Qx1@41+dQIedieNWe}8oK0LO'
            'RnwY74D!ZMnmC&DxuvuzQLrf04Q7$p%O+lz%6-{awoYk*#<xx!tR}Mn#>GW!pFDp2@CkY<dv2}NrOHIk*)q)4T=tjY2N^}xJs%!C'
            'xlrZgWr7!(DU}t@Xj+spIQbaM32~f1W(tkjA&6h<$J9$^PR6N?@uJD#&nEAHw-F}&XWthpVTOvqFuY<Fp(?6{$CYfUL#k*lGtvtM'
            '*w-nB(c{SYw9gNNB#1Y8l#daF`GOL^>R6MTWKXNh!W(L5l;xOWgmfZ%8;OfoRO=DPtP=OpZ>3f8W!hhM5yI>ax-iE}nD3jkc!~OV'
            'U5qgy3!+RWpmbh2yL$?m9e5VyR~&wUMP&J^#&G303^Hc!(u9<#A!jINhwMOxA)7@SnVSqISu~|;bCsQ6t6b%NVlgR1Wd|9gr$75g'
            'b~$`l`B57ifp=2{hdbFV<bkhmRa4h`tEw203`3x#k`{uRLmgM)3Nnx_ncbt8Y^BD^I_>eD->fxs0Nn9$r<v?brJ;jK7^@1*Kq4vu'
            'ykVk#wKy0W;m*ldYhX~zrlI@wc@sEyl)gJX3HM^?nmY|Idzy(8x6x#*HS`tpeUPZ5I6_rRAiP^zldXr5T2UUvaP$^YJQTMC`ghs3'
            'Z{lztt60K4@UgrS$dCe@tiNw5Ogd_JelXT=hyG-~Z!av`Jo(qQD{7m9&eNS+{r4723vOUAG-`JwF24NGuoXK;<^t&4I#1yH^&RIi'
            ')}GhLx@TnuXu1MOA4WVvdz)@mo5Rc)-%foi3ef<KDxlv;i#Cdj4q?&*vY&s>Al$ph`EUbi(;u(D`<wwZ-E%grG=|eiYRtRO>l(Xd'
            'H$Jx;UVPS;?FLDHAoRd%C?LSNUAsVSme_k!=LeEzZXm%rJIGt4N%Rq;1H=_DY!ThGg{m7o!LWwDu%!fV2eRYA$gjdAN+N+drbMs_'
            'K;-UwhPl7Wuj4Gxi1Eir_R+>-C@&s7l$wfT%4_w|C;_3_5Gwqj!#J*jO%hzmdXnx@0OD!v<yM?aZw!NM^)!vkFU+&nFew`!?>D^~'
            '?WtWZFx^_CmTB8a0z?r86n&H%-f6;)j{%av+Zr1}O^i!@n|m{8cok*qC|*T5teh<)f%s{!q%w;)P}g3KHmjwrq%HeU>{>yp67czJ'
            'HuvghqhlUewMf7_;sLAH01)zuIr_hx{Mkh%&no!$xg54R)`ol$WsbaXr+U&=<-e*7Wu_ND>c+^ixJvmzwkj@)C&QzatGPJ)>FA7O'
            '%XuzG3AzZYx2Y`TYQo1%Qi~+TI7QR>w8(&w$1pDnjRaX)Tb8PXcyOe{HRquU3Y~@sB*E!Y>fbo-tA}n1Qbj0p#7SO$6;9F_#xz{Z'
            'o}6E4cNPjWbw+a@iYm>Q;#I0XvK);a70|UZ4e9uRU654*RutM@=p4pEK;2gqztUA+Hd&d}6H34`b&NWg&El>bjisJv>^C@`MMj<p'
            'ow4~IIk|OP+d-eI5*4Y}0CZPGf!g;mV8f~%)IEfBU%O7rloGH<3pkc}TUH0{nn##;di?tM`PuP{wqy<@!0=&W<Zei{M(8e6Z(U&;'
            '0$5~NeWuNMh^YxKjZ6}K@eAxl3<!9m69*Uv?}js<$*f{Y9?8gmdB~C3k?9|E%tuG**km5as$7+f`q4?`%v_&sSv}<mt+E-+G1>qh'
            'oRaxkRgR}qk9FBuAEIatmjmS7nK~KTD<|3Kv@CK%pTRI`AC7)LW&&)?l}cSJzx2Y$(QbF+s01pICtVj|@+x_TPSUA}VXhLQ87=5*'
            'H&Gl@a!*uRa4$}WF^lH&kQ`O4!zc-&Tr5frfe_(1=T%im&R)mJ78IFE=JT8~xhIsR4mT{TB}>k*`edA9(vXc!`8bgq=O!jhm*IT!'
            'w^3|~v`6DNWgNeiuccPcPQAnDOR*GEL!rYtWwz9j)2y!xeYzy4V}y&?Z~PkB!W)z#dB^garZ^2PPr9SnB{q6LGfDw_5UnEEhAD<P'
            'f5eK=4wR}Uv|{ct*p*^V?D0yiiY)?goUC*ll~b8)6o^|TCs3;k6&_1uME-<Z&s~Od&v@cpZ0-500n&Y!xF6lQCZu!-DW~$)=+*+a'
            't!o(2j{>O8Ce^;p&fN7H>93z#FlC-Q<FZlJF^xxEw@rU_1^@0~QUrjk3bLarahzTm2*HSt;RTD3gh1%QNwGl;)4QKXz&Cu1GW65N'
            '7;cI2pP4V*;`Z+5@#KwVv!%@n>(;BhrnBCqtv1vd?U%X2oYP-^JbQKa`gr{Qr&p(MkKc`dK6!Ec`g8{dIdZ`a<LEIueid#2t8{G('
            'WznTiQW5s{!Tc`UYO~3|l?ZUAc?xM9tnnHY?Ly3fj(F+E=?&TpE6KDDOmP%Lv$`OMF!6_)^~45Ot^?=Y6qDBHW!ZS#WNR+^Yckx)'
            '6gm5IAj_P<F!RoX-p7mE!>#8x%){a~>mR9iXw^g*8@pd$V_qw99c}^<UEwLp<}oT4K=1~3M5Hj>I8q8UV%&77uIZ9Y89J(C)Ae~z'
            'YbWEbU@Gp`r2jF}QK)^9hG96^WS^-HoyBN&tiODF5?!h~irr;Sd+WAW0d9;qvY|Vis$D%NDU%b!USLwm2g1ZUY}H?_A9*I(!zkj8'
            'Cg_qL1G^c;2AM;_feD#at-x@3>*cv8cSMshv?=x?`Pkk%uUDAO6{Rz6mAzH4MhZL_UJnQOqk0ew2Y)oZk9yPLHNF9k@WbJgN7R_o'
            'YEvE;%%Ou^5Ivq|bNuh&Q*IP}7|&PqW>a22IvlJmu)&nta>rqS{|$nvnvhmjO4Yt0Bk<b#GWx{yu^!p;-D|jkByEGw%-u+(T?BB8'
            'nczTft#<7j@h2(6$e{a0isbso<CiDzj#Vr_F(l%-%#<J#CUbft5zL|A4SLzAtY)ML;Vk+A;FsogC9f>A$Sp983zO(Sqd0#Bbrxdj'
            '*-SCY-*ET5NY^SQQYB4HQ_Mh(17S!irgR=<dBGW#`wcv&657y|tONgHaAh3sDL)q`EY910kZ0<|cGNqcD;ugvrltc~`H*wpb2FL1'
            'g-K$o(>C9~ZEU@T3;APQUg2&buStuF_TNPutk6P}p_UTf#j{8nN36KEwm7j`bbqC#uGD<@CW-arNA03{)uth|YzHn^{d9a;?T)Gi'
            'nl){YzgLSE)=RIRco%YJjfWBaIPl<~RRNHgO2?#*D!q}8fZUO^vpFLn@7ze`j@>b7R|p~w++vpfRyj##gcvyzm&vWzxg$C*gzlGb'
            'XhWi%k)zczPFK~hmR<AKPTSFVqIR(Yc9IXPyL@_|?o~z*2ZDOB8;HWeabB)D53kI>B*S;DybAiX_Dl5Yzfx7+Y;i_LuAZC{M`i@9'
            '2DzEBy-!P|bL^g*DUL3xOt<dy61H#Tu~U3gV8sfqKg3)la?LQd(xu)P`@4cu%<hfJe*n;|%z;`i%Igr;V}Ey6tqPivb?Jh28J<d<'
            'MSlI3yO8Q;Xi$|VPt`eBSg&d;f4(-=Y)@3V{U|Ft0f~O7eO15l%xN{|B*yTP85N#dP^hc1XPlv$3z%C`Na_ujFt@^u_N=hF@J4=4'
            'hR(2vtrui2qX2X)(V+3pumg^MKk)CT>-`*cbG^@9YG=jS=j_?98X2Kp9H`UKrAY^^Upir1EeUC?*40wp(YiUJ>W)rLVN`c?ae}|>'
            ')>bPc9PCb*COSAEYxgQ2oHm&E)ALhy<MGEdT~(o62YmxoxxqZ0mK3g6g?l~rt#=#yopDrlS`OZ^nwT{H@*Y>5oT<&m+CV4oUj6CS'
            'o1@p#hJdPRodcl3fB@WjnnWNK2n-$XDf&cFh@?5xQCl<VIZIbej)|h#)eg&cQIQ)sJ?J3t`t{GqE1jL4Y7H}&v2sUFsb8=t5%BNC'
            'vinN9PiQ8T#+W7%<p#S-IZn0u$4HpOsbbKJ{yG$fRYEJoNw%e-IEEJ3=n%MP4xg89`nA_zRTKV?pbF>h_$-C^{{0o=`~OK4;*L+n'
            'XKTk@hl%~4!)rW2*b}5bTw>1X=M1fDc%y%jf?cx{TEDKJbe$GoMZ5l|2q;ht)Ncl-r~DhJ)<4CO;@@4#ZXUvRz2}~(xBNr}=%=VU'
            '@6k)<z_oiw6{*GbU^HqO7oVpuwZK)H*QO7+s?~oid&M)yn{MuLj$o`Us7`vzSi4)-8>-UR*H!Re!~D?zwcQTle)?}=5NVob{rjeg'
            'q>=F-Ypt2%KdF7;nudvMQ~wwXlWMPVZN}V)D(rrmnWi?-CVNfO8tedZpH=WwfQfe>dylnb_c#7b;D%9$#W4ic?!Zo(CE?W&Gpga0'
            'QNaCx39W(XC!^ci7vWgBci|3ui$i5T#qMJ5gO=tDI#oTxI41i;=bnD+T{M7_#f~u|yCwp*gQFDK7tYX3VbB<zD<d{SbsM>3tz<?6'
            'vMoq9!nK>YP_<a;tU&d=38sEqTLCSAHo)WOZSd%A13VT{YvAPLW@!Af2CRxp9mx#fO(<E#?a88|T?dREqo?C@8-Y1tem*5Gv2R^M'
            'CrlWM2PLFJ&-M*YEjmqCvZ<6p`y(lVyvm=zQXEya#xVerM3g)%eD&uvo9o{>rgcF21mDO4Y0txyULb2XP^wP=n3MWO96a?=?l;+7'
            'QjN)cafh62$yAz&Z!K6|@!)|tgolR~ugqGsMkw-CJD9yH!HsF*wH8>v{-<e|O}m4;O}pQSQ#b-)ie)7p_YaiTA`5%G6O{1$=;*mD'
            '+L}inD5*{YI3^|_ki9-RJw7$-OBA_?gAH0M(LtUi0aYQ3(gz?ghOM*XE(89RP`$m}07nWR@0Uay1=QEoNA{oS@$7qjafDXEf;2ts'
            'Q;ErZ40x1XPE_UeP{~5`*P0?W|Ei*eR&6i-q?y;9iqc>uDyduHd129;I0`1Z&}-grGNoHv>zLGSO#dRgx;{9V_t_VGE1?>k(-*Kf'
            'iN!I3PK%)FI|ol@gA8)V06?Kfn~qc@vDY;$$CQhmbS0Y%4L!~*%xe}U)$mloD13oO4YTr_?&S4vx*VCEzEz!g8P{?JGbm6+2S{-h'
            'arUBTLPfvy=((&9g0^sWBM4T($G#%lRu)m@lt0y0+@J~2ic3U+6Q`nz%BCvErTrO>5^?nQ?Yom-k6&26%~3*1OWVnrRB)tLMJ9}#'
            'fR%%<1V=AOB=fbuv5X4ff>+T8S!0d)j+WYWzACCl)s}i%oryOwW&2`pU0ny+C^~JI(8l7^y`BMW4G$1$0>P;}HPg&xq_C8nj$qLF'
            '|BZzi)m7$*R1E<lwRYXAc!x48QVF=RE^mT*Zu?bb=F{(@dDFlu`~}R?%pTPnYG_@Haxg2QApn_vcz1seg>bt!j~y$cSYimVW>qy{'
            '-HX#4YgAT6sAT>UJ-LSOnakMxIcRaOugTKAY9YGy+~zc%#)0jLmJHwbnCpd`{`nk7(lDa7EaG=h@S-lUR^s}6U+s0zz^vO=1Tf?D'
            'y)wB<;0E)y``s2Fos%TkfH-O-zE)Y9k$V!ITTHRRyL*>Od%jPN+fmb~Q=>2xt=k{NxLAeVeJe+H9fRCt*_NkKz4;dCUJ+%oLKP<L'
            '%c4yjC}KOqpo<6|8*WB6Z*9A-)Hdec-bZ!jwwAE^KCVSy;*HYmVpP*A?5xSK+wVvvJBTa1CT9=iXUJig&Uyn`P=y++GhH2<T)%At'
            'yqiOC?^|{M<H6N6!e&ulU%dYsW~pVTW7@I`1G1@MCJq)aRL#@`SWNP<V56L}J*tbhz@QkGW2mAm)+!{JF`C#oi3b+NU{+R;B~0?R'
            '&=gtl>K2DvJDP`B_?VmsgG5H{7?Fe1WjMW}qA8Rx1g)7lT&;QP>uMGiJlsKCG73JVna!vO>b_%HeEa&BQ?gMSR@i;|GS|cn;<<!K'
            'kYr^g)C^yv$&zpFB$d7>W5b7Br@`0TY;$0DZT_C^77=6#D+TmrPMG9=c-Y`kp}RWD(I4+qb&mJz==IrOj{f!4>8pR$)$wX!<fni!'
            '*8%W3047g??1^X^X21Xft13FH@?&)MAh<%`)&w?gnP4=A%2?5@K|5k>mU>1d5fw<bk>W}F<?)AgoMvJ6n>P#e9PqPn9vpxO286l|'
            'a$#nAev{O?Yu2?k0d^-Vn&tP(f)7pj!<;+{{_NN^wt~{FdvbAxX{y&|{A7PE#YgXS*z#_5zpbVSbX7!{rNgH4+wL7z3=u!;IJTot'
            '%@d0vw9UGOh*cEKO7VzN;vFxtV7=6?m-;c+=b@LKJv&30GzFSU9^cm1_yYYwSRXU097GJC5cUQcS-v5@e7vz9Z(K~k1Z2B0LEV{s'
            ')Q++zhJA^ughb|nzWXN7Td0<7|C|OoyGVeuIll%m|6;%1nu~~{rVP_FRokw{M4YqE4XiRKSJn>8g^I=Dy89GYfwS}0w@kV@+xA|q'
            'bc^4x=Jo-o66}sly`GVtdo!|Bb^cy0fi~*!M|DGhU!~U`)ph$e9Hkr{)l|%8n&Y~D+=ZfYAJ;YCc1u^SKWa1<_<QYHwnq&n0K2``'
            'SdZ$eZioG{)X}3Bm3CuAr*gu(Dswa9)ycp3DRQ4o20C?2=<b=@m`kUJx&oXzblt{l=r>N=vTF>f4Kx{fw=E;nP`X=hAJ1_$9Hxj~'
            '^UUp&Jl97VcgMS}EzQ=ub;r9M=F4D>YmP>`dmMjFjdq@Xbp2?`&+$2%WnXt1jHD#)a(kF|`V>#xZOsRD>sy(qM^$=w-%3F}GIA`t'
            ')_Egs^ys1LHahfW3S;A|>ilj$ar5-do3{|qi4$ny2xJTTW9!{x`!suTPvXtHF~%x;G3GPHvKP`P#lC!fXG#}JZAc1BjwNco@wf%0'
            'v9DP#q@IwqvlI;VBDxAO!~84@{wluH%s~-ep%)r(cp;@5GYmh#rAX0<{qT@m$Iu{309vKYqnewT=anjH<G(uhHjYFsx@&l4FCb3A'
            'deC4H;a#j*MDA@6ajag-CW7OuEMVPFiF|8Q*TVqQ>v<e3NPqhs8euq1G=HtO^<S*amxzO<WRvMeW(+hpOyzu{&1?$gk?xez_3U|c'
            'RS@nZ(Upp=YVbmnpRM|$o2K&1<d+o{YPBTPmSZ>fS-E!t5wR8q9g}NCjIh{&ZEQp9p)U>|_f?JHmdTu}Md7Nm7wAHBuY&Sz;ABO0'
            '1XQc3&i+KTE^8`GsY5Z)76yOr{(!2+eqmh#p4?54A&M-G(XViUYPDxFwHGEek~E{SI=fw){T`#;vm``va+w9o0HpPgaiDOEzWo5L'
            'EOK0>Q4%hVXQj7iBtx@NR;e}+VWpqAdeoSXKxG&j>msD6m%ZIlZ`~+P+~SqXJ1Xs1s<l-Dd%7eqb7P&xa+!v^mQW=UW(yPUx;^Z3'
            'a-KnbGBeaEB?4}kVjqiB<(%gm#m@BV!nl8U0$&Fqom%P<tG%o9U)5=p2?eWKj*e--*1luafGXou3jilGs8J!It<<}9I;xuQ^;xLG'
            '?+tIP>hCJUc}tadgVZz>c^?yPY8jI`>OY`Q$75P9^RZktXMB!E;cL#LB?-4Y5km{VeD}Q~QA3AI?NoUOnI-^_F)N>mqFmWZ7r2t5'
            'nvQ;fN$3FYeMdzlyhe%^CN@nIcjyXis<vaiAg)j0!kidtz(1e7`Q`NIBg}NCRDW_lNa$Q`k*9QlzPSI$oAs6rgBHz$t}XyrTqevw'
            'vKQ&9jByKg4WwHu^KtLX!5WWcZkP4F)lDJUk5zys7>uhRg;ym(7&r!G4&-Sd1T!5mYP^jS;4=_IojUh!e4#h2dFL9uTKlD8^lg0S'
            '0%(lv7>bs#Wizp;<G)WAitW+g*v;^U$M4i0R=u{yvg!PMwS}wh8hF=m139Cp-DXHE+-J>WAey<Tk3bnv)G~N2QK)XoH_uRcR@ZUE'
            '%2irUo%p|c)NO1m)t*j!`E*y8O=Fjh+r@MhgdXOfPD@jJhozKFWnN=TX0)rOP3k?*WYMd&Yl3RA?^V-As5|VEh^zOtT6%RKZteL^'
            'BWU#kzXu#v!Ku@B{aH|?xqaB`Y~BW;Awz&FS8+#<06sycx*6m0&A(gPDhJkT{hFgqs#T@uZg77Mo?gc`aB_9;4UGw(W-WGrwOa0m'
            'EWLx#c9KUHT<yQ?HfaM`xwQ`)yyJIulK6j@uH5b<FQQ#qTba#!=cg5>w!u-JBP~E`Zt_EM4~@Fl>@aV3Zd0y%HJ5m^SBrhXtNFdV'
            'of=F9p6lIi_mNhL<}PNjnY)Ytx>KzN&7JlD#a@}WAt}JJ>jvJ4FM&em{b;E6=~-GdXk%;D8r!0tZD@L#H(u`5C0)DU>(sJ|NPZ*U'
            '|McqlPp0o9UWz%V4UKZC=s-UMG=~=C1KAhvISVqmHSW3P%Q*e3%GoT-o&`FCGO(b9^w)^f^eSiV*N~pWJ~-ge=fW7=^Vi2m?@q}*'
            'T5(NQ+UF=y>@#L~BG2RofVwNYOy{BWFDcN0zg$a?THUHh1!-8JtphBW;3`zUAvAUEmdU=}GCIBQV2qJP%}S9|lkZ4os(W3U37>5U'
            '<Dig=M6a=ovgxcT>lW3WHC@tV^ic?cBi%hN-LgjLlv9!{EXBSBb{iwb%TQFVaqe-c`8naSsXc7#WqqXKsnf4b%ta=?<(12nWQ8g1'
            'G+3tGPv&`Z2W@vWgH)@>RSk(s4dHUoq<ba8Mb<KjG(*?c<8*7b^gy@f7w&uvGdtZxD0?WIJqAHG+mYJl&}7m#hGIT;2hyVB!o6&H'
            '<v?gvNg=|vy{k!f$`7k@b{{%Uiw@B_s$ApHr42q+a+zi(sz?lws%eKgbZ}dhITpH|1Vs_ou7CiuNiQ&KA9~i&h1Uy{Fk5VZmMsz@'
            'u?Z0ByreYRH&HT6Z@75#hGIXWqIL;z4t*`f6RAg<Oy!z`WFt}{+PWr}?@-Y?)lQTnKKkz&nZDng7&%K|2Fk@R4i|8t0Q!d9+JV@?'
            'vIB-&U=oHix&@S5X_veLBZBtk<V>X|l>U1Tv?*K;D|Vn~cRaZ+*csJ#oFcuzP=Is}Lz@@68XIT*cITU7v6LwW_B$6xU0ntEQh(Sg'
            'BeFA@B^dIf8e>3N)5$%sxfH^bwCNe9xskCBJ$385voR_jQ1HPVXyejM7zs0=N(vY((RbzibTw}9bXFo<!>bw%U9aPabS0HrG+bUy'
            'U7dTi?nd3rh_NNE&V{i?*Qm=j+-c2h>?d#a17r=%>WFn9<n7@;y~jmh7&03SEX}G3!)}V5Nbg>M<%`3X6>}<B)K<>0BQZ-|JsMAt'
            'NOK0K>S0i*hwgzXh(C7Q*Vrz*ucbo+v@Fyo?q|=mi&~=DUoMVL>UL)#NHZgq@%TE8rXdmr`KH|ivX=eLh3!7opc`D<8Va+1TAH}_'
            '^l<IpY0n$CD_<O35_s2)Ln_-iv){IV%n|uWR=!T-5>u*GdAJ@v_C*q8(pc`kxAJ_|{=tDU7f|(hx;ww!&b)T*U%Fp$lY8YeYyU`F'
            'qO)BI&hCQNnEC9kjAuUqdi3J(8pG}|e`Or@;KV6;4DdhO$Z8;P*fF`+WJ6sCPux@4kJg)HLcGXeFvd95auQPFja4Ak78vNt%}`>1'
            'n`2t!D&tO-woI34G4hxBVu5TGRUp+YOBGcJL?&hzeuZAli{TNY#UcJe2j+f9SF#D|JF$vBAbZ4l&=OtmgWX6d#t)96GOvS#s+=`)'
            'BT2I_!w$JntDP5Ztegcn(-e|WoImY`%ooR%)!CFpi4fN1B*wT#j6K+JiKUKdZKBjpAU@p$o4l*aK4}46eqqR**St)aWqW?~26-~c'
            '5el}Dj;c!odEN-akFR937%+BPo`-`}Qj{sM(!_~S{%M)_jTIiqLDGooMt%a<!H_R$$mdd4dRc|x+Cy>J-6|$TayiQ&7dM!z1=&Jo'
            'OgAA6=PjT@$|YM;;|7HGSfV=TFpw&THjI_EFPBt*r9xsgQ~4vTvITc9eOV4i&WCsY^rxd9T!O%xYE0&J;3C5Cr>xU8ElNTh^k;^_'
            'AH8`2+#KQ+rQ~i{0-Z&$NwKodLc>9$BRMcNZ(r#9sCve(`s#TPYe5T2dn0F}Vofo>AIn1_mMSF<obh^!PM0%LYANGVFIP2~OYy*X'
            'UNf~_5_``Fp3;wnmbVO7TEY+{Dbw)k!2=Z8@-PI#0ZU0$&{$l7qldm4ShCwr9r%qnA#NLhrV~;@W-RtdkDl%W8LiRn0|Qfp6H=*B'
            'G_NXXhJ%1xRDx4q=KSb@+i95teqsqTpj6Lf^nQ}3aY$fD+`=<>g{rO82y0U6F5r&Bgl0zmHC$;5bM+V!4`FV;Ol75_d>ti;5^cUl'
            'cEo@9M04y^ie`4r6Dz@tT!JoYnFXy#ROM6kJJYAiC=J{UxKDIthDCNAs6^fsRlVK$&8;W2AjQRNyZhbRX!Vi(+5@ftw~V>v%gw{K'
            '?_Z_-mvysSAXbe$E3C2+3_72|J3`T>2932}+v>AoX2_h}cQQD5vozlX=ww3WRae0b6UH@t1IsqoGHFUpE~bw#=W6Y^-PS5l-sF){'
            'FH37=r!bOn*30K|cT~K<vENx8rU`Fl0!(+jeflKj#oTYt0%{}x<=mg2mG;w|#+PRQY@DN$3e@=C-zp<${|i!r8pD&BL8+|hD#|%!'
            '-8>5889HtQMMaOr(V#y#Fb+TfEYvP+qW)dKu?jQ4Gd=YJg;|KdlRBC;*2^?gxKHv9dz%F4B+fr4-LCkdn!%odV0xbH-8m<`nj=m$'
            ')|Gcl$&*^gSgWqS3u@qZ7gBG)BX1U5MU;7$O1_Ja0WvDHV6_OyU9gTyXd&B_`KpBbZm6>68bhc^%RgS;)PJTF`#Yr+`*W38NTrJS'
            'XGXDqW)%BpMzOD(QS2Vc#O}@`#&<vKkI_0rrelm8wDNdad}Gy^dB(4u?dvGPME)!Ea<&8f4I*;S<T4s=BX{mMPJCBZnSZpb4WFE}'
            ';U719!k0!5w?q#A4Wov?AY$0Y!@pUiFdr2g2l$zZ0CfsgGurruvBckDnyDLOx?dJ+?3;7=cMd%Mj)F_RDEzn>eEqOv^{P={VcwQs'
            'BR=^&;3xk;0m@dLgTGIpaxwU4fbxIh0A;!K4|qFz%6-fTT#h($S#<d0qXO03&oJKZ)tl3!H{#^Y(O+JP7sr2k_4>uDcV;x8M|wrf'
            'J{AZH76chB#Z-ntg6Z`!g~?YBalT_df5U)h*F%B~Oe5r}mq35|vW(?D`CMdSnF@8hvY7Xtdfje+8Gevnx83vM!IKMVP5DJx$84h*'
            'Cu@Cubawnso0t092}e7#AYJmRI=JYBHX%zhiaHYzeG8N$rNxF!HK~ZxWKo&cP(TdhIb6G$DzA$+;yvYm7Zg<@+^8T|!90{@w4_PR'
            'F^M_mu#ObjB*ZE?uKJwXm#(&wDeD*;@@h7+JA>0zt<=huXt(vGEh4@0i)jMYX4<$L8e-L0>dnU5tIgK`-UHJN6PcZs34KD_Mk)1<'
            '6_)deoAAFeVD$f=QTnF+WycsQ9h@7|ZxUnoy`;g|%nh)u2+}hmj=<ZkF*Eiogd@fl8kugcK@WIWT7f$T(`oY069tTF72|QiDCOFC'
            'WgZJA{;%FVKYo4u^YQ5`@#FEIU%&dx$vZb=VTXg8OS%GPz)4`4qBO|@o^G(oI5{F5XOP2v(dvRg8f2<D9cA67Y)0tA=%R(Kn{Qf%'
            '??9IE`T}$8!ic6AY@K77Yq+B}81SCvm}?g`)-~WDA$_I5{Dy)3y+W*;JpEJ%LBp%o?zT&y3&DQ-sZrW?#l62%ymqztb#t>fY%@gv'
            '<HdWohi!jKT=X45-Tz*J+`GcFzg|f8r-flV3JmlVc2pWWBMc3l26Ww_@Nm4F9qk7O12~U9DH!of0ub9{4%;HW?h_C7AIHu93q0pP'
            '$xZ)1rJMaUMxQs!IZXQpze#)PTKmsw-@Qco%W*Yg+(Zi2*$6<>v_$XBKDyy)e+3pPQA+Z~Rk-oB(j4SG^hqW!0@3gHbq`j_a}QR6'
            'VQ3EPeSGn)(#w1I0L3|>36g9}IPSFm{{Z&^<8%'
        ),
    },
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'viral_prediction': {
        'sha256': '18870ba7aca2895774557ef7e2c3542fb89fd10476f857a2b858de31367800cd',
        'encoding': 'zlib+b85',
        'data': (
            'c-rM!+in|25`E`a^v;Wh2}#jj_+jal0VS3b0kRZG@*0L944RxK+2P!FdPXv><-hN#zRX>=vx{IKmWU!vPjyvw-A+~0p66YQQt@RZ'
            'm@0T7RUubAVh=Ls(KO3NC|89{6Sm|^L^2U9N;h)F)RyBNTkc2mSAixDYT}v9_Q{g*BxLh>uusdPT#A5~MH+K@AH-Z0A~$5S(daTQ'
            '#4=5HZ2pSzFqA~YqiE04^;+Z#j98kPMRsxVe)^!Ml{%|5&Uh{rRP=WuD@KWBHjPp`h?2=lFj+9E`H0V%$Ym^&0`DMenkEy5Ev(oI'
            'mgQ+!n&W1?*p6S1XqB&u62z6nSQOhdRQh<^GzxJ%CLVZH>a&3nFIXs6G8CFAB&K;G!coskzmw9Dd=P9aa+p75%e2@sXiaJtLK!Qw'
            '6hdX3MMA5sx58%PLFBB=qLf3IG-p}4jHKEMQfyyrWwOD+G*QfVw2G(VG0k^cz-;7s-e{etF$;n<9F&V7ASuul3}6v0NR13l$O}=x'
            'Y>vV`PZ<6G3p5;!?5m`VvpwUAC7D~wU@9!&zbqV$Zf;L+KimZWIKB9AcEe8C(}>~YH9g}^m={9#D9vag$RfTHWNVSrt6V%t@fhUd'
            'pQVI3y@}z#GdS>po|z{TRj$`^C22df<+QlwGBU4gPlF2&_h4}tjozNUJN<BR8@xZi41PVoxxKpn6kK0@Bqd+YtGB;iUA(>ea2ve4'
            'KK=Rj{OS^K=d+jX&6dP&epS7^zPh+Lzx*Zmcz*fz>LXsgn2$!IP^_!)3RYW~UqnHSIN^z3<Xk2}xL&|^v$r_NyPT8V9hnTrZ2E?+'
            '5eEhP7rTTj7TO$+AWt6^1UWu;)gNBX!n<cDCk>YOnuA=on9M|+6??;+KDj{DBu27dbJea{rDalh_cMEl@tBzDG_!Pu855BbTIW%Z'
            'V^j2VT)Vc*(+Iye@aYPE{|?(t*rQCs^l`ytQou^Xe)=;kJX6SwfXaqYzUPuR40z#<|5w&0qp_2=*aA7@8zYVTPBMFXVh5cgk`3SJ'
            '8W!;I_Yw#Y{$EA6p$>=xsr(0`oOVZ&ig?>OcOI=)8`d(467J8<1}En$5^|vnt3L-)WV3`PW@p1uIqH0+t8a8THmt|a&vbabGJ^>q'
            'e>?*K`D1hX`D|{Z21qUd#f>n~g>(K_{eBIU1&E;_&gSe#K-OnJmWl6ob(s!w8nYkRs|mQh8c*1iel<h2W%wVgC`9K^r#ENF^0VOm'
            '+3l}aZyQ*&6=kkFp{cdr)bQ{Y&(-NkccU#uQ<EXhjqM`U+gP7!>e8gSdmxK6cIjOEBJwm)d@TYM^Gx`rbx{*WU59!ml`IiXk09WR'
            '=Q-ckJHuk1i4(oc7*(@s6YHe`(oiJm9SrGGIi6$q*RB|~Om=X!1F6~ahP@EezqvG}%%ayOu6ijt3?Go7<MSmv-jsGnobC}vRvYs?'
            'MpSFj?(qj8^W2p%+ue!4c|iqe9Bl9BZ{fEz-w#wVOA)Mdu6vDzwSJ8=-Ckl+9qub1B!&~Nv$_jj-3~<iotjQ#Iy8PaStJX^YR#F?'
            'Bi#rwrJGeW_N<9XV-k^?`gUw(o8Unc?98JWwVQj^%PRycmLDW%?@!N;O~Op^mwJEt+u2O>xz@f@?CHqW(4z@Ea;G#;XuNZ|Qb!B+'
            'WP0!*ftm3%o6Qb~gJER)V(?JVX9_plejPxM-a1Jcl$9Z?+i*pbfujBz^F}NSIz9Egnp-7vM(r}KC^8*D<TDIT6$*CoBat+sfP%+='
            'fe7ome~jYPC7rStCjQ;A!M1)slNe&|7L`ncT(wmOh%4H~{m8<OJ^>ZCgDJJBvkm)A+`~G7lY7paVGxN`Gwp1)*Jb1GeuAIsJ@F4F'
            '43SCvYGsL2PVw4g<`|MSof;L~Y9?(YAGda*IOxsrU|r9epH-R^GATt*o|<7=5J{+uLYt{Qm?O|CG-0!*wzWy~Ys>D*jd_iut~A-Q'
            'M8E1XN+D6LxQaO**Fr6tb@}OD;XmW3u>#z_fgTMFbVS`Xg6O`&PWSO}V&GN|n}t69o*>XMcvafh@7#)}X!Fo|1lg*nsP)Q(z0_TY'
            'Q5?=EFHiu~9aonTr}^1>aG|r%9IB=}G%@7Kb1?|&Ivl+^nz3F#=!0paVA=lVFnL+ZDC}L41UxS!bs}Gb5Br8j{D3ZJN|bt4<efLD'
            '+lsJ--h2;yP;^F-FlrdK6N4$1>$8~B7>U1A(w~HYe;fn9lWi_k8a)8=T+PuDbD|@vX*Rm`fa&5)zv=_*Pvmr^?|+7`eW*Z?P4qSZ'
            'WH>ESw0_}&Z8Vami3!K_ajI27&s2+N&TDy@7$N<8sELN=Os;mCLUZd}VHrrI?NR!O;}AN}9DrIpK5~~TM+`l)hQ#jO$&rRxv}h?<'
            '9ChwyYT#@{m$53m)j`A2U~mR0HHN)qo%8@0zj3Y8zxA<=DE<uUS77J&)p>K$%Yv~s;*dro#~1xPPc~FOpFF(Hsl1qQ@!(GTaB!_`'
            'Un&V`kN+AqYgC8yGEVGuW3_zO-gDocy=pM3HN3Ox-M5;C9v|vy#xto>z<3OB84t3eqkW4JMz}U3JIA8OGIxu*b?92AQS8`XPMDYO'
            'y!tea%?_T|-bSoqU;v_4vVFNjObNo2<Gnz?q2w1dhy(TVP+zl9Aax?-n<Dd-*s+#mD~<V2JtrU7-_f#mV(Y*Z*ydx;8m5}?EPl`>'
            'NUB5<m}H%}mKFBZPM~^}tYxT2;fO9vb&EH#(Pv{Y!7s1hH8z39dB$2|DB9c2pKPV!$xzzWy@g?6Y)&1E*We%XM6OYIOi47PCr3{`'
            '4Gy}*6loUmn3JLC!e4%=JRC|R5cd0qH~(Ap{T7T22EucWW<tr0Opx3>LT?!I;Ycw-ftJndh#CrFy7xcQR*0mLu8o9_jEC~*P_wWX'
            'JLryTTQt72IQi}ow?EWQdRSbSj8@Fm^6r{_R6RI!GoZ7_by)Y!{#zqR#cYg3>vaY#I}WC^Lr*0N$#&(c+`%^;&dxKhO@D_nle0~k'
            'mr+qhB}*f|l#$H6)^N~0P-jMHrnj9Y`$b?U_9qDYCE~?QyIqny-GATde!DXs83|zB+tZSL-`<2Q5P3CRG(AxD#P?sls_Z12D_9yL'
            '(P=cMg3rE{HnqCiV@%k5Y<|40+-$2Imios&^`Sd{bt|KuJj&P`M&UQ~4bR++5589NS-Mx2%%i)GfJg^V-7YcRZ>(B7`ZZFzuVQ!b'
            'Tiv$#Pc6S4##P@}Vs=AK5iAraNeJCg=IB7>YcvyzzCKO(Ub3Yibd>uvuN7$5gl7+6d#tbO?bSdS$$cmpRqL%>Em8ccGdPV4zD2-G'
            'R-)6?c<uH21DBwcQ14S(-j++jZnkM&pe2enX)cSHE2^hENv8)e>>{0>bg}WEWDLPrKzU@lG$|SS+LEn#4l1Gpl%R7BMFiXNt&A8{'
            'iDD6x^{tu4-&Sl{<{TNiD<f22i62(y$8mj0Pha%zgre`1p^-ZcAu&7s?VQzahx!sOeBD5S>hPxtkM;#!N&!pKF9?vIZ?BKZj0tIX'
            'yn&(fx=P7Cj&}O#@|@kAeZqY7%lXCI^J{mq*3etaGOk75^@3s2C9?ka7VUUw&=I7fv<5(G8vA=$)h4w)CJ5_?21OTLgJO%JjiQQ1'
            '@6OA{zJdO-p-W@+xjoe}pTXhwt%KKVVz9G6cMsB6!)1F*wcOY>Jg-u{9DLEVqb?yNq19#91W^<RH2R^A=7G)-H`%n9*h`Nm&t<r`'
            '7-(gQo129i`#T6x$6M9pHVxFS>DoB$rKa|Ew$go1{nVh@U{atjmGr=}sQyg@MyW^N{j-KyS*%)9jM;w}{ToA|miq'
        ),
    },
    'approval': {
        'sha256': '119794218b882b74fe3deb51c7ea08faa1cc706fe6145e48bc4a40116a202d9f',
        'encoding': 'zlib+b85',
        'data': (
            'c-p;OTXWmSl78o}n1ehBtfH9O-Ad{dT{=~YL?_%RlGl=x^{P}P2o6bvAi%}N3a!$A-><u8fB^_f%4YZY!IprTo}RvZeVKth&$|ec'
            'G>O6>R=YGWPFFh7Sx`i2qSoa$NKS*@E=#}Q^(rXzu-E(Tr_bu{r^&7Q<>vC!)mV-Hb8>rk+dJ+2(>u4bw`E?axmJO?9e+6gcz!ol'
            'd7*czSO<k#V-cO?>iqrP_#+-^RpvUw<6H${Q3i3mSGm?nk6sxL^+E+%5iK!bs6GUF9wjR^%fl?%6;okaFV60=QqNSB6e*KkmPyEa'
            'qh))RQqJ#Ws@m!xkq4H%igNH8Y<F1Fvy?p$m1)6ND~K=d@m9i2GjCC$PEWDnWtxFfvKMKpFfHQ+Hugm;45VR0bw~5CMv@j7sjR?z'
            'B=7+d!n;6a<$MmAFe6aIt!&6qmrjao4<%x=IWcEtQ4bmJ_OpSS6-7=zVXZ}Pmg!`nvuRmGu{>SqIGVFqAoL&%HDq#qQ*i~UtjaCc'
            'qUqRoqS6Ed^%tG>^cKv5;+IH2PBZ=AWt8c~O!;Z1X3HRo>6dTSEK2fnxs1XHRHlod2=KHwTYyBI?uhhs7YCtUr!hF2^`YG?-KL_p'
            'RL!ggW|i6KKf<`oqc6nqT4#FDBi@*=$F&wUk%mZo<!%>8M&q|XouB^eci*dY$-nA9NZDbU6tq2TMRuYW5cdz0>&waYkLvuxhmSYE'
            'o?ofE8+CDWeK)?oJ7W4`|6|ByUkfkOVVW)SdJ|K#+%#VYU<K+SP8QKh!(9wtmW|$gPqykCLoM_l1;iPYizrn~NEg3+i56OZf!Jo@'
            'T8EpdUEJ@_K<X4Sg`{$^R<qC&0&bZ6Nv^^&%SaA}!GX$COt;1aUki-K40@oNl}SPm@9u5~3i_r;Y+3Y>C<|vnS)^M`580)o1Z+i%'
            '$^&<_)3nTcmAqgo1-S_%kz-RyfmvaYWqTqE4Z*t}gNzh(ofd0)NcvahE>43*K2W=I9!L3F=gbss1)C#pN_6p<W*eYN_9Y5|Yo6!z'
            'SPawYvMkC>Pp2x{?$V459iGV!l=ph(8K7wmKe505o~MaSr`;40Ai=z`zeBP3`)itL8Mg}xB4CGoz^}cYv92Gk#?zl+aJTAA`5r_O'
            'YKAxRi^Ll!PXtdteM!MR3@M{uiVV<Iy%PC*{a)`IKnzLO(osZ5P~~!bF}a=GTvwVO2n+!R7g10>9l?<Tddw&Wf?|_75n+SXVU&}D'
            'Mn$gUrSiAwLX*jqS%`nPDY<pd#+{NsB&!&Y)3Vr=1*icP$-;zVQqwY8_1WS;ukuKeIWHJ&;MBBfHKU!}dv<y6VKxLgak^!s0ks0k'
            'jJn|(>uT8vn_IL4)+OLRg#mJais2dp>=-kT({KY62Sot5m={>iVaW!S7h1$kvj}pS5XrB$Y!}goJ*f9$e0vKdnSQuBzn)t0Pk$bN'
            'hO+C8kkVFgCehL|(-2pd&VhoVa*$HZgPivOkWAw!+>eMeLgj=I8S#ghH&{b1fS(p6hrlVtpFOEt!6OR{HYzqqxkE_R6i9}>>+!GS'
            'k5k7u>4(YX_#+wUlea=Z@t%p}bF0E_uowKLh;Nr?EGC5lG$$X&@Y1{h6I)VDB=%^Ak`g6z5TFt8n3qv(unHS2lZvnbMAAUuO6Syx'
            'BcNFMih5*wh=Y=uE!JtSU3yUhv&<!Hz)DWcue=209@kO0R&(rN)5;gfU^C83#1a!q5g3?k76hcrNqy(nmx@vpti?{nu^1i}Edk5-'
            'o;4ir;obw$MZ_TFJ-ov{f0?iy88c@Of&g>%K8SPR*~jsJe1ecKrys}H(9{&c>T+@iQ)5u+Dg5yY9j-a`n;L5+vtYe1ynzg9<x}=?'
            'CCV4xkh@|p1)@+swXoC!9Nm1J{5ZKjznY$Zx}4nnMFs_0rtktSX6-P9wR#J^L~;_uC46ye1WbvQUqjm>3X8rv{RjUr60)8*Hj{GB'
            'ZT6(HN(B^mjIoM=3SopHn*32CejqJ*Iw_N_E>1@3o-81L7^st4i+KDr91frHP;#Sb<J~>pJv<MY;5m}J?XK7>Ll7{P7Z!SNuCG2*'
            'R#-4_iBSfwA382*82kaT(qJ=824-o&icEi@cQltmE#fD;M~7*Pn3OuTsSMDq=6_~ho+=%H9h(ZOWgM(3AK-Kg5JC2DsJ-^b-$oL^'
            '&i)~GJ}@{}yJ)|Iu><+eh_x<XWHHwiBtQ*o;~R?c@qCgC$BW8JvC+JYIgp)l?u9Lr+rM#l$g9>YZ(vT5fjbU3grbrKQtD`gXpNNr'
            'Wuq9eQQQMjAK*Xt52ERsb8@PxURvqG_hfTkpTfe^b6+0T6YLA76D=K5GSawV-&QW9#!<CX=4r_J`NDtEkNd)@Rn}CZ?wM*bTOpy#'
            'Rjp7O-yk5f9L#j**}y-gEagxzl7F-W)0ji)Rf05>Tn3;L20LSS*3zd9*8EB=uR&im5CU&;iR>WFqE(b&i=6XM^KxE9MXYnLnx^3s'
            '>NxcCIN_8KjN=kOql4>_%Srv1WkJ&#YUjwy#3c7I7TGZShCwg|(ffgK9#Q4NI3sbe$BX7vDg;TW{ezJMc8z|`&o}CVb9%`%v#oJu'
            '!Bjz7L<7oA3W})Ax<#76%~Bj09WppIVCB74Dld?by^?n8`K|^JW;T~0A-$}gmsFH-JX25A1dm60BO{smxuYEktA72Y?~FTC=IC$>'
            '&iD=k^#t(zj#utVO}8C2ol3?%n=C;p@F|tmKK(U9RLwXqz`s>m@V<;=<s;cb#T%0U_p3I-TL0lx?g34M?odj9x_f{6cg5BF(1NxR'
            's;MJ~vbfI+6Yh9kf2flXO!{6~EKmRL_4`9CAoTryljqR(@TOFV`sw`l1HPSJ2lZ^`ziY1YFV3%Tt|u4gSL)wyQ3djn6n6X^RVjka'
            'M(=a35%L{U>AY_;h`MQ*r&25-<e`?10PkOI$<Lj!sxl1X$6%kUJ*BI8^`5C2p=El7S{`_o6PH+7QNYVc97t;*JYs<&Z%T3UG0Lf%'
            '$fazisovKMK@={-qkLn^J~HS45|#3glK5oND9|9MD2`A0fm5rwAy!ne05*|;hl_H%%WG7JGzS=xQ`aEK!zem4$q5N@N+RSwB!1Vs'
            '06`<o@-yEXP!t(?sB@vo0C~e1^d%;Fp!*k0#<2E+bx=FVr+&j<F=iqHdubZ68F1F-N%(z?kY=Vp6JDG;5_1Gq04NjzH8+tgT$jm)'
            'zzK@Z{5aUo7s1Gk98#(3fB)Tg|MoqkHHU9B!AsZ}mOIj?&y*SxGw%-tWSehXs(=fBgZ!8MexN$)16Oo5$?xbR+J1%UmT*w{V+VhO'
            '0TrofozSM4R<WkS*XGyEfZ1HD@xw6*<g1pAWJ)i$S_Zb8VrZIDx%y*hx-CU%;h6DtOQ|s)84o9wkKE1tZF2V$J&rPJa)IC6%&sf$'
            '#%!m9&&|vCldG{<mYHX&mo}G6N`6c;z?maOR}fLV;#U-HdYom98K(7H7cWTJBJR)_B?R-hR@Yt^%Gbk{cz!_QM=Eg23Ahmf0B94e'
            'G#Me1z7haba9J<85I2aZWgz)M6G0GyNw+oxw>y2cL1(lPXCMC3n@0#)TfN7Auor;E2C6Ev5)~xT{9E>{{JGx55Nm>GZQ9R$n`B!i'
            'D4A@x6>APE$ZjR!FEy|FZZ6bH$Sl2q@g&MyFv31=`trf{57Lx2=R<;I6@^r~suN4~gxq6Qs(re7haxK2V`LBp8#RZOdWS@#Jk|(T'
            '8{l&mDIhsw!d4^y8a4+XY}76&X)31~{lB^;!$z_Q)Co~MLF5PxKeQVJTKMXQx_@v9eu<E$S5dr(vZD+eH0;h9>^3?q#q@6KO_`<;'
            'qeZ(=bKZFevT|PjX%R_>2Z^2#X=vSL2nt8`f@~nv#)aXw8>qAYu1oK2(*R&)(dJDC(3B_VjSeFnf^_=xZZUFKo$)mA$n<wC)H>Ka'
            '5X7ZcDgm%V%E0f)X-;`nDkfmfr#{Q2=EEI2lgK^O4>h=@0*7F=dVLO+nMx=R9*628%`$1cNhb+TPwjCU&dYHV-z69a1-JO^H2Y)6'
            'aZocA<BWoL&^9@KD&_k9Lp3%m_2E!@crel!rxB0AXd3yaZPYX)Ydtrtbc4!ctG@+hL8W<VqBWAxO`4SXHt4cYN_?Y2>4mY!!2z5F'
            'aj_4+MheAP@;2<+fdDdQ>P_4~c~r_hih#s#>4>R*1{lxep{^eyLVHf3)nu6uWRa9&_&iNx-_9|Qr<9^Q?Tdc@nRCEL?qHG{9=ZsV'
            ')`Z!xgW0f)+3<B{yZ8;?#&1aclA%1j5oC?nJD{OH$E%G_x;&ISAGX7)SBD_$Of!*|4j29Uaeo-)QWU%vsFpxo#6=X=8I^KrzDK};'
            '-Nt1SY=cFFWTPgFG{ie52VTwLz;q`mld~IqOLvkJz-c|L*#mOOxf`z^cW%66(<c4w+;2^--fzcVokdNpA!mhiO?Adpp!Ah;qj4Nz'
            '>y7I4l5fZ)S}*vrhg8Z{buqv*vRzw>bFZCA@gXW6?WO;%!*z-@0(FFwxT<Rg_-$F#SZtM3<+I5$rPIH<)k2$r1JOTD1IA9A(h)^T'
            'j@rr><Z-{*MRKl0V8%5vo<=vw#!9@CO1pl^MG?7CdltW-1gp_Qx__`uAgOltPU90hyl*o@&GpWzvDRR}MZsa0W>?1@Ko`1O<5VmG'
            'Uu9`?V0T25VJbd`9Jm>cd<j(KY_uVG&JXy40NKy+Bi_-=+hO^?z;QE50#Jfg9PAMh)yY$Xp%cr{N&opo?a~-J+61T-Gvu`pO&nBl'
            'ZyS7V1xn99a($@pguwoRfXo$%b}}XvCY_yeXNV4oQ1E;nUoz^O`rU6@1|7lebEM+MLCRygK3aJb^~mR;bnnYECpvsuKd0<qM-Ud6'
            '$M&pJR&(6r8{iS&PGbKhl&jCMsG_$0Y0f1d*K{way1h`HxJS7|uRi&%Gc7tIh4x-NIc-E9dUJ0Y!w;?Q^XqogvYnQj(s*w6HWuk0'
            'o84)7Y$fmM>_y#NpMO^4>mSEgR}*R{vPmXTH(zFh);DQh#xV-8QAa+u^qx)z>SXwP8YRMzIhB3rKM&PM2^~ZRMw@VkMrj@@xgesd'
            'ecPSlZ4wk6$i4R^O26nd*dY6_*iZ#behVHKbp8WUaB1k3+5+9xUIP_4`WOvQt?pXK-vi}@B75JgbvI4k9!a9A`)u>Qp#sZ922DTH'
            '#TM@{qRFj3O_$*eFEa{45)jT#EX5PPM+uYXgYHNgQ?nyH&Xl=u@XBpA+Y)RZ*av{Y-<V0|(|o=MVh_3`8ZP?7j#d-3x0AgQ{j$u1'
            'SU|uWYSj1t+?zQH3A$Wm;@bo&FGK19EX$ZGe8U>S5r^#R+9lZ8!V;*9yq8wCZ1xd=r{((R$i4B%FhLhinO>ZxbSiCf2m@u3YbjqY'
            '59~6&L0aEK?2}s3dBK--ilI6u9mPTqoUf1o=XBC-u4iTXcP`Y;4&Gx38Q0l~V8{tm?!E%Vyao&Ld%hy*TwY*mey(3S;DmPGes8Gf'
            'G(=egN3ISzZA{)C6ik<?$|8$a>q4#q1d?4fFBbi&D@Fgzc40Gbc769aUC$Y&zNT53#GoiZx1qKJp%MX<+64Or1@3T*f_K<l>ln85'
            '-d1IMN9J3GhUz8>$#Fx??YwmvA~Hs(is21#w+PB^rw*$$FBnggs*%6YE4sN#segvBpio^43Js(`giT^Vs5fBYkX*JXS7lNrEQqMt'
            'ku#OgH|oON5BXvI{sxXvg=jv(mo$THNE#^Os|+lBkej!X`fubU);*|al&UVo#u@;SJFsx#oH>**%965@;p+*~|7BkAhK34<YIyqP'
            '_;PY}IT_iL0Wmx3t8jw2iK8l7R^faRk<PjTxEt{8G&n~im?M^V1ZorBC9rc(Hma|YBFeMcyQBHh-eX66w~4Mq_^zq9(%_7~BD~Nf'
            '@j6{>^+u7M@6#;jgVDoW((kJcZ8<qe9pqAtL`tr1-can2@nsSa#l_i*>txTG)Dg!W(jocI?+@ejQD<Q7Z)Xh-?DMw*#PZN&1PoZV'
            'YR(;U_I1<frA1C6<v$%^@P70!-@_P&!+uK*OV5HUjPUl;`Ss5e3l4h(V*knAhC_uKD*2WaG*lDeK{#AQU2u}8Nd^GY2|`(7dwVda'
            'G#2Z4*9CU4zzm*XF^^3V6zpaX)ZhPquO0ui*8WPJ*b<>zHXf@JNuC*edIOQQipHLs-@+sKUP}|}?Av?W&Y=3Ej`5vi=>Bz1SJp9f'
            '>B&iPwMww0F1bn15O1HH;QQRB$8)FkUEyY1rs876t%t$k8_@JdZWhorCZN8n{g`2xq>sM+o?%!PA$8VLpaZgR-)-e^5rB#OFC|&6'
            'rp%GgV5oEJc#qqnF{MqF!<mEp=9ex<i<_>#n*Ac?pk!#23i7#Ax<x5ZP**7PMVY)cteG2{K~ZP?EUxyw;HtJkA<1VR^n=*L$+LBu'
            '>KisLm7DmFU6|`=)?$3;0J=U%>9=XgJrsV;KMSpjThs4=xJ<IpESp}UX1D}3D>!rEM&*Q1)f(@J@Y`<0)}I08>h|vZYRpP-`yPBv'
            'maO1-rBRg_brDwIHKZe}^Q+q%y1ry{ngyj?;$rou8)}vE#}IN!h_4pVRRQ<%gv+g*a>0MO+B!XzhTj*ykdT(Ewev}oeAI>Cx4DkV'
            '3`Ma<c+ILn?ice%7miz-%SLi{*ZJK5%m~w~TfE)PSBkn0MW?$+>)m?|sH$oT#??s9ol%%c-QEIgd3w@E(b4zXT}vU~>$yu+Tr_&6'
            'm1`ZG6STmud0ub~URouI(0*~-tuo}WNrC91jw~m2Ky~KFEv8n8lEgu)xyelVm=6^c3V`S<V&NV^KcHf&ck~$A6=tK4ZGHHSjqBIj'
            '(w$b3mqD_LT3OXv=fO@wFCU(O%=3#@K$ne$T{j`L8tO;}U(=MyEPfncARU?f*T^zRN?5?ZCA}gO&bJ3i`qJ}l?vR{4SuUzv4miP^'
            'KC`J_o#hSHXOpp+>=R|DA@yhU_>KD2!H6=khzd@xZ|+q68Lep{3GR)WxW}y(LyobtYHHW%24J=RioyA4z!^{1_5axg;%nNiYs6Z?'
            'f~ulXEtzr-4F_)N=(JXer_D=MB4z6BF7ffA@z{a;E2t;X-GaP>&2uKJTu(EQ5mAQEa1lq1>&130PLeu{!FH<jV)M6tYumgp1oBrK'
            'z#q-W(1r79srv2w?&7EMrCf^?R^__O&8Iu(R!dY=4N|^di8?2@of2+p7@{VHmM_J{a|rX}6n0%*5VWo$^{@RHQOw>s98nX@$><_7'
            'b!+XR&a5HNnYooo-`2}rI_GP8{ydZ%m-{VhCQ)QYA#2u*WBh?pP@NXZ64GJhj};~5__l6LQX=Di5y^*92cO~}?RUSkHM|4+QS<9_'
            '>F-uwxN@JpO6A1q3=GEf?I+*BKImyST7WON8=YZwyV2IVRnKux_mhtG9A9f?SoCU#z?XZKsyYMwCv-C=*B9ff@h@Zb^ZDn?Nuxq`'
            'dKtVy|A|hAk8sw1p^sqzMRa(yZ$Sr%PVeHNLaywjvqH9S@4be*w2M_c?66pM%)uMEHq<rde^;y?5nX37`dW4SYyz&hIQ9K8-yUkT'
            '8m<*qoB!rP(3&m^O#vfQu5HZ~`eq?pl)G5p3vScj4;8dtqzQ@*WCo!gF(q<#{=%c``qrOGQ1llb<W$(N|1W@6T}_72my@kU`k0sm'
            'V+XNlKH8GA5iX78<0&~}q0=JgKM=_!!>F#39DXO@4XQ6u&g>@;T^|lqn=#*I&`y0B)Hnz|x}o|7Nw(knA1&#y3j'
        ),
    },
    'google_ai': {
        'sha256': 'e82125022c0237de8bc42f08ee360f4edd0c6264a5cb32a28d46ece49c8cb0e6',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$q}k`Yk9LHt+rCzw$CgQL@n!&gVWv%Wr4gTv~MpwzfoJoBSc$BW?`;0ka1'
            'CwWKjBkCII!uQ9wE^2hxr}wuDXny{#-#>U`Pta<P7V<bM_ds>*l)nMwTyh>4_TaF8G92`dToBY=4USOKXh=5L*isMOliqP3hJcre'
            '-@$DLZqZB{|9;tP|IKTE+-?8yD*x$r+E<@;ThE_9{GSFKu7lH~-b+{R_nZFFySmNRC$hmP2q*)3OXLG@n+2M>H09ZH(6(~}^au%Q'
            'Fav4@Okn#5@v9H?<tF$TnuMw$dv9Z--eg~FYsA_X%{zbP2|vY;OyG@$W!p|m9QrqYvTqKT3xp}D>q@Et(Col!kj;3L1Sf~Y)C(uU'
            '4RQgVM8_OyTbZl{9CKJvu&tVrn0Ij8`xPX;XbIvDoOh6W7I3assYmA#F}`!-XTSlCAsIk0?khCY5Fgl4U+Im<pa;=2lsPpaTz8g='
            '`yw|$gEBvAC0JISJA|k)TM1m$0Aqw89vTCj_!aoT5I!IdQMN+K_wK0%(xbo}gBTo#bAVdD)CVoR9DtC7ij^0h`BN=E^q63w#K)Ed'
            'VB9fq*gK@=iodgxvCNd|LSL$Z$dTU0_mPw*t`mQefT*x@4O%_@%VeJQGL!*G+$18aE>NWHrzK4}zO+TNR6t6D__y=Z6U`g=k}gHn'
            'YphqKI85k!GrqAw^NLhVT4Xd7c*`U3wvucmbhU-{XC%}>_*0M6Xb^U8Zjo>Yk(aC#mjY!k7gy+q>!x9xrLK$WN@E7dkF=NgfQ!)5'
            'fBC74*I7#5oB}n$pDw`{X$FplqMz%+Dac&6nff8<yEx=N8b?|pAC(@6e~<v!64v-F-m#V?572+|tK7FM>gl1AOIbLF4o(3&;6L_@'
            'd_wR#|D&RjnjN&uK>P5U)>FssAtwrL)qS3u@31T1Hs!yCz6?+93je|eBa9fj{~vS#ts+}#Smzh21Z@(k;VgxQn910R51YmZoVpR|'
            'Z9Z7OP5F?3J7|4PKxN~T))A+87_Oxao*4w+7~z3|gS7-XnlyFPpF2ucHs)tcEypA}Pu5&Y>uTf>XrSy(;~3Y>J{s{GWNV0S8u`#!'
            '?2=(}?BZ=w{tf)gTIsnu-7S3#+o?B2!x$@-WZyZ6frx;TCQeQj^+CG}9`vf!ezJvKI(7RwuG+ZYATX%+3=^l&L><U|3jqs_j6=O&'
            '#Q&<MfZfuxPd(_;uY4kmsXUlO04b;>0($p^_zq67cQg0)Q30bXe$mD&@jHN25lYQz<4+c?rxV=$q~Y|y7$ToY$UGEO3H)!W03@5r'
            'zaIF^%ZE8I9{#N88BoV8A=;=GWG<13s-|=x&5gfssl;vwR8oI-r99^{Hn|`$06H~Mv?ux}Nu1(EX~&Q5f&@?zB@1k&2HJoE@qpF|'
            '1DyRuyCQgO!ylua7gZZVZ$;xkxbQ>ftHcrUl5XxrOJHTN9-oX7`Uoi6hD=or0PPzM2X+q*Q`1p8H1fcrB-@!TLl6qG@okf7SW%fK'
            '#jQ$*Hu$Uh>|Xo%Gmy*@P*7oGSlB@IRja7CUIKdx@4b0TY<ITdf81?BqSAJIn~ZgHKMYKr)YuV8$!H6tT%y1(x3p<6Msp+dU+F$k'
            'qY6ft|7T%G7s)j3W3_~I#`8!)UH~elq!*qHh~^+#1vhBh0HFgd9lVPu7|}lB4Hi#K%1Cs5t8M-)Q1R6w;tXuyEnj#MxKSqJCh~4a'
            'cvA$tytknN$@8QmuGRYS;(w7!<g*bjRg+9}l~)Qwq^n9EPOKVDK^5w*k~%;v3}ZH-XY#V*9Plk)tZCiRp!f{dF0mP8Te>e9>(`)W'
            ';|4o8aXm0KjdC2&yYJn3Xv-zBA|;yK#A7sb=*Xe4DD=QM;-?44sXrgn2AWi&xxWHkr+hs2#V0kRe@Px*fLwVuUS{xA&*GuMqZ_=o'
            't7m<wI$u#61?euDcvbV3gOD+xV!UF4M7*1&t^Az@Q#CN{{Hj?1vnR)ax71kC7y_6_CO+=Jj*v_nCM4tv1GccEn)nf|Z{o%WasZDd'
            'I(ied$0z;_=mSrfPbhAkd17&D)s&t((L8aA#$5BMX31*bq790=N?t)7aui@CLG+cbP~De=P4zI4;-KABQaFpo5HXX2d@Hviu_Uk1'
            'qN@ODQOc@uEVV%WSsukEuVu3OWz><b&f*(%t5f@_)G4~trwtk3g1D!Y#WIg}=muT+MT#aX;n$}AkN!9|-@YabA{`h36fzT&7)YP*'
            'u=O#&l43f_x5BBo)_OO5;u@!j!i?w%1CfZT5o)K5RFPOlsh{nH@pzUi%y{WfzMtfv(!fJAMRP<$`to{New_8~Klw5yUV3X&%KXgR'
            'Rzng{aF41E^8SVdOKr<CSprSM(eP|GM{wZ$$d8bEF@?0h?L0XDdmO*y@TsGwC73a2jT0AtBeplyj|y?7Bv<w(6Y}=ZRuje8>4xhA'
            'dTXK29$vC1cf^%%=_=bB>$^VkjOhGMo}71EXG$n3oNB4`9OnU_Mb8Wk0}H;uJ!r0J=FcBGnv*=pr?lb+b!WyJ<;a>nD>eCpeS6DF'
            '(A_rpaZ}q%oo%rrwxt$qJZ}lxa0*j^<1-1IZAscr=!}n*c8gm<NL0G>%4cY3-(}!6%;|kHIvL8B)370^r~L_K@Xmk4LDVev)XM0C'
            'UV?(E0nA%JT)60SmNN+(fa+8ft>2nxwTkvsvURh?I0$c0GEhcNfyAiO_!c)+9O1xz`Y3xhD}3=^pw@XCOmJpwiXpN!TB7kiO_{?v'
            'U;RePyk5>R^_aJ?>WUeibLCD|nU88Qza>}YqD3Z7R?Ob@Ie3X<A)T?jk!3ssc~@cVgMmg;r}1WneuR?1we{6&wC794M+Oni2E`(d'
            'Lm588&T~whq#M77aS&dLCR!k1L0jHRNzWmDo1EV-`X7)4N}m|y{xqDF0NCgCu{-u_M+qo4sG8m)z$+^D0=EPeban*EBb7JgFy#>a'
            'pLZfZyN{EZkkbgFJF>Kw$Us2?&*9rD(5)h;AFdXantKIKzzk<3blbKeV*ZI};>Gs@wD-H^$K?)!B3%JsfwpEO9nAfYPN%4}%b7n$'
            '#EcvJGH!3#1}Eh}_HK|<gjGE2h)<OS$@nx0M-LTKw0BY6DcH7INx^D1-1v$i3(#?35G!9pJ}%-r`8Qxj=zR>(77$AWiF{I45j#xM'
            'X@y|be^wf~l<2MwJN?sT0i@!1w~QS!&%1`r1~4fzEOINdJ8ZtujT2ve8;FH5C|=Zsli-EC)>Jp4=vX!P(8UTJ!RO;Z%$6K$^08B6'
            'N?K1AOMtTf{1)uz*b3_c76Q+%lg0dc=Q^YRO%@XlCa7{7J|{Vvq5!2=;&#dbbm$qVd`Xv4fZ<;h?KVcIjB}6%(R$>ki#Vc?qX>`^'
            '8X5lA((C#jG&9hD*omHEz`heOrS2}d7BA-T1M!62h{E88tU*#O@|IaV_jCl<SLcsC^0Rk`{lB6MS)&-Z(@+i0hi4ZD!wZl{u~{Ar'
            '=5jEYZuk1^^y1`@TS*jQ8=R}733S`x5PdcpS(fTY{o|M1*9!d6uY0dS8phY&4Px8aRlx&FxIzzxar65Y20WrOXA9xm5*hvvfV?H_'
            'P?@ybbXu^Zb08(%2D=hAOarIsHP<BvJ3<IQ?KT?lkwwrggUmLCe=__BcutLdb!EqY8y6OaIO@X0p(Reg@^wb9v%1%?Sn6$Dq%;a^'
            '*YhYmH|N8kftLKkcw_k!?(=<~`OF-JAlBGt$QjSsHvs<Ec*Y_FJ6pobhCGw5u{Pye!oBbw-i{W~z9Dp0L~mbUpi}$rOAiD5q?mf='
            'O92Q)`Nxil^h~vbbYXUkZn;U6H?-R{PE|{kPLO`24vPURBnLkd!ginhxa;oj?h@j5pX}ysN;s_JyjmRQ)zVR3Eg9q`zo5O9)KY%@'
            '@4gh?;+JxZK?+>XeKS^-q{K}qVI^EBR_dDpXRiu6Kkj~;o!<KzEBdcr0PpX?zP!n4tFSSl!^Tv&|C7#-M#LJC5l2+-?~{K*yk83p'
            '{7<r-x)43O0FZ&g(VUI57_(OqlGkZe;>eS)gY`um+5J|aC<wlX7k}#?4UhZ#h#K>t6un51j^s=yem#BLLcieE(uh3J11S`9jmX3d'
            '-IF~uLZY-Bn4)n7x*qNelx>;|_M}IX4&FVEgIN}cf_J?mHlauA^n$F^mZNU4LBsQwLlRK;DR07_u?Ks`9#~=y`f%6CHZH|IG)hRR'
            '7D|wQAs32E{^;dVFnY<4;-4H2SNlI^m5dpHoq5#gmr3##%!~lo$00?+wSv{&t~{7eO=!{2d;o-wEr~yJ2z9(IXjkXM{@bJe$#MU0'
            'k_s8|vY>{>055^TV+lIostDY|S)BV>gwDC!IPkvcbUNZ6_`lm|3!4S25FI2w7M<}c08go4L^EE%d6$9OF$dD-!BCFa4DfTW{jB|T'
            'Z`2O54XLEUPYLq9=LX21xzF&op@hu6GVUoo(@n>>{@k<dM=!{aQ$aAGbSkh}9RA>~5(-&h7?lnlt>ww{QQLd=-+n#3)2~3I0NwB='
            '@$OLCx0I5Wh!>bPhFpW{w_Ccz=c8zfwEJWG$?lWqkAsOcRRyQ#3QoHgWin2hf|l{kMyrBg6~Ty?3#$~koA)1{MC{-N9EnAMrV6G|'
            'H(r9n!WkpJcjqOkw+cW?5N~_$dPjpwDE`e);x-$D>6`?ql~ib7np@DbMF*pxg(Xk6A_v3&`hPt*x<I2qQ@JAhAI9O5LTNGfR^<h9'
            'TG}UPiQ!wKT`#}EM828yCbvs&)NTptAUfzELKR!IOOZ$~M9lV@83_$5PxQ`@dg2DdAV1ddJ>>2%%4hKPI>BrG<SW?U7p&*er7#Nz'
            '8UM!rJZkb7)PZ9xqio5+s372dy;&3Z|2xe%7nT0mXZ=7yVZ&R$BHvY~E{%^&Fa@Mxa5C(lo%atgp%q(B_lX$25S|q4*=LY!u`2n~'
            'h!W*^+WJoMlG?(Gce)*EF``(;qrv%5yV2X0By}-x>!ofKM<0PG@|_kZG;#yu(l&Q<$E|<??V;$(R`s%Od9Zoua)5(*O^0-7by>6z'
            'UWq@$R#2Lc^6Lq|mP-gySKE>~^3+wf=CzBVNEQY)2F7Dbr7FWTJKLfI9f4i%F|Y^B40bCM%pj5x7}um;(~cB!<!W|#8CWp%;mYgx'
            '=3Cq~3^BzEtvD6UckpdrJasza6rDU(7P~2|9vwW{^%GGwT~f%@bQy|ikfa&K_+p6X@bukzp1^>!$jm+Dt6~u>@FrimS53Vr#e?zH'
            'ZvcP=4*PBoeNB49!7r31pls`GRTvFhbq2FmNlbR+XI{E5FOJ^YpQz|sVX|sfimMJl0|m17?oaK#pYXam1XUsk<7~Mvio{Z;Tg>T<'
            '4E23|dUkPg@$2as3<jDUjB=9%;@R#qj6z<n;sh_*8&NFKFBh?2d%k8d?9M%f(WM>nF7y!}mJ4+Gfw5Nc5`9%B)DGH6#bc)gTufOo'
            '<F_z<yh=uL3c@NsgQ5I6`fjdR{G~`h#C=bADNJ}3hz$M#&Mx9`vP^Kwq+l(hWfCrD(4UqhGcSZdt|vGkg)@g8@lsnta_btUK=i~4'
            'h&_C`Be(Csyx-XnOVrjM%KM(?daFk35qp3duCMQknpMhhOSOBYFO8aOC3@(vHSC?gb*Y;XOne(>^O({kX!)#oWF&Lu_#2h;iz8@j'
            'nqs6PDK+EOqr=L33$jXN_02}haORS`3#MbTzt6``HoOZG56B}DasWcMrlrb7QMDwiVV=?gX5f3#a#7u`WLneBa?DaglI_Pp9r-*r'
            'BIwr=6<Q^=t^65@4N5L~=O^tZnig7o(j0A#6l9I;h~5ZuaER`o(YL@hRU*0P1RaDo!#9JIw}X?{?ot02OnkErfj(7TCK(3Liu)P;'
            'wm$r#_WFV>st;?h^xMp5!wFTMg>N*Y4X0E*a^GM!n@yzpP=BiLY+SF@UH#}b59<lSSN=*hG5N|LsU$>S`5RS4tQtY&MMf2N$W4~&'
            '5ek(cb)z7dXQJglF=jkc0yyTQ)v26iETkkqXZ^oi49@z8E^B;i$&aZN))R$ttICsFZdN=!<rdXwU2a!{ndOGn$W?Avi67-=Ra;yz'
            '8Ch(~xQeLYQkMCBEt9g0hXv$NzUHhPw<!^b82cE6a3gb<QXV^yWqa>?zlxVQPfqOUVZv>hw=KP;RH)dSGe9GjMHo05lKc2R+WVwU'
            'oN^Z_hA8uaBhy<br&U(RG3CJA2kF}TAl-pDO-XvqdLX)9%ESU0rHlj|=WLy%{^ES*apbQAYFPWIL1rMjjwvb+M<;9WJeagOfkBJ%'
            'i>J!D9=4iO9E{_Xk~yCD4^B@GDMy|E_oa_^21=<WgBX-GAWtW0r4%FtikFxwkfH%NM>Qv@!<&lCOk=04*fJhPWdOy5HUTC5wuK4G'
            '3YDZPTU?q2VMq?gRzC1D9J?}|I3)~A<$*u${nZ7T=H11RQ@@-WZZDMHNc4*CPF?0n`!b{m1Z9jE#@L|+YMjmg-1~8NR}9b4Yhscr'
            '=i2WC1sefup3WrVIK0Mm2?|-V+My(Ms8p$JJ{5@5sij7H&)ug#JzGPFtOk`d$ZAm}KjupXnwMqJFqOvxk|)n3^TfhiQR+}Kadg%Z'
            'u%Fz$KR#Jcy38ikM9XYZNv@0rYB>k{x#|WzDB&Qcl;*{7npi+B&Na$jib14P&2{RDp8VE>G$NHS6GP6Md^e{|N0^89A9f}+v}zdk'
            '9!a!h*dw|d3zorUatTiErjHZaj-ptFxf2a+0;M&s6M>izDY=PtWL#|m=Aw0}`Ye&P4!@2zBo&Q=tz>+PG#?I!{o{9})jpFDva!kr'
            'PfL^yZN+>R6!?b|<3nUX%^tZ$R!(JfBMXu$Ymj0pannN|uZNV-pi+YaEot(dD1$dYuKY<%T?ZX<aIwnM6;-;9hE@xseCsWpG)BqP'
            'sbk7Eu$yuqvAKUz6`QIbY$>H)C5OkBXgV^Cmy?)${YaQxdQ&AMu+GHXi-IZg4rwxc^7R8<awI(&yy~A1U5pmPL<NQ?ZZ2E8YDOJQ'
            '57#L79uAM(H>Ve8h}%z|b(9s^od$k5X#yi|QH*;_9mztt%V99i+HC5i9>8tl*>XWg9`n~TCr?#sq98E?Oj8*qf`f=xgR}FYvg8Ac'
            '2ci{>_pRLfI-?w-@-@5kr`J2z9T4E>^ke3)(rXR_&9WpIq4}01lIt5J%~{ga_tBK%Ky33M+s-xcLQEEuVl-Vw1yl&_*qA*+iivde'
            'HlC=Q-l{f)CQstdN7wgKYzn}j!quOLmrzwrjSv<BJ%#C)I>9TQZbNU3V@E^qYRmhlLm*fb1kuOkLr3wHiFaR$>?;~m6G^r>+g}*W'
            'e6d|)@Z8o^h4LhH6<Wb(KwZLT4NgLU8v~!V?m;pc(cDQvUV=lu({L`^dsjMP2d@KX<!VE%idNwH*l``-Zy4HPXXAkd9Q003PX-6Q'
            'BXMwg3<u|MpYsuuWLtBy#aC1;ROXe4er9}Eg#XiaJj2jJ?z_D&E`v#W1rI18#kTt4VqnhpLkG@@qBLHJ>=P%z(UO=ro>CU8+cGBh'
            'd;>%G`FzFSZtDxP`&+4alB`N9rwp-@wQT4fj<eaTOCHfIq*^M|86Ep|0H<s@C)r=*au3gAzigpLfnKM?#W~r>$O>}S`PRBMkvzqn'
            'HaD4(Oh!T1gxX+mIkYGpyibjI2Evkz4`}cj;K3zqT7&kgK^(scogMYam06aIWH7GE3-wT5oMI=mJQ+>?srt0eIiti}C(XpjI18{3'
            'PR$b7Eg)Y({#9cod_L?|;IoIVszCBupxK^YwGgzMX=uuDZH0359N}y2X;o)6V=29_N+Z|_q$(mo%h>%z6lJZQ()cgm14+N+aG)cj'
            'R2y+lBNrJc$6=^k(3^gsFo=ti9Qxswb+J~p=HqPor6|7NoS2R{WY$h;Xqd;e2&1G0MFm+2F{#yc;ky5tzMsSk@iF&vl5T<RKg&Ea'
            '^98C_R2{@m+L>|jOSBTbowQiJOp(p#WR2yWA%~l590^r!q-w{5lff~pH=Tnf#YvyN8;okt#OcY=uR3CPkp!q@O9BqU1y*oV#Z0Is'
            'QF-%>uwb13{cRlbV6b7pBW_fh#~hgBTYo&$;l6l2skDf<+D<V6v#f7|G8zr5W`<GR7{o?(PI#|C*0O0dY&u{h{x7stWhOfCcT<&@'
            '=q$6bRuoF;UKWya(Ym_{qkkkMaZ}{Aj!b@(ODqIVUiDN>8e9F%obYW<;Y;v(s5_1u5b&)Mt09J=S8=Ikc;(2fD!HFWS6t4kriGh2'
            'FFS8sDQ*!i(dYnp=Wafl%u2O(#M~pKFqrunshL(dkvKdYpX~J`RQHw^n!(gS0#1=H?omjS)sXI}rW_+MokcdOYL(GATXIAkglbrU'
            '?t#80oIP7wDzzP8n~PBDfKsdx0;e8H7^(SQB!^=&<jCiQvpb+yMVNIho=!QZD~RkjH=KfJ8DSVP2<tM^UN6j5&jC;r3TR*qApMnP'
            '1}OK+&r2n^i_Ti>LCzTJjxd)Hp1KE}EXVXDQYp(oUov*n5&u9yHF+A2nYgveIB}^)ywdH=ujj9|Z8lG$sWMr*c~~fCv?Nb1_3)}F'
            '!;~UKs~yAYdu7$Vvh}?<p0wJ_tkA0v!O4BBhs7S{z*GYg#g`?B=onRQFaInFj&{5%x3wM?d(^qhLGMkR>59|LNf-=ZVxl2q+dWj+'
            'EYFqyx+-Nil`@Hi=xmE)$Gkw=IN7g-Z6Zy}c);;0%C=XyNW8?cuFRakl$ir0h*?0T2)d-^T;@%RCyl02%n?PYM^H~=;3;=-m)~e0'
            'w)FH`k~Fj`$cJMM9!%BvCBdP%iVjv&h$aPVrFNLMIJxf2>sDPqF-J7^lq#s;_b~vQewgYO?G^*IOcWRz^4cm-a!Zek_{41)D4ai+'
            'H>N;lc(&Mpe-%fO;FVOUv7ZCHob-2z*ogGp%2&lHjiXr3g{6>22~RacGgsZ<@DJR{ls|75i7^7hRsC3;0!w+uH;mKedKrb9-*<vE'
            'R$0=`DEKUNZ-EZsTI$W76+`FvW+=w_!^$2?;xgDFKf0=Kt|YQzAAW9j%tcR3v2SEGDP)r_WwxwMgHtC8(;k0PiJ{G!3#~jKbYmG|'
            'IqIlkN}vtRT@^MLxVclki&Z8FGy6cb*a+?GURCwZ#Rs!v{?zDWBDW!+OF_YCs9TtlfQf$?BFAI^PaFNPb2ym#oRZefED}|@l@R;@'
            'DXKwao(nYcx#(NX)fl)^7+PAYDufyR^5bq7jw-frHK{8^0<QebkI<`C9{WJ7U=KsXX!rxgDTP?{H3>otS+1((Bg2*EAr`YhpHwVC'
            'mb{RzzJ**XRCsLU-GtC%j0J$oQGt>gA(N=EpX6ekoc3jDLy&A^S7i;dU)xaFO#iIbCT;qw=F?R_T7u0qmHL0HI&+WetJ58A%6uhv'
            'Sj&T}ZZlC=Q<6ZX%2v*wukiy&S1PkvZz6jW*SBV?zE}{bE0O>zTwTR8(fm{xM&<vj*A$iTP+m+Wg71oEF>Hq8%0K2`DngCnUjBe('
            ')#Z3UuA{Y*T)IL-V}3pJQ+zEG26!0NZrzK^3sS!IXS8ua>;-y~-Zs?}nGsyu^#CaqcijiQG4_-zfJ_})h%J#I)slfsjI0fNbct(o'
            'c4tkb4162oc+?E&Vm$vN&&0(Lu)mT^1+6twf%Ax6UR$@&IVu-Fa$;gH=E~VxpL(^lM9V(b$vy_rG_JRg@tpGZiBFaOn%u8jHotw2'
            'sZJF|z}9W!D~|Y6H?JYvB%pQ)7n%TMM_<sz-oyo=hE3~LHg&+XWtgQv2NFd=*BjWkKHqr9<H0tKQS2t<{xTc-2G;oelx#ZY`or`E'
            '>vgS?HRYpeo8-MQ)|<}Hdg4?4*K4^kJCAr3T+7tCfi5|Ows=%Whey%?o%mQrHlD)INL5`L`(c=S2PRk~I?d#jBpI*u<?&`^ZMN#c'
            'fLN}s#7&vls3)|hsLxi;U(LL`2HHAer#3}#t-pp&S8UnLEEFZnA7$)Cij0eN*5rL2^FDT|5@3g1*_zZDohCTjqaTf9bXDj!mf5uZ'
            'Q^Rp8MsRa@H=F#3X{VuvNCzGE0n%V}9>G-T^B9~B<-S>*PS>pJQ2D7c2jemLOv3*gjH|1EndUzm6d)Diza$GbZiB2`;YH<mq~;qY'
            '*(Og=@tqjQNwQpI+!>E$;uDoDs?m?$N^X5!Ikf6ptKwM>RO5sg@^lV4yzXYuQB`9X5fu0$u+A}IXiIlnC&sQ3#=p(Atd_;4$Wwn^'
            '>XfPKG5eY7OcZ?mM>lK5>%TQuf9lRt`a=S3J$>~`G_TJtPLzw{!Rg6KUwO8l>jJ^olv9^{zM0@U9nPw4Ug^nuycbm3cEyEI$n#s>'
            '!`e4|O!92QPU1zmKS4(3EZ267FdYsg(Jb8flg$%#P+bHk^~V8+NCpx7k};FMbRY94oZ>WF&at>NG7C<>jegG^oiFg8$#I0ir2HQ0'
            ';Z!4x*ns;|XA{eAO!$E)y>A;WiR_!Ax+e3cMI-kBCGT0Qon^6f%rXMfQe9=PRo^!6gG_sN?s|&8oxvl4?#`9@SrRa+tkVe7OAXtr'
            '+}4X<!n~REL2db!_o?ePsxBN&ZCImF9i5LnYH<%v<>c^IO&9$xcynYywDj#8Dbyvqyx7a#tF7W_HsW2?+r>-Lg${w}mVAwhd9|7)'
            's8GXhZf}qcG1nGiW*Llp_0;{;D0!lzv-*SG!P#(E%crV(n+*g1nv48AwNV6UF<)dWl^)gAZ%F7;e>qOkx5^sxp}s~(iL7jdAYUI4'
            'b;RlSIBR{G!<GRsUC9nt`3+pXx&E!~2_=3|Yn6o%*xmaNU*Fxke|C0y=AQM2efMZ^JQ%uX?~YNVqVq<!P`U@MYIj&_H&N|g4vvn{'
            't)Hs8V>2K|7^;DaI@_Do-7gC`vK!W)rmW1%>{dNHIR%vlL(oXjW7m}!7%0{6k!{ZVXTS8%D4{fEaFN#~^6hYN+y@#@DWZe9A7hqj'
            '*{}DiKm3*Yc~hT4=%&C{LiK)d@=Nb%a417Muw4+5is)7(uI4U=-f=uQKL;RSw+e14+h3f#Jvn`kVV3&VgUg@e3Z#b^S5(MiEc7Nr'
            'aFX#d$ixUVubH20r&O`nA1^T-aa*}{P&X`ZK0NE4oDX1DE`Rgw;N*}pD45V|2@6_|ISVdn2U_^}Hcv8wyamfjNq3P5!*6JRI$N)L'
            'M@KJv2XBktqw`dSajmMShw7<DftE2q<*e18_QtigU^S}j$<g3#-+hM>H)kjLq^wHfy}?KCa<~0QE+X#!*1r0*x7&J9?ul00=Ffri'
            '+XyQHTw=qYLE8NXqHmnYv-^*c?<Icnj}L7mH*ILxhBnaWxAgnV{q7a~^V|6kwPQCgfBx62=Fj^y?jmsc1C#}Z8J5#^F8}>se@lOB'
            'Y+wDbW{T;oZB#`!*W~B|)aM$jPYJ2Ha7A9MDv!%J!%6}Z4zTia-Yt!zbVexLvXxt4sK6zAVbWbKg=xv)qI4ByXeA$$6C`)XxTN#`'
            'aSsmD0pQcGpxeb*Er(cP?rlIG|1FGWZO79#h`~292ne)a5T>vcNC!A<aC3vfY^nkZ6-fd07?nKA;Ee%KGLa(%p)w{@HE9JLA`xdy'
            'As8|hWnW~J+FX_rn92g8f(eX_u1i5x>LbO<)V$QCw62&UKTU;xYWY<$l;Rl6N?LWZo3fBL@feJ3h;AG_v6v_kjxdH(q?mh&8z6&}'
            '8sYmmnd#J)vikN8U)AkUDw=eefD*&;xIl^&|H8CcadxXKtmFkSWsr^$o<nxL7!!0+F%DG|3wo7eT#QR+V)y0EC>!)m8ecn36~@;M'
            'WmA>9A!}5(x<ShfV@vuCPh^obGAq*14Sd*;rOQmCBfph5sb(YAQIRwQgAolbGL<{wV7Xc^$=C}Ne^IqfOA4|-Au&M>->RHJ{#L7x'
            '`8^_n48LU9bi(*v-Z9`zJM?GM(f6wa37prn-MyB0w)>Qp-dG^|rgYy3kPfI$cBIF`7Rv}JF%@{g+JDyj-?j>dLo6bIwu+p9mBy20'
            'UtB2!nD1*g7X=z?8Qm`BgTm2rDW>?HdoLA6jl#9FmJ~OyXZ`r3G&?QL?8zT#Ow_;7zs~hP+<MD2s{k!DePm+n4N>fe;v!$MS5FjX'
            '<6?+k>SExRtHegWAeF71$~u7+f@j^R6A+MAn~!M=C0mcTIWA!?=CT(*?mof)d5Zt@jB%L43nHMDEU>PJD4?@)PcOk0uguUQU$Is2'
            '#SM^x#VIv$EhrZUqx(2f^=9IG6ZG>%Hvr$CY=ntYYSUk3CRrMK#ML}Y&w**)2BRP|(n_y~Ah3HGyuJxUa>ijjmlC`3eOQlIRhbY~'
            '<Y%;M&b?%Y`Tp`rbD44Vv}w4l`FN9x*?jS*{3Sj7Qg=a)IV;W*!`2e@PPI0?qi~+`6k>;E6fS2p+(P;M=<C%-Ph)MaM{2cW>3M>E'
            'sa2$S0<`5GsMhVu%mV47gE`$7t8(1%Bo~k*2TI|G7qNbH?*$}0cmm`ZwDO_UgQ8i8**+pn72+sKlhc?aKA`am<w7js!g9wPtstoV'
            'jy)3Es5~bMhm-~og(j8~m569Y#dGvAxrkr_5%Q&n<@(ZBU#4-C+L3MX{vuq!q|5kGxg!)$#XnK5!>lIeBPA=FIN(R*A8!n>@IdIM'
            '?3sCcNycS)R<YYC7T94dGTw`A>7q@K8vBji#&+IGy<~$ysforq5D-$lr384pq<mu4+86|9mNTkj#oxpcNtrT@Yg?ks1mj_oRl7$@'
            '><;hZp&xoHR$r(hYowyTju723X+blPCD$Dkpx+FK@5Hq+(6zV)f`@gzJiN%piB51#tP-X_PzV(#X9$clhtHwypW-fpi1R7nVOIUn'
            'j{9^kZHQ^&-OQ=}TcqPQ<N@gDFkd^>h0I}_dhGObi&=dvl;e4)hHPoea7vlWrMQOdQZIQH8dQkJvPCK6oNBy-)P)G%FyPD74gd7i'
            'pP;>;{o3xm0?oZq;MediIf(pD`NFkia4r!2x%Z*D+rfnM<YFSnxrAY_obtW1w4gS1dB>J0smaCvs3NGYw+z7JMH|SY7tL5;6_dO#'
            'Dg*e-gtxi^ftK<Dgp>W`dtmQbPAKFy7j`E9EVS>))^tsRgH(O9T%nMC<mu$%o!B_`QLI#FS%>5sL#fua!<ixc+*aNTqjdrA4mzB4'
            'rkO~{{!F0#t?{#AaB2M9{B80>^XL8FI`FUavy=YZFtimMWuUQ`hAw;i5t`7u4r=v_=AL62j8*Cd$ww>JzF%=pD0KnQo=hU5Q`Q>`'
            'n2IAgbt55fHl0y5kmx7C=-&Yx1z5n76oK`B03_BHm0__FJaPokNirxugM3~j8f!72;@YSRCUOj?bmg(pfi32fDSI+*bP*x1_3?%&'
            '#M<qc(5MTt|8B`{GbB0=LO^c0*&(GJx}dFwvM-rKw<+fW+jBZ9>B9sIN{XiE$jS(qOv}joBWufZ@-j>tf?UH(Np%b{?m&CJG)Ktu'
            'R>V74!kVMY^-3I_p1wWjlpG!s+#So(DQWLcK27Pda&dM9+zMz!ZkN(|TN0I*I?dvQvQDk;F`?QE&PhbsEp!_9Le2&@YrvpYW*Cx6'
            '$w714_=J^Y=xBFb3Ve0lhkfx09z0Z8nCinCI6vI_YPxb-WXe#ElEp%?fc@Wp|NUA2u!kAx;b%j|{MOHe7G!^a(~s7Dj;%f!>Y&Jr'
            'Doy02Z^#avbUA8<-h4Cxv3p-M=lax(AIhI2+QBW-rreieqqUda`0;9M7Ok0cxYz_3UtI3X<9GE9ZtmWBW4y6zqq2g!8K?0<M;okD'
            'v_!!TrO$VVtkfW2q|oe7F#or_)qDQ+Z}o`dEO>fuJe|VV*%hrRljbe6@xFG<Rq56=J-RqoYZDj41Hl9Ai}oMHWQin49G(wbqWzPI'
            'G5O(rkSbm60{uGCy_funStk7`DCY`hU{U6a79#+Ot7tK#vd;+>0Edw|BP5q}^TiXu_;Um+iFe0TM{RF+e|MLrCNH1%(#&Xd65nCc'
            'q(togQMPlKat!8+(3jhi7nqTCX9TImZK;2vnHr{STgmaJT)YpWNqnEmyiFgp5&Js(hzq@@{7snjX0j&543(zVfRe?j{|0{#;BVcE'
            '0T0NY?^?%8l?!!KwQn|hK%j%Mw{sMyZZEp=LqBae4*FR05KW7T%VBFpEO~waBbwZ_lL6L-T6S^7&W?Ec92nUnZs%I!U4ckuF9V-Q'
            '!7U+~1d*|cPg~+?OYE7VgPZc!c4gxq47YH2o_3qhMVsGs?tOm-&(MeG|B63A(?2MWbcqwzm-hDbrzpu-{w3YM`eW*K;d?yP1P98{'
            'nx1>|H?5^~`1AI&c^sjuqWuv5e+^y4qKH_ArOpo@XMz?9V;<%@<ut5Eau(LJb(8qx`WZ-E0@8sY2O2PcGLb?2Nl%pMLi;Ce24D&X'
            '-$%oHCGo28KDR_GIM~}VJr<=QL2VvMvb-G^BOeo(;x%>)v#Ky9(A1kzR2q@miYO9|(C*{LLb}43<NDZ#T)oH-MO_?*^#!B-qj4B@'
            '9StH<?i9vjFSOcQhk%`%ALA}IE2bekbnqNAoL25L`VQy;3AmeXwR)AaE?^2QK4mI_9(yJf_fWIt{|@`88Ns7q-+Tvo1W6rn1E`m)'
            '0A)rYjpO+!09-M*u}m2EoX^UDqeln!+9A^9JWfZdCfcrD8e><TPdIXGLLw_`YLA+5r+ttE<_kd~0fFHhFbNV$#fT|6<5X>*N`{np'
            'sq)oF$F4XUFOvj~co?HH_9Ch9guX`!^=`dz+Q!diek3Ks9UcFoLsX2w5d58oi*y!VLJ0zAFsmm2k;<}c{s)YIjA!(xl%8QdUG9j2'
            '3sCs!H2A=A9Q+&BG;MUC-KUt$J&Rz{bRHVIdxzmku7?x}GFi?+s7GkR7o!xkA=gXCL7?i~V{M!$>pmgH;5SVERSFYfV5Rv%m@is_'
            '*Qwku+*hy*NAY-8^neF|$0^o*iP0%W&<(>~;E7W%Z1Rs&7t02=IDwak0%z#Asg=CslCtaTI0pU`QPy9ThMTuz8<b?2Ep(%$CvD8*'
            'rQ?j2XpJSs3~Pk5yc10L))_6kAT&ymks2KZ(J~hr0f-6uV&J_2Kqc@E9am*47Oo2`&0H5`h;YhqX{pI~_sHzAh{Lc8zkY1-0@-Tb'
            'eGZSz+%wR>W1<o}005YAYdQ!N!=IX3%g?lAU-*ptX?#X>g>EKW<S*^v-04J;Z*zH63{0-in6A1~l;cYxfxeN(%*u=V%o9OjBNSkU'
            '&umG!(r9cf1W*<$!4e$x_koPTbBTVyv*-`RLG}Cn8AfI}^=8d*x>aFp#skpD`KM&G`V12{2Cy0cY#Xpb{R?<90A)@r_Rpo=-LEC;'
            '`6$Z>_G#9XzKv!0R=8i=zU76bo}Sk%(b6i`Eb-4>jXl+ae8<5m@<x(^WK^9r%;k@-NZZ5LF9Z=6e^N6Nte!5YMU#@`A&tPa{Qw<`'
            'b)mFa%z309y6MK#rsAs7(DxTjeFPor@O&_6!}kC{7g?xd`BZ3>t=0JeF~?z?`c2i;*{Z>LZ5I7J6tF^&^EA^({lnM&v#KQ1d{ao&'
            'RVJ54kJJO(^;KEfQ`1HIU~qPDF&Mfp;Q+pc!G~|o`sZ&>kI;qqY3c33DZDv-e}dL8Xt4LA_+C8U)qLkKIE^FX1Df-^!z?pcI{*k?'
            '|9m(&?qOX?<O+Nr%vK(mU?Vrt=Cw7BPBBBJ>Z+>Mdgw(L%!M~X)*x-G=&x|6uk}=#T~@s5dYSXO;kcnU!XT!JxW+sgSXg4h!3EbH'
            'G02$qrEKlM3r-MzC~h{QyK{9rUKKnGCjv`$-1B97s`F%10S~^%M4PeU8mzLyu?HlYY33eTjVV7ReV_O%394$gdR)piOrS!`2q09R'
            '6Z4RK7o_r*!slSPn1Gkd_{w<I;_C|KS7_MeSn%J7(@HBo$o=j#stfF^i;V8Bi$`6<mzgJ%ryM-Y6V2PlQAgRs%uv*n=Ggs5wG}-?'
            'MJV(7An5X=!iZ$+>&gj|tFCQ(%9l#jMm71U2td|jqGOZt2Sh_Do-|pz!~asdX};w=NfJHmAj|q3!y6}?Vpj`w7%M}4D3t>qd&)>v'
            '4sB}u+CS?biDB=}@!+I`R?g)Z{y!2cKbv6;Bjp0YYK`+H7h;^`X>H2!#OS#vF!fD_iEywG`e&rRG1A}I&P1%Vy9(9Nf1$!82}*V0'
            'ggT94uDs7lN>s>Qr7)6{m{lQnp(_{Il!?B|Cp*u;^y#7a#*3CPEUmg2Z2FLs6GB{#Sq+1gG}c^cvLBxvXP+OGjf3rZ7&F+OpA=*5'
            '-OZB})c<E+Tk(gaM?p@$$`SareULaP__Irt$2DoHWw7E}yLA}g)QIzvZ1FhtO**jOQsQV9kN{H`IvWP)*HTI|YF}Owkv$Vlrc}5`'
            'i3((7*s78rHS%FStQ`4liBDCGR#;VK|1Z<^OY_8B>A545v8{SqtwNPsEAEDdWPh)0pwTT``PT~!pshkIIIHcKGXNfeD(o=VFBjLb'
            'dBWr6Whi*r*c}qkx+WlXfNZ(OjZQ+P#JZ{r8aZM`q~&^UqjNQq*Qhj1x$HIx9d;{P^Go-hY#405uOyfD*Flgnsyl7;q0+P_<Ox^R'
            'R9*<w&m%9C?UFZU%{ZJ!d2JI=0y1h)$0qVtz*^Tf8pm-6FpOOCO|5UoELA(-Ki1rc{d1|%m&`CgQGj*Jk|Bhy&bdh1-=tBuHW^q+'
            '>X@qvX{=d-<kopwzlwn5YMPiI4lC_3<9y*zRTQ%C8g-irFP(M0vvs|*$9hx#;`(n-Dtg<QjIF-bD^4SX_gAhlrsP6JbDfI{kvywu'
            'lRs7dbbd+`kYfW>0QCyclgiqd-Ii$9t{AiqbZ62dSDxXkt#Ty2j;f|z&82Z`{!EYMD#~1p^&f)?<(5Ob!!2GPLYoQ_E3?B%isFcW'
            '!HbVo6~VpKt*+>HWL4Rw8U)Z`w`Rzy!+upVw8;BpbIw(i*;cmj(Iys6Y_~4)95ryNjBc`P?Uh&cXH>68FTRQbBPWTI-JOzO!k@e1'
            '`EH$?FT=YEXjiO8SIRIOZL#EQ<z22ndF%LNd7Sm~<kwi=m#yvhtgD<B`*YV|2!x5>o`!rK_<Jn^WD+Kl$Z1R~q{5b{Ua)fV+^x)F'
            'D$PyrG2Vp@VaFHeLn_}e^3k6gwE(nE(K;xbn5<+ejbHfRX{3wCr4-3R8OpW$f5D;#>f0~<b6to_7G}w*3W`Z+xJH7hX%Oa5VM%32'
            'Wg5U+j+P3qkrjN&`e#_Q)m>#-<~2m@vaDNA#qkzShIf-`ll<kIH55|^GO<A<Tf%-!f=5dVHH%UidB)!*n71ko#(@AHGHVnwxN&fi'
            'MU82qB@LGve#T7=-`Lj^Vj9n>yX4{C8_{CxW3u08QDdz}-itrU0qVb`Gvt1vjeHQzQIG|2Tt3l)RBGJMqj;sepdw2z$e`6;-kJ{='
            'S{u2I7Lx+wEy{Kz23gZskgeh|nE==ow5)cgZ*ZFH%)5nCw=X1abGX$c4wsZ*m`2~O8h!ifVdoPJ{(z1y26#aZUbmQe<znXR7o({='
            't!KW`dM4g1n91<5;L!!~7v!4exTa%cP1(9NWtD5n<eI9<IKBGqcQd;m>W2R_e*bg8#6{XIVEyN(3?VVJAYnx+XhTrsEz;XK+t9r6'
            '8Kk?~I$p}|^jYTcdgH{S_*eI#M{E(Fxvh`Qm#r4-Pk!x8ZGCM2agFV<sy=mk@?&*<>NMoXD*M!Fip72f^MmeYJuiQ)U_i;gX1%y7'
            '-BBoWQAv?2+3i`JjMMCQ%9BO=B3VwVWy$YB&YN*=x0c9Ll~p8Ht`TM)5k_0WHruRAPg;n%mA9Yrw%yHcG3hP9x?6k4UNf#5LGKg}'
            'zV0gi_fe*Lq4IGq_Cr_fZGAD#XA8=1i`?gH=s#IaHkcAi36=A)%#!Q$yMH2^56Z(9B35gEYwV4@P*up#e%ZADbhfOJhgtQvN_OG`'
            'xC`dKb#~UWf~@7y5?#X^ss=iKx~Nz+>b3b&KW(V3nW9Yxi&mr>%bPibWfCty0~VB%C(_0?Pv+*V*=%VzUx&u_rvZFb->^UNojBy$'
            'h~zy*+N4Z^k4Y7TkxoVGbM;Oln;Y#aT-nj=xWSq*A8l2lzWpyl#u6Hv*!Wmy5adfF|Fe*k(7L`u8BNKuXS$S2qh7a)aX8iRr0B^i'
            'EYTT{YOK0N)la9I6r*Lh<f0~+e(IU?0-m5H1>?cs>l06?e7ke+W8*RFHo8~23R?R3*UII#_WoINdF{sJL}zPL;-RN+UU*kcTFNRd'
            'X_S)E4ZDVnR>~u)(WqeUl)Mf{m-$~n2BYld?6e)}XgxVuI-U3p!!V}3>mtMRPAB<`Bkk19&UcC*l53pubZc2225c;oD^OWQ(qJO1'
            '-LvBx%n57T%X|3|(2|sEoO%@nBiiY`w;(@pj}ie^8_!fmX*TW2Vm5NU81ubomeMo?sc*!yS9lTO-T^+7Z4C4DB_Uw70FXxF1~ksu'
            '*cTt;1dR-nWg>Kb?rmSpywwB~q6djnbOp*!9wsy?mmw@_f|P`*80cFpa;TPAYQ<<o?f9p%j#jp$i`W_mg^K!^R(5!HxndnWP~b{l'
            'h?JbK2#6`ZnNW#bAO!x$j#a2oTP8eooJ=Y0xW3CY%QBH~3>PUa-+$VcP#%l3s?(_ZkKLbk7V)CVjf;swHuh*yIh|>TOPW7DRpk?y'
            'B1!LER(IHkW#4q7-BkN|q=v#*RC&;h$PR_N&<?pSZ&X{J(gjhj6*k}N;tQ06K~2(%`|jk)pCm<>&n-={ly)7#)Qx8rjPjrP+NvSb'
            't$tjBll`Yzc>}#sC7ATL<iyF<sE{q2IUk4jt(Dj{oUOYOfYCV<+{EY1hSCT|eOg3d#Uk{m?k)6f=2hy?58ulC5>CfRA<A(&C8KyT'
            '*@x}PCcjoljl(zduyOQy%Se@p?3rGo9kQZ~J2#PO6K4L(&SkZ)Go;Axnk~pVEQ<f`?Th{&p6<z@Wk00MDpVr}(2uv(CH99`8DlIO'
            'hj$5S@eN;{S`W91N!xER7%EH4gHqibnRCy2tAefNhLevHO_G)-FG=!?9i(Qwh6fXSJ^?+r<J3?s_z^_Lh^p`~Zq%I_&+wqwfy6q6'
            '@D{#Pwun-`gM(xVO#MHfwa2$!l2UjVBbyqd#ax|45*-^xH*Ldr$Kqz`B@+|XICo18rtO;Lm<&|IPQ`>hGS`I>X2-3{VRnVWHXoXM'
            'EjXQ#T<^|~c;<YA3>arrRD&z{a@u<T11T0dXMHN3ihunUiR;O-O&?Y)he8jihDXj>ka%jr6O#-A{$jX50#aP66ZDv6cxrhB(>mFQ'
            ')8}!W3s)uhv)D)dK$ww)J4&9R*;Zc5>#wT%jx2IJ9QKdj4bQ)rA8y3VNdGkxUI3Y0PJ)b7WLY~1Gvd(^K%Qq*5W?5pZG`YPPYAw8'
            '@gO83VVDZ;4(1gIGfHj<;+rl4^Mhg}#!yP)-=?AH<XuacP}vp+%;_(n;SCP^=UfT<`i07Qr}8~k`k`R$G)nxj9KkZPCD$R3Ek$E|'
            '8wX>`PqIqIEvz<GfuiqGC%LY~fUYc!q$6RF3S=NwQvzZPgTch()Q3OFSuQBJR7g7-g$Q%B3?|ga8aV)J&MHlp_@(67RBcBV_%uq|'
            'nAzhB`oR=)Lk7{@UjdPmL*HQ7Td*%a;c)+w;2o_$3$f}EJd;(5jnSP4r{S1oM%4_O>YOzcEbT;grK#NbW0WgC3c0;BKJ71j7)LHl'
            'Mq=4NM2;gk6^<cRXB&ev^=~R3WW6_E=T(L(1OU~lNSTd+F4ga=EejrK<I^B8*?_Fdj)DM6{(2}3;c=X@8YtUWUKnmvYelMHt*XNC'
            'RPKo$7vdOlUs_%Io!aIVouk-Q>vPp2svJnyS9sPsPNTH{CKW(+b=~>jCcJ&vzN!d%kzk2}dB{pz_(JH?xlGuwAlN+MO44)hW2Zcb'
            'Q{}yCOLuFx);K#%tR%`7s||N$%l=<3250?4w@0<txQkT+PnOF1jCxw;TcfNNjmqV;=rX_>zP99+Zf8|AC;L=ZGa<Y7Ieq<7gMs;j'
            'L^eEDSEzvMJY`yC4x*|f@fj7OiRAKU@`79yC5be?7HdftDruF1nd?hP*HxabEGzxl^L)Gj^#(KFxHk3YgR#1b^(K?5!8NqM6?c72'
            ')3QfLQ)0bJm}=sgRWbW_ty@hA#DPL}HljtF06+!<tx4HMwnA-6DEctw^QsmUo$^%ynQ+Ul!k@Ep{_)^!JsST{r@ve+&Qz7EMeb5<'
            'K`u?Vzjw7&@l<I>csE-X!yV^pYpasCT0C{jvTrKhx@CEU7{`v_U_CM0miV#$U2KK#h(>NzaAx_-e5cv8tl-X|;LMA_xy|FzDD}&k'
            'b)(qX=H89lU_8=w2CL9pH1VA8NYbQJ+lU#EQVK$KUKCii8guO3B$_2-;mxfAFMg1!KHGI=kw(nn(9|lasw`hh%lbVfw{@|ir)L+~'
            'uFN5(E0&V0ajxZ#yeN`(r@5D6=(~=g=NhO|X*d9kN5h~5)Zmaw(jo@&W<^;Lea?8=ZtH9SoUyf#FkT{bgCa^f1EyZA!K*cQC7XM*'
            't}3MT9Ww$Yc`6`NfMh&Qf{_&lPA{9qnx&5igs#b`$_S}ypV=khbNhE=NNW8jz0}KXJd4{`55E~VS<?m?B%mF^pztpLFf=GGs?fH_'
            'H%Pi16QU7c5*EJa-3vk?v0PyAD2%+#gle)aGh;J~@jAEFDBCjetS+-qQ_``Bihax%go$&>GDKM<K`fRP9}d_ccc!XwQSm3U;!<u8'
            'm4+pkjBomLR5|uTmg{x+SIK~d4;gE2VcY0(BNx-O!I2WmmWI#mkqSZWRPd?>k?bk|(Y>ey5?A_gsANHsRSZNH{lF&+tsDVc+N(1V'
            'mD*O6=%(c<rbLY_^lZrE$L-2H$##6bl~(bOGGq##E_ygL$c!`BE>KuPkfQDU%m@&m9;HA5dA}_)*RZLEIFH5=^{!jk4P(;M=sflP'
            '336lF6zZmZt`#Q9!OCgWi(_=m{@qWqpFA$V6T^5=S0X<qZK^=9DH^|S^#X`~Hk{59&>(3R;>YsgkeJ(5Jj}ONC6C9F<TV>ydIn7>'
            '@6Jx&3|<a~{X_ig<<aTE+au5&xE#X4VAw<5uoRx4VQa0W`^lKnL!&0Oxht!|8#Q;Q!HsEFE#Kd1l;^}48<am!SHLUhE;0x#zy=i7'
            '$d+!^<K18|E~*F8uFc0=oL#M(CyO-gpD%HlT%r7$*Lcl0Nw`<nHI;|2b-nZuI3vCRj?Ja~@XD0Z1>$2Yzquo_c;PPOzcct(V;NtT'
            'bi_NE9xa|ugE6oopyYJp{8Zd5p&_O>F-J^q<7GG@lO);(gDh2dLt+n;z$xDaToYwx7U;a=3vZ%Jfe6g4jxt6(53;=SO_0juh9sHY'
            '!}wW@M$b5r1H$L@lsrnfCke>Zn}bAwxjkU+=ur@j!{r2H^DMFD@tAqj7Qy?$q8r{IJ>;br3fPh3J@>h+DCmriEK1<F-sB|*O4!KR'
            'X>>p?tx^VuHg0gChC%N;!AqCL66GHD-VIctqtplXg9MAegGr107u44LqG7Q94v;WJCl|{9Psz!MNTShPg>n%of#E#EiA2GmydDlP'
            'E<NUKp&)MJp+N*8osk}pYZ2QJZ3iRLMxj9&^m+h(K;Qh-u!jYWg8tQ8TA&@Kic6co<}VlU<>Fv?an?UX+M4f-vYSxZ_VSeU=<{F>'
            'vejxK-Jm?IdBC-ncR0CNZBL$8gJaOGXKD#K{V}<NocQZAbUcG;OMAkjQ?Id-e^R^fpbYlsROclBIuHZc-&qLww~CXFA?N4~0_w=;'
            ';qsG8v&Pj@D9O#km0A%I&LT8Boc*IoFD*%k-zx?}?v^fP7e$5+#H}ecY!_33erTCa<}LQlbf-}Zbg9(u5+xgZi|$YOBZ+PbRi4aZ'
            'PWPx#6)PX<vQN<MMV_G^Z+74biTo_y@Z8wN>_0I5Dolnl`19@5HB92#ywx>kU2TY=LNjPV1))&ZX34vNXhhjKtIWo*cU}5lAb|-W'
            'y-a^!91|wQKoZ}J<`eP#_rij!<KXk%&X3{;9IV45VYYVhL?2xq@6W=Bl2#kgq<4iHHt<B<r<2a>*WR|0GdgY~2n6F0JxV1@=Re%3'
            'M=5jF!-76u&)~JnrzC{0ccuIF?sUJ+rS9WiPC8dxeQr8)hPksZVkWR{HPn;*97<@YlecEXvWu`B?Q?@e>H<XS#xvtY)(k1#m0ab8'
            '_PNeo$udgyJvYiylp7euF1ErD<v_|aU52uRr!?qDG4WcSi6$R`mSH4CHb9d=3xVP**EnS~X0bR12WDwhQO3buh-+iZn`UQ&Pvf{$'
            'LT_>n+rDCU!ZWHYczn$z$qHq-Pf@6alcfHZWp9xSg2>5I$4jjBj75Bc2pz8Q9-Yg?2PvnE#Ea}yW~v+k_F)Ndz-qF(t0c%W_oR=-'
            'AZLcX3@sgsOtnQ%o+}|O`XCVu4u(g+iu3+oE;w5m{DC6FSsw(2lXE4IQ7Ho<BRz6qsV72@tJtF)3-d)8-@`SqA&v<>3t#?S5HA@Q'
            'KqXiP5z74>DbX^#!Z_VLxhiqyMz-Wly@-k!Dan~dxXcUVn~q){R-VM%e7JWcc^AqSD$c~53Rx8MuNGIH<Rtfbfk__nP!9=WDAqaP'
            'Q~p9OnpDzBMI?mE(^q-6bau_!56h~OJM8snmjJ~Hk<B7rh+s~(G1Q{S4O_~~WD2ZFqoIm}wj{U{S#(|&=fv%YKa_65G(4Z!v)#R&'
            'XS+|ecS_EMsMw1%&S@8~dmw5)l7SaA-_#uFnRaujgJnx!bEq;k1S@@+$RbKU=8qpf=rPr9)2d*gLe^Wx%@bh+atKY^jcKYQo9a`q'
            'ifO?&s6rT?^OL@IlfUyNAm!OP?;ZDTPgJ?>W!TCJ_sZS`A2=VLzB}*8M9XAJ_2uc^(dAqLGNTnn@yq`CctkT;Y1M%?G8<k7XK|a4'
            '-*!u`h?tf7*wh;(RP?)0G7P@sCz%_?ZgrwbY2Q(-5OpT?6H!8^V&2My2Yu&OeND%9U&H@deM0BLG169V<u&k>M_tK_Fn`sp^nbIn'
            '{b>EI%MKv9prn5p(vteoQi2ixW3$+5$EV7ttqQTITYgj+|Fis;jW9zgxFU11LJ?{%?-UB+9T!8eQMlSG=Ul6}uT#$Zr5hNdmAZ>5'
            '4TOb!qxHkocg}DBsyeH;72Va!^#&9?e;Pa#<$0tAWjLrEcF`{uUbDfi;(bm2w+0{D3IH1tP5JF3>F10fS3cJxACtL1-i~X`mYSh}'
            '5RF$TieehQ&Ti%XDifl0cc{g!v0h6HvpGBM)o)GdVavJ!d>muU+jDwjTEg*|T&h)&O$3zbB^e?)h8N&l)7*@cdwgky`{Ke)q_kzw'
            'aSQN9@`ZT(_O~=IbEB^Kx<8sU3pEdz;<m)oiY#;~RcbAz=7AS+c!$33B*5cUh<qfUXvyAIa!vi#Dw4*_&2J))IbA-6GeC>qy*~!#'
            'U>A;lG_v^~56;gAC$Bfz*G(#%e$$Ph>%ce-$Ya8C{{jhoPR92pVYTx}YS!G%O1D$)F<--BCde~}6>nh}CS`JarQK)B&XQGZNmO~+'
            '#IrKpttKH23RMVKuDnbm-&||}@6^=k<}3_T7tUHmg4Q}0rE+wt9m`Z5pIS)NWQVCb<TzUq7cWPy38KxFU-2C&Y5>TQzSP{Zzb#Ko'
            'HMh8`OS=0=FTHl1aHrtN#N?%5fw8g%<YIXBYrR9(V`T+o3q&F0-+C$IqK&xT&#r!B%%K!w&ZcY3yrB#q1p->KfK-Htv-;4jz4<N{'
            '4cD7W?`x7jl0E&LMmpn$Y*&}P$|JJ*#H@zE<`&b+f)PmhTbA^TjvU3=7So!QD8Kqo?i3Xv38t$O9WGCK?xz&u_SwYSKErkd=T^Ph'
            'v}K1Tsyo*_I?;|x{Bjvz8S=WeL9aZtan20PuyX<M-FjV)&@Z6aUzc)MQ*eT0PPFAt7io6>NA<W5&e~P;Oer6Xoa@@=rf7l&GXLIH'
            'evXZ`9ScGV{@CC{rf|}%2ixn5U-~CUgSP^Np2NYuw(e#AO#mZKfT(0aN4#F%`B50mC>{wugg3TcI`?KA65pu(z?8wXld1|C;*i9Q'
            '!P(&GxPLf63&ey%X5=P4RFAcOE2{OA^(<!HcwZ`(RpyP!MYO7emS=_OT@2r}YGqL8e~;k$GbPo^|A$A3RkImJ&CJ9kzDbspG~(97'
            '^aq#^g^a1rekmu4>{)+`8v6L87?6YMRxz?dfZpfLu!2oz?GZG+H*fuL!Mt``GA_^<x19&kTcQ7L@H<bU#zS3y<|?mI*7%=#h&8u-'
            'IXF6k!|ML!;&j;ipK)UAdw<qRmI(=q@;de`JzO(FNgqH%W%1d6H9P-jAw79%Dm0Yt$ZI8W=dU~t9?Bz%=17v(5F9B1_*jW5B>xyd'
            'CO8Y8>W-zbsLn#3-et{jljTQ0%z_0b#!kEp)UK5uVbxclW|2Ra0tJW?Mxf!E;WB@gcnMB{<&_(sC}VE?T1yW9m(qO=apk;!_Dlav'
            '2`t0GasTvU$d#zdO8cK`+$k!jQtn=<Ipcp-&QoP_z}l%>$kj<T5KuGRybO2ldxh>E@c-4YESr038HSw-@PDEAdC(i2{L(ub9J+vc'
            '?=FS}y$X}ipTypN&6i4TYhk7Drp0#~-2_wNb2`4|byzm0R<+4V3(S?Af-bhV3h5iA%Z8nKUz#{-a|)H>FTDFgVZ4pZ9F-;xSws5E'
            'E-0-P6cR3NX5w)E%gxI*1;ol1RJoWX|J+!X`f?fSOJt{qY8XBuE2(Dk((;OG3!#0mvXym(N$ktJuWVS6+}2JgWyw#)J6UG0WMZwu'
            '1LTu?m804Bx3nO6)q=jEr2r}8U77M|_o%M8^rR<Q5Us<{%l-D=mAaYqE)mzRuD0r=f#O{<i6t*o@MBFbiBI0;Htps1)dL7d9^Eu-'
            '=ZFg$Vk4icT0+^PNSCv5H1=l!;|4n3>pM)8%^F*eiPNyWt<6N@Mq}$+$X8!OuBwrzKAQ^tevQv2KV@cP5F?v0ZEd9OeIZ5qTx#@C'
            'zC8Us+gziTm7!z0$<%9on`mv*Mz)+=7x=PvaOx!i=JRQOGU(HG2}8E8oCm2cSs~z!aWKmQaq;Wn#o5Ktaqn=@X&8A3e-APjkoc?T'
            'L(9Eu+tSZYUJy|TF}c@%V1HEUMt9yn?wt$=2j|};Kbp_c!dzOFQzDYfFopFR35~F%J&?52aF_kJ^V1WVMRCVV5^qHgtnrj0N>zNZ'
            '^u$f$In^pk#eL$jZ(NWuS&dUls^N+2>ua2am}1dxrz6hrMlKbV?#Qaht*mmzo_peEiSD~bJO;V7$kpg9x$^IFo?LkaSoOVTV-%-c'
            '6k7(($OzHmXzZ6jWydiXsKc={&7=XVYyP~C3v_<|E&aaP`Qc{=o&jOwe0|XNT)X4Y0Hqv%^t6M)Jd5U@V|`K8{=*oMW<XpkAjOsD'
            'HibjoY<$AlTjTG`hC>&W0fwx#_nbV@B>bV>40&K|upyubNSVjzF$&|GCHm}0sQE1DdmngW3D4v~&rCkJ!m0fr1rrQ5f^L867K;_G'
            'ordxGnYZwwxi=3|wRg4y4;^@;k~YrdlD`EXzOuY?>ThM&&1TBe@m2|0z@=CKLI;chi{~hPBf?hS87uGlw8_$BD^C={Nin-k0Nwc3'
            'AJ6coV`N6<js|rRy;nnwLZ!g89Z6~O*V3#0+;Z?|-(Z?NdOTpJeAC~bUhj+RPYp~2L0*83#=|v>zNySVDx>tEKde0rF;8ZQ=Tuzt'
            'yss%GHw^suC7qjiN*$Y#hd$gOcTLDoS3<yH-{&}V;446fN~!J&IWra93i(;r@0Q9W$19Kg6#g%02`o{M_jVX0sHW3#U7V=vK1k%U'
            'GM5&1%ZmQvMl$A)N2?0)tDB?D#Vyc0Dg0R)VDbwZ^Iwt&BeL%fLj<)ta!D<&%z1DFBfpVZbDdstW;^LQpK%98ew;6!Gf>p>j1hW;'
            'jYse>rWh2l=ywv&V2Owp%@V@Ymt3#(tXRJSFM_j+-X}ARK8ZuI{RyZ0uuXSf0(esOxh|y7Dis^~ypL)D3O+@jx3i*1%~J+j5^I;|'
            'ocviHCm<#bs*4~hQP2vaRV7`N-Cy(*XEV!MVFhaX76{a2zJUbSOR@l+ILSVS%aeI*#oq2W%3*u!J^ArDgG(nT)+t{eOC}6cyA6X;'
            'hev6SFjK{?|6vl`fHKi^)}|V6QrPBbmHDajMLCBt1}Dj|j#f&hv8w!;YqA=XOaM#)aIV={X4CdhHli2SM!>NaX8~{u%1aZ2u518u'
            'WB6KOH2xAr@R9o<_Qcd>BRJSVIBtO)hQ6JfhVr~1y$V1rYjO_nec7@D8q%cs{Kp@kK8F*B+hCIbo`#B`rjhJGTYg_&oy^7|Z?+|J'
            'HdTNDsxd0V)%Pu7MdL2W!#d--8|#i9n=568i>jhyGQvhRc{C%3Y|fCb9FWz@%?q3fDF)}dXyvdv+mf3uN0{IOn68}?WGxsdbA@}m'
            '1On%J3)ELJRg@Y3D+A-rboKJ%>jFx>5=}vL;+RNQpW7<YNFH6uQIMc|p%q^TnJOiEhcJKx>~;{fchH2`mg&T#GcK}pPENuRmXu8%'
            '%cCr_+c*h6vfy|Nn2VNf&+rX|d0d<wwODmSk4|56RVMT-#8I4No*9k(Pf>J;Zc<Jf;)QrHR^lGL=Vb&f2clN`C|)~q;Z)3Jx{Nr|'
            'i3_m^A8Vg(^Uh#?E%+E(6(*d^!lf^5zo1+hF)xiQP%Ns_TvZJ3HF-q4i(uhny10(oZTZ7EY0h;a|C(}Wm0@UI6_19ZRml%dr_PgS'
            ')eMqQj?S~j<7|;!OV<p9MoMYEag4+o)=W7yvjkp)lQ&tQTbriYwslofC1Y!L%KIXKd1U_F|M&v70u$JPx>U;h2Z-Fxyb*{*fdCz9'
            'KJi4ec5fUM$S{bc6dIyuWQBnlhQ~4}^-Bh3eLz^bE59)VtU$3`#6gsiVNuax<9+_Hu0?goT21s>c}iazfL1q>YPM-CTHQFG8I)Et'
            'R=H`#coi{h6$hx;$XOHPMhZxI47-`*tc`+O(_O_(RmGSU&holYGxKlCCsIQ->ch`~4R{Qb#(rLh9E&Jf6~U_uDivbp%3n9%w=ky8'
            'Sr^||?qAH4w@LJ0d6b%_&IU1pj}OKz&9N*IOA1D~S%&OyHDS}8`J^0ETp3VV{)Q!tQ7KIOL2+1{(wj51zu*{St(~J&i0&*1X!>B4'
            'Us*>f@j<5y&^-yZ-G$EtW8B(TXZ^lA9Q4oJ!``rmmHq|?eWPD>oHv@Xw42kKJy(^%$k}Y--}nh#ad3UGVeifH;N);{)>4JRR(>`E'
            'N}kFj^AluEDt1$;M%7Yg9<C!<*JZ<Q!O662cBz@Ux+cpXaiKJM7q1_B<$zln2$z{wv@ao(-S?p;U6fiweZnZ!;u|RN@BE~lvs4rv'
            ';|L9!AbigOpp#T9^et56=2EeTo_#|gH@LE(3T~M7<@EgtnXe|Y-W~Dy4#=wh3bQ)kVM%<H)nPDlmmyM-*r_U%uMLH$h29vmOyDc+'
            'oy18qP<B_W0fwU#EAvI@b8XUZVu_$r?xM&c^S*J?Inpd^^EGai`f;sQsy63ip4qHht-O8zJp<UC-FwMRD#ce>W4T*X0E!od7ZBC&'
            '%PVzivk#WdGUX=S@-v-U&&g|RqRG83O^}s{4G$!a%Ih#qkDF&P>{etkEDS$wd@7Ik@LTj-^ohUxP)wPJA$!%iLu*n}@Cu6vRQTK8'
            '^a?aAKdM5g(i3~!9Mkl#R%N(3r@NzV>eS>2<y+BizCs1nuKLlE5bN?k5(|`36Ytm`G3(k!(Soy1b6rM76bS9H1X;GKM`M9NR;#mu'
            'ANQ*Q9i=heI+Rv@$!Zrlyd^bas#R&px^#1#Y0gf4rhd0dG(i=oTt;f7+c=xYbN0zVZCz3XmCg2h*>RHuVbdT2#sUMjc|^e@iYC`Z'
            '>$M6wHM=O_Ea%F|apsTx;LcBljL!x_kig&4W0vmRFvZ~{zE8!DNXLL^K(;(`h)>5Lucqpz9C?lkzD3n27R}U$2^<}rzIV@h?_CT{'
            '>$$^ISIbL<1g#ajRX`NXQ`z8k!f(0P72Oz_u=Q4+(OCAwsv@Xot>CpptHr)`p{G=8{tPe&)B*W+3Y|$7m-^k6gYIZjMszIPF*sSH'
            'xZMqduYF&OBn*8;$uKj}@5K_ipZSOyOY*Gj8*YNUj{V#{kZYO;5ny9<ligaIfY8N^*m4KdH!%0L_C|29F!-~_VP1mXW<Vb;!<j1s'
            'o@H{@-vRd9;O4e{7l+HaZ|1+k9CHfvFUtosh=4+cN|Z>@HzA4dg*_|i7G$Bn!zB{PE=oq&%)2o*xMiGUPT&{%UbI{Q6_whn_F|Up'
            'jcL-|H1LNz^aqZk&qR^>t*L<VW}9;tBpy(H;9$uj@w4B%JMiG(Y%rvU_MqrwL=NZpXG+cpf<A8McTyYZP~JG@E=2NlTI!7{Q_0Mm'
            '8*xi;saoD<fMJP&s>e%zAR8(yj^t_SRZ`0zKNDhAS%3MyfS*W^94PK`kxEdQ3Fl;T>by7A-4O#6OeQk(ohQzZfpdsxIUfO7aXaFL'
            '%s{CYp=*_3`Ezf@0!=bUU4{~md)Ox!Q(DDkl5n}jAd*?|?xpxOT4v-a#?B|6QIC-(@@u>tE=N8dmn0aIq#}VZ#Dkeq>NHII8>l>Q'
            '=UkZg;_N8h83(+JDJf6#vfAfckr{dl$mwWp^)Y^^MMJ=g{JK*o95b=0gcFshWO_5|!CkH&%s!dwioNu<Y1vjyhjXWH+`lggP&MPq'
            'w;PXZ<21X@TaT>P3Jmc6&iAr{BWdHcLPsyb0ORxCVgH~nwntcMJz4@<;H}lIHzM=P1P~(F#;fNgnq73(>%{UQwQy%fRvQW%v=Qhd'
            'd@dpk!hpR<tl$?G+w314_y1P#I<-AVP4~)*-7)<h@NkE^Y_|5bObuKmc9JR=%j+D9msg#9bj3DQ6@pjnW8|Ud<ut}qP6&OYuN1vN'
            'T)$S`k5zB6!#8mH%>iCVzxPMi0)3xS(Y(&T$;C}z8u$LFvrK{=$@${B7ev=mh*Z@^F0b-z+}cZYOm~=+mNQRPu9i}A@G>D+>-8Y('
            '%}8hi-}L=RRt&`AgU<R|A(qF>+Ik_HPwNsWW}%Ss#xzRBJJp;0^%dSM_X7DTU4z~H5^qO!$s5=qW1`p)y28Uk!5i4bycxX4+A~Q|'
            'u8M=x$F}e#{0CFl1DN+8ujR&BD*dSAO(sp~YL6^8ZB2+0&SoFKSrzhw*avT|>GQ)1_o4=)X|cV0>QsgXNfeXQtUc+PUamdqSUKz('
            'J}R^9t6tYe1s+cFjNCCzm*~q_#hWC$qZhOyl_0|fbE*|QqTH`znGzT?s#50Vdk<&~eb&#5lgkP4)!HQpC|4VlZ`%C8;1KwU)%qGv'
            'RVRmWwE(DUL<i*Lk4X1QwW!km%NGjQsFStJvVk%~i{bEG$x(;SK1xFUcRdVgm0Gtp!AgN0CxH7z9!=z!mk4Q5eceAEoD5tT(f#Fg'
            'aL_;3EU)o5e{s7?sJzc|Ji~tvZoP!$&Bj~2hJt@z`B4&3n<IbX;lHQA@9^JV?CUbSjRQcPAW3dx>eO3ial}9SpiPI~gu4v9g`e@y'
            'i}?sv98&A>Ch$j|8tM?X7&KMt3|tU&X4%M(f)N!Q@=_1}0*#5L<w1*yA5e!LAXyT5i5jaHQgjr)I!FBTXeF<`akBit>khF;0=ky$'
            '9wg+G@HgPuk&oY8jsi8>FIe7S9(cgXsrNac%=mW1pT<8zP`rHl#Q8GxrXXOnwwx`McwcV;2DuP?tyf3Alh+r$*Yq|^+QYNP)_42j'
            'Fdi@2)~;k`#0@qf?u#eS#O%$-me_mV9tC8u#SGNmy?A$UESm6p%;o<41UMvq<h*zM4oe#&Ds`c|ySuw<zIZ=49KOLf^zh(K@8qO^'
            'gebFDm<-N>?1&$fZHPZ`aC$QA{}onr`nG?<0oc&{=@a^pwC^h_M!4*f<DXqfEL#G4U($<~;a??>-#=4%w;)1=u2kQRAG-x!Y&t<w'
            'gAJLO&HA}3vXAoUSY4)~WnOfqG?}tcQ1+c<?r-}}R?H98+vm?Zqvy{iJ~>n3^Q!D_MDlpssY#3Gz4d6c1>zh9egM9C3iM16??`l('
            'F(8a)i}BpeV)x#=lPCrc_Ju+!4=JdN_VRAwT%b`Ky-dQiYo37`mpwVqeK5&xyTyaVVFdU^&57S9-XhgU6j0C;XM<O-cHZ}X>F1|a'
            '3f9JnmoZ4vOAbRDaDWyw<LeHr@)~o9$>N<O*Z`Eq7ooSp%VqXPl7$2)M#5i!ww+((u?H&KeUNZuATEH4bnV~=Ks82d6)M}(gOAn('
            'YV&kT)o-xEKujgVXb!e8jcp0gln_vZU3{<M3_ndU=0*QpFAQ!_qF&dMA%ghVKu9*k4{{xy1&ATd#-9d2ke@z>$7Hx2&nfC>)If+&'
            '=b69`c}3<z*Eeql@c#{@ur0i)RTHr=sFkm=8cU4Dw5$cddVX@O?~nLcS1v-&-||l8<^O>Au1h}#(<zJSZW?^#gQ!uTpX1dDyIdoV'
            '22n06o<k1~u;V*s1gel?{1oLSEK2mgcl35}@|uemkivk1oJ8CAZ+<01N>98R{I!25PG7w`?+@u!&Al(z!N~8(pFL~i<kaGT1f=4c'
            'l|~O$(<zo;c>0`5&0&E<(&g}K9GA;X0)rwLO`q+ypFLv??ueJmj6FgFX*<H(sL;QGkHT4rk*TsPctZ9XIhIeQ6M!K4z&Eb3^5ON)'
            '_2A^y>9s08%8om7l02{JbiVL!#3TmE4^&f{q^^rM4L)EI1YQY+V9tFH9oyMf_WtY?pv#cuH7;UcLa-?lS-^sEAGQOiWJ<;{ihRuC'
            'Sfq3G0st{Gg~NyzuND#nZ<+e?(_Cp>m!!(C{kRq|03xN3s*ZT?g)>>5k9Js9=p)d{P-L|M_OX{s1IgjZR-W8dKK()bgrRnb(8K|Z'
            'n`O4rK|g$P$?f-bw`X}!5pFeC&%|T0jEsX~bj3b-_n)qu%UvrjNq?`KAB;l!j{}e(7h0XcFfyeo#pp1PuI?UaKFerKj?)C*`7+Jq'
            'rWHm*=rLXfRA?j@-f2OI#1iXCvoZ?B7>6DsFtIt=bep@a9Fn3fn)z}aBQ6OR!l;U7NH`6XHW4IL^^l1~BA!fm4rA%<O|Bu{gr=KM'
            'Lyg{rJ^X<?{2*vH;$Ov{MVoeuk%lbs=@1FFN(T`$=<eR`lV|AKzF=R-H9We)P_0<!4j$g|4$fysos}J+D{Z{*g9gAj=~habnAtxV'
            'whh>UBwz0+u8_8bXrJ5;*cm^Zwj@K0d^~o*^`=u!@Qf#ixun#Y=`vMD@lxHNAS-Z2D$Ub{?`5|fF(RuO<DD6mEc8FXkwb?leA@|9'
            'h8&8d`u;L^xzfS|iad2W>KX(c7iFV#@BeS_TASN8vg~*L3IzEeG8QQ*Gx27mVWyZ79jhWsF3HZ8SIZ(n0-`v4E<nhpHT>^;U)|_N'
            'H>ii5nc1mLr4o^BH2U55aqhV<$%LVvH-@`sP#(Y!h>owOy$jVS_n4mJPFK1SR0%X|${oUf5I8jD4y)Q}{)z3|sE(1FULcRzScB_D'
            'F6ZJKP>QnN5{Z?ly!@~+!;yHnuhm$09U~vMI?F6{o47TP<Y8<ycM^fB{FlcFRHU-U<rOcoHFCu*rg=v69Wq)hOVfnoxhk@2-5knm'
            'bI=t1N+E2k9_ck=6$yTV+tn(k=C@R=RqV_n-=^^^+^M+c>8w5aV;4_QC4uShe>^(;UVcGw%YgZ)m)onhD*-MAgpUslg7;QRUp@L4'
            '=h|~0XbFxQ^!>rnA6|BgLy<t7a+7$)radK*8PD)6C{*EnX)#(&GuNbUeKr4{yI%1dgQBAtHhUYBBf}gl9P~q)2p$*s^wO=SwCAZ7'
            'Q$}yKyvMO+UN*}UQJ9j|)yqo}Xet3$%NYFqfb&#V>IXII#r{5CX?AsLOIXK7!NZ}xc6eme5V)roKCD}*Scv!xKqeE!$I-J*3Q^8V'
            'PHyA4;6I*oUKnQu#qP-42?r&ohp^ygV>3>-Dum}nfJP}MD&m_Dg!#K7=DCRoROpF16z}RA71g+$Vnzf8Qow_~)hesWM@`=$uH{>}'
            'IF(oVj|T?_t{wUcYs-R*^Ro}^p;4HDCW!xnOd;v~hJ^VQ3ZhYfnMtOJ(m@!+D*2gbGYZRsVUYxX2C9`zR4`rI=q&M`X7g26b(7{o'
            '4mu8k{d3Fco8$k%GF=@UOVcApY*q{Q40#s4LSd0UK1015nFuI=ze(d<OT*e<{@l85e)E@Z{lBvp!rM`|BKomyQfpn5T2&X(S-_Ij'
            'E#?i$g-fm<YM-)5OA(E)v5V?1sQ2ucz`Uc1!bp$IA2z|gdU6w%hr0A?^M-h^g$f5rC3jp%1gup)&)E%SE?Cg?dRhd%Ta1S~m#bs0'
            '-IljCnvZgf0~s30V0}j2Ep^P1)`m`?c${($Bb`<@bb*AAfoNpQB$_abqH^Zp2dXIMJ)y!Ik?<O=IJ6W=Y3|zt*jFkaCm|%PIYVQN'
            '$VirJohPS@#WJH@ZxUt7%{e{4Iz8#o9_{<!m3^INRP0+AQizG?g#vQ*5d?7%ow>F&X;6Pn)}RDYR17;-X9dIAB{3ccrm}`3qw=@K'
            'KqWG#?!h~VdNP`-+%2HRze1gXRqq7zQlh^Xq$Kpy=xNmEg#qRy#p`pPA&Vt8CZB_JlE~n{P8g-Hvbd6EP@i`4(jx`P*DRP6K8ko2'
            'J`N*cu_Tp)dq<Hb+&KvlDAG}<GOl)ZRHf0fQGvD`g8TimiWw@C>3BeBV`$AZxmjVXIwNQr4Nlm2OZEyC*~-LbLdmp8TJ9jZjgagv'
            '<(#K?oJ@2bkRVPr2OM)Yr9|%KFrG^|tsW|YU)@GA4-o2Q(_~gMOUS-gdYAfKw|;FdBRXwn6WEhA{*Wo!O>AFJSUCBy!6L@o*v2VS'
            'aj)RriiJH&hD2J1t5qgJu^x81BH5dpgv+CGl!Nk!UkZ9NjE?1Y9Hkb&v#*khEPa(xpzqpOzxFOK-oNWyT~lwmK96`g?2~G=lksiM'
            'dshS>k<crOr3xIP?+Vr-t@o*UpaWmIZ5<Ekmhj`75QA~%vt$iB-!RXEOoogsRPQ(G2ISSr=cZe8Jrt1rGEHvApK#51j(@dkP;A3U'
            'hmhnlU&+!k69F-l^K(fq6!Ec>A}3z<r$&27O-%%js>GoJt73pm>!DZ#+e=jym>{F_g7(N>8LEPj3im#>TJZl+A<X!S84s_+_WP65'
            'i?Hk47Rw|-3>Z}jW&LRZLhEwJ)op(EU+5t%0HbEuJqZS*ta{Eb`~}+B;Uj(6Y}{H)Uu}HhAch|mG-2ff-0Ed5<AaPskLL^`ha4U$'
            'sF8>VE?&$*cZzdEU)IY<L^<PKzQlXfz@29>1Sx}#yiQT^sn6F3-6wX<=oa)cbmMP|ns=TXapDjD!u4M-;X1v6C~{pqEfS@aeok|5'
            'e#0~nZuUpxjO~{5y4hunGuKnnVQ&34Xs;Ht@enQOm_o1dF~x}6Ia*`+uFvxHj$%`g79`4UDK~c%u<<)0HXW7_q&}AEL26`ylT)j+'
            '!S~}LpJGf)5)s;$>Jj-4r9th%&>my7!<C)LWeR4+(>Plyu|mrbME_2=^d~K5H2E{0>?mUtJdg5x69+&{do}%55l*tEK+9{oe4)tL'
            '(OYa_@!8#Nfu~}I(LlScU}>6*k5*Ya)VKitRbovw4_7ihn%`JuJr^H6QE?&aMW+`yVdxkaEg{6zNh7A5Hr$TOu`MI#KIIw9Z-c&P'
            '!28JT2lA@8_b6s4K<HdrSb>hU4^wF?-y&{3pjZ&?#VdRj6hc>U_plzMOjb>+8;)75W{@!wx>_}Y@GbY7x_oOsXOGNM*kuZ`a&5BJ'
            't3;y3drSo6;H(S5so?8<d)fKzwDX_GET9HNz6&QWqWBpeT_UxyVA0BMrTccR%y8Z9$!yChW~vE?&AQdshI6gWjz3mVs8Vw;l0T57'
            '|Ah4=4z?x|TMP`Mv&pD!1|nNDL3g0|scvAvvRl;Nak33>oH|%?gu_f~Gfw1d>nD#n%S)%TYf14~{l3(Q@@3|e2Mj0L2u22V8aC57'
            '>8<iP#uj5SYse;JVo*(8e4KkRo5}!_rxTc0)O$r%2x@(6SX#RT2Hfq}Rk$wfOwemZhf(T(VSXsoh~QGejF=I5nIK*03N0$^?Z;b|'
            'R;?IH=2if33<|CC&fvS_WT+f30ed(27A{QemYY?@|Fqw}y}bCXeRian#>y~8Y+k>#x%(QiKvoneyrVS-_F&6#fyfHH6Y=nbjRFP6'
            'H*1Ahkb$489>J4xFfngfUvoN<grG?-`e;<*AlR&cOrWe$SM~_ai9jQUQ?buDYD7Y_A!0m-Jw9gNzRpcRMbX%?TQo*&CN^c1GEqec'
            'KgnQ=<#WY{5%g_Njy?m7g5@;NRAkeRRt%`!{{SP19Kl<x>;Fuv2e<0~sM$jd7XmeEJ?ND29)5K(JYe)!?-}-oZ3Ruc!vnk*Ml<XG'
            'I`)|1V#*9$U%^_Us&dRFR&)(drjy$HIrfwXY%1o2zYw#r4|wdiuRPDj@&r4}b8RiZkG;iXbMbs-JIKDo{_+?b%vZ9**o*TC#uyYx'
            'rx5|#tXR5CvS6v;nJ+QgsCFr<%^(zawf?34m900PH|%^J-jIJ|gUvH+BZWPr+O+ZiRtwi^At9CumBk<HKEsAwsTHOT>%#pq3&%go'
            '0>FuPsx|-M3KpOdWQ{C$obqCvX34%vfr?D8{8b>CvxxCkCqf-1Lu9r14D)~~PmK!KNt{gZ1QDU$cpMY0|906vd(-}}e;EseFDRTT'
            'un+;EIfU_p&DHIy^xx?(O>XQ~;7|%zUdkQ<MdnR&wP{Uzs<ooV5Tq68A}Y9A6=DmB84Qyv{kc8;h!`PPeXPkE<>f`Ttv-)*f4}dj'
            '5&goC9&wGNz(+{yc=h^W)l1_Ps+B?(ZdoIY3Vl#(ptgdBi6)Ba+CtZCPz?Ji+6%UOT^$p&xAaw{7zSIRBWHR9zt;6GiG91>y0Hi>'
            'lF*i^i|lym*jwe^`A#E4gW}G&yv)`xTFekuIbtR+b-7O2VXYFiw6WaeT;ioPKE`taPGrp1oX8myBdY`ATH5FE-@kUwI&V5xr@_yi'
            'U(ZfITwIpZCWjZAdHQ+m(q^yyo%Yc9`ZlKLyS*Kkz*=Kncl>_$avO{vP4ZM0!N;-Z3BGl00$;vap|CTt_B~AV>oXhH1y&a}LX7+H'
            'RIc=m2VP)2>ATk5=Y6FtMrGg4?N;M0X4La7M*gCnZ^WN);s!JUtcJH_30q}NJ{#QgRxdNNPB_bumIQvyVQpu-!ER9=jY&PKI?1G6'
            'yTJ(Yx*Yrft+<7;OH5<7b$;urnpLBLukcN}e=x+xbbWOAva95%>Va04B<r(9{0yTLCwDzfhplr?m4YA2aSFAXe-ju&XUW6z2)~QT'
            'hP}vEy*!J1N(m_=Vu})s=5k^u!vD$f2w(Szv#)p9`@t>ZLC!qfEwc?Qw7O3z!VrsT-IOBBNq!v(($kVs37hGfuG*}pZrXyu*qX}C'
            'RzBdMGaDpvOnD^4xR0?%&!V<f2UO?H&z+N#)AL_?C#P>Z=U1l}=Wu!d$IBl*CEi?|bj|{1NrlbRNK|tg--kOGk?(#_4)XaT=;Iwv'
            'LErsjMoq%!V6;edfD&7VIQ`8|@lG;i!j*@yn^&6bIaK+Udf9k3o<=Dr4G1OGi%JAn{22T_V6jrMCk&@~wnAQ`QSn^yol~4)k2t%r'
            'unw57Bc?IYwErHc627O2Y@_;7`k9{FxcFf`tekr&)5&eboJNUEwS?>&p>2s?;O0kYD0;2fEwyEZY|lr7+Zk_i62eNEW}fBKDmotA'
            'jU!qu)Tjp0JDk4@RAvc|6<1<oLvgZ;bPkKU=w)U(xgtGj#2tr7erA3{pHjUU^xc)+7oWyz(a>4fGwb40Zz6FV_5jRBPWTRs-;hE$'
            'mmnA<@v(-^I(`>AmG+8CdEymtkK$xOU7-;Vu3onra2W{+RrHaiVVyCHCMag;E=4``OH)Q(lv08hipW3%G_Aa@C!JL$8S_XABp(EQ'
            'YbV>nTDJIIii!UH+%KcWEC;y*6lzS*Iwz8by-Ej5mCt0m!^P-Fx4!4~v$z#M!tY{XH7m|Yx{S)F0uh#~$I*9&u|wC|3Xi5LoSQml'
            'QAPfDPUtnup6qb&vO&{;JRN>XE%hTWm4>EwL?nr0;25asFY6XU@fikT<LGHsFh3cS?Eo2z-=1JDwgxNENqBum^g9rs7im_oRVl{~'
            'Bkf@lt6IR(VnwIu0q6~X@2833-=63V&<%jbPN1=Q$^LD`5pn%>G$h-z__SAgFDE0x>0q&j<M3WrzmY7b!M)LljT^jXw^@c>c`mLw'
            'Wblr+5CyYwP2GAqTuPp#eiDg>Ec7S16n5P1-CI>{g3&mQ#{ry<s$cTfjm!ftr?A42q*6-qL(la@Y)|2p<+GaKE4$)Rlx*43rAPTQ'
            'tQ+J>ZVd~jrj`a7I_<S+6)c~#RKEOW)e}^h|FnwmP@vu^0oB)5Mx3g1+O2{jZ?wstiWy0+Z86O;_|n(>etNRfv<r$JvP%_WW(~uX'
            'NwA|mab!vWk@l$JAK0u&?}Nd%OlzU@njl((0-t}z?7EXxL<i+!I*Z`2lqEh*(tvh-D(6kSptwO;cb|gqf;Eler@ep<%Fp8^H=K*2'
            'jMj^c(x13AktEe&`M{0*Ls4=dCD*!Ms&rtLq%q)XTfqU31V0NOof{Nxd@C0ww0$21T&mzZc~sF$%@=pcR4LO1by$x#@Se8t_VnRn'
            'PY=j>hd~2@8Zw`z?i7t86!ml^2>#3mQ$hgl7daZ`#u91!uz7&)rNxS@K2jTVk}^9DrMbco<~hg725}BjFbdDj;3R>qi4r$*Ra14N'
            '@tshpQVtn)`uZ|qm;36B?4gY8fODWCX^Fdt-7zGF8s2*Jr<_jqxLVtUV!nwJrFlTqBDF*O1Ba$TZ?A)N_)^Uw=7HCVgnN24vMtkz'
            'OznurT1-Llqv?eH95ny*y&LR&qUysKZ*lZR!*r2h3>GqikY!1c?y6`z^&&T6IcC5ZpiaVVsy8*?EwOmAO^*1H(jXeZ^5hD2d;2JF'
            ';d^CeP^hFmpQ;jzi_?0mIR(Wu(<VUF$|ELKT5nMZ(BnU~jIp&PR&JK&Y7MDL1QVgAg_cc%7Xj6Yb6w_BO=0KMq()>pUAH9p!0ae!'
            '%&yJ4x@w__ouW5bn|-6VPtlypOR-{Y{P=^rHhxKyBnIg(Hn*Ly`7>j=VH&A%NN58AzQ#KmNVpdlSsUaDMXHKAe)oZyEcc)oc8W8I'
            'l@-B;F!6f`#H3zX7B1a|pr&6QxekXY058TdCD5pxdZT04;S7b16s}0M5~%$y#h?&+klWmOwq9L-(Igx<uPjv?A{zz0pb;D*L=FVd'
            'eSRYij_@kr<>sl^UH<ByuJeta0*etv6UBjoJt=EH{75MxU*@RTJ!&XGEv5lP45DtLXxGwAd-GhI&kl_--EiN{CDE0wqfgZyygh;&'
            'vnDsEY!SvP%5Ubj`NI#MEAuUvQ}&yH!OWO5p#i57E_pm<`8>fU&66l?=qBh3Q*9Y(id;$dox$pbl&IG|_e_ni6;%WlhS3$dX4s6}'
            'h6`qtMt69(iyiq@{fL6Y(=lPfVOw6jdS#l(suQH4g)%B?)a3p>vpy&L12hZM3_d;)wet`5Znc2AMQdQY<)iave-*U7*+pwub^Zxv'
            'U6bqdM3O7o0Q<}L=e_p%yVGAT+rOQ@`_Q}WTy@%)e|@b+rO)X+C};zT3}L`kyg6gobyz#=M4ap^*CFU4t6AT%Tg(Qai~GqeM!BUU'
            '6a(zM#eHp|WCXh8s#tN3)F8@s3ie9WT^oX}C-43g1iUs1eYg*x{(gBT`@tZOM+sUhsU95b_PdMA55jgP_yrw_1pA1>R#5PWqxEhk'
            '2sjz_(^Y0*`vp<aH>a)-2d4wt@XD5I**Z2%+oP@wn!bHeZQ4|>Cr>WQ!s=<ZJM@++jGeWxYHcwF2E(t?ELy{@IIko5&pcUY$iALN'
            '2w<8;!9!WNL(s7W?1a$uz2J5{9E?*2wHJ`~W|>z0%K+`(VJ#!vze-SlSYDe@sQi^D^(Q4cC>0Ax$XV6O-xI6P!K0ImKkB$;yH7M6'
            'PXe3`rzwMcHB2%m)A;8_irRFXr<f@%12F<b{XxFYC%mJARX)MI$2hZDG^sl&#%15O{2kSaQMaa`{HZ0?{1Z2spmQGDUN@ogT1K}7'
            'C*V=S9R-71?tkl^`?Yy^SIPW9jZrpUFVf?G>kY-jVs4|a;d6L4>jNbu@;e=zgqeIC)DHL5(r;{E-5!U;R}0fuuXu!=)W9|PHxHRk'
            'cMNBNHlaQ`@1<j7wekiP`I4-~&<)h?x1q&al2F4hk_0_aD1|;)daDYSZo*!zTlEp(*V3VU(mC&Zxayn*ktqX4Ao}IUE?)ROqV#4c'
            'LcvTUwpiuG-YXx-gb;?~IG=G&Ck7VJlF$00&5|{o3hVJ4>GwW62XZwQ*PM6_&ouG4B?0*H&`ulHS-HTr;5@CXEE=H?KXj0r3f<|#'
            'HZe_tLTFZqOE%c@LBOJoKfHEupBCu6C{`AmORKdg=-mfqtDFMm7Cm-{{QjxNU!_UXz?}5V9?NK`V3bE`tdflXss7bdbyM<2L_54e'
            '{OJ27sLS!s2!>AL+GRdRI&zYx<kmniyle-A#iy%Yfjy<nfbbaozxbY2RWy?6^p+ZIGl|yoXu?TmL)th+hAz+`E|Z&_NSqf)S{PHp'
            'h2dhkPKmWG4yb<om7;rz-yO#fWHBU&#xLRVlr=ulnvhj;xuOtiQFG3a;W>EqifG>VgPRBqAB~T<;|vMc_Jg<UE6kO9Le^jSJRJ|x'
            'MTRl|nD?omtK)?=kEZL4&?2z^ju|adorrgj`QBI2Y>7A!e9t~1^LDg7R@fhPrWBn;Y_!0hGal2cbCu_zBq{sne8={VZp-=0y3kQ@'
            'Z_Ml_$cM01KmraX(P{?6Q9hbKy#MvAbB+n|C})GNtp}=Vy9e4Zrq+1@zkSFj7->HV@azUzF2m$h(K?Mrj{J=R`!E8z9r4?vBuysM'
            'ac~<=rl634RCzmI28(H&kMJ^BC2LTHla+Pki$z8+Ww>|e<N2sYOA^-Vk$DCfQo#EU?aR}nt(lPt71Rt~M>AgL`?#P5MXSfk2wLG`'
            '8_#iq5xqMm(9J`G^>YrY3lxU2;m2^`j9N*r**Y>Y`jenl@L5O1{4WWOPcQ'
        ),
    },
    'provider_adapters': {
        'sha256': 'd33b57415c3c0d24c7df31477ec5c9ec45fd39faaf4d9a0519dd129127b5693a',
        'encoding': 'zlib+b85',
        'data': (
            'c-rNC{Z|{ulIZXHEBa1O4l~yp0d_X;9ujhvVA*Cb1~15Q?y{VrCAH98jb@bj00U3|`>U${n4S*_VUyg=TTWuosJptmy1u)*#&Mj}'
            'MII(GkGgD>im{0EkVpP?!PBuA!=oh2{6(7FgkzC1KIV&Dq#1KROnD~Q-@IO@^Xc6`*x=pKIeUNl`r~n*^?yA&ADnmm)}PL&sfbxV'
            '73^)2Tt|YXNs$W{W=wpZ@*>M&Q1|S^YnClwDqS{D#v)?h?R*E*3)4{%<}874JOt5MgbNXcv0&LWDWWkO3dV;3Gf&tkfrZ3{U`fn4'
            'dwulgP5-QaGGJyYzRzx_;b@9O^G+nL!#tdGn2KpJn)3KsbXlC_?6)GxxtziL25|&n-Asti)io_6;9>9J`1tfwaBz5d`tf8C{O8Bh'
            '!9j2a{|t^!Pp%MACrLgH<7+k*DXhDQGgicr$TAk@nT32rMVRG1cEILgmSK;XxWjp#oDLX|nMl(l-RmF{)rPQCERqzUT=5$oMu>#{'
            'HvmaovD+}80tPZ3!vgOCyQ$zAA|)QvHjH=_iL{f2^F?$A=)Msd;H<|^5(8Y^z&~M(>&_5VF&e?FCq)FXvLb?ALSS(M8<9*9=)5Cw'
            'Cl`VXh!PkF_Gij(!X#zxgu_m!KI{PObIP)u-(@tNMGCu+-myI8aW+cBp-8)(53tYac0YoN#K(M?QG04`1{e&^&CX?r>^@BTj>RIs'
            'P0|^yl3gbP2LXz^o+2K@_@82g1b0QFAqKCgCE~BkvV?}XohFg!038;QqEnHM!VJ2FF}s3M!pU7g>+3D<B));9i2(VC=EmbY#CpU;'
            'JYFPWoFgp|Qj0~D@Nw2<i((js*;L@4X{hilfF=OBfUgWj;Sp?K`MlS09H%o$lQ|25Ns$++2m%JX2&;e*foM5WY1ZkeXVZKh>A(L8'
            '6d?PKf#iYkh3=_8cNzYB4|^)xEqFeS!l7>V0X}x*H$B4v`b)RN;OEc~^qH7+=@F1i65WVEt&E;&!VDA`BJOU->vX=^V}}Xs1v{ta'
            'Ear3CKaq)25JgEcW0NEWZh;+6lq~CYKAfHYa`d`?7M%AF&-y?d`|N?nbFS0mH)oHz@{g`3r{RpEq!>>ko(lT1{21`jC@JDR2**yB'
            'RlX?lX^<x~5jz?RzsG$5niiue!Lc6%z{Uuo@+<|(lMsL<E%{80lKC7cr73+$RAvkrX2chK7)D`!*VBTON4LmC&>PxJ!Zgc&KzIsY'
            'DH1ZziJ}Qk$|gYR8A$H4(~oc8fz0e8K_?*g(_ZKB?DYIRFi3<ZgT95sX?Q&iek(XmAjrk%9M?);;SZperUYv}0RmFEQQr{KnS(-{'
            'huhbS+{ZQUzT6o)m4?=SK=XeeO}5u`(degn-6T;l7hg85-)=>`jsLvreeb{gKab9-Z12C?8T$Oy(4Q>+R-cxb55;&4q99u}A?I4m'
            '!#MO!OaJ9#Lk~G1V|Xq6`KxMEl111m1NIZ~Ncbe)fJx@0y5}IWfplWf@u2kYKp90|ea2e+5<k@@ZQ$30xcl?b+0n`2(Q%iJfvB^&'
            '7>AzIie}ZthxNb$ZeIByzJ|$$1hBrdSwa37vPowUFW3Wr!}?>;T(J=240M{Mmq{al{Sjs*7l9@c3-}0}1$qc15Neyu0(Oh$jv!qG'
            '+z&i$xEa8ZjniXC&>2mDp@2OS>;T0Bd>DZ_1e|LDo1_8+vt)GNXnJDgAs@{wGbm3(G1dS%NbhhAT3z%kbO=~xfLQ9{cKd@f|6n>7'
            '`H3h}*xt%kRHWLv=0=MBf@%?_;Pc$SO~V{$5S5Hvn(V^{MLGkW$8M4k?1^6I;J-ee^@G93v%epmzYBgjJvt<j<F*WE=Fp#uNKb(V'
            '9>C5V8p>fdtuvGe%Aizxh$ah66L2<9vYbWXOn?yt=HnQqFXgA)aLx`-Pu?88{djipGnlxq`)>|D9uMTk&%tU%oTosHYT1lm^DxeG'
            '1Tco(kAQU3tXGnB55)8~(h5BGfu?Cz*|y%Xp+LI<7!zTEHP$&iKK=Om&G7*s2(0Y>aqvt3?C8zWAsLqe*oA$>l%#xX`Jdzn?BGbP'
            'dlt@NkznwJKyow5r(ya7n-yU7#B)%qpaMW-Fh|0A(&USKfD4K%X5-{KEh64?P>*-UVq#RkCjY>)cU}E(511(JDg!6D7CDHQeUjD8'
            '|A~#mQNAaq<v3>&UZ!|Uc0o$l>_rURHRc$ZTjKK~A$yv$8y*$F4g~H@fCK?+{9#M?-s-X~!(aG88qN5ZHSeuG_ArA5?^SJ`<x;X)'
            '(-t>8tqppxWucoSuo&bs3K_7Y{Rz*3Uct)E70aTnhre;S47R#%-t+)j28|vy4JYKeP>(&s%qgvvJvh2C2ub}XnkIV6@R%lDiUIxO'
            'NL)aJiGwCcJO{ZJgB^Dr@w;#o0#Y=<zobhtnM#yFuZQK5VA>APzD8u#LUi$AjUGI-_GndTuF&cvJnbv(vx`v=EEDigvw-B_#m*&l'
            '8etD>*d7r<=6V`eBcSYf5?+Il8eeK8D!5G)RxtBr__T3(9o|PFSnhn9^OW6zS?fL^!S((LE`zIP;Vr!%N>qbHn}_#|=fJz;(5b<$'
            'qTTvbwp(_dNE3s!01Z@5+iC8v^=YY?e=MXcS8_j*cIIxv>F36C-70ryTj67i&G~{hSIPnH0GRprH0LyStrR-7ous+p+REV^B!rDb'
            'Se;rgEc9JAc!_>Mc7nWB3#xCTgob#%sPjgBsCJ^C88*4B?>qirlXC7-yg-MBd;t|c2Ttk8-f~-mY!)sSVqBNsk~-9f(o>-awf$}E'
            '$_g4Rp&*5s5AY2sA9#qwgMP)V$!U(7XGy=f#RVeOe1%<$d{N{YRN0$6QfE&stE(Oah%2HJLj^%|5Aj=|rg6X#uy(F{adPWw|GHP?'
            'Bd-@`2^#1;ca<q+kURnMY2vtsRuw4gf{Gj2fxI@JdOn=1F&{&^{Km$*T+=2<Zw(K)wN-kJO))$w;XLbxq~f<C1bf&7ja>7Wx2Ls+'
            'd4$RxbqlVqp+ih7X^0Sb_=khl>Vma9O+fn<DWG^NI0m-%C6#0NCKx8;JM|fs!PTF*2*Bz7_d8xWnu^Qd#6<><2DlRP5eU_Y--i(p'
            'QBL2eAcBmHfl*RUu$(!3jzMDW8+bkSNBfl3Z#nCIGxct}Fs%R3((s>hY3jFU0OaZ2UIp5M-(g&hrYL9I!*SiR3lNZ4<g8D>Kx3@x'
            '|KJ2EzYT_WfF2S73LHlFfcBR44K)oUtp_9EkgkT;vDr`WGf>qx%zwTgfc6%=>}Lkrd3$#HpTXg~gF*1|>=+Q^O!It^?QL)KMc6~P'
            'o}2>Zq2KIokH9n9hDVD?P#DsY5C&%lC+CM}M?d#Bgu{z*m~2<V5dceaB8F-M>_f30l#ug9uEgPAl|-bEI8IyisLj=ArAxyGipy%!'
            'm)6k;iM2wS^UpG5n1O2vrvE<M-Ff+4)9N{e=^h;Uh(Vr*=;b3uS&30R*uahMv2zf#85>1{$KbqDygyFR!!WiFSu~OxR$DMSI8!w$'
            'Y*&K7R1BJiKr<$RfhPMXgYnWvomX=MYHk$FS9P}S%gYN1S=v(IJt3E_70I23h>*P5wb&ftxK+dJ!|C||=3k-v>W*pw_uw29AotSn'
            'o~vlt#Q9l(pPaIXa_Z9ASUM}@o^vQAia)qpknzbEizpls#BCH_&a&FG)Bz6-eOwY5J~SUzid<(dGH@zL#9cf%X+pNX1FmXH_Pv~%'
            'v<`s9mY3FCN|_2&A~DPUvd>=b>{MlH%0o~ZXGM(SyH8qt;=CISKClPbYq^JirLNi)?_v*NTrM3;&SU&$UvC@G#grmPKodJa7HSPh'
            'A$r1>Yu#QGjeMe^LpEg7Ena8uKtJ2o!G&~78>e0D0QzA$0q9hY`VX-$7f!yu{MHWBeT{`9^;lggGLENRqb8FplT;(nMp?TVicz3%'
            'Y}}%}!vGuxJM{U5I?|zI2nx>OQ3Pu}iI8!%5eB0ukT*A{Aj;apBUT-|6Q9MXKvQRUM<=bxZLE$!y7CAZp}|8L5|k&s5k^PT@{m*='
            'Jr4o3vzGIbLI2l5aCmThOlKt8Qq*VAqz(ca(p6@xJUHDWncrnK&9%}p`f{}+<z7?S)ivA7zB<Mnly-Ku?|Mzeb;wij!#R6%aCF>%'
            't+$0(S4PYMDe_>ocDe=nEsRw}PUi;T895zg(M|wG>ivTS%riX0&!@QK6x~~-!WV!?JWucHSeumowR8gtzSD8oG>OK|eebqyI!?hr'
            '1MC+gJaZlUlB+SG7>-F?_t+^aVhW>DfVd^aVVsCe0mD!<U<`FRjT>9>YlV)3GOU?_ewRrLa?u%ZBAy{&+*`6IKo_x$+JX@vz%-Cc'
            '-;x7f8RgG`3=mycgj~=!0-b*hi33EK(L$7=0ePT+#zC&Y7=i#Kk0l6mKx<rOb%f5-Je=?mdLSd<>P%%`kgOnR@EwpaS}$pm3+jUu'
            'W0o&5NXV43hvF8z%w<TRi#6L39e@Mrd`23<?1zcS@09sH0nrH}ikpa`eZ-@7v&@Wu4k}B4qhCbZj6lQ8vgYAiP;5ytnu>7`<H_kA'
            '0tTx9bRNccZYk&EmX2p4Uc{74qvwK#U?TB=Ysqd=!q`z?DJ_8Ak<+Ro1hWGkmc~qz52c1s>c?aI`%-hYI4kIY=(w^p5KVW>p*G;#'
            'qyTNbjK`LqZut<OfIx5SxoScZ6^c4xR#6DPr9HI<3Bw`*4;Popq(n)AabOoIQt2caH?|(Ytg6z^ZB;ciqGhz=Kq&XnCI(<=>*$k3'
            'X>K%!krOIcOUWqvY^4sNArVBywxV8`1XKwm5Im)4%tM717y~;g$4Jd*Uuo#7GN^o7sRc@9Vg9nU|9;)JxA(Cp1&muU!6crHnWfsw'
            'O_mS6HE{xIjt!F}aw|RIvA4EgX{t3l>+jY>(573Zt9-bAWNi{w$5foUtZaB=O~h1aB3HkWRwWUp<9I!25H4J=W_?TUDHdoCx(`jG'
            'vA!wI7Ejxx(%5aC>?Y=JrDN^eHUL#~d)4=WIlFLM_sKDaLe8_0<pp?7<M0OtLjxy2>@1o4u)FkF&Qm@CDFz}N%z!+M#-Y;+u&0b{'
            'N$YLcYa=zMqF+~g9j4i1%aSr`4;J9)8IgQW(sreR)TF|+t@CTjV6$Uct5X8TUSLsKb3GRIuh0qd?~9hPFG1Gn50$31aj_katbzh~'
            'ybkgkM36jXBZ*!L@W2Rfi4XTKnR!kHewC;$%_%%7lqO<zL(Q$G?F%+>*#AXgn(}+=gzr9#!43_<+aWR&m{bE4NK;>WhTWmB0#-v?'
            'Jy2!`(p(Cz$>1i4Lj2VdI9ri!vp#;ifvW-58JMnsXEPE_r+3S5oVIwFI@qhMz?c<1)|yfJ;q2)B!P!5IqmY1MF0q83#jQ?7R+gUl'
            'j3JjW2RoE!vrO5TW{CfZ(X?8k)}JS&Rt*F{M_{cMAESeVW@rqFE<mb7Xo>*!V;$wU*y3Lk=-ND5RonmHFJx7TJyD#&dpv^mhex6s'
            '=vWhudgeoy)~u`NK82wJLMsEgNWpqF>B;s<aCz7czbogylgg1xEvsu3g@B^%oA0(2*lhcu9ALT4!)*|q+YdCxvT_h(O>2YWm{keA'
            'Yt+iugs(OWoz>LNW`e$wr}|9dz6$fPBNVkkE2VpTF1r->bTY6}&obHlk5UvXC|wGC?bbhCb<iSKXpPFL%Aev;#KeXw5?Vf$X6D1P'
            'eu84#rTWC$p^!$HVcIH>N5a*2dypSInWTg~X0HtC8bLm~C_5+hjxsvH^UfaAc-0K>|DmTeG-x0bf8}vq0BAu{c|2$L4_29qP>j;7'
            ';CIy_4snZTC81-aSxcPI1pV~2B#Ig%UWY}l&R=ApT8yR%22ta?TYlH8&FVs78U3gBjGtP&uAYLoDu&pqn97S7F79r&LNVD_D>lv%'
            'Se-BMCs8Z*HI+YwVxgQwjPuf7Ue@H7+F&w+NP1BinV$Z+^^&^SM63VT=$sod%99jP@49Q$jwc_=fSroa*VTst^&|~q==6DXiB0#v'
            'pEgkoeO#f8Va;(Si39voS9QGDx%B?r>Zj_lRX3!v1ug35`Te!(RiwfB@8PPE`jMBX@;q~;V@j@ua*KdE0=!vEdWh~p89mf;>8bNY'
            'PEj+KR$qE~fea&F`dU`Z(^Q^k%i!yAI1?(%o-SL%v}L9SsCeKsJS6SP3pnFqgeSD2OvkuN#Z4$~wK$h&#RT!B4<rdNra--P<9UHi'
            '6OdbkiUh{m%J5(s_!#f>+DD=wKB$pCUu?TPwwm}F7ux6=906RZ+kZj9tmWlcWVx6Dd&<i%h%=Z=hH=yf0`r%2$(#*RTqlk@BtRmD'
            'uB4*cmf~To*L#BbP+Q?M84z`-&2VI{B{AM`TBZFPm!dCKfC8^VZ`Mm)n|`9jfO!CcWJ73=H{yFD17#p=-jt#FE@gzMt6QpfHjgR+'
            '{>j#lFdaSv;hv(z-gkC;I}ZCikM<%SU++sMc7EEjdnmHybFuG8T3I?jJy<ha{<uw#fBJEopsVg*XS8&xM5jYQ4-${c>gd7t^(2CO'
            '^AL|tl1VP(Ldi91+8Y7h?+qs1ZQtxVD@>fGRyiFl{DVa}_s@%=iO%_d7pQ$PPgNRDohRi);%2-6Ot+o%{p!Ehv-eL)0ly(-g)l%-'
            'PW+eO&8GMM?)Sqm_vcBR_>0lJ2B<BZjAQmhj7i<DQ{)r>Z}vgt^BG{4F+;KFx_58oq}RQ6)vn^U>-6)Q=|!L)H@u{_K#>n-7K%7F'
            'PU3Nqjtgs=t7=4`84ysh2;GuCxmxjgy#-hRU^~xe;}A_w{E+R-YdTciBS>cS(OU_D?iWjR$P=wcOu@SFuLaceiWMkz!k1ftHRV?^'
            '3DL+l%V>cs&mr*JM&n1VNUc=^ty2T7N7vl!{Nt>@bqZJ-ejpcq53`Rh_aia8MnuSGu=KId(I_=#TDsNy|K4+(h^z0(Ya5JartZhf'
            '4|IhYcyL<8;}lC2B+3xV5L{W*Cx@k6s}e>TEN%|EEV@$}d0>^vJMACO4hH?#<~sX%e;~^pVR0n@N2PBxuswD~16^TQKwxGT*0s46'
            'QRGu;i7~tS{MrBYSN{Z5n=~%3SUBz>pf_QJNqe&J5vN?ee2U3#GSd!h<q*{i5Kb0iQ3aVcI0urY{5+Y>M|bimSGUM<*;IOgt~<0Q'
            'ZtW>!6;q65C`sithPspto$peO6x8}wax&fMSN0WF1K_4A5F3gpxm}b0TTQXi#b&BERoW`fv>l;zJ4<AN{_D42RTe?*Za{FTf4hG{'
            '1ZnsqRhQ!5f9SuJssbob-m#_qGwwmFCxD=oCy=q1-?7=d`|aKDebBetsBhVa!}sjd!7s9M)D>`lkQQ+)QoL9N^cLLmn_vVqKFh8U'
            '6eUVxZfU|WPlJ35Jio{EN-2pz-%Tc1rhtLb0MpD(p)J=UIJwz;G?m$zg2~||DNzU$RGWh6sq9(GW+W|)A^((*@}Arhpt27i2f_Kl'
            '`wz$c067Wp`trMAXJ@C<;PBnS$w{A5!*}%^LP{eIh@XOR?3S({d25)JX1ukl#!g-N*%WNnaarAfXPH{E%)Ui#*L-X87C^aDIwfin'
            'X@u&GF|ew@s_?5BBrXOYsE@~f+Gnq5tIRVDM_^*<OM0vyfo9;*V#?iWDo?fNZnNjE>=_sHmY$TOC)@ehx9kQVFZQT!0SY*EJzZ@='
            ')>6PAvUV4cC&|n#ePVMkiY~DhX|iGnAOs8A?Dlp&^l7v38dH2>h!K`+aJOFU?tHuDQE9FdDyu?pun^t>dk?&J^x^mjT-&2_Mn&QJ'
            'uj%`B%77|nQ5s67vzmd<9T-Ptwm4nw>LQDRUwjMpgd)|RDq*1rfPm50m>dMp$T~whUPOGPv%)Exn*5_k^05-DQfgARDGK9mNAJU&'
            '&#?X&HT08-pHhi8b@#K9%?$GeZv<d*m7mT=5uY!b9z1S{&S^zt$_JRQ+{{*XUhI5pdRi8*in=22?9~stWR(w<5!p)xt>1Mh<IAbo'
            '>y&Pe3Gm2LuJf(Rw#3IsVze1Ae5ZW4yXO<-jl*kTo?5ykW^WEN7i<VuLO{X=He$PVXP~sYEAfR{5*OK=vuSuAG7dZsqkxFZFyRnG'
            '4A>S!4DJgbKz{2vt+{hJc4y5f*J6Z@>pioFv%#;4V3ctSCfR`v!%;X7!56;c3^<2fhr<vaVm5xd;IUVxkvDe_d7L7|MB|OGU9!d|'
            ';Kk&gh3YR@l2ZwO?DYPP{kSgy%dtHC{Ia*Rc96>M$knaiB^bvCpc(~QknCgTzndmt%eii45<XyX$76p*g7Lg+`o_c7i!EG$h62KQ'
            'M>=|2{?_sV2~U0HEx&bENkmyCZKX{7|Cu|V${lEVWZ*^JiL9NJ{s?+rKZ$<<Md=Id0TmL<?qb*(d1U@q<H@1g#SchoUADlhOP*E2'
            '36urdo2YDhrVNuiUx@2KrK8>^={OkTdUzTNs_co~kE-Ph)b+54>TKdA9GTk*hUOJb?26s;%+&j=mG&h4rb?Yw>U$=W300RJ%37oL'
            'Aq&e2(2ZG5qp9MXFytsqI`%VNY)##5!4|UG1DL%wd1CFyBE~^5#H)Mt4$H?1#}<P33Co+6Q&8u3v?2SIec6}0Qw=;JA#2pbd~i?D'
            '?7wrt_OZ_L<a(&Y658yPHC@gaIb860i;kq$7fz1hy-N}-G^*27k<tNym1BHUb)$2|%c|lNQ7hO_Kt(^xmE5&J|3-%ZWD#B95Fc-Y'
            'hND^bgK`S8vV>+<4D&EY2LybQUISQ(F>~dmJtJS8<I-#^d6k`2S4UE8FPU4d5lA!>IjnLUW{sW){@-P<Rsj)I9!1HAI*b<@bK^p~'
            'c7WXt052QyHaiw)lTf7qhGLT*a)4w5ZM_F10P-YQjOOSGw&tl<j?N|%DKo;jRAuEoDaC`ZZ{U(*>Eh02Axm@X%|LJ7zmg?zQ>@pT'
            '21Oo5Sy^pGp3juMB@Rr_66*e{#;ylW74^`|#bb_Y{Il})6T_+Buuo~2=a>eckg-0DWWi!A6OBQVF^Wnml?>7`Efy$pN7BJoc|DQf'
            'ScZ=)cscAOGQB$gBPRp=3v8Gqvy4hCVp4q=W4Yt86-JPJAP+gjd;tqJi$bj*WW{6>em3jw$<IKv4lX*^icsz_y18b%iNp-vFVO!j'
            '@r<!le;STOFyt74K;y%YFkEyaD59b&2*J}k`-s5tLwtxO$oF>W>l@0y_BkF8jOaTc-)#4LtT}4;<0&1TrS6I-f+wN0G}4g-uL5sE'
            '6;=6->VU}etXf}TyeRU0i-joHi|R$oNVFmkP@AKfecXixfkq&)EG`UBG<LLMgF}Il72sC8WbV)tjDEX#GbI4Sg$%|W+xCMNMGSN{'
            '(?#u?A^x$*c}Qn%dInhz43^>$Lkw8}hOI*wsz^uHV|o8gX03GA9=e!d7Z_I|#-p}48;P3~mV@_fz|4{pF{Ym6>_iE&Br`@UuBwzt'
            'o}GTCmH3m6|C=$pHi1ssU!M6Zl`cP!DmR$~FLvJDBW<7|v@(6IO@daE23Cj50f9YsLiKV_POW&k6&0b^1HgZScYStx(q{*+4?Ya~'
            'XYA9_;2qF5J79-r{n!1I!O_7ndkZ}9SiIzsNQxb(G?&6D8i-YQ<#CNP5i#+gQjD`#t^cdfvQ_b4XeAHIWuTZsA1J(_yQ6B#F8FB1'
            '0Rq|lkt}mY7wjulDorblIZ^E%npZVCPM6*B^v(BKhF8;c1hT83$)G2i>@sGgP(Dx*elk~Nw5OD_;04}@^6t}HSqpj0(TtWw5Oif?'
            '%rTR6kCXg@S0t>-9Kp@#s*3-cB#g*p5jmSp6GreN#@M!C(S*|%_6d|O$XYS~u$~ARv$EW>tW|K0XWy3T$FUq4Ze*4e;2Eh}4pgBD'
            '>!^X}0c@V~+I6)B&$31o43XCw5cXD|pH+%CKQ(#9_>{D4bS$YX@OAjq<P|#x)QD2+fRIUCF$pJWGuxu&+VV3lxfJv;qgY+##nbF0'
            ';&XnlCs@g?K-n6oh{uX}GX7&FMar`CE4%1`3;pu!W7Be`kDi;ZLIU>kB)7?olzl1hHR9#NhV|O0oL6&K(V~l!-}oMT^WDxa^Q+Pi'
            '*lZ}uJC*AWf=fU@ANO~5cH81HWtvgB%Fg=SDQxKDJQsWho62TIT0{kvwT|$XEO>xeE7I!)bmuP!uzc=%%63*(*R`{oY(%#NLOM-)'
            'YV)TFg3SzSJNN6}eXt@!<`LIkb!&I<pOyt;IA~20NW0%=n>m)1VOijtS5~gj{|0#*oo2ZRT><tfIUAk!$_Dk6j4#aBcyg|Wqh^*_'
            'DSs<kTH*@*{*si;SCUc)V;0G@LLmae3s3mF-h^XeLOsw5&!%3WdWs$w6^(B5a%!z>kwDCj#%xK8p=1e|W)<u&tM^*gN#0bbfqaw&'
            'jxfY1z2gZVrW6Dlf2l}+{)?*BvRfwicbtNWlRkHj3uQ(qbDxT%tw@R3ARS^^IXp*Frn+j99YGo2c6AZ1MAZ_vk22)*rV7;8hWKs$'
            '{u=_M6L|`U@LO%7)b7Gntp*9rjE7LX&iab8st*BFFH0!mpjx;>E>-VU^=|EERpHjvm3zKxw)Ik{jY=K5>glTufruE>ZpN}wk1F2L'
            '*yo0`Iv{yaD==tY4@SoQRf)DMQ*v;efb)WH%^B+#Ayr-wk;qnWj?T^p`ZZ`h_QAlF2`ToA4d@*t%H@N<rGcjWPge^IlqXl}>6Mlo'
            'JhS{is;*W(RzrKzmH1S?V#a>COc9rFh>=+#bpH<>ETC}=o{jLXcYtB^EA%iWBWZ`PF2Aa7i&dTL;?lP6^?o%|Fk5S`*K6`P{S$n_'
            'N>IzMdUJ4m{PV%#-(`5p>8`C`q$<JJ3&U3mjknjYxwup>M4)nWur!%4Mz0r;sbewQq)1p-R|unCSc$(;B}}~@j_S|sbefg;P3<t6'
            'i)BWtgDT(zTe7yi|3l{?lqXxw%3*7Zb6KU}H!1l3Sg9>#cxgMxp%+hWCwT%oH=KbhHs^QTdXI=xs|VDma<<V$h*j@psOM@|uRsvP'
            'm0R@FJ_<8;w@at1t#Zw)oo-Rl$n}FXi@3?rUSOVB<JPX)$LA~N_GRVVwC<(*7VV1Ix?FC`eU%d1E6WsKARS${T`iYo7K3aw6?1Mb'
            'tgH68H{~=GUE8Rxc5f?$rb{WxPu{vxX*R7t=@ka+%Ennv<|v%yV0ADPNy!8<0xRz3yzx$#R(R&62WH)-IH}qbo0Xo*^7eLaT9VJ+'
            'u8OZ!DTrmXfIM^9q}*PkK4raTo;`AF(#J#NhGs(rIo<X*Vzi&AR`GP9K3`U@erjS}0b*S#wdHEUt_i6hpDd$Rh#dJ<3f2i;_z9%I'
            'soZ<AUZI0uy<pCN$NwF_{`g=0<H7s>9y<m5c^1msxhn{}*Aj+t4uBajA}4dm-t%-e#$vw+V2&akAVzU`KjMr+m{G|7<NWkQg(7-Y'
            'fQ^K>^v;r^jrBc?K>b3QY9<)i($!rhUZLS?>H3&eO*~RUz^al){<h61!3Y;+;MTjzL*nYJtqQQ-R~Z`CV2=wOTkyy^*MyFN^#qwC'
            'g9jy<5R{d7fZD~CjWZ>#w4L0x37e+yxRj8mcM`YgDZQC9F78E?qw<j)I_9gikLT5lFl*43Jl5Yw#MZ<tTGVY)wnmF<U_CBx6`cgK'
            'P%qZd<+bXtsyJX6o1UIkvo(`UWhzZJU%6DGsNsy!%f{>($QOiZgQK&$pegwMBbjiERV4k=$dIpnWFUC(eUJEZxgToL?ao?}QJ>A4'
            ')=pl`n-c)x4I-T0J2~VTPhl4KKS<k$ef-Dkk7pl`-ygg_vf2E3T)JvW(&g#^OId-S46~FqVxBE1vL>qW6aZ^uD=(h}3KpB($`_b_'
            '`FyB*P_3lX`ugGR1yA{6n(|Ee_RCrQ_hB3!zxR)KU-~yMTMB?$^*?@o+xOqUQpNJYWwh#c%A6(nw)x7tUukOQc-%xobU}JNPL_o>'
            '{A_`DBtr5a@%7sJRt8-PlTnd*_*vKUP<JgbZOl52zmneu1nWvim*@x|Uy%CQT--^;u)xy6@ik1L^<?;K11h5x;ocb-B2oCehf-Pt'
            'MXl7!ezMbqjGy*+38!_oTAMw#p)XXcQ1bQukkR(W`uG>PMy1Ko<`-W$HsMk^u8D<*y$987i&nmMxh$WUvNiD3X$8GO)#lYNC_vp>'
            '3MzFUshx^;*PL%tx~3LPLFK*Z>gfAieK{z<t6D%`S9GdbLaAjy*Al0mE;v8~d%Zj%kX2{~`S`YkYRw=z{!oL_jCM9R%*$83YwEWq'
            'wvj<jo`V|RYdv<g&qbTYW@~w<l{fp%bT_H*w9!(kXU31$cCw2%nZMV7udo{y1Cx6A!I~WKn1Qr(zRIlK=&Jr#QpK|bhbJ;RY*qS_'
            'FDMT&IzWk;EvX$k%Jq90@6Tbgx7zI`ftKo7zJ4GE*UYRO2|86a^IhW=2TxVoohQ)F>Ql+wvX&poZ~cNYiY>JiyI)~De^G(0O_Odn'
            '>RV<A=^FUewA;;~NJWUSDoAMB%#^ev-=F1HOIax6nQFWA)}L=mo0&jGf`hHBlP0l_g36M`4B0&^r__4amdPf?i`4W)Rk0Eal`tu*'
            'I}8OIf*hI2_q4UWoQbIfTMT~86Ic`8B~ZC%czr>?LrNu-S_RBhvA0yrd8l6V1{21VQ_XHeF{L*c{V%Y=tL79@QOBlFZF9Hft=lZ!'
            'o~iZ-1=nt#NmeMi;?WGT^@_<(rP27pXHT=%%O19^zR~N1%C28x!k4{U4fx8uYUVqB_aqBjc2MMttr}TxEzQ>?sv!tkX`3C7y*r(z'
            'Ti03|aTSd{;e8{U`hX2;Y7IU$iWP0-3O={zx6aX1!7cBpFUeW!6;|JH^9YUT#v4-zUDVvrHrunRPs^Jfq&@HjpkZv@1&EnGI$uUr'
            '&_Jc9E?RVzmZ|_F{exFG@cn_PZ19$Fc|tGmH8`?DCK+<Qwb*G|g)ML7`G4~&{Q&'
        ),
    },
    'narration_presets': {
        'sha256': 'e20513c4b0498c302ef1dc00560e424a09bc306ba8dcd504158b89f2be2034b1',
        'encoding': 'zlib+b85',
        'data': (
            'c-oy<ZExE~68_F#F=bI$3MBQ@0S6nogU?n{UsAiU+!RF-$SZOsZN<0ll2(+!_uKE8S&||pIW3wGt;L<4otI~xncXl9Z&j*e%9ORL'
            'G<im@+$GwblcJ~GM4Fa1PpMpMvU+7)X?Ij4mBVtZ6Z53;J2qBFymt}=zx?nkUD8kY)7kWaE^n{t_U?fmez~JJ_tPo8xtYGdo(1E-'
            '@Gtlvr>iVJaS-KMvZGXOG&!9)O=Z5)u+dH^yjm-pXCZA(xyE+VpCVK0I4G?$iMAaj#!;4+;#;hB0nc;0D|NzBD%<55j&~abXPMiP'
            'SfVIWRxhh0*lST`siHNc%A6kG=*$}XNP{?!s#Iq<m?9XYqEcH#&qU|+YRVR=MWSB~X=zsL(gkdT54Xu&sZ~t*lH6JqCR>pyC-jpt'
            'mj8xeN>vuaM7A$O@+1yU+=$k!Be43(=q;Zx5~uhtC>D_-K}=#D83*INS5cl8$_uH;=FX9=GKXY7u)Nhym&6%xJssE6tDD))-7SvD'
            '*~e_<ZS{ZG2C?Ls3sh0q{7EIWR&Fh4nhZZptjbzYoX?*m@-d($9yZ%pM3uDFRtGDqB&x`XKY^Xi0mv35<1ua5CR%fTEr*w}JX`0s'
            'tQBInfCB*x_PDRn&=*no#IOp24|g|LQ<_cRPp=-}HQmev+Bdf!rne8%>p#PEFcsS##m`5APUNj}I4`kC_YN=xJUGCQt2|#Ndalgm'
            '?exc++nf2r!)z|c&mKqKwIDUt=C+P;wMdL(1isa&$qYfh1{1ymJCCw?e)YrU{pHoe^!{e{aC0?#M3kf4#-b!ZEXG^>IzTJE6=SG@'
            'vy-@M03MZ)#gae)z#e_H&`G`xu#CS*RWVl|A;|EV^9Tq5uUx`Z+DhzepfLAfbA#a{Em5tGlDvvh>1IhPsFA1{B#e94QWPJ4`O^)o'
            ')IN!>DhlL#5`<wG2!YM#%c`ubp3ljog<v6rq@>;jUS`rpzL7afKoER;PFxl**7rU@-nIz(4ZvJOw&fTU%Y>C(<%Qw3hGh?PIL-^P'
            'k%TcWsB(}N_y`Arbbdz=&2onbL7me{!)_UEttoNNh%yNpWZ4=-Ju2JSnRpVM#0R~mF9G3S!>H#J-emA0MbMsfIP!~cK^fI2Ui?vU'
            'fSd1^$x3Ht!3%G#dV<HhX6Z&1x~v>8T_sxC=KH;JDq$WsFPt?~u}Vs{(v}6@W%)MI@k(=G_Qz%>Q7Ny#F*a>J-&ucN_uGtFTZt+Z'
            'XV!e4W|2-*?B)N1;w;y#<(}SUN5kRG(z_hK?@xFRg4@gc`%C7Exo?M^h;qA`ciIg$`W0E-=pAc15*qDA5``*bq3grOHJIKG`;ytJ'
            'LdfP(9*coG8~;4(N(c@l89vYZrElpMAgDyK#`%`ZIZ}Whh1)fNw+3O#)_;f0oeFdUg8!q^&bLMKT}a!Gx~)kP?y0`-Fx=N59%jMJ'
            'LDK7axwQP?(a~os0AQR~sCedgraPv8;c@N^OVkO{IXQ54?kQg-?Dn8t?0}su>DZwhz3Slc6?c?WD+j#CBTzTDImJmJ04FIa?CVSw'
            'Q)u0vM*Hrx<*uf<C`+3odvMrfn$L)OInA_=-Q-Z~Jv&cu{+v86Y0w)jjBgLk+@mvXS9RWMDm_prEyyuVD1<4#pt^>kw<%$$g}#_+'
            'maAPQ$e!;BrT)a_e{rbP^8-|Pdf3|)CAs6&W7dFW<Eo$q@`(A$8e#En{10^2P5TKPh<Q#IXw-jH=pp6qg18R9=W_qPrUD$zHHm|{'
            '(Q+yev?=Gb6>Z%n#nm_2OAb2)<1L<7g^-))zi!b;f9Ot`2zS8~jm(JbyNj$TO14G$d0WOl=4GCuX@<UpiFL0Lf&fWmHFsOkR<x06'
            'ZK2nShaU_AKz)dI+u|o|=b5adr*jG;7>tl`A4dWc^lXmizHDUyfJJNR)K5Dv&07zTYv`T>EkINLqM(aWT^UjeJ#2f07#;VOuk7v!'
            '?XyCw4XFhe`#;A2{Ei$fZjz<oDrz&~D8Xb`mp2tE(&roo6g?ugJ1We!h8cStu7pp(mkGUNX7zW2da#UCl8(C7$|QE6abLiwa&eU@'
            '&*d1|*~hwSWaqj{6_Wj)XPUIYWQ=$%6{HVTZl&8}u&y`5jcr~fIKJfDPst2N2|9dofe4BhZ1O@Qt{TEfITz_+v5QN7&cQ@YGW-Dm'
            'n0||Tz^Y?VFzGu!4Ht!!GYaEz2RbqoX)T;*Vv(tsF(Zy{J#NDo;634)9N1O-A4vH5ZD}}RhZ<}5CUor`LtU>&jh8s7D(j0G6BdYA'
            'tybdpQC`v<s2;a>z&fQb6gc=z3iKqnuTc`XIPG+4+y+CIwi^v#W(jBAT+~R6(KP?}9e_%)S}WSAC*?-8sj*X`u#SAKkpo>Rm%>UV'
            '6Z%luRYj-@ZXnfXy)&Q@zes9>J6xt{VJ!Yk!2bst2OGGF`(gdRC)R~Vm8#D}#1K&0fF4z%EJ2fElRoE`KRNSxLYE5#R!q#H)}>Xe'
            'l&APStLWFecc){B(LuVLXOng;aJ~NHFRm@0O@uk+r*v}WYuC{Tm|*&-FB>gQ3Cr*;-E)KV)GeblDk~`#tL1dmbce!a_k0P$;W=**'
            '0`^&zMMjt-98n1ObXYs|JZi?ar2>h)Fs=EuQ$yMZ1xnf(zBLc{S;69ZY(9T=OuL2HsBye<%?(~<J8DEvOmH%Uj=nbSh)C0d3sV3Z'
            'nLcxbtbmjaD2{4R+P*mnU+8^e<Da#LFnqDF!Wps;)YTy&ZV04y3K-0cy`Z)F3{V5!!lG2|8u<1tI!NU{XPCjSr*AHQe*Z8(oW(JU'
            'w;4Mc)aUP}zw)qNRz;!*M`Qdj2m<V-<6A3@|0P%Ma}Sr08=+&+s|N81e3kihPH37I<xV#6FbOfw$@RU=HMdv*m#?v`5-7#a)WNvl'
            'KFSv!y@0?`dq74<22XGUPhsA98#LUfz5H)Ac0LNBV)N0B_q<pAjR6x?@I=$6MzJUS0&||v>Fi5=!kL_K_VM|wd$;9|!k(iLn9+9m'
            '=749svO%>2QB_Hb!JoHmOp5klhT*UkZV`!~_#-E{Lk>Df9d`ZRi_1e(`|I!d-@3cgF}eVoc=n}L3YW!4?_g~QhPcXRt)?1GFg#D5'
            '^n71L-j{(S_tl7c1o4_nPv91tXb(50PcGx$d+s91qmAICLE(t|JUXGn#Ync^i%I^Tqxj7Zfnde$jSs`eDwp@adVuY{BKikC&(u6X'
            '`=-uMeNc(P6uafA3-jrJXLq+_SNf+(Kb}&)_#JFP2e-8b^76v@nuiz;1zY}7O%Ep!GR0t6mx|QG>l5X`@L;uGu&o6yIgaZ-;}RqK'
            'ppr^Y`Ms?V_hq<1d(2Y3(^=jBQ-0NA1~X_8e8e~Q`2g3z^g?eE+{QA<Fgyk)`Dggtfc$S2>yhcpur>{810>l3Z@KU*zrv#J?{W*H'
            '4hk1O>@68Irt}-uN5MevThD7=eQD=6ulpN^e?0*b<L5T$d((Rg!99f03=_D$ILd|th!G12V6NK$_Lz9S^tM0u3%Aqy{b^`AjcYs$'
            'd_3{Aa*)42ZJLezsnw5PnHpXSF{-Bbscw-q_xN3Nc}I@s0?8NHrq|rAjROtr#{T`fZh|8W@<}ProkA73<NHQgGf<<rZE4b*`u*%{'
            't?Y@haM;WG=6(%qM=j=W0+avoMbKo`9_Zpq(;2?>r)EDT_HnYG5cd}F?@orlilV=q{{wyfDXiJk{d&@ZdH*#ITTi|l3%wqgIy|Yb'
            'JD2u0bXGc+Jom~_f*t%HK~fHW'
        ),
    },
    'source_strategy': {
        'sha256': '68239bf36d7adc6fb353bfe38fd23a021167efefa6cd9fd2419a59f31ae34ad3',
        'encoding': 'zlib+b85',
        'data': (
            'c-qx{Yj@kWvgmjH3Y7gIv6W;yr|sRxsk@dEtEo;LJJ)vFo$c!-Q4%GwqDXB%;>fP=Z@=>ZK!6WN?ruL^ue}?aBnH4>FfYsu9LIUT'
            '@={+Mie695yeN)>nHP#Q&XbuhyfpRGbmK>v@S?fcCh-jZ4x(i*%@Qy3mpicyy(oew{Z{L@Up|WKUq)Br-NosLv!NLNb98lm)#}y$'
            'v|jm(IPpcc3R1C&=Xr>~`(opznV*O#&irW{uf^~AX6w%d_7^NynOG$8MmU}t;iq8k$1VSJkY=ePvRF7t9>HL}aF+&Y7ePh<b2s%x'
            'I!l6WW<9#TzH++aF348Yw>QgjFKody=lLuP;z;Oe`q3<&!#Lt7+=&2s5Mdkx-X`xy?}ulj^WhjKJ)B?zM25Itc^Pyv_eGGkg3UGt'
            'v`uk}vFLcN$owz_7^{5aMFQ~_M}E;+ng?0RnAv(s7R>U{OXQ5Zt<KaH(*Twu!#?KzEJ$&v02${X@e*X>&P#a&fafl)UuWiuW$q<j'
            'l=(h4_gb?s*!E~zh`GrB1lZ5we75pa7~$3n<_G`I<IL;L;wa5GFgr1LKZ4~}gcIOU{e+kE-QfImaD8$4Q4tJa5&&g)vDo?vEJ;Sh'
            '1e00-w$tNQOTbU!r+JvA2R!(3L3aJ$Q<$1S;=IG*-T{I5UYaMc6yNGcm&4Z=@7@j1Pc2{twT_WU6{rFjqKj967amaX`j?BVVXJkG'
            '!`01^JYfz*WVo_0SDbW&2oI9EZ~*+wU&Z0vPaOJ~rZo=|f0m6AzvI+T0zakU3Gohw7i@tTRx!{>jPp!I?<cWQ&#zW-lBIodkp#;C'
            'NLhIKJczN+Elkc&KEdn(y%HQwgT{tKo$n%V6U=Z%!6(0W7sL2w-j;~@TR=?&V-(w9>xV%^v&d6_-j|TBXorZUANfFu*aR05HVG^k'
            'jOzOG!|Ur0mxHsjk1)B^1A3kxiwO)V4nO(hEz-ky2_&3OB*3{J1-SYv+$!`QwqzRXP46Sii4Y6k8n&N6Y{UHD_&{dC!Nt~(2BQNB'
            'YY=6tB;LZ@fI)?Q#Bk&#xMSvHKtY`B)GYiE@Qyr#w}Qg()Stm`z%f!-YvCj3h~V{sc-2c{*p5KIYD@Oge7g+;-#{_I@506h-UMrq'
            'F?O(l<TUULjJz)fu!HU-t3mWiT>7|MlQ>_lj71~<I68kL=PTfS=8Mi`8OH!?>;)4T+=O9`i$$5Zl7}tA_%7A(2)a!Sau`5bxQ6hL'
            'Tz_v1#Qh17Fk5-R%kl{Rjh09Ut@jsaqt_oL&u;aQ+MyL74sbZ<S;MRbr@xM_aFbu0pMC6$32)djBfR5Igcsg<yA)bxc@p6mC#hVO'
            'F)sVbIq<s)QP;%Fvv{oDp8V{EsXuv%{C|a0pv?wE0z)N!>k--X<;ZYf<9McV5+Nae3jDiq;{Q8`tv6@**p|4PT>c499$y<cgAr=1'
            '0hz9P(>RC5h+$X|g|EJBJ68Owg9GRS1F%^Q$drD+?@nAC=MWg`jweGLmvNmIEJ*LhyEqT$;`Bm<uys?EG82S$<0u~Q+)lbM(z{hK'
            'Tj8|*&oBm<ng*p87Uv*lFw_Lt$ZPD8`&aUz%d!?wi2(U`+9mvg>78ksp8#oYa~M3JC6nKH$<hb;lP7vJFd`+9fw))%OZB#p5J3b3'
            '4Ccrt>@g!DY~X9$-aEwHc?<~9vKZFYo9`xF<h%H3Pc@_``8JHbIsLlLr(uw;==ap00SyAKaDh>>A&@SxQQXOM9CMn*C`@p8{g&f6'
            'EmAPX<3$dn>yO6*g?EDb50E6yC2h6jGsJQjOx2$#V7NKXE(?I#<Qw%jU=IF!55&*kZM_TusL$TRpDljV*JzRNKpxN5<6_R~%6sws'
            '4`Z#ejf>v<%oh=7BrkyXNgy5K7N^6X2OrL^$LE8~%fa>N;(Yx6a(Feo?$X=x)2ptMs>J2Xu)8g{)w;SKTz|M4|2jDPFuW2c;(?kt'
            '{%4T+0n(Kh!kz#+{^W=879WgFnF+5-d)MJk9kp5UlK|9@#Uhvm*c>qG;lqU&g#5ZpDFEWJ)k3Xod^!RR;d=b~Z1jG7HM|_r7@ewo'
            'cKCC~!0>c@{mT%U9DG}nG*0XN<;Cma)fI4WQv48Lr=hQ|sDlbv`}$ZY9SV2&oR|V`?Rk6`fimcO8!0qVT_yDlH8vD}Ums)Z2`G#a'
            'Gx>>B!^Fkq=*{SSa3*zzyCld!<sy(I;TX}V1Htw{1`yIWpqv1}fc5-{v>z0Z34+B93Bo~)GG>w~Gt@ic&Df_>Fp$Bftf3<-0DxK>'
            '1qQ!(3EW#Dl3DjW@$OQ5t3VX3wG71>(}E&AMGve+$k|UNZ%Y&lm2*TWcS(#d{m)sy^@h;@{+GcOua`r{!&r%MBqHX5QXC=2g=Ynl'
            'B~_X~PDMOZlPqPTMo#GPQoloW2V`(Y64lnjSk+JB-=SAEUEH&wPfWRA&`Ip0;+fH6MW0BQCXGWu1P<~NamF)1r6;@d{fIgCUCbhg'
            'HCZNh%YM;Yw}F(8g%eAibWHR$PV;FNWFhRGZqcwr>c__Zfi354d21^22RY^e(UUreic|<f7xkqWDV_}%w2(*&;VP%3jgo3PSChas'
            '9kcH8<vBd)RV-7$vFKbUIk0KRaf`LCQbqlDUz1Mah1t%#4?wW^TM;h`UH90KzwK09{B|x-Jx7tN2w5wf2rP_A5)uNCBE)p3O~$D*'
            'p8XwRV+v{(;4Fyx#i$LsZJ1Y;c8{fqP(P%O*;EhMn4eU6=dqqRP{{9OwZPJX4nxCOV@F`K0wpei<P+i8FkyJ=E3p6|pyD!b+QBk|'
            '$p>x4$HBke4gU2~0E?Ug1j!%376s<9Oh7>rJ8v6=8}HuOr{;8VSN<mxG$$t3F;Hg?hII<00pnZa(-3xHkbvATO%F7Q3`XYP>(Qe^'
            '{*0kZPEuXR$EUX4-$m#p(%K85B?)SfOXTf97N8X#g+PTHxw1?xjJW{tTteyE150<^Kx1=%8H5d}tR?!|NoI|BdTKd<vnd7ZdT{w>'
            'cs)M-AQh^s;cMVdKODB+4gNX4_;CIH18}B`59inL^a!4g&gIgL2j_3jhU3@23@%Ypy*MOp`7_8lG(SOi6{!c(0p;Eu%s9S7!*h{`'
            'VQ&HI5E$oRU}VUAJa#Jd-n<g8<ItNPoF-sf7?PO<<~#-i6jn#c8DLL9-K?>u>{nVCoujG&20Q5dpi(2SPUO+2!9DzUMJjd%ikU0@'
            'U|?5;p<(#=(#ur1t7zB(=^*JH2ziw7iQki6H{_53SfHe^!(&$V@wqmIfK%Gm1o|NmT;%)n6otv$&w!}=t)D?>8(zM@9Gzc}ho^6b'
            '<5wTANgp^meDR(5hW=>*zD4h^`vGWIYN7=izeSOc%FAEeS!?buBx1)#e`7WoI;bObg+E{V<LNF##)gAA5%q|1QD*^V^q3WL$2kQ&'
            'Cg`t#Rh?P$A$l|Yvb3yycJZAuO$HH~k+5KsDNq`c%^yfLfBd*&2?yi(>5m_gc;p|Gz9{t=CoSbcfn6YJ2ZiK3SS&yZgpsf-NB@1`'
            '&jT{|K-B`pH}!pj&_D^;_-?gh!A*WXRI-ZQg;rl*VT1lfi~<7Dd?W0-^fXvdQ+&`5(lNH}xW}4_Bwhep?Pqv;9%nzJyJ46lane~R'
            '*2XBD=3x$K^5y~XbA<KiFnB~8bV}e;uuMBH{4>U<^jh+O6J@0K<?6e}+kU=9%M`jK$bDRn$3Gqr#^WDOpLbu!$+Jc8ACAi{$ypw6'
            '{YcW0lS~~~cq#M~%|pMKpWOw&?V}#kDM(Mkl-Qyf{bW+6#quQxSiN;;<9+TNGQ$x)Q48>yV_;X6YO(S^^KXDJ-cPtyedXa=EH+zz'
            'IflL46^jMGqCXfX{9qB0n~OHkJeXyA2VKGdQtdc7Aj>+ZU)g3O!x~+ai`Zh4L=bX2O;jIGkA*t#1kobyim5jPo;`QTR1Bium6Mcr'
            'lb@0m7J+f#Co?2XHo{^&&j4L?30n`>KF)DB051+a+IcTgEd>O5AZO6<2-qVzVG%iod4*}8M@5Y$ifJn|X927`rNOc7C+7exn*s6s'
            'L@$`&VMt1+D+5NcK<hteZ~XbiHT;=*Az88N;K9UIyq4ZYViqJUisP*m-IZyKE?!XFffWHWTjuQ3+h?<*k2INU+k+b5ThI}JdGd#b'
            '0-=~6yw*F<)KP%0nL5=6_72D0V)GT@!^{NIn*m0V7$|fz091N=7tCpVrN=8D&)I68d+C@c5MGs(JbFj8Tk}p*Bokj1tXiAKmOgSt'
            'tKFkwuWAp_`orsjU$7s{q(S8ZGe{A+mIi?amv;|(>Du?VsoW1&r*C_UM2d?HwAO-MnOh3k@P6Bo+WVhpK3Fb8`iD+g(3LwgL<14i'
            'jfH?)=mhw5w%ht0_}J}_$4E5e@gv9uc<|^L&54yUlvb+>Q7s0-l)u+}1#5yIpoYr=hdfd#dQNcmsEa*3y8We}bsYH)Bmq3Vxux$H'
            '<Xt5%RuHMy#UxR-%*Hq$fXHtdR`J5I#Dm%3L|QbC^jBqi>;;+L+Tx?;19Nue&v<v-md3_U-Fm>hamr2P<Dw1Nzm>_z&C@twes2vp'
            '@$Saj`3`GRT@+Y=jhFG7=!dF_9FRrSsz-}&V5bSHC!Gao-C4)%7tHkka&c{}!Dv=wQhVnb=1Lk)SN!UQIsfkN(Yc0HqH9)j^UcOf'
            'kX<U<HCTd?n|4-lCLg8bEUt>-;b$-#XBoRq$=M=gW9@E3@#mwvv&$TX#G^bWQns`d$b}~tlr1{);}i8%yc+&|aXFM993ih#UT!k3'
            'CgqXx;VF82Loj2(2xC}N#%W)SB6*O6@Fw0Tkb!s<YYMk%oJofo&w%$M+DIg4!5U(fq8Fk^UQe6cx}YFyw-VcF3vZePGc?|9NvN7x'
            'LQjvUhM~(PqC+HjeP|ln@pepOtxAZE??p}T(yi~$R%0+Nle7d;zMA_{8f3fiY@61|Gz0alA6LJ@NECQ#e#@Z*yLkOlr0Y1r;EbIY'
            'M#25e6Ur|UL_2hwC+Oatdpj=*5~n1-kZlDiDjXhYdlic961iGCgy>(QovB2gZJ&s8Ysq|yr;gjuuRi$;fwGBNL`7Ue(Yx&Sh7p1r'
            'vx<^woIVpJO_O5+<L!4MZX_>dDsV}(kW^9ghgq33vcB%3yvn+Q*q7_Fa54!88Fs$ugkaO4t|nU>J}yFv0~S!FW5y{RwEA=8SODv='
            'f2idje`Yi=@R@lD+GqSij+l*hdPV%NYrXbDzz+7K-y8N0>mfDJ;KdWL)LR3RB@@c?`xg2@<EwutG1@_!0*o;JgN9~~YESOZ-9*4y'
            'LA;sVMHz1mEQb7VjT{D-RIOA4vtiMOQ(|fZa`}!#nk2qDz1reP3;+p80uTq(@BZLQOw9cAOZ-fBZW8l>xm-#<n=AMeNU6GE)H1``'
            'W9F)Gt>Kr7+gawd-rj$?f6aChVFaBw4MM<_vSI%YCmR#NxO!O7;k|cEEWE3x&3{%kk=Gc4#|_Pq_d$SuiW6uC+N2Bu>%u;#pq0Wu'
            'i!#~^m5Lryyqe_bVB~=1p8Gwd8edYD=;*`NmG=p~3mad?r+{9titpUMmhd<|^u&6w>Hu#FYSP*#j`=<8|C?L<e@n>~p0gA@T4^;l'
            'Ie|kC^5T1g?-Oi{cK}Axk<PEoqeB*Njb|gDyyRug;R|6Y5uI;eeD6x(hGboIr$+k}?2i|33)nq^4T~XgG2ktqKKL8|-q7F6BQAh}'
            'Mj$#p91J_z3hmrPjium{XgO3E)GdBG5l8<xtijw-Bjy$khDC}81>?AXxOhw@BoNEfAi<-Ng?Ar%Jjq$G2Gu##csw%jSQ^X^wPQJa'
            '{#bs%p};V_dmK>1I1U@eAr6n9*&{5A_tIL@V1~vh@QEZ0K$3;`u)5IaVLb^|)*3sB!&#0*F%MKjy#u7oVhO9olxteOIXT;{FD<Fn'
            '6^A9W7h3@8zO$ue9ZA$8=7AtQ3J-KH$r@e~HW(Mv828l7kNmYo1BR(82k@e40Ed5K0Jt(l0yL6MzFFsywf5D;c;@VOtZ8O2FCqYA'
            'k!5Tu3exgg)QDMYIaD)wFn}_Y4<F=nr1Ab?1E#*PQtLR%(~SpwWSK`{zRZE18SHWYXz>V&baIeJFM+LNkndU_p)T`Tn8W<O-;ao;'
            'Qp);xz;?{fupy6tO+|?Zn*{$lYzK(D1T=Y&`zThE4Qlbod%-Zom=AaqB#Qi{gnuwkT3JJq&Uz5eoufna+T(NAK2N0)I>%bn7_L&u'
            '#o?v;)p)yr?*mo^aXD(f;IMT+HBNlT-Zrq!)5n%2&`XEaPf7Exkm0&B(60ckAaTbk^by0Or*RxsCH}eWpeG|OYo&gOvYfk4`P>Cv'
            '0SWY=oZ~tknZsl*hyVM-=yG@^JW&imMHd(2qX2m6$`3Lf3Ca+etQA@rZtCq&J)r2DJ3JcTQ{$;GW01%lf+1XL3VY}e-<9--|4dl^'
            'a9AgxUi>Kmg%T=*RgwG=$OZ#=w4``(*eE5BUi?*J5+K>5*r9rBNvxy&MAnmn(da;l#3hiP9|Eh1fHd|LZ%ialN;I0(Gm-P4Tn=p$'
            'T>JJ>nbk6IpT-)1SoRWNQ0qJl;X$1`@i(&lAOA|OKhM(~gmE47cVFr-nYq98L4yMGD=~S0-C9`|CNGEhQ=>J!B4Xozn&p`yqHPjy'
            'wpq@L0g~NYVo;A-ga1G>Sp*4OAlR^cI%=ajOA<?pqe2$J{Ga5?&(W(L$v@Y@CIp!aGg|6^Ef2%Jf^ZX{+l#T1#2HG^Di!W&q}WkT'
            '4Sl1X!LQ!>JFxiQTwGwh(BS=O{C4=!)q^4XWupd*;blcg)N|)++cwgTST_0iV8z|X2IKGsZ8*JDyV<h2{y)_$3a?b5SmaBLCm3@g'
            '1+%g*dI><jYn5Rz{}iOCE6m7d!uH1$(L&i`m|Z9S?|&brcyO(9JZ2P0g$MK~<D+~tg^%I|`z(=m(X*y~s5%Kwb9ODqpDCQDJM&};'
            '7wL#{r#kxS!GZYiL)YEkZUT^nuiJpwEOlr+N3L7;Fw4eBGvzS6t~v6q(Wf0FvM&Tz#g5qNQPM5T5t$q?QXCpn$cS}vET4Q+jXf6?'
            'r@#F6%Ni+$dsPE9`NxW0>B_-_F6?C#%OL(>>Bk3iRn_)Y9)$B$OzLwvC-S)L9cOW-R!)-ik-oaP{zc|o>eSr1<=$KQc|yPODTp_X'
            'KS@=U4+k-G)y`V$H)OJTa#tNwxEK5F^N5pwF&c(Xz&Ie5^8GjgLFS>MCT(Z7O*j~lvfuQ<Wi`*u{EQZ&bh5?)KPP)e9o0~jUPSea'
            'YXH7+s$qeAZrvC(Sn0wo9Rcot5L-IiDcE{r!2XxqteSNioWDLI`bqmvgQ?pU`}{n$?_4bph{mLFp|h=+bF^g!L?kM?Mf&)FzP-IE'
            'WYg`f4%{f{dMrg90>4!(Ry8>`G7M~=c^3G+DKuRlYm-nOG)p-r&M(Z|U<_Zs3rvanH6~elxr`l~t$YuqQp`av>57a--cTXua?aua'
            '(^!LT-wFeusrYlr>8-`khN<;*?e9)NznRWGnQ76n@x}VWEhiOVz(oV88T8{;M`R(lR$l{~#Tfsibe25G!5FV@Z`$+_jF|(_Tm;6I'
            '=DTHH=*%!uJVf=-9|K*Z&vcisQ-58j!aYSjwwe!=Kp%c}VqscE(KM0Sab-t511ko05HO%6$4Cl$z7(>5EFRh~MZ5ny(1pE(ZKZ#;'
            'Z{0_y0>0k3Feynh<7W=ukqH|xjOKBpOMB_l*|u0x+8l_81}B=ievRtw%HMcn%--evgd;=iRJq3-A$2UAilBAOQ^U;5a@24eGAxac'
            'BpNCi9V#A~>`l&a0SO=A6oYj@h><Re1ne}#4p^~k6$YRVQRczRata5Kr??uv9=#ih(fQTjT)Z7z4lYNqEk%cr=ebU83WuSRz-W!X'
            'P(rI*+kO<^byT`-KhI`vA0$1;M0Ko$5W=5xtP5;Z(qULC;1^~2fT=A$3l(}#zgXsnra@8<$PUqu>OQn6()`?dQBFF`R*4TnB=nY;'
            '@`qXLUZU!$%vM3<_b}KNm1;87IlLF}C8L%MXw08gKB|g92A-bu8%G<YW2Hw`kK`@ArzR-KQlDxfeAsS+_on6K96xDM)hPr9Bu-r)'
            'm%?M8$l9@wx5p~~q}I|a_906D%rA7P;*DYAnW+`8fYVda70iv5^;{^&-;7kYD!ag17S7gun);a?jiqVZ=-dw6<PY4ZsJ^Lf=|i`1'
            'TQmbGzH-grC>1SG8+^d?bpM7Jg7-zq7hHnBBVwSYSW1Ug79fPGG4)$RK6p_;MP`O~bx=EV2->-nSBr_VUMS73YnjtRRu>mrk+ueA'
            'swO#DTiCEBH=atgq~hLB_~@Zf1bdvv7!s`@V4PJ^`lTrRLQV)(*NOSel#E3-lF+f?N0~X3WrE9r%3~g69Iv0qfKBXB9!<i04sH8M'
            'X*m?>7KuuZa5)R72ubQJ9F*(P#^3p9Afh9R%2R=Af*`+29z|#BuR#v_5j;bg-YKhG?xXVcZR0SQp|iO%1lDWn6#zmQ0C!|AG$sKM'
            'pc{H~dvkc(Rj#edyQAAjOE09yhy^qK-QrlZ2#SrSVVoHx*%c;xrBpE$M>bBpXq|Rt>iIZ{W9Ddx5>3D(Aa~05mjy%+Jo&1~YbB}j'
            '3LhSc!P#$vk5?)HAB!^uSe=6klF59Mg9+zffZZKLvoN3IUd8A-%5vmB81b-wQ%d%Vk)d*GeH#Q?kfa%R-lf6=8!rkLAQt#9oz6}~'
            'h$Om`LCiAh7xMym2%t3Wj0!BiO2?*{8fn^3eLw2c^iuJn-#@1vDYp+DynzCVg&EZBB(BfTVV=ANsq7HUp>XsSUI~opqS_78$w23G'
            'Fh?@dyT)A5I*`+}F@8rG5TF>wc?z@Gu)h!_&qR*o@CZht6E-I%Er_4Q(S*0%xh&R$;AE8*oDLN|LcDZHkYNoPpJR;aqq_unL1~?J'
            'hK<W?K=znUfLj3$FFssfjZTN$$V*qy!$cLwnXrqNur-qvZ)Bn%o!@P@bcUJ0eu$`emr|suBeN<EzEc%{EVg1$y2=1DiC>20!=`|@'
            '#2Czt!kH1$Z9)`rVrEKU;__!C<pi^2F=Dl2V%v4i%sNRK7@2Pj34>q>3K+OD#Ls+v00x)@`M}?OD~^s2U(5}AiUy|apa)xBr#8;9'
            '&}~au-k}GYC%ji?UZs~xR=7qE>bZ!c8)|V|%;ZN3rKESYDNY1PP72T>v#m6O+|rj49cnX-jSMYX&7vV-tNb<jFP$>T1r@_6{9?`E'
            '_{)^|-e&uq38@q$iHp#pl<(K9K?>kz%iLR1Nj|%T3d(=3jfIJ_M(%+pUY}iD4NnCqOUJ4n%O@0cTsRC_d2Ci%;S)R-(^#HhqB<SV'
            '$g!!!@~1eM5inV1g(?#vxp$e_V!~7p?JQXgQAP`-0W2l*N|~F7xx%u<ON7vn!>Jy-LYEn$VvEe$m?d~choBIwDfVE>S&3ARuituo'
            'aeh5KzaF0r&fk0(ys;`0MZFJK4vN+pTO0C8w;*3<2x?J~u0-d<m5VQE#rV@o0T_poHkP@3s%_(UNlph!&Tq)JP2?-&u246EDMw(D'
            'Cs^kxm<L=HOE^z^C7r>>M;;mpIS8PKm^2`C$a&F@<2N*p`;M#~;bckg`qDxQKQWPVohNIs6HIhu$rsUqt@c3&7ak;+0wSQDgUY+a'
            '*?a3?76|TK?7aK_7qRdPXo3UF=0PZrVK*L4mTS)ZMrAKcyJ(3;vQW-x-4cypS!_q|ui@t{Xj))^LCY0aJ^(8^qDnpZoYh}+m&T!Q'
            'MdPe+&vbQ4ISE1h7F1`>!9iFmix$NJ4<ThM9Yi#p)NvgKKLi?A#%h_?F)f2hx<x_J1d%J|U@1{l^JS@^)Cb0fM<^(b*<RS%_D_m2'
            'Lx`i+*G9palnV*bIAl>wEDm*!j+HP-Wi2L3l9(sKLRKy-N>`=`&6}<7JS<HKWB|l1BeivYF{-^BRP^uM;~;K7%VdZ$*OActq~E$6'
            'o}Ugc5fwj=-i#$?j7_f;q46D{e~TO)OUqCqE&0!2+^See@vYh>_yxK!xMq=M!zuG#ecB=J+7*R8r`6zJrE)RX+fTYyq#4&m?SPKe'
            'lq***J!zAmkZ69W^1rJV+QiVDW2<r~G3J2k)gi1|NvN>;NN7imnWYdSn+BwSnzd$0)1m?my?~B7W)zdgg$M)Iz)=~T8Z~~2U=+P*'
            'BTp6u<W5k<vr8C)l8xukwZ`37C-sLs<pW>s`JIvTpTslEK|poWKEhOM;7J{QL_IU?$%=q^!k#Rh56sa6h1R1qFlVrIYh+be-eD#U'
            'OsCYyG&ob>zW%&Z8A4%m%4s7o%fKrYsjRr8y25hBN%K!Xo9^V3Z=O0Ht5}Qnaq&}4YrJyLgkY-M^7NVy_{LPpl8>)XbZScdPkC3@'
            '9C|<1yj&mgna!on$>)Z=E`O%)KiA%{7ET;hMp@Q6>XPdw-4BPFXr37}P*<{fqEl~&|9pQodOf<9HPBe7Dt=fxg(u_Dpxp_HKxhVB'
            'm`j_^HE*3ODcS3($}(J2!ph}^55cA<0mQ|Ikf*6o0R)F8y$lO*w!v&Ii@6~~-^ID3A`mhz)uu|<N>4$hc<oD45_oQZqW<iU$Uc99'
            'fA3cDPwoW01}z!fu|cyt$SO>=londB2bqXU;SWVL2gmAIhfyx_<eKxSC)lI2^k)><N~r+lmT05H;;cb6T*8`w(qCk;S_`f?bMxWn'
            'ml!jR15|>N&(G>*U*})tq-%%OuJSCO<p(hrxn0w_*)CoKIh0RynH2m)$|{43Wgco-HgWv}ZuAtc--EOmUk%TOuTfmS`iS|u7&g1{'
            '?qBi&Cx=%84FGYjy~Im`c@b4vxa+arK$#RQXTIs?w@@hGn1Y-eF%TV+=kC99_8RSbx?J?!bhfOIXGz<ek5%eU*)>zQFB(p*WK5d5'
            'Zb#zBlX4&Mq#RiGq{rGC%~9{6-1rg02tw2d=N_y#>gj+ByVFOlJ;<j<1Ru`0^@Nyy8UuBwZAIvUp@Y>NytRs5A@TLSg)cqGA4>^y'
            'HCU<}`1d_%C4kz4o&PkFRI^7Nl`0*5QXzXlB;hko635?=A=SA6AN7O`=hjapB6tS<7f^I$JZ@w>qlc|+sbnYRRj;rr%=MsSdp|`x'
            '2#jt)Q8fe>86Fl~43KkpV5)XFZwNKnT|hZl)K*152Z>wRPo#=T)@EvF4ZO`}^Cw@@`6W^bqAPwB^)(esIb}~Gn2}tH7~_ZsihMuV'
            '#YqnO37SSSv+<$6SBl{WVdf__UJQpxPKRz!a|8+4Un0L%Wiah_R=|IWhkRw_TkJQ@$}E|cVg{W|IsidE3s4->fG{oBB(5X5Z|X*A'
            '$L7FzStWn&2<r|eK~m$M<vavl2StD+*%+e-=+mr}TW!FnB2v(fS{xT~fi4?m0QVB0%1<6v8e})fGNy9<`YeLONm>0KU0IbWB|I{3'
            'xf&u!$l`Th*&q197+J}Isnw7OKOMdvT_K0WVE?`yF<Rln@_^rvyV{JmLIq^_g|*)*Q8G25p*nCz79Fa7Pz|;j8}-l3#`>|@q!jmI'
            ')45sIp*E_c!oaJM4#Ydv5rhfQ;7%B4q_~uAMtv>bOUD|nvRWGJ7z%was`_arudQg8Tpb3y@_7^;Lz+PVx*^?UQ@RWy!E#e(l&AKy'
            'tK#WMs;0NSaCCZkwt`JW{$lbL_pNLP%PuZ(TXlq$ACz1RwWV}%uXHT)S1OSvst0;K*`ElThmIq}w`8m_qKHLsbt=kb)aMZlwr{{2'
            'MQJeit1Fk@1=)<wG-WsC<>UQZT*8CE$~Rlk$ncQ%z`Az@#;(Gu)Y+>Zyn%0R%vS>PnyzA~vdd1{BVdK!a4TjFWmpwZhMrb`Z-n_{'
            'ogAI~X)<xkp{si-1Eu<7M)05vzNV-faZ($LV-MgjiO;MU=YMT<rnFZPH0D6&DU3D`=lM$+R5?0-4Hn<K;nhgI8ooUneZ0679W1&n'
            'ydBERd9ay#mGat+E(Z!#hMAX@JG1Phx~}F}iyE`4qrCJ$#rsunA-J&?;nr+htQ54Qe~q^wI`B#=UQhhsX3NJKT|q%2^nE|b{7u@S'
            'syNFV9bt28cB3x0+8+|~nI>q>X<~bkb43zTzFIY8#}?dWF)3<-b?SmJr#t6Z`x5FtppaYRuRj%XORhwDQck~@l>e%ft&5~Bt2&iW'
            '%q|x(ug%zwU9DS%XTdG}2!IT$a(>yjp~{g}lp!NxRI60`M3I`H$R(<;2Xf5X|Mo>XDgd{5#B0v(^5UUaEdgsSUHN-^l$ExR&k3Pb'
            'VzhE(!S2oi1JcP~)lHx2b#F?gtH!fk%G4T9UuRgja>4XLTBo{>nHTb*ULK|1g6@4p=Bdx`F8O9OadTT7J1aOa^n6Cg|DevLbP?I$'
            '{8U!ahpj|cGa7ZF`G&lv5aBrKTVbDc9IGlwGAu{;4$T;t(o}Zn!%4udmv_#U%TZHDiBGpV%^lmPOoeN)js>9U1}1fDB_x&H&Ahbr'
            '6%TL9GE_u<3jYgWDEF9a!1q09&~$dt=$|gj<DxqruufIB8jE(RU0e3*YA-*ZaH7+vtRZx@sP7u$v0D~~btG!Zylx=_x;E49R#<>*'
            'S=&4|=_SYE+m^RI^M!Md1-3%4{t?;!It|vqQ3%Fk;ejvEEGOquE!)aPT#X1Wud0`Zo+vk$4r*Ogt1w=R{xwRQYFuhPq&UjJFVy44'
            '5t0uakGq$-&_G=D+dGxSVs2Wv%3*}6Md2YYM!4UxMR{Sq6rAidcv}iw*(B%cV{}^uBhBk68maM-S&&8NLHF6V-n3m9r^|Umcq49}'
            'd35m7;jlrs$tQ!i$q~6frN0+ydv)T*Mxtrk0H2zy*_H;5A=g>3_R?SnB#RgSq?=#|R7HGDU@91~2P%<-^&yaor>rwu!`siRY1&j3'
            'vZ@&76|)JcOdn@g7OphaT2E)W4|Xtbe0?BiRSeT#eK}sDiO!#llTh#rdauJ~&8>f2vE^yxJxK>O?nh&20!x1YuX-{_&FR$LBlV}r'
            'q6m~M(?#JxFr9xzmcP&0LPcbe<rFB>ko;b*XyuZ~FR!S<xl{!kJUHKZ3EgZ$;Ozy~^i&!m`BiueSH^L}fNpQ(SGQjpSVN|&zKaUA'
            '3^`8N>G>{CH#ux{7_vR8HuXomtzr2W4h;CAdBhUJ?bB3Uc6O-DU3={<*X7eBUEc5PNHCr0Na4iY{?pC&j8WBWwCcccyX{QnKq^O;'
            'ZlDyVCe=W`H_Bh*0g`-5L`IcMtcg?)br0P}9YPIC9-di8CL{q1@nIGg!%;k-X>WB&$MJl%LdaWDf`6o;nnQ0K5TD@8gZm(&jOD2x'
            '-E-`NOlGDJF#CKOB=A2x@AL0{5?^LDu!_}Hv*gXk1~lFp%dNG?KDDn%!nK}zsumQ9U6i;Tb7XqsR)aGd^0b4kq&U2S5a_iOq!BDE'
            'i>oLLX4Lx36S|G+WIt=ZUTbOe5fJ`l7td1EYVx@^NSF0!eEH=(b_*$GJG!P~nynKXM=%D->LNhaA(gCos7ka`>WEpDy|YIZd@~Qn'
            'z<9NMJVU24L&evH5Y3}?gt=Q_)6!wTdFrp`cw^7+maocyZ=@L2v%EDIUqXJ`LgUq<1J|9W%F7e*kR_^w@@PkeDkFvbEPSUlpA3vC'
            'SX-Lm)-jB9vXqY!zThB1N5R?X*I|+IX<rB)#Z=$YPb`my_B+r9P&unAkLjRLc4R!b9ob%-ztSNN-ukA)(Y;0HCx&kWW25B_ROZ=^'
            'btcyruky@JTLrX%9fRM-sf;T2@&2k;!?TOuR64y(gu$eFO4qaQ@hB<?vCYeI&=HN6!~JADJId9GSmX(%ZQ${rbmXuHM>%iQxnFT7'
            '2vuXu;%%z$$&=?##@$x`&I7+l#tEp<s4fGJc_<WO$!ULTQT{gXj&+2Aw)O2kroN%n;@+cF6_nyw3JSmH3vrF(l8&QeSuFzZmUxo7'
            'w%PEnn%Z=AD|DMRZwbS8#^4yMcUD<fAZRsRPUJ-YW~=gKZNbFJ^P<jDZAD*0c=+a<`c#Ro_~sk=nQbJ{K)=uszp*op*Zz)feB!_;'
            'Jeb69T=56)+K%kVbt{Ew@GC+f03himKh@9vq%QwJ7<%kK0F1|Df_m~$hK+hVMoUr1%Q_&1|K!0M5`6NqLAPfm!fWroq()$h6aI#^'
            'EBWTzreZdA3>zVh>5wU9uZP@NGGSbX=d1gL#%km0ns4Z8CzhS%TNQi*`?({}tyHC6mZ&JO%asQtOCAW&40N7Qm*OoWi+7XnTe7&j'
            '6;*puoVDQmQ>=Hv90M;Ol<Jb+_84Xjc-j?31d5B$$=$Akxxs?M8w+K)bWwwfu3BP0I~{rmyfY!xmuO=;j2^TnMNTOwnzQGnxHdEB'
            '_bnl?@b1yPMKHDD!zRokhvRu1RG#Q`YV=xB=B<6!=a2}PX7z7VK+6mRn@Hg|=(nUV*9GZQJQ&h8({#VAEi)AZ1wMJSqvtMVOT#=D'
            'w!l!4+jtKiwR_a7SJ?;+I<|6O<MBXoQdSDOXht=&8T%lfZ?)cdWQlMGCx%CqpRg;mj;T0uN$8-S)+IGprO8M7wUAx4)im1ba?CVe'
            'EP@$U*eGftCRF=NZOl!c%Db8kc&|q1r)JPnWtaAPMqzM6<udlMR=s3ElIpMnJOVFe?U~K+K1)~T3}afD41eN{h!@XtQAnB3P=Xl6'
            '(-<_mC1!T$1UqSR6|ptQcd)w?sR66zP3aVtX-36h8NiK0<XpXJ@Y$piP&I<JP}vYFUR`z6X2y)QubXIPPvJw~&f4>`80y!N6jgFL'
            'VW)PT6Wxd=q8Fv;eu|2KPt}c6930Wezlu*~tP6p|%ZU;8BZkBpXC7$%=y?{goD!tEmt6|ZxsNe$Q*~BQaGXji(iIgGzeh3VCq1S*'
            '8u<Z<qK&uJ(Ob$WXrmJD&vG_Qs19aP0>7!a3x&Jg2is18n4+eO+kINpyRQ3;8ZUo!nWSG(diE9`nGIVn^1=S;fK*TY8C^wy^)vD;'
            '?&)9%_iwp-X|c&v#x3!-&nJ$eEs1oe=O7t!vL~7XGHAh(wIqPF=-uN319(*A!XYcB(}kw_<qBTNSe1%GO^)>q_juK0w{kP1aXEs{'
            'm@r(KT;b@Ts7Z{r9X9W!TVSui&}DheR9))8cT{pdy2QmMAnT6|#7LHES$SNYMn}Kk4Fmm3{n<UDiHj1j)r=s!KA6!cBqJIO#v&@@'
            '#1Y?!|Md?ePUPpozfe)>iMNA~=YzLmbcJzj;_4b5;Quqa8sU+w;lpyxsJRNbh4WVWz2bD|@iXACL`Fsoa$Ch9w_VpPS+a1h@E(N&'
            'Siv2vOJj30{J0JKxGmDC?LJ}w5Cays<J;S;=Ecp=ZRuBrmPdy#z5`23yy%J}0{QM$yEM4h@jV_RVK9Vf9vod?ixc}?eaH9I(yR@*'
            '0<1RjvYtFxAFEpLI5k-H>RU=YFOTv)-P&#b)Q!@*q!<STio)9NLETW?MwPi{jkg|MRukL@x(BO^sy~|1-EO-#$45Whx{v*S-va#3'
            'yqbXA=9o_Q$xGVh0_JV))oV*W)Bbqzwr!*J4@#Cfd9WH<*IdA~{b(HQf-zRoJdtQJKiAh7Ru)ZODN)y`deH?_DPJ#9EZA*rZFaic'
            '&k*3Wj|-=yKur^~Ce_6q_&q2sQiHO-cGOYAe#L8_pLbYJH>AEo57j>B!^bxt&Iad$SEIAhrK1Zh%M=Dx_Es*RGdzDYJUbf|DE77i'
            'LgB)9oKf;|@P1H%)D-%u0G9x~8~n>@LhTJlN8AdghN2T$NZ92<!r}p^_$Wl@0Xlj#kCn+m&BX!HJ5!8O6&uCNcuGE&%K8fNEhfcb'
            'gU1qVnMGLwt}{CKgZ`BuWTWcK4b5Ka>4#;>x27;zI2Y%`OOOxx4%*E7lm<KBRotLNMeTBor@{S;8eIkU8bd_w@~x#g*UYz9#%9fI'
            '9E%ZML>PPxlADk5FkdevW>lV1eGY1wC2MdipYPL7d}hnd;A#?WNDNq^?dI>Y(C_ek`1TBz<=A|u%9&Y5y~m*Utiv1+lhnxuC|dkX'
            '*6zEw9R1Jed~gPUuT&64*9Kf)BZ6+NDXvfh94>lmKMXd$zEAC2<uC`0DKoPf_vygMHiPZmCQ<Qv@aA%K2HU%D11O)p6Xq$j##cu='
            'Scu5H0_Q+plM;t8Uo|w)#Di#$vBB)-!7`{OE=DrXP+~`30mFCM(4in2i+fLArz>{V1GaHt4G}H+4~b>^dk^6EK{N~S@P8M{@VBJj'
            'Kysk1VPR=xs)+Z~k+9?APpaf?z9zp+*V5RmGRD>B)@#&CtdY?sleZntf(_W=pn;NW=XCgHbapzrlx{h%%_Zz}_z-pTSy%ez&|v43'
            '!LV?$HL1H+SyAWzL=LCC6Kip86Nk)9Hf|^?wBD?FAE#Mi&MC`=s{PzyP%;&uE&c?l&z8#FsxXB2Kh4?SIPlOqkY)j9M#3_${3yQ{'
            '7w3bI@Hejfm#l&-z1`-8PHLU!w3#GqzIkbK2-U1H;<2Qxo1DA8%cofOFD9q1Z@v$U);g^^G_Id|aeggT<ocX%w`~t{WQ4`ps=nPp'
            'IRLL>b`46wVg;ZQBz@}Ew$#8j$5&N!WMByCGa<<kud>Z?ZG<E}HB~eGscK!~6|7=iz_97bi1n_O;5)oO0@J?4CbwHQB)%XynE2?#'
            '2*z|#S5>0Yyg$(GVssCPltYQtbA$G4kKa-ha8tiUmOUu6ghKryFBiKB$(!>9ucS=RD%7uym#j%`e?59DNL8<KPD;%jMlmoo{)+1M'
            'fY2W0$<Ct#Is-licJzQi4To*1BkkJcejK;KG^##?QD2Y5#}B^_5Iv$}24`*jd-vE#MNq+EHDPcZ|MG-V`ZYzJ-L$C!D|(!9^D9$Q'
            '#iN;wflPFe7_wavsaCT5p3-WYPe>lYR<=_cR8Hdj#B!+=^w74yFYRJ|GGf!=dpxSt$2O?*$eTZ~&cy;)9yyygSc9?<$;%kiV@Q5n'
            '{W9pGA7Q5y#y=~c{~z0nILH'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
video_analysis = _local_pipeline_modules['video_analysis']
script_generation = _local_pipeline_modules['script_generation']
tts = _local_pipeline_modules['tts']
render_utils = _local_pipeline_modules['render_utils']
youtube_data = _local_pipeline_modules['youtube_data']
viral_prediction = _local_pipeline_modules['viral_prediction']
approval = _local_pipeline_modules['approval']
google_ai = _local_pipeline_modules['google_ai']
provider_adapters = _local_pipeline_modules['provider_adapters']
source_strategy = _local_pipeline_modules['source_strategy']



In [ ]:
paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB, use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT, drive_project_root=DRIVE_PROJECT_ROOT,
)
RESULTS_DIR = paths['results_dir']
YOUTUBE_DATA_DIR = paths['youtube_data_dir']
YOUTUBE_CACHE_DIR = paths['cache_dir']
RUNTIME_DIR = paths['base'] / '.local_runtime'
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
RENDER_DIR = RESULTS_DIR / 'render'

SOURCE_ASSESSMENT_PATH = RESULTS_DIR / 'source_assessment.json'
PROCESSING_PLAN_PATH = RESULTS_DIR / 'processing_plan.json'
HOT_MOMENTS_PATH = RESULTS_DIR / 'hot_moments.json'
MOMENT_SELECTION_PATH = RESULTS_DIR / 'moment_selection.json'
SELECTED_SCRIPT_PATH = RESULTS_DIR / 'selected_script.json'
SELECTED_VOICEOVER_PATH = RESULTS_DIR / 'selected_voiceover.json'
RENDER_MANIFEST_PATH = RENDER_DIR / 'render_manifest.json'
NORMALIZED_VIDEOS_PATH = YOUTUBE_DATA_DIR / 'normalized_videos.csv'
COLLECTION_MANIFEST_PATH = YOUTUBE_DATA_DIR / 'collection_manifest.json'
VIRAL_PREDICTION_PATH = RESULTS_DIR / 'viral_prediction.json'
GOOGLE_MANIFEST_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_capability_manifest.json', GOOGLE_CREDENTIAL_PROFILE)
GOOGLE_QUOTA_LEDGER_PATH = RESULTS_DIR / google_ai.profile_scoped_filename(
    'google_quota_usage.json', GOOGLE_CREDENTIAL_PROFILE)

print('Sonuc dizini:', RESULTS_DIR)
print('Drive mount edildi mi:', paths['drive_mounted'])


def find_artifact(filename, primary_dir):
    candidates = [
        Path(primary_dir) / filename, RESULTS_DIR / filename,
        Path('results') / filename, Path('/content/results') / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    return None


# --- Faz 0 durumu ve ORTAK faz kapisi ------------------------------------
# Burada tanimlanir (Faz 0 hucrelerinde degil) ki alt faz hucreleri, kullanici
# Faz 0'i hic calistirmamis olsa bile NameError yerine TEMIZ bir "plan yok"
# mesaji uretsin. Faz 0 ZORUNLUDUR: plan onaylanmadan hicbir alt faz calismaz.
source_assessment = None
processing_plan = None
plan_gates = None


def require_plan_phase(phase):
    """Alt faz hucrelerinin ortak kapisi. Onaylanmis plan yoksa ya da secilen mod
    bu fazi kapatiyorsa faz TEMIZ durur; hicbir istemci kurulmaz, hicbir API
    cagrisi yapilmaz."""
    _plan = globals().get('processing_plan')
    allowed, reason = source_strategy.plan_allows_phase(
        _plan if isinstance(_plan, dict) else None, phase,
    )
    if not allowed:
        record_phase(phase, 'skipped', None, f'Faz 0 kapisi: {reason}')
        print(f'[{phase}] Faz 0 isleme plani calistirmiyor -> {reason}')
    return allowed


def show_review_prompt(kind, artifact, label):
    """Inceleme ciktisi: artefaktin SHA-256'sini yazdirir. Onay hucresine bu
    degeri yapistirmak, icerigi gercekten gordugunun kaydidir."""
    info = approval.describe_for_review(kind, artifact)
    print('=' * 72)
    print(f'{label} -- INCELEME')
    print('=' * 72)
    print('artifact_status:', info['artifact_status'])
    if info['content_sha256']:
        print('Icerik SHA-256:', info['content_sha256'])
    if not info['reviewable']:
        print()
        print('!!! FINAL ONAY ENGELLI !!!')
        print(info['blocker'])
        if info.get('blocking_conditions'):
            print('Saglanmayan kosullar:')
            for condition in info['blocking_conditions']:
                print('  -', condition)
        return info
    print('Onay hucresine yukaridaki SHA-256 degerini yapistir.')
    return info


## Google yetenek kesfi, kota defteri ve ROTA GOSTERIMI (calistirmadan once)

`client.models.list()` bir modelin **sifir olmayan ucretsiz kotasi oldugunu kanitlamaz**; bu yuzden
kesif sonucu acik bir allowlist ile kesistirilir ve kotasi 0 olan modeller (or. `gemini-2.5-pro`,
goruntu modelleri) otomatik rotaya **hic girmez**. Manifest `results/google_capability_manifest.json`
dosyasina yazilir ve TTL suresince yeniden kullanilir -- her calistirmada tum katalog yoklanmaz.

Kota defteri (`results/google_quota_usage.json`) **YEREL TAHMINDIR**, sunucunun yetkili kotasi
degildir; gunluk sayaclar dokumante edilen **Pasifik gece yarisi** sifirlanmasina gore doner.


In [ ]:
google_ai.assert_no_account_quota_rotation()  # politika ihlali varsa burada durur

GOOGLE_CREDENTIAL = google_ai.resolve_profile_secret(
    'google_api', GOOGLE_CREDENTIAL_PROFILE, GOOGLE_CREDENTIAL_PROFILES, get_secret)
google_quota_ledger = google_ai.QuotaLedger(
    GOOGLE_QUOTA_LEDGER_PATH, credential_profile=GOOGLE_CREDENTIAL_PROFILE)

google_manifest = google_ai.load_or_build_capability_manifest(
    RESULTS_DIR,
    client_factory=None,
    credential=GOOGLE_CREDENTIAL,
    credential_profile=GOOGLE_CREDENTIAL_PROFILE,
    run_connection_test=RUN_GOOGLE_CONNECTION_TEST,
) if GOOGLE_CREDENTIAL.configured else None

print('Gemini kimlik bilgisi:', google_ai.describe_credential_state(GOOGLE_CREDENTIAL),
      '| manuel profil:', GOOGLE_CREDENTIAL_PROFILE, '| otomatik profil gecisi: KAPALI')
if google_manifest is None:
    print('Google yetenek kesfi ATLANDI (anahtar yok).')
    print(google_ai.credential_help_text('google_api'))
else:
    print('Kesif dogrulandi mi:', google_manifest['discovery_verified'],
          '| onbellek:', google_manifest.get('cache'))
    if google_manifest.get('discovery_error'):
        print('Kesif hatasi (sanitize):', google_manifest['discovery_error'])
    print('Kullanilabilir modeller:', ', '.join(google_manifest['usable_models']) or '(yok)')
    if google_manifest['excluded_zero_quota']:
        print('0 kota nedeniyle DISLANAN:', ', '.join(google_manifest['excluded_zero_quota']))
    if google_manifest.get('connection_test'):
        print('Baglanti testi:', google_manifest['connection_test']['ok'],
              '| test edilen:', google_manifest['connection_test']['tested_models'])


def show_google_route(task):
    """Calistirmadan ONCE secilen rotayi yazdirir. Gizli deger ICERMEZ."""
    selection = google_ai.select_google_route(
        task, mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
        manifest=google_manifest, ledger=google_quota_ledger,
        enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
        max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
    )
    print(google_ai.format_route_line(selection, GOOGLE_CREDENTIAL,
                                       raw_media=ALLOW_RAW_MEDIA_TO_FREE_TIER))
    if selection['skipped']:
        for skip in selection['skipped']:
            print(f"  atlandi: {skip['model']} -> {skip['reason']}")
    print()
    return selection


for _task in ('hotmoment_review', 'script_generation', 'bulk_classification',
              'grounded_research', 'tts', 'embeddings'):
    show_google_route(_task)

print('Capraz-saglayici (opsiyonel) durum:',
      provider_adapters.describe_provider_states(get_secret))


## Faz 1 — Video Analizi / Hot-Moment Tespiti (`RUN_VIDEO_ANALYSIS`)

Videoyu Drive'daki sabit `source_videos/` klasorune yukle. Dosya adi girmen gerekmez; klasordeki en yeni desteklenen video otomatik secilir. Gerekirse `SOURCE_VIDEO_PATH` ile acik bir dosya yolu vererek otomatik secimi gecersiz kilabilirsin.

In [ ]:
SOURCE_VIDEO_DIR = paths['source_videos_dir']
SOURCE_VIDEO_PATH = ''  # Bos birak: SOURCE_VIDEO_DIR icindeki en yeni video otomatik secilir.
CLIP_START_SEC = 0
CLIP_DURATION_SEC = 240
WHISPER_MODEL_SIZE = 'small'  # 'tiny' | 'small' | 'large-v3-turbo' (GPU onerilir)
SCENE_DETECTOR = 'content'
TOP_N_HOT_MOMENTS = 5


## Faz 0 — Kaynak İşleme Stratejisi (ZORUNLU)

Kaynak video seçildikten sonra **hiçbir alt faz, onaylanmış bir işleme planı olmadan çalışmaz.**
Bu hücre kaynağı ölçer ve iki mod için **yerel** uygunluk skoru üretir; hiçbir API çağrısı yapmaz.

| Mod | Ne yapar | API |
|---|---|---|
| `direct_clip_series` | Birden fazla hot moment → birden fazla Short. **Orijinal ses korunur**, mevcut dinamik aktif-kelime altyazı ve render hattı kullanılır. | script/TTS **çağrılmaz** |
| `narrated_theory` | Seçilen momentler **görsel kanıt/B-roll** olur; senin yazdığın teori/yorum açısına göre mevcut Gemini rotasından yeni script üretilir, script → TTS → render onay zinciri aynen işler. | Gemini script + TTS |

**Öneri bir seçim değildir.** `assess_source()` her zaman `selected_mode=None` ve
`auto_selected=False` döner; modu Faz 0a'da sen seçersin, Faz 0b'de okursun, Faz 0c'de
gösterilen SHA-256 ile onaylarsın. Kaynağı, modu, momentleri, teori açısını, çıktı sayısını,
hedef süreyi veya render ayarını değiştirmek onayı **otomatik geçersiz kılar**.


In [ ]:
source_assessment = None
ASSESSED_SOURCE_PATH = ''

try:
    ASSESSED_SOURCE_PATH = str(video_analysis.resolve_source_video(
        SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
    ))
    print('Degerlendirilen kaynak video:', ASSESSED_SOURCE_PATH)
    _ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()

    # Varsa mevcut analiz sinyalleri okunur; YOKSA Faz 0 yine calisir ve skorlar
    # yalnizca kaynak sinyallerine dayanir (uydurma sinyal uretilmez).
    _existing_hot_moments = None
    _hm_path = find_artifact('hot_moments.json', RESULTS_DIR)
    if _hm_path is not None:
        with open(_hm_path, 'r', encoding='utf-8') as _f:
            _candidate = json.load(_f)
        if not video_analysis.validate_hot_moments_schema(_candidate):
            _existing_hot_moments = _candidate

    source_assessment = source_strategy.assess_source(
        ASSESSED_SOURCE_PATH, _ffmpeg_bin, _ffprobe_bin,
        hot_moments_data=_existing_hot_moments,
        google_credential_configured=GOOGLE_CREDENTIAL.configured,
    )
    youtube_data.write_json(SOURCE_ASSESSMENT_PATH, source_assessment)
    print(source_strategy.format_assessment_panel(source_assessment))
    record_phase('source_assessment', 'completed', source_assessment['status'],
                 f"oneri={source_assessment['recommended_mode']} (secim YAPILMADI)")
except Exception as exc:
    record_phase('source_assessment', 'failed', None, f'{type(exc).__name__}: {exc}')
    print('Kaynak degerlendirmesi basarisiz:', type(exc).__name__, exc)


### Faz 0a — Modu SEÇ ve işleme planını oluştur

Aşağıdaki formu doldur. `PROCESSING_MODE` boş bırakılamaz ve varsayılanı yoktur —
öneriyi kabul etsen bile modu **açıkça** yazman gerekir.

- `direct_clip_series`: `THEORY_ANGLE` boş kalmalı, `PLAN_OUTPUT_COUNT` kaç Short istediğindir.
- `narrated_theory`: `THEORY_ANGLE` en az 20 karakter olmalı (script bu açıya göre üretilir),
  `PLAN_OUTPUT_COUNT = 1`.
- `PLAN_MOMENT_RANKS` boş bırakılırsa Faz 1'in en iyi adayları kullanılır.


In [ ]:
PROCESSING_MODE = ''  # @param ["", "direct_clip_series", "narrated_theory"]
PLAN_SELECTED_BY = ''  # @param {type:"string"}
PLAN_OUTPUT_COUNT = 0  # @param {type:"integer"}  (0 = degerlendirmenin onerdigi sayi)
PLAN_TARGET_DURATION_SEC = 60  # @param {type:"integer"}
PLAN_MOMENT_RANKS_TEXT = ''  # @param {type:"string"}  ornek: "1,3,4" (bos = Faz 1 en iyileri)
THEORY_ANGLE = ''  # @param {type:"string"}  yalnizca narrated_theory icin
PLAN_NOTES = ''  # @param {type:"string"}
CONFIRM_PROCESSING_PLAN = False  # @param {type:"boolean"}


In [ ]:
if not CONFIRM_PROCESSING_PLAN:
    record_phase('processing_plan', 'skipped', None, 'CONFIRM_PROCESSING_PLAN=False')
    print('Isleme plani olusturulmadi. Faz 0 panelini oku, modu sec, sonra bu hucreyi calistir.')
elif not isinstance(globals().get('source_assessment'), dict):
    record_phase('processing_plan', 'failed', None, 'source_assessment.json yok')
    print("Duzeltme: once Faz 0 degerlendirme hucresini calistir.")
else:
    try:
        _plan_ranks = [int(_p.strip()) for _p in PLAN_MOMENT_RANKS_TEXT.split(',') if _p.strip()]
        # Render/anlatim ayarlari da KARAR blogunun parcasidir: degisirlerse onay duser.
# Render/anlatim ayarlari KARAR blogunun parcasidir: sonradan degisirlerse
        # hem onay hash'i hem de render oncesi drift kontrolu bunu yakalar.
        # Bu hucreler notebook'ta ASAGIDA tanimli oldugu icin globals()'tan okunur.
        _g = globals()
        _plan_render_config = {
            'render_mode': _g.get('RENDER_MODE', 'blurred_background'),
            'preview_mode': _g.get('PREVIEW_MODE', True),
            'caption_style': _g.get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': _g.get('CONTENT_THEME', 'auto'),
        }
        # Anlatim ayarlari (ton, ses, dil) KARAR blogunda TUTULMAZ: bunlar zaten
        # script metninin ve ses dosyasinin kendi hash-bagli onaylarina yansir,
        # ve Faz 0 notebook'ta o hucrelerden ONCE geldigi icin burada okunsalardi
        # ilk gecistee None kaydedilirlerdi. Faz 0 yalnizca kendi sahip oldugu
        # kararlari (mod, teori acisi, cikti sayisi, hedef sure, render ayari)
        # hash'ler. API cagiranlar (or. Creator Console) isterse acikca gecebilir.
        _plan_narration_config = None

        processing_plan = source_strategy.build_processing_plan(
            source_assessment, PROCESSING_MODE, PLAN_SELECTED_BY,
            output_count=PLAN_OUTPUT_COUNT or None,
            target_duration_sec=PLAN_TARGET_DURATION_SEC or None,
            moment_ranks=_plan_ranks, theory_angle=THEORY_ANGLE,
            render_config=_plan_render_config, narration_config=_plan_narration_config,
            render_root=RENDER_DIR, notes=PLAN_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        print(source_strategy.format_plan_panel(processing_plan))
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"mod={processing_plan['decision']['mode']} "
                     f"cikti={processing_plan['decision']['output_count']}")
    except Exception as exc:
        record_phase('processing_plan', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Isleme plani olusturulamadi:', exc)


### Faz 0b — Planı İNCELE

Planı oku ve aşağıda yazdırılan **SHA-256** değerini kopyala. Bu değer, kararın tam olarak
hangi içeriğe (kaynak, mod, momentler, teori açısı, çıktı sayısı, süre, render ayarı)
bağlandığının kaydıdır.


In [ ]:
if processing_plan is None:
    _found_plan = find_artifact('processing_plan.json', RESULTS_DIR)
    if _found_plan is not None:
        with open(_found_plan, 'r', encoding='utf-8') as _f:
            processing_plan = json.load(_f)
        PROCESSING_PLAN_PATH = _found_plan

if processing_plan is None:
    print('Incelenecek isleme plani yok. Once Faz 0a hucresini calistir.')
    plan_review_info = None
else:
    _plan_errors = source_strategy.validate_processing_plan(
        processing_plan, assessment=source_assessment,
        source_fingerprint=(source_assessment or {}).get('source', {}).get('fingerprint_sha256'),
    )
    _plan_errors += source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': globals().get('RENDER_MODE', 'blurred_background'),
            'preview_mode': globals().get('PREVIEW_MODE', True),
            'caption_style': globals().get('CAPTION_STYLE', 'active_word_yellow'),
            'caption_theme': globals().get('CONTENT_THEME', 'auto'),
        },
    )
    if _plan_errors:
        print('PLAN GECERSIZ/BAYAT -> onaylanmamali:')
        for _err in _plan_errors:
            print('  -', _err)
    plan_review_info = show_review_prompt('processing_plan', processing_plan, 'ISLEME PLANI')
    print()
    print(source_strategy.format_plan_panel(processing_plan))


### Faz 0c — Planı ONAYLA (ayrı adım)

`PLAN_REVIEWED_SHA256` alanına Faz 0b'de yazdırılan hash'i yapıştır. Onay yalnızca hash
şu anki kararla eşleşirse verilir; kaynağı/modu/momentleri/teori açısını/render ayarını
sonradan değiştirirsen onay otomatik geçersiz olur ve alt fazlar tekrar durur.


In [ ]:
PLAN_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
PLAN_APPROVED_BY = ''       # onayi veren kisi
PLAN_APPROVAL_NOTES = ''

if processing_plan is None:
    print('Onaylanacak isleme plani yok.')
elif not PLAN_REVIEWED_SHA256 or not PLAN_APPROVED_BY:
    print('Onay verilmedi: PLAN_REVIEWED_SHA256 ve PLAN_APPROVED_BY zorunlu.')
    print('Faz 0b inceleme hucresini calistir, gosterilen hash i buraya yapistir.')
else:
    try:
        processing_plan = approval.approve_artifact(
            processing_plan, 'processing_plan', PLAN_REVIEWED_SHA256,
            PLAN_APPROVED_BY, PLAN_APPROVAL_NOTES,
        )
        youtube_data.write_json(PROCESSING_PLAN_PATH, processing_plan)
        record_phase('processing_plan', 'completed', processing_plan['status'],
                     f"onaylayan={PLAN_APPROVED_BY}")
        print('Isleme plani ONAYLANDI ->', PROCESSING_PLAN_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)

# Onaylanmis plan alt fazlarin kapisidir; PIPELINE_MODE plandan TURETILIR.
plan_gates = None
if isinstance(processing_plan, dict) and processing_plan.get('status') == 'approved':
    PIPELINE_MODE = processing_plan['pipeline_mode']
    plan_gates = source_strategy.describe_phase_plan(processing_plan)
    print()
    print('Turetilen PIPELINE_MODE:', PIPELINE_MODE)
    print('Faz kapilari:')
    for _phase, _gate in plan_gates.items():
        print(f"  {'ACIK ' if _gate['allowed'] else 'KAPALI'} {_phase}"
              + (f" -- {_gate['reason']}" if _gate['reason'] else ''))
else:
    print()
    print('Plan onayli degil -> tum alt fazlar duracak (Faz 0 ZORUNLUDUR).')


In [ ]:
hot_moments_data = None

if not RUN_VIDEO_ANALYSIS:
    record_phase('video_analysis', 'skipped')
elif not require_plan_phase('video_analysis'):
    pass
else:
    try:
        SOURCE_VIDEO_PATH = str(video_analysis.resolve_source_video(
            SOURCE_VIDEO_DIR, SOURCE_VIDEO_PATH,
        ))
        print('Otomatik secilen kaynak video:', SOURCE_VIDEO_PATH)
        ffmpeg_bin, _ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        hot_moments_data = video_analysis.run_video_analysis(
            ffmpeg_bin, SOURCE_VIDEO_PATH, RUNTIME_DIR,
            clip_start_sec=CLIP_START_SEC, clip_duration_sec=CLIP_DURATION_SEC,
            whisper_model_size=WHISPER_MODEL_SIZE, scene_detector=SCENE_DETECTOR,
            top_n=TOP_N_HOT_MOMENTS,
            preview_dir=RESULTS_DIR / 'moment_previews',
        )
        youtube_data.write_json(HOT_MOMENTS_PATH, hot_moments_data)
        record_phase('video_analysis', 'completed', hot_moments_data['status'],
                     f"{len(hot_moments_data['hot_moments'])} aday -> {HOT_MOMENTS_PATH.name}")
    except Exception as exc:
        record_phase('video_analysis', 'failed', None, f'{type(exc).__name__}: {exc}')


## Faz 1a — Moment Explorer: adayları İZLE

Bu hücre en yüksek skorlu adayları gerçek MP4 preview olarak gösterir. Zamanı, skoru, nedeni ve transcript'i kontrol et. İzlemeden seçim onayı verme.

In [ ]:
moment_explorer_data = hot_moments_data if isinstance(hot_moments_data, dict) else None
if moment_explorer_data is None:
    _found_hot_moments = find_artifact('hot_moments.json', RESULTS_DIR)
    if _found_hot_moments:
        with open(_found_hot_moments, 'r', encoding='utf-8') as _f:
            moment_explorer_data = json.load(_f)

if moment_explorer_data is None:
    print('Moment Explorer: hot_moments.json yok. Once Faz 1 video analizini calistir.')
else:
    _moment_schema_errors = video_analysis.validate_hot_moments_schema(moment_explorer_data)
    if _moment_schema_errors:
        print('Moment Explorer acilamadi; sema hatalari:', _moment_schema_errors)
    else:
        from IPython.display import Video, Markdown, display
        print('Aday sayisi:', len(moment_explorer_data['hot_moments']))
        for _moment in sorted(moment_explorer_data['hot_moments'], key=lambda m: m['rank']):
            _start = _moment.get('source_start_sec', _moment.get('aligned_start_sec'))
            _end = _moment.get('source_end_sec', _moment.get('aligned_end_sec'))
            display(Markdown(
                f"### Aday #{_moment['rank']} — skor {_moment.get('score')} "
                f"— kaynak zamanı {_start:.2f}s–{_end:.2f}s"
            ))
            print('Secilme nedenleri:', _moment.get('reason') or '(yok)')
            print('Transcript:', _moment.get('transcript') or '(konusma tespit edilmedi)')
            _preview = _moment.get('preview_path')
            if _preview and Path(_preview).is_file() and Path(_preview).stat().st_size > 0:
                display(Video(_preview, embed=True, width=560, html_attributes='controls preload=\"metadata\"'))
            else:
                print('PREVIEW YOK:', _preview, _moment.get('preview_error') or '')
            print('-' * 72)


## Faz 1b — Moment kararını KAYDET

Preview'ları izledikten sonra aşağıdaki formu doldur. `CONFIRM_MOMENT_SELECTION=True` olmadan seçim kaydedilmez. Karar `moment_selection.json` içinde analiz ve preview SHA-256 değerlerine bağlanır; video/analiz/preview değişirse seçim geçersiz olur.

In [ ]:
SELECTED_MOMENT_RANK = 1  # @param {type:"integer"}
REJECTED_MOMENT_RANKS_TEXT = ""  # @param {type:"string"}
MOMENT_REVIEWED_BY = ""  # @param {type:"string"}
MOMENT_REVIEW_NOTES = ""  # @param {type:"string"}
CONFIRM_MOMENT_SELECTION = False  # @param {type:"boolean"}
REQUIRE_MOMENT_PREVIEW = True


In [ ]:
moment_selection = None
_selection_source = moment_explorer_data if isinstance(moment_explorer_data, dict) else None
if not require_plan_phase('moment_selection'):
    pass
elif _selection_source is None:
    record_phase('moment_selection', 'failed', None, 'hot_moments.json bulunamadi')
elif not CONFIRM_MOMENT_SELECTION:
    record_phase('moment_selection', 'skipped', None, 'CONFIRM_MOMENT_SELECTION=False')
    print('Moment secimi kaydedilmedi. Previewlari izle; sonra formu doldurup onayi True yap.')
else:
    try:
        _rejected_ranks = [
            int(_part.strip()) for _part in REJECTED_MOMENT_RANKS_TEXT.split(',')
            if _part.strip()
        ]
        moment_selection = video_analysis.build_moment_selection(
            _selection_source, selected_moment_rank=SELECTED_MOMENT_RANK,
            reviewed_by=MOMENT_REVIEWED_BY, rejected_moment_ranks=_rejected_ranks,
            review_notes=MOMENT_REVIEW_NOTES, require_preview=REQUIRE_MOMENT_PREVIEW,
        )
        youtube_data.write_json(MOMENT_SELECTION_PATH, moment_selection)
        record_phase(
            'moment_selection', 'completed', 'approved',
            f"rank={moment_selection['selected_moment_rank']} reviewed_by={moment_selection['reviewed_by']}",
        )
        print('Moment secimi KAYDEDILDI ->', MOMENT_SELECTION_PATH)
        print('Secilen rank:', moment_selection['selected_moment_rank'])
        print('Preview SHA-256:', moment_selection['selected_moment']['preview_sha256'])
    except Exception as exc:
        record_phase('moment_selection', 'failed', None, f'{type(exc).__name__}: {exc}')
        print('Moment secimi kaydedilemedi:', exc)


## Faz 2 — Script Uretimi (`RUN_SCRIPT_GENERATION`)

Bağımlılık: geçerli ve hash-bağlı `moment_selection.json`. Geçerli insan seçimi yoksa ücretli çağrı sayısı **0**;
aksi halde `ALLOW_SCRIPT_PROVIDER_FALLBACK` kapalıyken tam olarak **1**.

Script hücresi manuel bir onay boolean'ına güvenmez. Faz 1b'nin ürettiği seçimi, güncel `hot_moments.json`
ve seçili preview dosyasının SHA-256 değeriyle doğrular. Script onayı aşağıda ayrı bir hücrededir.

In [ ]:
# --- Ucretli saglayici secimi (fan-out YOK) ---
# Varsayilan: Google/Gemini (GOOGLE_API_KEY). OpenAI/Anthropic fallback KAPALI --
# key'leri eksikse hata degil, sadece atlanir/loglanir.
SCRIPT_PROVIDER = 'google'      # 'google' | 'anthropic' | 'openai' (google = kanonik varsayilan)
SCRIPT_MODEL = None             # None = Google Rota A zinciri (3.6-flash -> 3.5-flash -> 3.5-flash-lite -> gemma-4-31b-it);
                                # bir model adi verirsen router SADECE o modele sabitlenir
ALLOW_SCRIPT_PROVIDER_FALLBACK = False  # True: adaylar SIRAYLA denenir, ilk basarida durur

CONTENT_MODE = 'spoiler-free-recap'
TONE = 'gergin ve merak uyandirici'
TARGET_DURATION_SEC = 60

In [ ]:
selected_script = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None
SELECTED_MOMENT_RANK = None
MOMENT_APPROVED = False
MOMENT_APPROVED_BY = ''

if not RUN_SCRIPT_GENERATION:
    record_phase('script_generation', 'skipped')
elif not require_plan_phase('script_generation'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir script API
    # cagrisi yapilmaz.
    pass
else:
    if hot_moments_data is None:
        found_path = find_artifact('hot_moments.json', RESULTS_DIR)
        if found_path is None:
            record_phase('script_generation', 'failed', None, 'hot_moments.json bulunamadi')
            print("Duzeltme: RUN_VIDEO_ANALYSIS=True ile Faz 1'i calistir veya hot_moments.json "
                  "dosyasini RESULTS_DIR altina yerlestir.")
            hot_moments_data = 'MISSING'
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            schema_errors = video_analysis.validate_hot_moments_schema(hot_moments_data)
            if schema_errors:
                record_phase('script_generation', 'failed', None,
                             f'hot_moments.json semasi gecersiz: {schema_errors}')
                hot_moments_data = 'MISSING'
            else:
                HOT_MOMENTS_PATH = found_path
                print('Mevcut hot_moments.json bulundu ve dogrulandi:', found_path)

    if hot_moments_data not in (None, 'MISSING'):
        if moment_selection is None:
            _found_selection = find_artifact('moment_selection.json', RESULTS_DIR)
            if _found_selection:
                with open(_found_selection, 'r', encoding='utf-8') as _f:
                    moment_selection = json.load(_f)
                MOMENT_SELECTION_PATH = _found_selection

        if moment_selection is None:
            print('Gecerli moment_selection.json yok -> ucretli script cagrisi yapilmayacak.')
            print('Duzeltme: Faz 1a previewlarini izle ve Faz 1b karar hucresini calistir.')
        else:
            _selection_errors = video_analysis.validate_moment_selection(
                moment_selection, hot_moments_data, require_preview=True)
            if _selection_errors:
                print('Moment secimi GECERSIZ -> ucretli script cagrisi yapilmayacak:')
                for _error in _selection_errors:
                    print('  -', _error)
            else:
                SELECTED_MOMENT_RANK = moment_selection['selected_moment_rank']
                MOMENT_APPROVED = True
                MOMENT_APPROVED_BY = moment_selection['reviewed_by']
                print('Moment secimi dogrulandi: rank=', SELECTED_MOMENT_RANK,
                      '| reviewed_by=', MOMENT_APPROVED_BY)

        # Istemciler yalnizca AN onaylandiysa kurulur; kurulmalari tek basina
        # cagri yapmaz, ama gereksiz kurulumdan da kaciniyoruz.
        openai_client = anthropic_client = google_client = None
        if MOMENT_APPROVED:
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_client = OpenAI(api_key=OPENAI_API_KEY)
            if ANTHROPIC_API_KEY:
                import anthropic
                anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            if GOOGLE_CREDENTIAL.configured and RUN_GOOGLE_SCRIPT_GENERATION:
                # Istemci TEK bir kimlik bilgisinden kurulur (hesap havuzu yok).
                google_client = google_ai.build_google_client(GOOGLE_CREDENTIAL)
                show_google_route('script_generation')
            elif not RUN_GOOGLE_SCRIPT_GENERATION:
                print('RUN_GOOGLE_SCRIPT_GENERATION=False -> Google script uretimi atlandi.')

        try:
            selected_script = script_generation.run_script_generation(
                hot_moments_data, HOT_MOMENTS_PATH, selected_moment_rank=SELECTED_MOMENT_RANK,
                openai_client=openai_client, anthropic_client=anthropic_client,
                google_client=google_client,
                content_mode=CONTENT_MODE, tone=TONE, target_duration_sec=TARGET_DURATION_SEC,
                moment_approved=MOMENT_APPROVED, moment_approved_by=MOMENT_APPROVED_BY,
                script_provider=SCRIPT_PROVIDER, script_model=SCRIPT_MODEL,
                allow_provider_fallback=ALLOW_SCRIPT_PROVIDER_FALLBACK,
                google_ledger=google_quota_ledger, google_manifest=google_manifest,
                google_route_mode=GOOGLE_ROUTE_MODE,
                google_task_model_overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_google_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_google_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                enable_cross_provider_fallback=ENABLE_CROSS_PROVIDER_FALLBACK,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS,
                cross_provider_secret_getter=get_secret,
            )
            youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
            paid_calls = len([a for a in selected_script.get('provider_attempts', []) if a.get('provider')])
            record_phase('script_generation', 'completed', selected_script['status'],
                         f"ucretli cagri sayisi={paid_calls}, saglayici={selected_script.get('provider')}")
            for attempt in selected_script.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
                for _google_attempt in ((attempt.get('google_route') or {}).get('provider_attempts') or []):
                    _gflag = 'OK' if _google_attempt['ok'] else 'BASARISIZ'
                    print(f"      google: {_google_attempt['model']} -> {_gflag}"
                          + (f" [{_google_attempt.get('error_kind')}] {_google_attempt.get('error')}"
                             if not _google_attempt['ok'] else ''))
        except Exception as exc:
            record_phase('script_generation', 'failed', None, f'{type(exc).__name__}: {exc}')


### Faz 2a — Uretilen scripti INCELE

Asagidaki hucre scripti ve SHA-256'sini yazdirir. **Scripti oku**, sonra bir sonraki onay hucresine
bu SHA-256 degerini yapistir. Onay, uretim cagrisinda degil burada verilir.

In [ ]:
if selected_script is None:
    print('Faz 2 calismadi; incelenecek script yok.')
elif not selected_script.get('script_text'):
    print('Script uretilmedi (sadece prompt paketi). status:', selected_script['status'])
    print('status_reason:', selected_script['status_reason'])
    if selected_script.get('prompt_package'):
        print()
        print('--- PROMPT PAKETI (harici/elle uretim icin) ---')
        print(selected_script['prompt_package']['user_prompt'])
else:
    print('Saglayici/model:', selected_script['provider'], '/', selected_script['model'])
    print('Tahmini maliyet (USD):', selected_script['cost_usd_estimate'])
    print()
    print('--- URETILEN SCRIPT ---')
    print(selected_script['script_text'])
    print()
    script_review_info = show_review_prompt('script', selected_script, 'SCRIPT')


### Faz 2b — Scripti ONAYLA (ayrı adım)

Yukarıdaki scripti okuduktan **sonra** `SCRIPT_REVIEWED_SHA256` alanına inceleme hücresinin yazdırdığı
hash'i yapıştır ve `SCRIPT_APPROVED_BY` doldur. Hash eşleşmezse onay **reddedilir**.

In [ ]:
SCRIPT_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
SCRIPT_APPROVED_BY = ''       # ornek: 'insan: birkan'
SCRIPT_APPROVAL_NOTES = ''

if selected_script is None or not selected_script.get('script_text'):
    print('Onaylanacak bir script yok; bu hucre atlandi.')
elif not SCRIPT_REVIEWED_SHA256 or not SCRIPT_APPROVED_BY:
    print('Onay ATLANDI: SCRIPT_REVIEWED_SHA256 ve SCRIPT_APPROVED_BY doldurulmali.')
    print('selected_script.json status olarak kaliyor:', selected_script['status'])
else:
    try:
        selected_script = approval.approve_artifact(
            selected_script, 'script', SCRIPT_REVIEWED_SHA256,
            SCRIPT_APPROVED_BY, SCRIPT_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_SCRIPT_PATH, selected_script)
        record_phase('script_generation', 'completed', selected_script['status'],
                     f"onaylayan={SCRIPT_APPROVED_BY}")
        print('Script ONAYLANDI ->', SELECTED_SCRIPT_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_script.json status olarak kaliyor:', selected_script['status'])


## Faz 3 — TTS / Seslendirme (`RUN_TTS`)

Bagimlilik: **onaylanmis** `selected_script.json`. Script `approved` degilse hicbir ucretli TTS
cagrisi yapilmaz. **Ucretli cagri sayisi:** fallback kapaliyken tam olarak **1**.

Kalite-oncelikli varsayilan: ElevenLabs `eleven_multilingual_v2`. Opsiyonel uretim fallback'i:
OpenAI `tts-1-hd`. gTTS **yalnizca** `ALLOW_DEV_GTTS_FALLBACK = True` iken ve hicbir uretim
saglayicisi basarili olmadiginda calisir; sonucu her zaman `development_placeholder` kalir ve
**asla onaylanamaz**.

In [ ]:
# Varsayilan: mevcut GeminiKey ile Gemini TTS. Ses uretimi zorunlu ana-pipeline adimidir.
# Piper lokal; ElevenLabs/OpenAI ise kalite karsilastirmasi icin alternatiflerdir.
TTS_PROVIDER = 'gemini'              # 'piper' | 'gemini' | 'elevenlabs' | 'openai'
                                     # 'gemini': RUN_GOOGLE_TTS=True olmali; Gemini TTS
                                     # modelleri PREVIEW oldugu icin cikti HER ZAMAN
                                     # review_required kalir ve yedek ses/model onceki
                                     # seslendirmenin onayini ASLA devralmaz.
ALLOW_TTS_PROVIDER_FALLBACK = False   # True: piper basarisiz olursa siradaki uretim saglayicisi denenir
ALLOW_DEV_GTTS_FALLBACK = False       # True: hicbir uretim saglayicisi (piper dahil) calismazsa gTTS
                                       # (SADECE development_placeholder -- ASLA onaylanamaz, bkz. approval.py)

# Piper ayarlari: binary/ses modeli eksikse Faz 3 temiz sekilde 'insufficient_data'
# doner, notebook COKMEZ (bkz. local_pipeline/tts.py call_piper_tts).
PIPER_BINARY = 'piper'                # PATH icindeki komut veya piper binary'sine tam yol
PIPER_VOICE_MODEL_PATH = None         # ör: '/content/piper-voices/tr_TR-fahrettin-medium.onnx' -- yoksa None birak

In [ ]:
selected_voiceover = None

if not RUN_TTS:
    record_phase('tts', 'skipped')
elif not require_plan_phase('tts'):
    # direct_clip_series bu noktada durur: istemci KURULMAZ, hicbir TTS API
    # cagrisi yapilmaz.
    pass
else:
    if selected_script is None:
        found_path = find_artifact('selected_script.json', RESULTS_DIR)
        if found_path is None:
            record_phase('tts', 'failed', None, 'selected_script.json bulunamadi')
            print("Duzeltme: RUN_SCRIPT_GENERATION=True ile Faz 2'yi calistir veya "
                  "selected_script.json dosyasini RESULTS_DIR altina yerlestir.")
        else:
            with open(found_path, 'r', encoding='utf-8') as f:
                selected_script = json.load(f)
            SELECTED_SCRIPT_PATH = found_path
            print('Mevcut selected_script.json bulundu:', found_path,
                  '(status:', selected_script.get('status'), ')')

    if selected_script is not None:
        # Diskten gelen bir onay hala GECERLI mi? Script onaylandiktan sonra
        # degistirildiyse onay gecersizdir ve ucretli TTS calismamalidir.
        script_check = approval.check_approval(selected_script, 'script')
        if selected_script.get('status') == 'approved' and not script_check['valid']:
            print('UYARI:', script_check['reason'])
            print('Bu script onayi gecersiz sayiliyor; uretim TTS calistirilmayacak.')
            selected_script = dict(selected_script)
            selected_script['status'] = 'review_required'

        if selected_script.get('status') != 'approved':
            print('NOT: script onayli degil ->', selected_script.get('status'),
                  '| uretim TTS (piper dahil) ATLANACAK (Faz 2b onay hucresini calistir).')

        # Istemciler/parametreler yalnizca script ONAYLIYSA kurulur. Piper API key
        # istemez; sadece PIPER_BINARY/PIPER_VOICE_MODEL_PATH gecirilir, eksikse
        # tts.run_tts temiz 'insufficient_data' dondurur.
        elevenlabs_client = openai_tts_client = None
        piper_binary_arg = piper_voice_model_arg = None
        if selected_script.get('status') == 'approved':
            if ELEVENLABS_API_KEY:
                from elevenlabs.client import ElevenLabs
                elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)
            if OPENAI_API_KEY:
                from openai import OpenAI
                openai_tts_client = OpenAI(api_key=OPENAI_API_KEY)
            piper_binary_arg = PIPER_BINARY
            piper_voice_model_arg = PIPER_VOICE_MODEL_PATH

        _, ffprobe_bin = render_utils.resolve_ffmpeg_binaries()
        try:
            if TTS_PROVIDER == 'gemini':
                # Gemini TTS: AYRI bir ses-uretim implementasyonu (metin
                # ayristiricisi degil). Ham PCM gecerli bir WAV'a sarilir ve
                # sure/codec/boyut/bos-olmama dogrulamasindan gecer.
                if not RUN_GOOGLE_TTS:
                    raise RuntimeError(
                        "TTS_PROVIDER='gemini' ama RUN_GOOGLE_TTS=False. "
                        "Google TTS komutunu acikca ac veya baska bir saglayici sec."
                    )
                if not GOOGLE_CREDENTIAL.configured:
                    raise RuntimeError('Gemini anahtari yok. ' + google_ai.credential_help_text('google_api'))
                if selected_script.get('status') != 'approved':
                    raise RuntimeError(
                        'Gemini TTS atlandi: script onayli degil '
                        f"(status={selected_script.get('status')!r}); cagri yapilmadi."
                    )
                show_google_route('tts')
                _gemini_tts = google_ai.run_google_tts(
                    selected_script['script_text'], RESULTS_DIR / 'tts' / 'gemini_tts.wav',
                    client=google_ai.build_google_client(GOOGLE_CREDENTIAL),
                    voice=GEMINI_TTS_VOICE, language_code=GEMINI_TTS_LANGUAGE_CODE,
                    narration_direction=GEMINI_TTS_NARRATION_DIRECTION,
                    manifest=google_manifest, ledger=google_quota_ledger,
                    mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                    enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                    max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                )
                if _gemini_tts.get('status') == 'ok':
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
                elif ENABLE_CROSS_PROVIDER_FALLBACK:
                    _azure_tts = provider_adapters.run_cross_provider_tts_fallback(
                        selected_script['script_text'], RESULTS_DIR / 'tts' / 'azure_tts.wav',
                        secret_getter=get_secret, enabled=True,
                    )
                    if _azure_tts.get('ok'):
                        _tts_attempts = list(_gemini_tts.get('provider_attempts') or [])
                        _tts_attempts.extend(_azure_tts.get('provider_attempts') or [])
                        selected_voiceover = tts.build_selected_voiceover_artifact(
                            _azure_tts, selected_script['script_text'],
                            selected_script.get('status', 'review_required'), False,
                            ffprobe_bin=ffprobe_bin, language='tr', attempts=_tts_attempts,
                        )
                    else:
                        selected_voiceover = google_ai.build_tts_artifact(
                            _gemini_tts, selected_script['script_text'],
                            script_status=selected_script.get('status', 'review_required'),
                            raw_media_submitted=False,
                            )
                        selected_voiceover['provider_attempts'].extend(
                            _azure_tts.get('provider_attempts') or [])
                else:
                    selected_voiceover = google_ai.build_tts_artifact(
                        _gemini_tts, selected_script['script_text'],
                        script_status=selected_script.get('status', 'review_required'),
                        raw_media_submitted=False,
                    )
            else:
                selected_voiceover = tts.run_tts(
                    selected_script, RESULTS_DIR / 'tts', ffprobe_bin=ffprobe_bin,
                    elevenlabs_client=elevenlabs_client, openai_client=openai_tts_client,
                    piper_binary=piper_binary_arg, piper_voice_model_path=piper_voice_model_arg,
                    tts_provider=TTS_PROVIDER,
                    allow_provider_fallback=ALLOW_TTS_PROVIDER_FALLBACK,
                    allow_dev_gtts_fallback=ALLOW_DEV_GTTS_FALLBACK,
                )
            youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
            paid = len([a for a in selected_voiceover.get('provider_attempts', [])
                        if a.get('provider') in ('elevenlabs', 'openai')])
            if selected_voiceover.get('audio_validation'):
                print('  ses dogrulamasi:', selected_voiceover['audio_validation']['checks'])
            record_phase('tts', 'completed', selected_voiceover['status'],
                         f"ucretli cagri sayisi={paid}, saglayici={selected_voiceover.get('provider')}")
            for attempt in selected_voiceover.get('provider_attempts', []):
                flag = 'OK' if attempt['ok'] else 'BASARISIZ'
                print(f"  deneme: {attempt['provider']}/{attempt['model']} -> {flag}"
                      + (f" ({attempt['error']})" if attempt['error'] else ''))
        except Exception as exc:
            record_phase('tts', 'failed', None, f'{type(exc).__name__}: {exc}')

### Faz 3a — Seslendirmeyi DINLE

Aşağıdaki hücre sesi çalar ve dosyanın SHA-256'sını yazdırır. **Sesi dinle**, sonra onay hücresine
bu hash'i yapıştır.

In [ ]:
if selected_voiceover is None:
    print('Faz 3 calismadi; incelenecek seslendirme yok.')
elif not selected_voiceover.get('audio_path'):
    print('Seslendirme uretilmedi. status:', selected_voiceover['status'])
    print('status_reason:', selected_voiceover['status_reason'])
else:
    print('Saglayici/model:', selected_voiceover['provider'], '/', selected_voiceover['model'])
    print('Sure (sn):', selected_voiceover.get('duration_sec'))
    print('Tahmini maliyet (USD):', selected_voiceover['cost_usd_estimate'])
    try:
        from IPython.display import Audio, display as _display
        _display(Audio(selected_voiceover['audio_path']))
    except Exception as exc:
        print('Ses oynatici gosterilemedi:', exc)
        print('Dosyayi elle dinle:', selected_voiceover['audio_path'])
    print()
    voiceover_review_info = show_review_prompt('voiceover', selected_voiceover, 'SESLENDIRME')


### Faz 3b — Seslendirmeyi ONAYLA (ayrı adım)

gTTS sonuçları politika gereği **asla onaylanamaz** — onay hücresi bunu reddeder.

In [ ]:
VOICEOVER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
VOICEOVER_APPROVED_BY = ''
VOICEOVER_APPROVAL_NOTES = ''

if selected_voiceover is None or not selected_voiceover.get('audio_path'):
    print('Onaylanacak bir seslendirme yok; bu hucre atlandi.')
elif not VOICEOVER_REVIEWED_SHA256 or not VOICEOVER_APPROVED_BY:
    print('Onay ATLANDI: VOICEOVER_REVIEWED_SHA256 ve VOICEOVER_APPROVED_BY doldurulmali.')
    print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])
else:
    try:
        selected_voiceover = approval.approve_artifact(
            selected_voiceover, 'voiceover', VOICEOVER_REVIEWED_SHA256,
            VOICEOVER_APPROVED_BY, VOICEOVER_APPROVAL_NOTES,
        )
        youtube_data.write_json(SELECTED_VOICEOVER_PATH, selected_voiceover)
        record_phase('tts', 'completed', selected_voiceover['status'],
                     f'onaylayan={VOICEOVER_APPROVED_BY}')
        print('Seslendirme ONAYLANDI ->', SELECTED_VOICEOVER_PATH)
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('selected_voiceover.json status olarak kaliyor:', selected_voiceover['status'])


## Faz 4 — Render Pipeline (`RUN_RENDER`)

Varsayilan `original_audio_subtitles` modunda bagimlilik yalnizca `hot_moments.json` + gecerli
`moment_selection.json` dosyalaridir. Secilen klibin kendi sesi korunur, Whisper ile zaman kodlu
altyazi uretilir ve videoya gomulur; Gemini script veya TTS cagrisi yapilmaz.

Istege bagli `narrated_recap` modu eski `selected_script.json` + `selected_voiceover.json` zincirini kullanir. Render
**hicbir zaman kendini onaylamaz**: cikti her zaman `review_required` (teknik dogrulama
basarisizsa `failed`) olur; onay asagida ayri bir hucrede, videoyu izledikten sonra verilir.

In [ ]:
RENDER_MODE = 'blurred_background'  # 'blurred_background' | 'center_crop'
PREVIEW_MODE = True  # True = hizli/dusuk kalite onizleme; False = uretim kalitesi
SUBTITLE_WHISPER_MODEL_SIZE = 'small'
CAPTION_STYLE = 'active_word_yellow'
CONTENT_THEME = 'auto'  # GOT/HOTD/Targaryen/Westeros -> game_of_thrones -> Cinzel
CAPTION_FONT_NAME = None       # None = tema profili belirler
CAPTION_FONT_SIZE = None       # None = tema profili belirler
CAPTION_ACTIVE_COLOR_ASS = None  # None = tema profili; GOT profili sari
CAPTION_BASE_COLOR_ASS = None    # None = tema profili; GOT profili beyaz
CAPTION_MAX_WORDS = None
CAPTION_MAX_CHARS = None
CAPTION_MARGIN_V = None
SOURCE_VIDEO_PATH_OVERRIDE = ''


In [ ]:
render_manifest = None
moment_selection = moment_selection if isinstance(globals().get('moment_selection'), dict) else None

if not RUN_RENDER:
    record_phase('render', 'skipped')
elif not require_plan_phase('render'):
    pass
else:
    missing = []
    if hot_moments_data in (None, 'MISSING'):
        found = find_artifact('hot_moments.json', RESULTS_DIR)
        if found is None:
            missing.append('hot_moments.json')
        else:
            with open(found, 'r', encoding='utf-8') as f:
                hot_moments_data = json.load(f)
            HOT_MOMENTS_PATH = found

    if PIPELINE_MODE == 'original_audio_subtitles':
        if moment_selection is None:
            found = find_artifact('moment_selection.json', RESULTS_DIR)
            if found is None:
                missing.append('moment_selection.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    moment_selection = json.load(f)
                MOMENT_SELECTION_PATH = found
    elif PIPELINE_MODE == 'narrated_recap':
        if selected_script is None:
            found = find_artifact('selected_script.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_script.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_script = json.load(f)
                SELECTED_SCRIPT_PATH = found
        if selected_voiceover is None:
            found = find_artifact('selected_voiceover.json', RESULTS_DIR)
            if found is None:
                missing.append('selected_voiceover.json')
            else:
                with open(found, 'r', encoding='utf-8') as f:
                    selected_voiceover = json.load(f)
                SELECTED_VOICEOVER_PATH = found
    else:
        missing.append(f'bilinmeyen PIPELINE_MODE={PIPELINE_MODE!r}')

    _config_drift = source_strategy.check_config_drift(
        processing_plan,
        render_config={
            'render_mode': RENDER_MODE, 'preview_mode': PREVIEW_MODE,
            'caption_style': CAPTION_STYLE, 'caption_theme': CONTENT_THEME,
        },
    )
    if _config_drift:
        missing.append('onaylanmis plan ile suanki render ayari uyusmuyor')
        for _drift in _config_drift:
            print('  DRIFT:', _drift)

    if missing:
        record_phase('render', 'failed', None, f'gerekli girdi(ler) bulunamadi: {missing}')
        print('Duzeltme: original_audio_subtitles icin Faz 1 + moment secimini; '
              'narrated_recap icin ek olarak script + TTS fazlarini tamamla.')
    else:
        try:
            if PIPELINE_MODE == 'original_audio_subtitles':
                render_manifest = render_utils.run_original_audio_subtitle_pipeline(
                    hot_moments_data, moment_selection, HOT_MOMENTS_PATH, MOMENT_SELECTION_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    subtitle_whisper_model_size=SUBTITLE_WHISPER_MODEL_SIZE,
                    caption_style=CAPTION_STYLE, caption_theme=CONTENT_THEME,
                    caption_font_name=CAPTION_FONT_NAME,
                    caption_font_size=CAPTION_FONT_SIZE,
                    caption_active_color_ass=CAPTION_ACTIVE_COLOR_ASS,
                    caption_base_color_ass=CAPTION_BASE_COLOR_ASS,
                    caption_max_words=CAPTION_MAX_WORDS, caption_max_chars=CAPTION_MAX_CHARS,
                    caption_margin_v=CAPTION_MARGIN_V,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            else:
                render_manifest = render_utils.run_render_pipeline(
                    hot_moments_data, selected_script, selected_voiceover,
                    HOT_MOMENTS_PATH, SELECTED_SCRIPT_PATH, SELECTED_VOICEOVER_PATH,
                    render_dir=RENDER_DIR, render_mode=RENDER_MODE, preview_mode=PREVIEW_MODE,
                    source_video_path_override=SOURCE_VIDEO_PATH_OVERRIDE,
                )
            youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
            record_phase('render', 'completed', render_manifest['status'],
                         f"mod={PIPELINE_MODE}, teknik passed={render_manifest['output']['validation']['passed']}")
            print('Pipeline modu:', PIPELINE_MODE)
            print('Altyazi kaynagi:', render_manifest['subtitles']['source'],
                  '| cue:', render_manifest['subtitles']['cue_count'])
            print('Dinamik kelime vurgusu:', render_manifest['subtitles'].get('dynamic_word_highlight'),
                  '| aktif renk:', render_manifest['subtitles'].get('active_word_color'))
            print('Caption tema/font:', render_manifest['config'].get('caption_theme_resolved'),
                  '/', render_manifest['config'].get('caption_font_name'),
                  '| neden:', render_manifest['config'].get('caption_theme_resolution_reason'))
            if render_manifest['approval']['failed_conditions']:
                print('Onay oncesi saglanmayan kosullar:',
                      render_manifest['approval']['failed_conditions'])
            print('HATIRLATMA: hicbir yere yukleme yapilmadi (not_uploaded=%s).'
                  % render_manifest['not_uploaded'])

            # Coklu cikti IZOLASYONU: plan birden fazla Short istiyorsa her cikti
            # KENDI dizinine yazilir; hicbiri digerini ezemez. Ilk yuva tarihsel
            # RENDER_DIR'dir, yani tek ciktida yollar aynen korunur.
            output_slot_report = []
            for _slot in ((processing_plan or {}).get('outputs') or [])[1:]:
                output_slot_report.append({
                    'index': _slot['index'], 'moment_rank': _slot.get('moment_rank'),
                    'output_dir': _slot['output_dir'], 'status': 'insufficient_data',
                    'reason': ('bu cikti icin insan onayli ayri moment secimi yok; '
                               "Faz 1b'yi ilgili rank icin calistirip render'i tekrar calistir"),
                })
            if output_slot_report:
                print()
                print('EK CIKTI YUVALARI (henuz render EDILMEDI, izole dizinler):')
                for _entry in output_slot_report:
                    print(f"  #{_entry['index']} rank={_entry['moment_rank']} "
                          f"-> {_entry['output_dir']} [{_entry['status']}] {_entry['reason']}")
        except Exception as exc:
            record_phase('render', 'failed', None, f'{type(exc).__name__}: {exc}')
            print('Render HATASI:', type(exc).__name__, exc)


### Faz 4a — Render'ı İZLE

Aşağıdaki hücre videoyu oynatır ve SHA-256'sını yazdırır. **Videoyu izle**, sonra onay hücresine
bu hash'i yapıştır. Teknik doğrulama geçmediyse onay hücresi reddeder.

In [ ]:
if render_manifest is None:
    print('Faz 4 calismadi; incelenecek render yok.')
else:
    video_path = render_manifest['output']['final_video_path']
    print('Render dosyasi:', video_path)
    print('Cozunurluk:', render_manifest['config']['render_width'], 'x',
          render_manifest['config']['render_height'], '| onizleme modu:',
          render_manifest['config']['preview_mode'])
    print('Teknik dogrulama:', json.dumps(render_manifest['output']['validation']['checks'], indent=2))
    try:
        from IPython.display import Video, display as _display
        _display(Video(video_path, embed=True, width=320))
    except Exception as exc:
        print('Video oynatici gosterilemedi:', exc)
        print('Dosyayi elle izle:', video_path)
    print()
    render_review_info = show_review_prompt('render', render_manifest, 'RENDER')


### Faz 4b — Render'ı ONAYLA (ayrı adım)

Render onayı yalnızca seçili moda ait **tüm** koşullar sağlandığında verilebilir.
`original_audio_subtitles`: teknik doğrulama + onaylı moment seçimi.
`narrated_recap`: teknik doğrulama + onaylı script + onaylı seslendirme + script/ses hash eşleşmesi.
Herhangi biri eksikse onay reddedilir ve artefakt `review_required` kalır — onaylanmamış girdiler
"onaylı" bir çıktıya dönüştürülemez.

Videoyu izlediğini yine de kaydetmek istersen `RECORD_RENDER_REVIEW_ONLY = True` yap: bu, ayrı bir
`render_review` bloğu yazar ve `status`'ü **değiştirmez**.

In [ ]:
RENDER_REVIEWED_SHA256 = ''   # inceleme hucresindeki SHA-256'yi buraya yapistir
RENDER_APPROVED_BY = ''
RENDER_APPROVAL_NOTES = ''

# True yaparsan SADECE "izledim" kaydi yazilir; status DEGISMEZ.
# Yukari-akis onaylari eksikken videoyu izledigini kaydetmek icin kullan.
RECORD_RENDER_REVIEW_ONLY = False

if render_manifest is None:
    print('Onaylanacak bir render yok; bu hucre atlandi.')
elif not RENDER_REVIEWED_SHA256 or not RENDER_APPROVED_BY:
    print('Onay ATLANDI: RENDER_REVIEWED_SHA256 ve RENDER_APPROVED_BY doldurulmali.')
    print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
elif RECORD_RENDER_REVIEW_ONLY:
    try:
        render_manifest = approval.record_render_review(
            render_manifest, RENDER_REVIEWED_SHA256, RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        print('Inceleme kaydedildi (render_review). status DEGISMEDI:', render_manifest['status'])
    except ValueError as exc:
        print('INCELEME KAYDI REDDEDILDI:', exc)
else:
    try:
        render_manifest = approval.approve_artifact(
            render_manifest, 'render', RENDER_REVIEWED_SHA256,
            RENDER_APPROVED_BY, RENDER_APPROVAL_NOTES,
        )
        youtube_data.write_json(RENDER_MANIFEST_PATH, render_manifest)
        record_phase('render', 'completed', render_manifest['status'],
                     f'onaylayan={RENDER_APPROVED_BY}')
        print('Render ONAYLANDI ->', RENDER_MANIFEST_PATH)
        print('NOT: onay yalnizca bir kayittir; bu notebook hicbir sey yayinlamaz.')
    except ValueError as exc:
        print('ONAY REDDEDILDI:', exc)
        print('render_manifest.json status olarak kaliyor:', render_manifest['status'])
        print("Sadece izledigini kaydetmek istersen RECORD_RENDER_REVIEW_ONLY = True yap.")


## Araştırma A — YouTube Trend/Kanal Keşfi (`RUN_YOUTUBE_DATA`)

**Ana pipeline'dan bağımsızdır.** `YOUTUBE_DISCOVERY_TERMS` ile yeni videoları araştırır, aday kanalları sıralar ve araştırma veri kümesi üretir. `YOUTUBE_CHANNEL_INPUTS` isteğe bağlı manuel ek listedir. Sonuçlar kaynak video veya render girdisi yapılmaz; bu bölüm read-only'dir.

In [ ]:
YOUTUBE_CHANNEL_INPUTS = [
    # "UCxxxxxxxxxxxxxxxxxxxxxx",
    # "@ChannelHandle",
]

AUTO_DISCOVER_CHANNELS = True
YOUTUBE_DISCOVERY_TERMS = ['AI', 'AI tools', 'generative AI', 'yapay zeka']
DISCOVERY_LOOKBACK_DAYS = 14
DISCOVERY_REGION = 'US'
DISCOVERY_LANGUAGE = 'en'
DISCOVERY_ORDERS = ('date', 'viewCount')
MAX_RESULTS_PER_QUERY = 25
MAX_SEARCH_CALLS_PER_RUN = 8
TOP_DISCOVERED_VIDEOS = 50
TOP_DISCOVERED_CHANNELS = 10
MIN_CHANNEL_MATCHED_VIDEOS = 1
DISCOVERY_SHORTS_ONLY = True
YOUTUBE_DISCOVERY_CACHE_TTL_HOURS = 6

MAX_VIDEOS_PER_CHANNEL = 100
YOUTUBE_CACHE_TTL_HOURS = 12
RUN_YOUTUBE_CONNECTION_TEST = False


In [ ]:
from datetime import datetime, timezone

youtube_collection_summary = {'ran': False}

if not RUN_YOUTUBE_DATA:
    record_phase('youtube_data', 'skipped')
else:
    youtube_started_at = datetime.now(timezone.utc).isoformat()
    youtube_cache = youtube_data.Cache(YOUTUBE_CACHE_DIR)
    youtube_quota = youtube_data.QuotaTracker()

    youtube_client = None
    youtube_connection_test_result = None
    youtube_resolved_channels = []
    youtube_failed_channels = []
    youtube_all_videos = []
    youtube_collection_errors = []
    youtube_discovery = {
        'status': 'insufficient_data', 'status_reason': 'otomatik kesif calistirilmadi',
        'videos': [], 'channels': [], 'errors': [],
        'search_calls_planned': 0, 'search_calls_executed': 0,
    }
    effective_channel_inputs = list(YOUTUBE_CHANNEL_INPUTS)
    youtube_status = 'insufficient_data'
    youtube_status_reason = 'baslatilmadi'

    if not YOUTUBE_DATA_API_KEY:
        youtube_status = 'insufficient_data'
        youtube_status_reason = 'YOUTUBE_DATA_API_KEY yok (Colab Secrets kontrol et)'
        print('YouTube Data API key: missing -- toplama yapilamaz.')
    else:
        print('YouTube Data API key: configured')
        try:
            youtube_client = youtube_data.build_client(YOUTUBE_DATA_API_KEY)
        except Exception as exc:
            youtube_status = 'failed'
            youtube_status_reason = 'istemci olusturulamadi: ' + youtube_data.sanitize_error_text(exc, YOUTUBE_DATA_API_KEY)
            youtube_collection_errors.append({'stage': 'build_client', 'error': youtube_status_reason})
            print(youtube_status_reason)

        if youtube_client is not None and RUN_YOUTUBE_CONNECTION_TEST:
            test_input = YOUTUBE_CHANNEL_INPUTS[0] if YOUTUBE_CHANNEL_INPUTS else '@GoogleDevelopers'
            youtube_connection_test_result = youtube_data.run_connection_test(
                youtube_client, YOUTUBE_DATA_API_KEY, test_input)
            print('Baglanti testi ->', youtube_connection_test_result['status'], 'girdi:', test_input)
            if youtube_connection_test_result['status'] != 'ok':
                print('  Aciklama:', youtube_connection_test_result['explanation'])

        if youtube_client is not None and AUTO_DISCOVER_CHANNELS:
            if not YOUTUBE_DISCOVERY_TERMS:
                youtube_discovery['status_reason'] = 'YOUTUBE_DISCOVERY_TERMS bos'
                print('Otomatik kesif atlandi: YOUTUBE_DISCOVERY_TERMS bos.')
            else:
                print('YouTube otomatik kesif basliyor:', YOUTUBE_DISCOVERY_TERMS)
                youtube_discovery = youtube_data.discover_trending_content(
                    youtube_client, youtube_cache, youtube_quota,
                    terms=YOUTUBE_DISCOVERY_TERMS,
                    lookback_days=DISCOVERY_LOOKBACK_DAYS,
                    region_code=DISCOVERY_REGION,
                    relevance_language=DISCOVERY_LANGUAGE,
                    orders=DISCOVERY_ORDERS,
                    max_results_per_query=MAX_RESULTS_PER_QUERY,
                    max_search_calls=MAX_SEARCH_CALLS_PER_RUN,
                    top_videos=TOP_DISCOVERED_VIDEOS,
                    top_channels=TOP_DISCOVERED_CHANNELS,
                    min_channel_matched_videos=MIN_CHANNEL_MATCHED_VIDEOS,
                    shorts_only=DISCOVERY_SHORTS_ONLY,
                    cache_ttl_hours=YOUTUBE_DISCOVERY_CACHE_TTL_HOURS,
                    api_key=YOUTUBE_DATA_API_KEY,
                )
                youtube_collection_errors.extend(youtube_discovery['errors'])
                discovered_channel_ids = [
                    row['channel_id'] for row in youtube_discovery['channels']
                    if row.get('channel_id')
                ]
                effective_channel_inputs.extend(discovered_channel_ids)
                effective_channel_inputs = list(dict.fromkeys(effective_channel_inputs))
                print('Kesif sonucu:', len(youtube_discovery['videos']), 'video,',
                      len(discovered_channel_ids), 'kanal')

        if youtube_client is not None and not effective_channel_inputs:
            youtube_status = youtube_discovery.get('status', 'insufficient_data')
            if youtube_status == 'review_required':
                youtube_status = 'insufficient_data'
            youtube_status_reason = (
                'manuel kanal yok ve otomatik kesif kanal uretmedi: ' +
                youtube_discovery.get('status_reason', 'bilinmeyen neden')
            )
            print(youtube_status_reason)
        elif youtube_client is not None:
            youtube_resolved_channels, youtube_failed_channels = youtube_data.resolve_channels(
                youtube_client, effective_channel_inputs, api_key_for_sanitization=YOUTUBE_DATA_API_KEY,
            )
            for failed in youtube_failed_channels:
                youtube_collection_errors.append(
                    {'stage': 'resolve_channel', 'input': failed['input'], 'error': failed['error']})
                print('Kanal cozumlenemedi:', failed['input'], '->', failed['error'])

            for channel_record in youtube_resolved_channels:
                rows, err = youtube_data.collect_channel_uploads(
                    youtube_client, youtube_cache, youtube_quota, channel_record,
                    MAX_VIDEOS_PER_CHANNEL, channel_cache_ttl_hours=YOUTUBE_CACHE_TTL_HOURS,
                    video_stats_ttl_hours=YOUTUBE_CACHE_TTL_HOURS, api_key=YOUTUBE_DATA_API_KEY,
                )
                if err:
                    youtube_collection_errors.append(
                        {'stage': 'collect_uploads', 'channel_id': channel_record['channel_id'], 'error': err})
                    print('Toplama hatasi:', channel_record['channel_id'], '->', err)
                youtube_all_videos.extend(rows)

            youtube_all_videos = youtube_data.deduplicate_videos(youtube_all_videos)

            if youtube_resolved_channels and youtube_all_videos:
                youtube_status = 'review_required'
                youtube_status_reason = (f'{len(youtube_resolved_channels)} kanal, '
                                         f'{len(youtube_all_videos)} video toplandi -- insan onayi gerekli')
            elif youtube_resolved_channels and not youtube_all_videos:
                youtube_status = 'insufficient_data'
                youtube_status_reason = 'kanallar cozumlendi ama kullanilabilir video toplanamadi'
            else:
                youtube_status = 'failed'
                youtube_status_reason = 'hicbir kanal cozumlenemedi'

    youtube_completed_at = datetime.now(timezone.utc).isoformat()

    youtube_channels_out = [
        {'input': c['input'], 'channel_id': c['channel_id'], 'title': c['title'],
         'uploads_playlist_id': c['uploads_playlist_id'],
         'subscriber_count': c['subscriber_count'], 'video_count': c['video_count']}
        for c in youtube_resolved_channels
    ]
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'channels.json', youtube_channels_out)
    youtube_data.write_normalized_videos_csv(NORMALIZED_VIDEOS_PATH, youtube_all_videos)
    youtube_data.write_discovered_videos_csv(
        YOUTUBE_DATA_DIR / 'discovered_videos.csv', youtube_discovery.get('videos', []))
    youtube_data.write_discovered_channels_csv(
        YOUTUBE_DATA_DIR / 'discovered_channels.csv', youtube_discovery.get('channels', []))
    youtube_data.write_json(
        YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        {k: v for k, v in youtube_discovery.items() if k not in ('videos', 'channels')},
    )

    youtube_manifest = youtube_data.build_collection_manifest(
        status=youtube_status, status_reason=youtube_status_reason,
        started_at=youtube_started_at, completed_at=youtube_completed_at,
        requested_channel_count=len(effective_channel_inputs),
        resolved_channel_count=len(youtube_resolved_channels),
        failed_channel_count=len(youtube_failed_channels),
        video_count=len(youtube_all_videos),
        date_range=youtube_data.compute_date_range(youtube_all_videos),
        cache_statistics=youtube_cache.stats,
        connection_test_status=(youtube_connection_test_result['status']
                                 if youtube_connection_test_result else None),
        discovery={
            'enabled': AUTO_DISCOVER_CHANNELS,
            'status': youtube_discovery.get('status'),
            'status_reason': youtube_discovery.get('status_reason'),
            'discovered_video_count': len(youtube_discovery.get('videos', [])),
            'discovered_channel_count': len(youtube_discovery.get('channels', [])),
            'search_calls_planned': youtube_discovery.get('search_calls_planned', 0),
            'search_calls_executed': youtube_discovery.get('search_calls_executed', 0),
        },
        output_paths={
            'channels_json': YOUTUBE_DATA_DIR / 'channels.json',
            'normalized_videos_csv': NORMALIZED_VIDEOS_PATH,
            'collection_manifest_json': COLLECTION_MANIFEST_PATH,
            'quota_usage_json': YOUTUBE_DATA_DIR / 'quota_usage.json',
            'errors_json': YOUTUBE_DATA_DIR / 'errors.json',
            'discovered_videos_csv': YOUTUBE_DATA_DIR / 'discovered_videos.csv',
            'discovered_channels_csv': YOUTUBE_DATA_DIR / 'discovered_channels.csv',
            'discovery_manifest_json': YOUTUBE_DATA_DIR / 'discovery_manifest.json',
        },
    )
    youtube_data.write_json(COLLECTION_MANIFEST_PATH, youtube_manifest)
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'quota_usage.json', youtube_quota.to_dict())
    youtube_data.write_json(YOUTUBE_DATA_DIR / 'errors.json', youtube_collection_errors)

    youtube_collection_summary = {
        'ran': True, 'pipeline_role': 'research_sidecar',
        'feeds_main_pipeline': False,
        'status': youtube_status, 'status_reason': youtube_status_reason,
        'resolved_channel_count': len(youtube_resolved_channels),
        'failed_channel_count': len(youtube_failed_channels),
        'video_count': len(youtube_all_videos),
        'discovered_video_count': len(youtube_discovery.get('videos', [])),
        'discovered_channel_count': len(youtube_discovery.get('channels', [])),
        'search_calls_used': youtube_quota.calls.get('search.list', 0),
        'quota_estimated_units': youtube_quota.estimated_total(),
        'cache_stats': youtube_cache.stats,
    }

    # Artefakt temiz yazildi; execution_status=completed. Ama artifact_status
    # 'insufficient_data' ise bu ozette AYRICA gorunur -- "ok" diye gecmez.
    record_phase('youtube_data', 'completed', youtube_status, youtube_status_reason)
    print()
    print('=== YouTube toplama ozeti ===')
    for k, v in youtube_collection_summary.items():
        print(f'  {k}: {v}')


## Araştırma B — Viral Potansiyel Tahmini (`RUN_VIRAL_PREDICTION`)

Bu da ana pipeline'dan bağımsız bir karar-destek deneyidir. Bağımlılık: `normalized_videos.csv` + `collection_manifest.json`. Toplama durumu
`failed`/`insufficient_data` ise veya satir sayisi esigin altindaysa artefakt
`insufficient_data` olur ve kendinden emin tahmin URETILMEZ.

In [ ]:
MIN_HISTORY_ROWS = 20


In [ ]:
import pandas as pd

viral_prediction_artifact = None

if not RUN_VIRAL_PREDICTION:
    record_phase('viral_prediction', 'skipped')
else:
    csv_path = NORMALIZED_VIDEOS_PATH if NORMALIZED_VIDEOS_PATH.is_file() else youtube_data.find_normalized_videos_csv(
        [str(YOUTUBE_DATA_DIR), str(RESULTS_DIR / 'youtube_data'),
         'results/youtube_data', '/content/results/youtube_data'],
    )
    if csv_path is None:
        record_phase('viral_prediction', 'failed', None, 'normalized_videos.csv bulunamadi')
        print("Duzeltme: RUN_YOUTUBE_DATA=True ile Faz 5'i calistir veya bu dosyayi "
              "YOUTUBE_DATA_DIR altina yerlestir.")
    else:
        history_load_status, history_load_reason = 'insufficient_data', ''
        history_df = pd.DataFrame()
        try:
            raw_df = pd.read_csv(csv_path)
        except pd.errors.EmptyDataError:
            raw_df = None
            history_load_reason = f'{csv_path} tamamen bos (basliksiz/0 byte) bir CSV dosyasi'
        required_cols = ['video_id', 'channel_id', 'view_count', 'published_at']
        if raw_df is not None:
            missing_cols = [c for c in required_cols if c not in raw_df.columns]
            if missing_cols:
                history_load_reason = f'gerekli sutunlar eksik: {missing_cols}'
            else:
                candidate_df = raw_df.dropna(subset=['view_count']).copy()
                candidate_df['published_at'] = pd.to_datetime(
                    candidate_df['published_at'], errors='coerce', utc=True)
                candidate_df = candidate_df.sort_values('published_at')
                if len(candidate_df) > 0:
                    history_df = candidate_df
                    history_load_status = 'ok'
                else:
                    history_load_reason = f'{csv_path} gecerli semaya sahip ama kullanilabilir satir yok'

        manifest_path = csv_path.parent / 'collection_manifest.json'
        collection_status = resolved_channel_count = collection_timestamp = None
        if manifest_path.is_file():
            with open(manifest_path, 'r', encoding='utf-8') as f:
                collection_manifest_data = json.load(f)
            collection_status = collection_manifest_data.get('status')
            resolved_channel_count = collection_manifest_data.get('resolved_channel_count')
            collection_timestamp = collection_manifest_data.get('completed_at')

        try:
            viral_prediction_artifact = viral_prediction.build_viral_prediction_artifact(
                history_df, history_load_status, history_load_reason, collection_status,
                min_history_rows=MIN_HISTORY_ROWS, resolved_channel_count=resolved_channel_count,
                collection_timestamp=collection_timestamp,
            )
            youtube_data.write_json(VIRAL_PREDICTION_PATH, viral_prediction_artifact)
            record_phase('viral_prediction', 'completed', viral_prediction_artifact['status'],
                         viral_prediction_artifact['status_reason'])
        except Exception as exc:
            record_phase('viral_prediction', 'failed', None, f'{type(exc).__name__}: {exc}')


## Google yetenek komutlari (her biri BAGIMSIZ acilir/kapanir)

Bu bolumdeki her komut ayri bir `RUN_GOOGLE_*` anahtarina baglidir ve kapaliyken **hicbir cagri
yapmaz**. Hepsi ayni router'i, ayni kota defterini ve ayni hata/fallback politikasini kullanir.
Uretilen her artefakt `results/` altina yazilir ve `review_required` / `experimental` /
`insufficient_data` / `failed` disinda bir durum **alamaz** -- `approved` yalnizca ayri,
hash-tabanli insan onayi adiminda verilir.


In [ ]:
google_capability_artifacts = {}


def _google_client_or_none(command_name):
    if not GOOGLE_CREDENTIAL.configured:
        print(f'{command_name}: Gemini anahtari yok, temiz atlandi.')
        print(' ', google_ai.credential_help_text('google_api'))
        return None
    return google_ai.build_google_client(GOOGLE_CREDENTIAL)


def _run_google_command(command_name, task, capability, runner, input_content,
                        enabled, extra=None):
    """Ortak sarmalayici: anahtar kapaliysa atlar, acikken rotayi gosterir,
    artefakti sozlesmeye uygun sekilde yazar. Cikti ASLA uydurulmaz."""
    if not enabled:
        print(f'{command_name}: KAPALI (atlandi).')
        return None
    client = _google_client_or_none(command_name)
    if client is None:
        return None
    show_google_route(task)
    result = runner(client)
    artifact = google_ai.build_google_artifact(
        result, task=task, capability=capability,
        input_content=input_content, output_content=result.get('output'),
        raw_media_submitted=ALLOW_RAW_MEDIA_TO_FREE_TIER,
        extra=extra,
    )
    google_capability_artifacts[command_name] = artifact
    youtube_data.write_json(RESULTS_DIR / f'google_{command_name}.json', artifact)
    print(f"{command_name}: status={artifact['status']} model={artifact['model']} "
          f"fallback={artifact['fallback_used']}")
    if artifact['status_reason']:
        print('  ', artifact['status_reason'])
    return artifact


# --- 1. Hot-moment nihai incelemesi (Rota A) --------------------------------
GOOGLE_HOTMOMENT_PROMPT = ''  # bos birakirsan hot_moments.json ozetinden uretilir
if RUN_GOOGLE_HOTMOMENT_REVIEW:
    _hm_source = hot_moments_data if isinstance(hot_moments_data, dict) else None
    if _hm_source is None:
        _found = find_artifact('hot_moments.json', RESULTS_DIR)
        if _found:
            with open(_found, 'r', encoding='utf-8') as _f:
                _hm_source = json.load(_f)
    if _hm_source is None:
        print('google_hotmoment_review: hot_moments.json yok, atlandi.')
    else:
        # ALLOW_RAW_MEDIA_TO_FREE_TIER=False iken SADECE yerel olarak turetilmis
        # transcript/skor ozetleri gonderilir; ham video ASLA yuklenmez.
        _hm_prompt = GOOGLE_HOTMOMENT_PROMPT or (
            'Asagidaki yerel olarak uretilmis aday anlari degerlendir ve en guclu 3 tanesini sirala. '
            'SADECE JSON don: {"moments": [{"rank": <int>, "score": <0-100>, "reason": "..."}]}\n\n'
            + json.dumps([{k: m.get(k) for k in ('rank', 'source_start_sec', 'source_end_sec',
                                                   'score', 'reason', 'transcript')}
                          for m in _hm_source['hot_moments'][:10]], ensure_ascii=False)
        )
        _run_google_command(
            'hotmoment_review', 'hotmoment_review', 'generate_content',
            lambda c: google_ai.run_google_hotmoment_review(
                _hm_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS),
            _hm_prompt, RUN_GOOGLE_HOTMOMENT_REVIEW)
else:
    print('google_hotmoment_review: KAPALI (atlandi).')

# --- 2. Toplu siniflama / metadata uretimi (Rota B) -------------------------
GOOGLE_BULK_PROMPT = (
    'Asagidaki script icin YouTube Shorts metadata uret. SADECE JSON don: '
    '{"title": "...", "description": "...", "hashtags": ["#..."]}'
)
if RUN_GOOGLE_BULK_CLASSIFICATION:
    _bulk_input = (selected_script or {}).get('script_text') or ''
    if not _bulk_input:
        print('google_bulk_classification: selected_script.script_text yok, atlandi.')
    else:
        _bulk_prompt = f'{GOOGLE_BULK_PROMPT}\n\nScript:\n{_bulk_input}'
        _run_google_command(
            'bulk_classification', 'bulk_classification', 'generate_content',
            lambda c: google_ai.run_google_bulk_classification(
                _bulk_prompt, client=c, manifest=google_manifest, ledger=google_quota_ledger,
                mode=GOOGLE_ROUTE_MODE, overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                required_fields=('title', 'description')),
            _bulk_prompt, RUN_GOOGLE_BULK_CLASSIFICATION)
else:
    print('google_bulk_classification: KAPALI (atlandi).')

# --- 3. Arama-temelli arastirma (Rota C, Gemini 2.5 grounding kotasi) -------
GOOGLE_RESEARCH_TOPICS = []   # ornek: ['House of the Dragon 3. sezon']
GOOGLE_RESEARCH_MAX_CANDIDATES = 3
if RUN_GOOGLE_GROUNDED_RESEARCH and ENABLE_GOOGLE_SEARCH_GROUNDING:
    _topics = GOOGLE_RESEARCH_TOPICS[:GOOGLE_RESEARCH_MAX_CANDIDATES]
    if not _topics:
        print('google_grounded_research: GOOGLE_RESEARCH_TOPICS bos, atlandi.')
    else:
        _research_prompt = (
            'Asagidaki konularin SU AN haberlerde/gundemde olup olmadigini arastir ve her biri icin '
            'kisa bir guncellik notu ver. Kaynak baglantilarini belirt.\n\n- '
            + '\n- '.join(_topics)
        )
        _art = _run_google_command(
            'grounded_research', 'grounded_research',
            "generate_content(tools=[google_search])",
            lambda c: google_ai.run_google_grounded_research(
                _research_prompt, client=c, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
                max_candidates=GOOGLE_RESEARCH_MAX_CANDIDATES),
            _research_prompt, True)
elif RUN_GOOGLE_GROUNDED_RESEARCH:
    print('google_grounded_research: ENABLE_GOOGLE_SEARCH_GROUNDING=False, atlandi.')
else:
    print('google_grounded_research: KAPALI (atlandi).')

# --- 4. Embeddings: duplikasyon / novelty (Rota E) --------------------------
GOOGLE_EMBEDDING_TEXTS = []          # ornek: gecmis script/baslik metinleri
GOOGLE_EMBEDDING_CANDIDATE_TEXT = '' # yeni aday metin
if RUN_GOOGLE_EMBEDDINGS:
    _candidate_text = GOOGLE_EMBEDDING_CANDIDATE_TEXT or (selected_script or {}).get('script_text') or ''
    if not _candidate_text:
        print('google_embeddings: aday metin yok, atlandi.')
    else:
        _all_texts = [_candidate_text] + list(GOOGLE_EMBEDDING_TEXTS)
        _client = _google_client_or_none('google_embeddings')
        _emb = None
        if _client is not None:
            show_google_route('embeddings')
            _emb = google_ai.run_google_embeddings(
                _all_texts, client=_client, manifest=google_manifest,
                ledger=google_quota_ledger, mode=GOOGLE_ROUTE_MODE,
                overrides=GOOGLE_TASK_MODEL_OVERRIDES,
                enable_model_fallback=ENABLE_GOOGLE_MODEL_FALLBACK,
                max_model_attempts=MAX_GOOGLE_MODEL_ATTEMPTS,
            )
        if ((_emb is None or _emb.get('status') != 'ok') and
                ENABLE_CROSS_PROVIDER_FALLBACK):
            _emb = provider_adapters.run_cross_provider_embeddings_fallback(
                _all_texts, secret_getter=get_secret, enabled=True,
                max_provider_attempts=MAX_PROVIDER_ATTEMPTS, allow_local=True)
        if _emb is not None:
            _extra = {'dimension': _emb.get('dimension'),
                      'task_prefix': _emb.get('task_prefix'),
                      'task_prefix_kind': _emb.get('task_prefix_kind'),
                      'input_hashes': _emb.get('input_hashes'),
                      'vector_space': _emb.get('vector_space')}
            if _emb['status'] == 'ok':
                _vectors = _emb['output']
                _novelty = google_ai.novelty_score(_vectors[0], _vectors[1:])
                _extra.update({
                    'novelty_score': _novelty,
                    'duplicate_penalty': google_ai.duplicate_penalty(_novelty),
                    'semantic_duplicates': google_ai.find_semantic_duplicates(
                        _vectors[0], list(zip(
                            [f'previous_{i}' for i in range(len(_vectors) - 1)], _vectors[1:]))),
                    'clusters': google_ai.cluster_by_similarity(
                        list(zip([f'text_{i}' for i in range(len(_vectors))], _vectors))),
                })
                print('  novelty_score:', _novelty,
                      '| duplicate_penalty:', _extra['duplicate_penalty'])
            _art = google_ai.build_google_artifact(
                _emb, task='embeddings', capability='embed_content',
                input_content='\n'.join(_all_texts),
                output_content=None, raw_media_submitted=False, extra=_extra)
            google_capability_artifacts['embeddings'] = _art
            youtube_data.write_json(RESULTS_DIR / 'google_embeddings.json', _art)
            print('google_embeddings: status=' + _art['status'])
else:
    print('google_embeddings: KAPALI (atlandi).')

# --- 5. Antigravity (DENEYSEL, varsayilan KAPALI) --------------------------
GOOGLE_ANTIGRAVITY_PROMPT = 'Bugunun YouTube Shorts trendleri icin kisa bir arastirma notu yaz.'
_antigravity = google_ai.run_antigravity_research_experimental(
    GOOGLE_ANTIGRAVITY_PROMPT,
    enabled=RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured,
    max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS,
    client=(google_ai.build_google_client(GOOGLE_CREDENTIAL)
            if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else None),
    manifest=google_manifest, ledger=google_quota_ledger,
) if (RUN_ANTIGRAVITY_RESEARCH and GOOGLE_CREDENTIAL.configured) else \
    google_ai.run_antigravity_research_experimental(
        GOOGLE_ANTIGRAVITY_PROMPT, enabled=False,
        max_total_tokens=ANTIGRAVITY_MAX_TOTAL_TOKENS)
_antigravity_artifact = google_ai.build_antigravity_artifact(_antigravity)
google_capability_artifacts['antigravity_research'] = _antigravity_artifact
youtube_data.write_json(RESULTS_DIR / 'google_antigravity_research.json', _antigravity_artifact)
print('antigravity_research: status=' + _antigravity_artifact['status'],
      '| production_ready=' + str(_antigravity_artifact['production_ready']))

# --- Kota defteri ozeti (YEREL TAHMIN) --------------------------------------
google_quota_ledger.save()
print()
print('--- Google kota defteri (YEREL TAHMIN, sunucu yetkilisi degil) ---')
for _model, _record in sorted(google_quota_ledger.data['models'].items()):
    print(f"  {_model}: gunluk {_record['requests_current_day']}/{_record['configured_rpd']} "
          f"(kalan~{_record['estimated_remaining_rpd']}), dakika "
          f"{_record['requests_current_minute']}/{_record['configured_rpm']}, "
          f"devre={_record['circuit_state']}")
print('Defter dosyasi:', GOOGLE_QUOTA_LEDGER_PATH)


## Çalıştırma özeti (execution_status vs artifact_status)

In [ ]:
MAIN_PIPELINE_ORDER = ['source_assessment', 'processing_plan', 'video_analysis',
                       'moment_selection', 'script_generation', 'tts', 'render']
RESEARCH_SIDECAR_ORDER = ['youtube_data', 'viral_prediction']
PHASE_ORDER = MAIN_PIPELINE_ORDER + RESEARCH_SIDECAR_ORDER

def print_status_group(title, names):
    print('=' * 78)
    print(title)
    print(f"{'adim':<20} {'execution':<12} {'artifact':<24} detay")
    print('-' * 78)
    for name in names:
        entry = phase_report.get(name)
        if entry is None:
            print(f"{name:<20} {'-':<12} {'-':<24} hucre calistirilmadi")
            continue
        artifact = entry['artifact_status'] if entry['artifact_status'] is not None else '-'
        detail = (entry['detail'] or '')[:60]
        print(f"{name:<20} {entry['execution_status']:<12} {artifact:<24} {detail}")
    print('=' * 78)

print_status_group('ANA URETIM HATTI: kaynak -> Faz 0 plan -> analiz -> moment -> (script/ses) -> render',
                   MAIN_PIPELINE_ORDER)
if isinstance(globals().get('processing_plan'), dict):
    _pd = processing_plan.get('decision') or {}
    print()
    print(f"Faz 0 plani: mod={_pd.get('mode')} status={processing_plan.get('status')} "
          f"cikti={_pd.get('output_count')} pipeline_mode={processing_plan.get('pipeline_mode')}")
    print('  acik fazlar   :', processing_plan.get('enabled_phases'))
    print('  kapali fazlar :', processing_plan.get('disabled_phases'))
else:
    print()
    print('Faz 0 isleme plani YOK -> ana hatta hicbir faz calistirilamaz.')
print()
print_status_group('BAGIMSIZ ARASTIRMA YAN KOLU: ana pipeline girdilerini degistirmez',
                   RESEARCH_SIDECAR_ORDER)

# Temiz calisip yetersiz veri bildiren fazlari ACIKCA one cikar.
needs_attention = [
    name for name, entry in phase_report.items()
    if entry['execution_status'] == 'completed'
    and entry['artifact_status'] in ('insufficient_data', 'failed', 'development_placeholder')
]
if needs_attention:
    print()
    print('DIKKAT: bu fazlar temiz CALISTI ama artefakt durumu kullanima hazir degil:')
    for name in needs_attention:
        print(f"  - {name}: artifact_status={phase_report[name]['artifact_status']}"
              f" ({phase_report[name]['detail']})")

awaiting_review = [name for name, entry in phase_report.items()
                  if entry['artifact_status'] == 'review_required']
if awaiting_review:
    print()
    print('INSAN INCELEMESI BEKLEYENLER (ilgili inceleme + onay hucrelerini calistir):')
    for name in awaiting_review:
        print(f'  - {name}')

print()
print("Hicbir faz YouTube'a veya baska bir yere yukleme/yayin yapmadi.")
print('production_ready statusu hicbir yerde otomatik atanmaz.')


## Colab yeniden başlatma / kalıcı artefaktlardan devam etme

`USE_GOOGLE_DRIVE = True` iken tüm artefaktlar ve önbellek Drive'da kalır. Colab yeniden başlatılsa
bile bu notebook'u tekrar açıp kurulum + toggle + setup + bootstrap + yol çözümleme hücrelerini
çalıştırman, tamamlanmış fazların toggle'ını `False` yapman ve devam eden fazı çalıştırman yeterlidir.

Onaylar içeriğe SHA-256 ile bağlıdır: bir fazı yeniden çalıştırıp içeriği değiştirirsen, o artefaktın
önceki onayı otomatik olarak geçersiz sayılır (`approval.check_approval`) ve yeniden incelenip
onaylanması gerekir. Faz 3 (TTS), diskten okuduğu script onayının hâlâ geçerli olduğunu ayrıca
doğrular; geçersizse ücretli TTS çalıştırmaz.